# TAS v7 — CPU preparation / A100 experiments

Upload this notebook to Colab. Use **CPU → Run all** first, then select your
**A100 40 or 80 GB → Run all**. `PHASE="auto"` selects preparation on CPU and the
study on GPU. Models/data/caches/results live in the same persistent Drive
folder across sessions and future notebooks. CPU/GPU environments are
recreated locally; the notebook kernel is left intact.

Bring **only this notebook**. It clones the fixed public baseline, verifies
and restores its packaged experiment modules, then imports them normally.
The editable implementation remains in the experiment branch; the
compressed package is generated from its exact commit. No separate source
upload is required. The code snapshot is also saved to Drive with its hashes.

Automatic flow: blocking P0 audit → 30-image smoke → 240-image screen →
predeclared shortlist → confirmation (3 seeds + greedy) → controls → P0-D.
Final remainder and P3/P4/P5 remain disabled by default. A session
budget saves progress; the next Run all continues completed stages.
Shortlisting requires the predeclared coverage guard and a nondominated
observed hallucination improvement. If none qualifies, the study saves its
comparison and runs baseline perception diagnostics, then stops dependent
stages without forcing a finalist.

Full A100 weights have not been tested by the local CPU suite. P0 rejects
baseline mismatches before bulk generation. No scientific gains are assumed.


In [ ]:
# Edit these settings once. Reuse PERSISTENT_ROOT in future idea notebooks.
REPO_URL = 'https://github.com/RachalHys/Tag-And-Segment-AGLA.git'
BASELINE_COMMIT = '624f74501bdafe9b7d6a05ae739d12b422ce8737'
RESEARCH_COMMIT = 'df378562806154ea13f52a9e2c3e2f1d057f6aad'
PERSISTENT_ROOT = "/content/drive/MyDrive/TAS-Shared"
LOCAL_ROOT = "/content/tas-v7-work"
PHASE = "auto"             # auto / prepare / run / score / check
STAGE = "flow"             # flow / audit / smoke / screen / confirm / controls / diagnostics / final / external
MAX_SESSION_MINUTES = 180  # planned stop; run again to resume without changing the experiment
SEEDS = [0, 1, 2]
REQUIRE_A100_80GB = False   # supports your A100 40 GB; True enforces at least 80 GB
AUTO_SHUTDOWN = True       # flush Drive, then release runtime after this session's work
RUN_CONFIRMATION = True
RUN_CONTROLS = True
RUN_DIAGNOSTICS = True
RUN_FINAL_REMAINDER = False
RESERVE_POPE_IMAGES = True
DATASET = "coco"           # AMBER/POPE require STAGE=external and explicit verified manifests below
EXTERNAL_MANIFEST = None
EXTERNAL_IMAGE_ROOT = None
METHODS = []               # empty = exact initial 14; external runs should select frozen finalists explicitly
BATCH_SIZES = [1, 2]        # conservative 40 GB candidates; measured, checked, then locked
ATTENTION_BACKENDS = ["eager", "sdpa"]
LAST_POSITION_LOGITS = True
SPLIT_SEED = 20261021
COVERAGE_TOLERANCE_PP = 2.0  # approved absolute percentage-point loss vs harmonized TAG; freeze before outputs

def release_runtime():
    if AUTO_SHUTDOWN:
        from google.colab import drive, runtime
        print("Saving pending Drive writes, then releasing the Colab runtime.")
        drive.flush_and_unmount()
        runtime.unassign()


In [ ]:
# @title Versioned experiment source package — generated, do not edit
SOURCE_PACKAGE = {'schema': 'tas-private-source-v1', 'base_commit': '624f74501bdafe9b7d6a05ae739d12b422ce8737', 'source_commit': 'df378562806154ea13f52a9e2c3e2f1d057f6aad', 'sha256': 'e43d51e21362cecb39f59c1fe984cbddec1034b2bef8b5ee1e3508dddb3f0f54', 'files': {'AGENTS.md': 'c7b68fddf4d150f1ee44ac9992df39c51974731e9704b0c67c5f9b7026a04f10', 'docs/TAS_V7_COLAB.md': 'c09a2d527b042a61133c24b5a4337b1aec046bfe098b0bb2681205dfdc7f9bb2', 'experiments/__init__.py': 'fe7a0225cf031bed0d2204dbb763b431d530abac99b01ea33faa4d0f6b244526', 'experiments/tas_v7/__init__.py': '5ff13022425f78d78413600e40c3941e750ee983755c9b17a5d564609400edf7', 'experiments/tas_v7/audit.py': 'f6ea192ba422b551601e3c558e16429b5b7670b57ef4cf42fc42c7428df42a3f', 'experiments/tas_v7/cli.py': 'b16c5fbd3f0893697b332af31203fa97b686537f83b4663e71ab3b40861455a0', 'experiments/tas_v7/diagnostics.py': '2a1a56e626656d6172bf0c9355a5db5b60519f0ac82350f5003cf3416cd1f77a', 'experiments/tas_v7/evaluation.py': 'f3f8cac6999331c5dc67c0708c628b0cc81a25bf6045224b0c124690cba7c2f0', 'experiments/tas_v7/external.py': '7e9d9eb0f91f1d424c1bf34bcdbd45587d16faa8d2148eace5dec068cd97bc56', 'experiments/tas_v7/methods.py': '2913cc58e41e1a726d7e1ac8c7b2711a5259afa5dff4fa7e6f0d3fbf4d7ad9f5', 'experiments/tas_v7/prepared.py': '81b5d30c97e731f8e05bab40b944835333ba937d041c4cc135e601540d93fecc', 'experiments/tas_v7/report.py': '8bd6aa025a3ef41255b24cfca247f5d9d1779a1a6b32d5b2f246bd39bf7a0994', 'experiments/tas_v7/study.py': 'd4b8ff9911287ccd3091a646eeff4dd22964548038210bff925d996d28b8ccad', 'experiments/tas_v7/study_diagnostics.py': 'fe252e3e95b583423d59157d01364560a557f70e2ca556868ea00bb174f9bca1', 'experiments/tas_v7/views.py': '9bffe95f8996c3462b7e82b3a51587b9629d5dfe6d45c7654a17989ffb160775', 'requirements/colab-amber.txt': '7ee7be8843a02ea16d1333e4271ca0b8174445aab6867efcd46e57739a95389a', 'requirements/colab-base.txt': '281f3332113abfd949c62f499e47d382d5046c760f739800842ffc6db4afe125', 'requirements/colab-perception.txt': '0e98ee13388aa50ad35045d981a278d347fb0c56f1634fca4b9e5567ca6aa03d', 'tas_runtime/__init__.py': 'ebfdfb5371eb85b8640ab449c246efc862a11ea70e142718bbef4c9f7b5deca5', 'tas_runtime/artifacts.py': '37e4bc9271a13ca334df828af92b58e425706d0164f9f30384c81e8dde7b2c37', 'tas_runtime/assets.py': '675315d0c9b6d195e7941829b0bc93d3101a34c8f5205d9f3aa6fb9e0eeb3538', 'tas_runtime/attention.py': '94585b4980179e09acba88e0b3492c2161d94904faa7783f6f5a8391c51854fc', 'tas_runtime/bootstrap.py': 'b447c2601e4f7d48fae48c82d015b9ee2c5ab44f660386febc6fcc65fd83c414', 'tas_runtime/config.py': 'a58903c262e490dc90a3b1cb9ded9b6bf1c18c018a1592f4c9c2bd19c6aedcf6', 'tas_runtime/data.py': '39cb6a954ddf713a622630c584087fc3f5e8e5cd4da0d9a2377b7a7aa1be2523', 'tas_runtime/decoding.py': 'a5a370f0345e5ed6fbdbb612f16b9b285ab287bbb6241dfeb39189663ebf88b5', 'tas_runtime/perception.py': 'bf7179deaa1bcc8421d6c6e989554399155c1e391fd16c0be57185e43871e025', 'tas_runtime/persistence.py': '9fef2d9912d19ae2f20964167e121f2874e92ae3e3e695f3c64e96cbf1956a1b', 'tas_runtime/profiling.py': 'a77bb6b706612bcc2ecdf45a12756146f6db0b67f5a9eabc58f17ad7b9d8c4d8', 'tas_runtime/runtime.py': '068acf5e27671c6b1131374ba167cc464c86f74816a2553beb1758bac11d497a', 'tests/conftest.py': '5b20fdc9ab1817a64ddada47c22d695378148741eada4f98b1a745a49f7c73d7', 'tests/test_artifacts.py': 'b21d651d6110626e3e99471230b5d7e9386b91f30ebaf035ef8e9c5427d9f506', 'tests/test_assets.py': '6b51775e20a03a2c9384bc9922285075bd960131ae9368bfde5f73fec98b6753', 'tests/test_audit.py': 'c6fde3cbbc8ad8394a0456390e4b4f350871eda9fc7efa1fd2f30eb50bbb446c', 'tests/test_bootstrap.py': 'f43fbe53cf7c5f0e69ea183aef171516eef148dde867fccb1a2101638d1469ea', 'tests/test_data.py': 'c55b36a369a9da72cfe17fc8c88b478cde2ae3d466bbbca67b60013ba18e92e3', 'tests/test_diagnostics.py': 'e5cb5776067150a2b1b6aff62102f7e881d0d8d88ee6a53485265478b2e02874', 'tests/test_evaluation.py': '4a8e77ae0c9e4b2723fb1fd1a40f42e3d5c4e15075ea2cc2e2d54dfd350da98d', 'tests/test_notebook.py': '3ebdeb421670b43f6014aea1256ad12a2fa4425ca6bd3e85436580d2e0d7e739', 'tests/test_notebook_delivery.py': '377d9a8761c98b6ba6bcb2752d6f7549bb89f49feb0b109678ffb3ef5fbe85e6', 'tests/test_persistence.py': 'ae9b71a14af81561373fb273e58b773f8ced8fb8e4189a1206742cdbc682ffa5', 'tests/test_prepared.py': '1cb769c861c2f06cff94d17bd5027354a94f0cc8baf631ed3b8cc1019b797a73', 'tests/test_report.py': '65f0088060033ed91f2649124f7c11c7554154eedd3312b16cd7f3992113f60f', 'tests/test_runtime.py': '3db34ecd2dc499a6d56257bf95be6587b5d877fbd5e4add76341b4a51c285321', 'tests/test_science.py': '7303f63cd2f15c550ce45d78c5b6e904fe148a3d58e59f103239d84bb9ef2ea0', 'tests/test_study.py': 'b6baa535ffe00ffd1a6c9441c59902dd0754fdfbba2de2046aaef5d1fbc69837', 'tools/build_colab_notebook.py': 'a2fcbea25cc31bb6db0f5dc4448c267b88dd00d80fe969127b40858a58507e38', 'tools/notebook_delivery.py': 'de704ff351f24c0ab3cf43a892e2aeeaef3d5d5bf87cd047f14ebe985886e8ef'}, 'archive_b85': 'P)h>@6aWAK2mk;8ApmfEJXx~?0077a000R9002QpMNU*xE^TCOR84Q&Mi9O0R}9cYTfic1^kATg11NUl7IqV%v3)5Dak(5xTP=6#{g5sH`o398mQ(a%k@W7&oA)t4&^-8mSSMO#(<!HUGPzqb-Oawy_MG^6;9z=oiA<!<>nzXQk#Y1FUSo}ei-+^Y<=5NU$ExrC{^|d|{#;FdPm$)kwDyinKUkJJDjB?5v*joSo223*WS7veZb-R~nj?T%%fGkhb@OhLfbh~=)spjxTkkhg)CcrXv^38G@cSm(_q|CK-7pU%*L^T6<CL8=yj6CA+I4e6<4da|TjLHOdf&07j+e$6DHbX45PX-lX)RZi$&~IRm#QWLmGHA4^_S}GlyG)wEdO`J3b<2DK@ANpeZYMaC-jE$imtp2h>I~PYfA^qE2Vcp@R&^~0}6M^k9_<XX~V&>tx)M3Cu;tf-+r83-hBFUbunA5#!kI*Y6Xxnx4@=3MhNg^fH)`8YvsVqs&qRRn+J{@w&R$3&3ZF<^hLvFgmm#)*~kqgzhP%yI27&-;GswIZC--HfnCQ=?<#Qipe!7hxOArG%kACG;%as`yQl}1)^+mr0G?Igu6A~yVPtgL%M-^$@+Z>tv4{kO&G7`(9>BP8HhrWwoWEf=0;#ugEYnnUSIT7wsZgwk)+*(|U|Lzay;Tphy%A)rJ^fIJmKj}+ghV)tuIKkM;WaxBGNEwx>Qou`4cvmFZnLr7LW_or!^vvc(R4aGm)?JG=>6vpjc6l*UF~~y$3}zF8|aJ?*DCE30KML1dhb%ql`YPVJVNN?L%GI1`7MWne=yFW`jA^|G#w)HPAG^Xlgyb1>s2SK!ZCN;2??)>o1ebkIsshYa|*i?m~fEf7Fzk4bsqC7ghpzu*Bh9zqRaU`up!$bimalN3_w-D6W2wC=}`s@A57Yb7_+leWRL@#-^+p%P_J%Y2LHm2;6p%5HEco#4owI;$ExI7VVI}G*Ifxv$p{Ns7-H?ac!CTLc@+ORL}WBfMn|y3!eihkpkmCdC3V0|h@4y$P@e<y%V?Ik1|^tFWWz<MNC6U2%V^JsOTU)0YCyoYkOmt{Wh<K+z@T{VB_&UuRExEg(HAUeaS+NK%(^7&V<a?*Int-I>2C+H=%l&qQ>;lqc@?w703+{7!;@BE9=~7}0E&@-MizV6QJY;S?vmw#cGp<L!1pHvP(iQvh8nlD+`}-vFK<|Cy~5JO3OHH}g7tu`5z6hjKyb90{0mS^0|XQR000O8001EX$g0W0iVFY$f))S(6aWAKWN%}0FH}KOUsg9?Lr+XWLN0A&eOO&{+qe~d=dZwQ9yakvr0h7GblvQXY|Bo%j;;D*J8fpigh+@)4G|0h+E$+WANmjbOM31FD9PKIz9h6r0QckEbI-*qaXY;ckMBjNoRVEpi^h7ju=XKIZfhk}sl8knJUq9m(<{|@QP@%m?SySazWq7#(v2VAj|*$$B2NXa9`>TIWnyIC6t(KallHZcqEe0Oq*vt#TdrDI;8cmnb}j5u_}V%p=4<@lh?OmSqg*OJ<9K5m<r0tmm--ni+J0elAr{gpqZ=h;Q<6L0;vYeu?bseS60-MpCA}`BG3zwBYRsBH-B{cxRbsUj3nlv2SXtsE{CPs|rAhMYmALL3p_}kalI*AAj=vN6u!+kM-22D(_hFH4*Uchl^I)keJYul7cQU`*+Y?>icvzSw2dS7-B=Cmzhk1BQOJH$rJN;dW(@sAs{%mP<ReM_OVCNDJvlB*kl}dyVrG0A1Bn0jYl4jf}o(x|~=_PM`c`>JO*}f|%FxbP%M|PIB$iJ-lJaT1xQ{#1+r^(w?+yoc)_NM!ji8!1H+ldboaelnFHzF&VmmXA7H-7|0<2>Gm9G%ba=s49p>drQltI;-zOHDa5ma{by>=NLo#KndDZ8`)mB%We1pQOoQ$_e6wg<N{zPQrWVH1MPvipdF3XA}hkhxEof-T_!CRO{)pLISTG6SA#SJph`DUzKXvW+!ndK6l!A0#|3PCqyiJ-y@8=RC43317=RuQt~m_N95PDzutYmp50UY?mtY<k3XJC<5U1hemuQB`E-Bt`5&{Jk6-sk;^6mOAi;dDT{zVtMadBQ^WzPG#04e4Lp695bAi1yG7Ts2NLa=rIqXy`srCSS_{n)Y;*we*$`87!5JNx?)i0da#zfqN2rWCMzHe$v6+H*5wWvbWQZhBhJ_+fCQPL4Y4&IT4v?^d%xK<Vls2CjTY+uzdW&izu{`K$u5A-zxUp%x{H{K<!>D?BuF;^Q;Ipb8WDDF-LjFlnMu)QzWV$qkC@<c+uh7m~(tUF4aaQ&j~EO76RMBglRqg{=So^1EP=d2WM{p<(eq+%*i1Eu)w3--3HSjlzLsc*eTzB?5ejlvh|kfM!^nW1=R+xxZ$ht+-afKcVryW6Y#n@@MQr&oWw{CL~-$o~@V66`{(g3A=j81W6-y$BrGNFxem%qY=KVURN%KLEZpvT1N`3<23p5z>r77r5(3)Y5XP3oT9ZH6MV~Hxe#a&FeS$@2_j`TlaI8RbcLZk%F$X%GwH$pbawq*IR2`2K+3t6>2Vi5Z-0OuI>6wn%pSgx9r#!J*w+0l7b4bApkGj|JG9=V;BaiL*`nPH-$S*W?SOGi-8$KxJP0em~F;4N8$vShyYUeA=o)Y(nS{Rn-YP{x9pddT2_rkRTM7Up{pz(?22ZFCNbOgo+Zj6Nsf04Z2XIGEBm0t>$el3SF%!I=>0>nQsDa1z2OfEK(-OD4-O|I<~#?e^BYO+%=yGW*?<DjkfRrc1(Z8#ZwM^5o{R$Uc`1NT<y7JCa?Sgc>QNcnl8p(L`VM)d-{+I@so3#Wyng>KTKOgwXN<c}QLZUOBnVnRCYGPS&E|*M{M|^LosZ8)Nv+FLQLpqwTMd~N!cK&Y69G?IJS@mLYCkAmqy|hIB#3O0FiM3(|MTP|O29DjuC%8{83Q3G5X|0<hmj(%2Wg2F@&QIJ;oc((6z`064g`e19D;$wu55$eT0#z;i`V(d)o-)w>G|yb_UZx;mnXCP`TXNSI>|>uE}S*JSIM%I1&4T&P7dC%Hi=X~-fdV~oP3&ozSiiE2|qc+kUe5hgeo9(VpVBZh$`_jlc^@fLm}|i!0e`A*|qHKcaCNf*?<}ig6h`;00Zh{F9&#F+7gXu+YQ9G9%xx=f?KCrgq8D=Y$Vi958MU7MjRZy#27b)h(IwwC}Dymtp|IE>I){O&4)<uC6=ScXOnEU&rOj$x|p3$PySBT?JdY76_@I16AwOWHW`1yit#62^J#kh<?0fR^7JSn9Z5#gsc6K*9tb0lc!%aj#7{A*&~mVoCEdqOpY$)X<rs?LZJO}Oon6R<HWXjDTT=?o=5G&<#FHY9aj^#Q<Ni-6RdwJH(FEs)n1WO)Wl`B36bu9nYJo7_w-o*;3Oi76#+{oeTh;PWqeI55h7QazY?A+JyW(2?GYC7|#sXt1Dr{F$z9_N<N$Oz*dVyTKAfb_p>j;BRd2(hcHDX2hV9Y|H12&cI)?z3j$S~DZz6Q7G6*DW8dq_~yN=?o&SB#;4yz`xGBLRToWAOv>MW<U1lX^xmjnQ~ZOb|VhIPVcNsvY;<!UMNK+Kgdn-!<H08+b#69W@PA$Vy140na5JIEEX3!X_OFYlyN!ble7`!(Ndts6Jh+4*?-0(KH+Rh{(`HlqRP;l|ewHqSmAmU?aM&^`SUcoKMFoSfj*A5(>04$(jIx?Hw8~W)!q)8B&L&S3P_|yeOgI8SveLFbNrXB12(v4f_CJxmxI|x1ch+AQ*_%K(j%;u{?4pApD9>52qfrdI}xtgvPlje1noLRFaHEj)Cqz_U$bHnMNmNGL~Ve77jH|pxz3)rbVDe@58wx(#EX`U@fDLPHEHaEaqrp#B@Y^lQqEFP~{^6CpSX{xb$0+*&!p|EozQloARE8#{HnTp){~Bbmjxmy`%{f-Gb_4XBq?`?TfLGHVwKXrztlB-RF=&3dYhSInNJ`Y5D))7(bE>C0D{AC0rw0G%Sy@50e8=(LBhEUW^-&e^YS<-hJvc={A<8BwaH;fWaMzf@^gO%1004MiB?(x9R>Q-T!G<-mt*CwKUToq=)Ig$k0ddSHLKm*8ZRHBbDndkmkTKD*!+orf=V+2kAx{KB%ryhD<PO$st3b()4M=R7Nxd!pLLm3z^NsFwtrjpG?Dxeu;u_F?n%~MSol*ra|oEz}%=x7VBXe^3X@rn!98u)1sD5MbM(|A72NEAW=k+R3sOl=Ynw&%6l-yk@RHLaPZL-`J)(5R`{0|5J47D#>ttHuAUC4=4%QazN3IpRnb6rykWRV{c%hQ7b-MNP+*ad<csvJu@Es7S&HA7J5WBYs+>EIqCQ|>P=#)4;MM;jA-oWRpWAg@c_dEecRT7gTH{IX$0o9k^zP-rjpEjuY{!84Iv@w+)#>N$0MM=(L#gem3<Jd&bxNSjT4EIN&zKYy5l{AJlzi<U2YgmNM9OxX(xQMR#muCx&iY4;2@NR@M<-yx&OQv$uk1a|FVuU6ae}ucU9{bK2zwgF4<1#R8L$wG*V$5jOKZO}m_w2?5N&9epzP_F<JomKznag6TN07qKn~76{xIO-hn%lAcoqecqIheGw0!cHo2$!<j2jGi$JcIiM-mlVRQ0q&haMEu>w6>+5eoNR3lvAak{#U_><+S<OOrsoG`d}#T&2P2AKlgqbm0!{(GSz2?Q}(gD~U`BkI8B(olU>YIY47H5!I#s+2>TTgwD_)F6iDRf(avQo<q{q$>qiEulJ|Z+v&%Qyv@ALqHSdcdK4j#V!Wa!n%?Li%oX2~?VC_fN}@xJzloiGvqREU;#97@d|3N`d0&jjf$hbg!)?b{(>)Ju#-$tI{z9&V5hIw^O1^+(^$=M+-atW&{o#5L@l$3gNDGmq#dmeQhun&2zNjQViEA7}xRM!->cXvI%=734qwpZAS^AJ6X>!&ZGv29Ku7Qj%eYX0KR&k;S1T3*C;9>wn5U5Gt(6A71(O9XjdKXR$07Rb*3hNv+5(#6`n}^}17W*F&Wp2hP*Mb2t$^%q8;$nBa0rM0_B9WUA^7$k?>5dF2|1VHW0|XQR000O8001EX`qaB&M*si-P5=M^7XSbNWq5F9a%pX4Zgg`mUtei%X>?y-E^v7@$-xbPFbD<EyHgMbZ~>Qa1!8|Hv9t}4#_>J*u8)~d*Ch$UJK<{RNmH&zD&iouIXtyj%3I?P>R51ULrRd1f)+b7Zy!)g0|XQR000O8001EXjp|AGt^fc4;s5{u9smFUWq5F9a%pX4Zgg`mbYXK}b~i6yUukY>bYEXCaCt?JyAHxI5CnUF#mY6LrJ|sS5CsoGKy>Jc4`RvoA=?S$?{R3TXLht&>t=Prut4O*6x$BgzZ`oGnX;8oDb;iZ!Hek~r$0tFz9dg9%@8D`b!2^lJm8s9AK6TDl;NjWz*Y=9^=A9*asmZ|_2CR>?~$S{X^IRdWXe#28*v{|`rr?@{qzNnMvjHSurKpUX|0v2uZQ}$+19(}cBzl2&3=a^bfz>P7FvA)P)h>@6aWAK2mk;8AplImFOB39005;(000{R003opaAk67ZDnqBb1!sZb6<8hFJX0LX>=}ddF5PNbK^#Ke%G()zz+s&2$EuFAE@G1jkB7y6UQ^_-STe2VxgL3lY#{TSX`u`Rr>Eamu{dN7n0H>Td4{kG-RVMr_bd(mu`&bdEce^`XR}7;^Vbg->daQC1kb{QtXwino@0q`lQxPo#feNk*CS}Ac`c-YmsL{uIomo;z=dD`?`ue&%3zTmib=9@wTa9a2$(dU*u&Cb7pxhahB@hLbuh)UUC18tmQhDRi!F3pj1UF*Q)EBm+N~yeJ!iFY%&}l$+AwiKu9}K*7-hJ$G=v2HtfAo>wJT|)t&td0iQo$tNKt7c_B4XM-{J1nE_XH;}&R8o46>|Hu)5%`7WvTlM=d@o48VH6J7+vb&E3JCaJZs${v$4&-N;-<Gsw1t*Yu!R0miMnv>s@-fFa0^?kl+ceiaSWmXpaEM6VrWD^SDS5qY`Sf)O}oYjM#0mv;R3Q|v@M{Siv5y3P?@=2xfgF3)h?rR>#D_JAgFE(mxSdufFe7&_$tn+M}><l%b_$m~;5*Trqzn5tR9rp55oT;a{&L31(&40=>70ifjJC&)Dv_j1FJW<!CISYhju83G__~}jU{uFs>tq{2n!*-5Z?F;M!`<+|et!ArlUc`md3me?$5B=v4-SbTzS8`vZ&Pr_~SsvJl=3h5V<rR|+g0AL<^$|6Ju9T78Aq={Kh1|65^WiZ2F>>p~#ND)IUgzsPowLj%^SkR&;l_DjrT*TiY^{Ls)0Au41FdtKONu3KRE~aVwWIG^_2@frh`&50>Pde^yVPI5;(r_srIr@v=3XJ!4{GD-xrf>dCO)&3vj-NPdYbL7aKg-DU`u3=Q2eBFzuHK_Mn1DG-ggb(l5@ADHA7w|yCjq8YzPf*fBcxvX7XsIY6(AJV|^(*V}py~w`6N3shR>Yr&DBLw_p~wCgH-0Gx;ROXz^(~ISvCIftGr93gODW{pRZ7I^Pp6R+%iS`@Gh4f+J8R>b6!%u-^oHnQ~MO98x;Jj;_Deu0sF3;h(K3dEdYL`9D7V^uvGM{XuIUN&tHPi`HRg<@_&Y(<vZuI#rq9(h-P%no~<litb1&%LKUp4!mrMJG?8)y!5^I<kT+NsR^?|*fH?F5}RZTQ%4>P_bD%{8lW1;=dG1H<n{VqW;^sGnnu%rJwtJAd?&EEQ0p38q=_7oO#GG<9r>XcqZ@~2a<8zvnPTSX$u=p;eS#_hg3jAs3mmqLwrMVFKQPO>t<FSdgRm=~zK4Rl`Lu+%)N2HhTvdL6KE)3LcMghFz<pFT`1?o_yW@45SIT$j3|TYqPsmz<f84oT28*xfzym`}#1H+;8<ILtYBdI6eS5!094HL2ay}Q{ho67=qenhR%WXD5g}y*F0fT-+46;8%_;Cdk3JYwEK?IZ2!Dfdu+M<2Me40svXMr)V9tTcXldF~92Mwy12=NZlW?sSbZb-?WdyBp3P@8cq@_JCKz0Y&o?4hoJV&-X6Yo8G-q|2MJW_O6^xtawXdbZs4G(@qN?gsRTwdV#s#u;lnBJfIXE!)-?4X(1;0HL6VF>QAkS(f=z)yAe^H=6|FZ7{Qg1_?+!?&MCDVkIjDQMM901YuF%s|@lKbt&shd5Lp&_)^`9Og+MwDgnHNz*?p$y6NbGrhEqtRSl0HRGBG2stSA;2y+EdTArl`R1Lt(4mjC03yCgCJSFu#fEBE7Wd|T+*t)_Xfg?TkUO*DNJ^&pEwlS}Qaa!`p9`;p5;=KeaH?EVZAjpyj!RuhY1j4x0-!+~Jg>1P36-@wb3t8X8v%O3}nGEAt&p9)ooQX6S69?>`CM)ysBm89ElzEOAi4A@yiuO^oe}GmWWd$CD{S_c`Qq^((KtH)JXOl5eHRnmMgxAkDBP5Ec#7q^_FU~^e7T{o?2yl8+`PQsasW84(p8(|p6qx;t2m%Q2LXr#HvS`b5+qSHX4YYVHfAg&X!!xQFb5adM7E9|fiMQXlQ#k}U<pJ7_vM;We3lAD4rpTV%S*uN)$`xz@hIo*!tQ9q0qKblo=YpRKxncDaKmI6xd51yAzNkIh$kz#MCD}seYycDIe0BiM`|<9N@BZ~aajgtemq#V;u5SSdpDtGl@JoR$<Phgq0A(M(`?t&QuZ&S~V#7mZ3C00CO4j1;#v*|huwo$GP&*wc{i6hvegE;Bf5wOy_IL$PSCY~6N@LM2T07V!K(5G}x@c-UyR#O1vA2%77r1K8Lbkapjx?_Ti%b%dmmwJ2MimgrBVy!R@oT;kXxZhh!3pCifnXt11Q1T`Hm#XET(5yJk{zs*Aq@r@@oyYbo*-g0kFJmalJgM}Ni`s~0)1JA!|XYNmd6-fU|{khm<RqwO;*7$^_%jq1IWbut~m0TtaWeGRw+Q{x<=i|Gv6aT(ZQg}7(fFlX6hw75jKOdRtc*q;Uf+FLi`0l$_xiBK!`~d+7j0Sz$t*`RKiGx5H}L<&`$us$y)1@o{bO$usjn_2>23ku>o9#sGuUZs&UrTer_QPr!0F7+6kH|nygeQ0P051_x*c41X5QCu^cZz>pta8x)I4P18oPwzXZ!_c1SptSAaQ;t4v<ZGC|gGF{=YCJ9%{rA@f1suD>N+*KWG90%r$KIBJzQh9TNeZLI!--pTg(0P%Yr?b{y$*qg|GU_oV@@^{F)(D)L!o-FPb^p-9#x0?q$a<}i<#y@kV5}hjb03_xG4)a<~HKu%beRX$p3lTR!H0TYxOIA9qYuZO??6-~j3t@sn3Qto?EgSUasEusiI+%#WzbjSrJUVBhiSg85^dSj*b1n@lpb1_Y6U51X1!s+4vBy67jkkX>QmPoWUf??wOlSc3DK|w+HtazVLNnoech_DhJiz$C%_|sLG0C#Zmwq?e4aTWBSveAKB}55K&sTDP55CdU<zSQpf5WE(JPcdbZ0bXy+#xUh<jP_KrdY81VN%5;0)glwNt?%0r-^;B6OQ%h(UHgZ_=bJD^&L||A!KU{-;>}%#TM8E2lBKV!b%W3dajOTRtN3ZE~W|SOrmo=ZQGC*r{Xq{1gXl@bAXE4*U}1xop5_|uKf2O_3BcZ9~Aoy%FttY{W`>Qi_{~7+*-9;vmWpM;C2r{Hqf3XRlji3W+fmEw`e<7&l@~&HafybtG;`tP>1acl+lQgt->!SO5W!S(q#@OcJq!&QCrLhh9Xe3irZ6hc82|2Vj0N-A%Do==TT$#n12A3maUEV)Pi98h;A4~|4t0(JT&tGGkmuyIvm1U&<z;p%fx7uB32(p4esTL6tMIKQ6Ft)2+F6+IbB)j<F(jQLJkR1$I%xE!@!N>8X&kb-!y<twH@i2_@RR?2q`7k5okk^__I=6523u*D)~(A$qB<}Qzm!HuVdgeJQLt3fq;J6sE-=<8OD9gz$cCTn4upt_HzyXt>_)RRRCRJH7bGFU`X3?CWaivk;UZ=_*$1DR#I*I)4@rPvNIT=?XNd*Hm*-+TO-#crs7;vT}3l&i-ROVG__ic34;#>sbo<q%>QvP+oU&ML)PbqSbPz8pL7id@=t8D3dSUbs3{QXd`|u69s>zP%2cSGZ@p)XzI+%NL~)Ge>^Ob_oPmZHuVbop%fUCLvko_4I$kYs%hg=mL$pY(;QdSub%2H|fouyHv{3~QVz-%a%!KQgORIBx8)4qXw=+7j>Lv@8I=3+yzcR%mBf|>0`=l^`ujn}*C}b6O`DOkMhBP_QeTc-}BgBz<wk}3#@S+J0t0dKg!5%i3`pyGq`+8b^PDQz0Xifo5odB!60>ZFM+NG|wZksf{yjOA~et!2?@$rYh-Tl~EW)<qf11>1c4UEWQkGbwK7YipnLPZFilQ1j6g1_&-^)%Nwg^an}vls}uD<GX&JX+i=oiq<d;_{ue=;_ivj(o;HjF3#TzK@Y)7IiOlYIV~;Q29lsjT*6c4{JuchK3~}u8)`M?@a<M6!5Q1FJavs%7hlFZb2(*Rifwp<xlTe=(@+M0aXXB#<EX2I9LJm(EQ`GJh0XEgWr)pSvBJeLJr^YxNLaqVP)NVz>60*R$eha!oDYAb=IfIOZPaM^5vKRp}bz_>#fYSip~b<psR2l>~$I9SVHI~gZkb8-e<B^bm&c{RndHwlj-6!1(B7GBEcv<<^$K`g~vr4^Q$c`EtYP0Nkxxrqdq|n3kew=PiM`Z!z35&f{~+D(Bf2$7B)C&+N&AOYvum!O)cMCgdkk_2NnVP=kZ!zbcN+aH10@YVOTz$%S_$4n~nn}uiNxL+Q1hiJuzb`im%1Za(WEQYx+5pI{p0Is53YtltN&xCPOpdC~&8|<AUt&>6kMgDq87N9*Tg5o}n$bj>py*I&}n`vl+q|*bu1`ZfqhXY*hz4E%7SC@tovjUNTmQe@$ZYzokhF#z&fJd1M}a35s}!ax+Z-Y!!TK^1~M@+Pq{Of2&1{@k`OqTK=B}(KM63Y72XzBWI4%RJo1m3YIE$rOiZ(UEu8A((SHIxUMI3-8N5VT{Ip){?i$wS%D{$oFLLgql$9F5n{!qyCrA7Rs*|}o&Ir-rW5^bt0>3f5c^C#$%>}d=eRm5FXfXi6x?!zF52VFTS38kn>Rk@Yq@GtSsnxi@Sbznu2YA&T|x)(I@z?sc{~_Oxe2EUy9EhRNm%-T$XUYN6Yxgvm6!u=bMsM0)S=0XOl3(2+H?-e2U7gRJ#{t*TjnCazALaw>L(1rF{XZ$DFsWr2E-16RU`t4C=|@hSsli$X`UfVduQA1_uKn25EU-lJAm3gViQ1UU}<?2>kgG0gtm%9sgHv!ix$j;xw*TvPAa>Rh>^0XbC12$qTitlqk7>Q#hQ4FxuSF4mS_@XT!9gd_^eK4+6yZa)mgn`kciEkiSlPlEnY%#FRPABjP$Xzs4lCw5S|gKVJhd`w@J9Pi8p09GU)FQZ=TvC#@#ruu1>OM^lW*ew%Bt1_x(D-cxB4g7L>VGkI~Lt<uMCjLOP6CUb_*@1^zD!o4QlP4*k9fj~y9wx0jw@&e0ktc^nJP-TNdq3xWwQ!$1lX^mZmQlC#p($pgf(=wxyvs5y9W;^3s9-8}9Qs~U?L?Phtjze%n_9@FJm?F&18_vVg%4<JI7tNiI3voF)}D;K!tugtVwm2KH^vSz{Y&K7^eQ_MP=a~-|ge_ry{O~W^qM{69XkKl$X!KJ0Oto@S%$jhD8Ya+*9B+CUDPqF0t(PvLZ6%JIv63gg5$u2l+^yxKD&vSO2;XGA0Qz9d}J)uWoH3~3oo2F`dNBXr%tSgy38V^m~%zBzGuJO2dyL6KzZaqPONs5j#VB$rV;|8SP+Bn&h!Q^zm)31+Bedv`K`vs+wBg|_v?h8rB#|3Y|2fK<fR)CouT~VKn^=cfmle?P?LR>(onzR<@)Jx8;o19xeIk%4TCO`2iU(tg`t^hC3U+C4<mNz+|&n>^4hpr45q+|ek-pRw>p*D3@9_Q$p_4celCpgC-CI~%k7_U1(cCU9lx7hn4o^fgO9!~qFhCWulwT={#+BWaUEE7mg@P_eW;?(#wFTN0emF1qUwD0l0F&JgYhmMTm5|e9Clw{xRIrA5*X0uauG%+0>q4DMTiC`I%JUsTqGt~FrcE_j}p9w4furlmPYd#Au-U1UE9)|a?-wXwBuE37A>Z%>(I%L2U=>yx$gRjj4{X(v8znBVHPKFz%I9uQ=keeewsaKlof2TY0D;-r_s4~9Kn{xim@2;=KS2{_&5+^6nudreFIllL|mjwoZ;Q-WvV0w~5$p<hPFv&x_I@GF~8;<rLyhc8zT4s*Jkj25sT}7&o`#-Zm5v+whm<z>wd}TGsb|3R3tEnv*6w;@}X{1&vE^EHJsNO?yo#r$f86{OLA7zpvZuv(7B5R1+7xf{gQtL>1YpTlpGS%`p>xy#n&DE{<k?{T5u1CCFJ-0^sa`oN(xh+B9rYUIul9~18@?}E1w`o(|Phdj(hADauBmGw%ba+4IR;|nY_bg;@;^XxdUv&EH0}o+qo8Ol5>BUlgGQU#oYtv>cp?FY-x&E&RkX`09yKlv24l}XwqPmROJ0E~aXSCb`zAY*2hA$j-o7!4vSI)v-X)Lfx)MG_XEO5?7IcB{+7*12$+YG<qg?lhScrj4jm@D#b{N+J<R@UMPLI98^dSnxR+`?$Ol!WIS^?jK)yL)`q4QrW?vP{sv%vBh*-6r^IAMA~nq?@Mv-b#A#M&7&`JGdg!hDTSgQm2UuAWE_o+|qfmnMYszH&9Ch1QY-O00;m803iU@8JnOC2mk;v6aWAj0001GcyMKMX>Db0baO9sVRK)0H!ovsX)bVil~~_z<G2xi_g}%PFILW$lfAu{J?J5qq?^N`d%@-kv<L!$mS~%eB<e^iiLu!Kz2ESMCD}=f<{_4;;mmO6n{S3rlH}^{AFR`&<C<Hk+6DWJ1!Mba&v;ugdX2YATiI@zXszC}PRZ7?-iV5=_v}j5e4QmpGMjC*YFJThdfRJJ6iha>5qer>opv*u#WlT!Nh9L-e;d`tXJz7(>DQfBr7*+F-gq+|x4T-dqm4U!4Gvgt3f;F>HX<ukyOG-{bWg{+a!a$>tP&g6)x4d1q4d*?;jdKK%5qkV_#$hB-Kol?OlfxU(McI0Dl?UGOR0MsHWcoSZXov)`+-Hd9bBoUv4u0`HfVu}wd|YwLVt;DrIoxcf~w?`cWg08m_|Jbtp9k<&Q4#JT8Op?S|Ih|<9`uDjIAnFO%7YFYIE4|VI&+*;}TgI-gLDn0Fhz~9o61>LR*dxf5zrMT0gLh^q@^{Vmy$lVyyZgB5EVp`P6~vT|(~3hHr&gFx3i|5M;AA%)9e5+sIce#XSzpRy3Ue&Vf$0=0?=ArTwx3uBF{CjV}XHB_peq<l9ymD@!uGNxlWYhjpLG?+1Tw&Q~`VEAnb1TTZE7u<PaB*PH9r_44nR_rKxem*4+MUw*y651uYjgm^$Ls<kj<2iASH6}D((+go7@oS-nM2!zU^!b6n5#>FT&(x8Ru8^Q`gNNK|fy|XKytQAFX6>3F`QfZ`%BS>1fD(9tE#;{;u;+(NiQ*3JWLh9GGDxZib?#>s??nKLYZx!XW#3{A0ON~%f%<Y=$iB$nLLm7m$KO<?d5>rN&ZZ4L$qy0`0L6A+~6o(Bpyry`S$S-|&#6?-WIQP}yVs{X?s>VZGiX*)+x|M5L10rNvt&Pyn2+P&&<*#2Qv4VRtL^B%78klo<YZZ72wXImCkQE1SqD%y&+(CgaBH&ilvXqm6IK!<fylqrk35Bu(ILQUWpGBZ$XUJI9u}XBJB`BYU_xmW);Ns$t6)1a{+Eh?N8FbKvjfP7o)yJ;g{dltS@nq$vlLLtt6khqXMsXa!``hQ`=h52Z7{8v)>E!U9+Epb>0`XdG6ky6(#5KF5)H-VXnZh3+yA1Cs{zy?HO*H>KGy((Ai-O`>P5Z$~O97q?a4LY4*jKdR6}T2;N~s5Ho0fM#w4DtqvXHK@<wZ*?3cu#1v{KY}sFB5%axi5JdYfkAwL`j5lX{-Al?#kX{C3sbuD8^a9v;0_;L-tJXeL``U<hX;J=wNa>v?jvIy+lMPt6KI+zP#<Y`<pIj*~Qmfc~C&-3usx_o(w=YoPcVU^R54Z{H7V<djw~<dD<&kOaC(9zRXs;5R=&_91x^`(nq<E_r+;57f&+GIA9T?Ib(CuNAM7$FU)=Ds%P>;)d4agi$57hhdiIfc;bQIC#lrjUEy8{$I-ea;(f^)DaYh5TFb_2B8ZdLqiNr2!bIpab=_n`4<lOIh#+zg*hb3Bc|#FuA~!*vm==}Se%HVRZMlC>UaGdJL&X|s)UQ6C$q%2?TZr)6hkMAAdbAiiWwb{{sxDJCzSlNR6W=n<$=?Opi&9A67qmOh~sb^T**)3L}D31#LOouGly#{UhO<hGpL>?QAe@>fTuyiv8TJmE6Nmbm`J%mjIf`cxR>=EG5XZS7@RylOnH(Z-x{lX!a+gQawx(458WU2uE>iyB`{P=O6fWv$@eLH!2IG3f5&7sb=K$M)I*q=E=KkFshu&f^Y}1*9Rrrfhw1AzNDU$39r-A@HF+S=$qV#YZjTKQ5(MhgeYE;kNCMqvyrZT&hw%JuOs45%!Vm9g2ni|{G~uAPV1<$tR@2y+sQ=EP`8ofBrWqL7U7a-09j{T@rQY_$$an6|9930n*-^vGi=t9xQKX}t8Lz5>hb{ADx%B4-)V27tm*}q!OYd(Pdh>GGft%4zS-DdZSG9be`1>TSQHP=r*IwxXb5Hq14|MZN(j$YjA;|yBs(F&#lIWuz4?4VbM+O~JJ4cm_$Bdn*yF9rDe3q)M_w4BEi-AF467&~aqAR)y!~ZO(@P5G=>I=a=_dyOCm!mXXA>AOIkMEm~bV$a8JE<21lPHsk_E~BWljEJcfBkX$`<L6x``i1Qo9iTX9pyZ6h3fXZcst6a;0|q5bBs0Ys~=Ntv{rgrtI0jkIn|z@#}Nws%U|*D_UH?Ug>0V@a)s+YuJ@wGl#3~lH9b1|+RdI?ReedL>GAa-OOEOVhdgrJCI8b=F;P@U*YTO>&^I;*bPfCcfaQO*2=i4Cn#~3#q#K4g=?L@WlHm9Y{kx)n*ABLB?nOM36XqXXTiuL&%K2}oOy8vtI<@yT9DKn%s&isxcXWb!;TIS29C`SBKTK0_yb&UY&e^FuZarp!&N+YIh_CB6GWo_lJ&OSv#Q(~^DE1=cc83A974}8xrvoE63}TQsbd!WaN$qfhn<P~Nu2a%o<UI#qYC((2hFTa6(*^Q`3six#2&3v}F;BzrWgb2ZqsQ=DlRNVytq*38Dt#4YZ#fW;lr9@bVtbkf@kIRZY=Zr1$q=<A%RiIHqkjBS<s^5sN~Czyz2CseH_x8#_m7Sn&{X25i%w&*oV(ek>YL7lCbD4C-3RjzaAs5YW;bH>(2%?&dJS@06fJM)cRR{#QBVaGMM$n|L+R{4P)h>@6aWAK2mk;8Apn9)uv@AR008AN001EX003opaAk67ZDnqBb1!sZb6<8hFJx(9XKrtEbZKLAE^v9hTWxdOHWL1>UxDhwjg+GpJH1az)VZWdTKAeb6F2SMXfzy%gd{c;sSu<sHT8eLeE~p_q7>(*bDoJUk&E5MVqcyGkl}FneUVpXk!30oJ5|PNSEWTR(!5gTt;+F>7BWx7D&4DOvMUvQfY+<6R(Y&MlFD^n=qinMI2b(7^P-YwQ0c!Fx(4>QsfKrf$knYXk&7yk%cM%TN>pWCZA4KDDb})5;o$p?%-3nY2KhjX+7#vC@v_*9Qhln^Qfc^5)_IOJNpY9!s#J38&Eu)4HmL^PGc`yGpa=B?%%M^+qiQ27u>}pX1Mtb@Qe{S%+NYp1eBMg^5!tPZa;td&Nel;9)&h@zx}m88RaKWunmllox(59M5*Qa{sp2YETI2Lqb(T$JuJ6DES3iV<;cz$@tjc05qG(kE6BR`w-R_FA0%e>*4NW2|8D~;!n2jNFj{?wcm&sTS%$u#OHax64>~KQn`5U>zuYtG$L*gt3c0^upcLyQCQad9syiL`eleJ89Sso%Z%wRD1rxQH_+CNl2`=zYactDTD6)cDZ=C5wk1cgio0)A5SG)j}HfEA;MWc}tVtZ}qI><?2>)w@j1S6LyeK=k}s5Iw3h1(m}*k!2|l;!p9m$Q8ZYl?AvCN+kYbCWhbvNgAV7heVQqL&V@*k%pxLGU9A_0KH>{V!lMvteBl(hsa1=!D{ayG@>+5t0)?2m955XA+pj+jNBLeK}SNG(6e8Q06Z&GIq?at{l?;(nfOPS3|Yx68=-fEsa~N5YP25<aC3pdE!s@ncuzn4DP^ja_*Ft6yeiA091ZP?gZn5*C>Xbi$sQ;}AUj;#VZ(EO5r_l)+h4-pOE6MPwr`L9rFaJ9ID-RzJY0NPom-iK239N#O;ibv@5u+oB#}Zr<qU!A)aDIT;9XLaca7YDF*sbhMI%3H)1NmFIg%FJT0=l7@#FqCPa*RECdwLN!3h=gM(!Zy1_N}eIw#Mw>o97CoSwRXpLc_E5eQ;rLk=oHfMsAY7>mhwUTpD1Ne-+QJpV(Lg&xu1KqQc8)NEN48P6z9G*Hd02EO-_yV>vPbxutVYOpyvG;km{X6LTxyXT0<R(=Wkr!iEJhWXV?Z7h6JW-IiZy4EX>J>p32(fY5N_5R#9|9c7&?N=_X)>|lZTtt=%lvQl$VT*g?EW8YV&RQPR(%vj94}r>8mpP9z4r3}tq~%sQzloIbijhg7lrL3s)_*$b51h@EnK{fZ5GMve5{fsfOANov`$zmIE};J8U>ldjadN|ey^~!wgR0%<*Qon~R}X6K%gLMPH~)J4a`NK(-vW$B^&UMYtpEtHItjp)B{*AvQ`d-Fuy)$jjEg$2LQdpwke=RZU<05jhs+8Ev`!%3C0zdr=YR<X)l<pE4$~MSk?KI%AS7-dax1?nQH-R3Ql_~|LLpf5I8(CJPG8O_x)3+d-%w)VI;t42;5@610*AZ9JgD?j;|i!b%qp`f>8_}=1SXiTV2%lmp{az{dNwlMdYBoC&t^fy5cB)}xd3g(+6+ix1_o(LgLAovGsx!$We_*&-p?Q{UM`oor(;_aF}oOd9E)eOwhree^o9rVbaDtJj4+6khC^Q}GTM1k!&_vkN8_>6(&}?g{B;fpx|krwz$l)iO_^z&(?I6L2;Zh!<C~f2l`kgr+>2(Xvl)3tq1Et>;ogk*YtUy6NKz#u!)bhPbSmm<S65gr7QMN;NkhH4+II}&ls37v5fF(9A>BY1;F4v#07C>o15hjXum$TtMZ>wbuq}E<!4S`w^A@{>Kb<BxW;V$kL6K(knF4j9I8ydAl}CbC$jsN0<c-gm>sSQWkK-3U@CbU}qOcia8bUOkNd;hu-GTdw{^2bf&RXC42Q;0)%-CVtikC^lV|ZF)ZUAOz@MEI!i8`$^t)~Acvs&!znZ)7gGYio_p})Y;P_6zib~fOK4I7Bhj%4WXW(=poEdKCBxY1$>R@~EX*hWYu0RTJ9NiI(ZzEnd>cp+jq2lVaYb9#VWf!2neC*>2{Ef5555fynLZO;a)sW-w~I56!FRT*0N;IcCyh3xRuwuquw{tlh7HV?$QmZbz^!iX9i7YjDP1S90$A<8W*e{>6-)+)u~X?TB4FVeeH&;o>xJs~tO7uM*8B?HK>1BYHQ@oGq~puyP;dv$Z;VFB8`N)*O93<q#%ISRwkNo|0EuD7G!!UpU5pcaY%#N`9@tYo=X28k7MT$cc2WsGVTjYcN4=7VkqI<3IT()>+p2E=jkrDz33hb`Yn4{JON_yD2z&7<uu9}r?qX}hdUlEz<G65=01Y7O$g+GqX2)1#iJ<-rm<3{=+&o(gQ6eeaGrTEP*O34pJ5Xo}m8Ad7d8goGd?-HpzH%~sxprFs4Q=bKjsPS`fecXuniC5+pm-_hkP4*mZ6`)jYzH2tm%3E#3Uhr;IL>-HDUb|<e<&V=RQ_ElVD85kFOqqN#U4)it;`*rJwWPJhX8xLEpG{<ZU3uL5wXw1EuIcVSu4LUWKI8R$?XJVJX6Vs2JSpp~JO94sjTeT%gdg76ph5doq^~zRYdZFw2Cy*RHYX`Evle>ueYaL-F`|9{L#>HKjFYZYgISS1aB`Piw_zUZ>-BldRuvp|x|6|-@2j=TTP3d3h>L2otd2yGwM$)7vXvb5z-};|Jp*&{hs~WCxr>;lSpeYJ8YTO{XlXG}LjcBx<n1d?#ooU}i&#V@LtkZ8j9+2V4r6Xb=tyQi{cEuRas?>uS8E+o`hB<{a7OXMtmghxt-o=eC4(GEhyJaGQYEtaTY9t2y^r<(D71H9xtJl}>UzsBgOe(mWB*EKObof@aj{3q5lymGN+6C(D2AV+YqAr`pfdF@5*F~ACW1>v$Q|PieGLWC!plc2hCxlZIZ&n~XrZdFH@BskoQz5s@bX^y<HkNO8%It2HT{N@T(Wjl+6TxN-JhV1p!Sv(6-;dzEJAX66TEQ)DlHzHfnAs_YsD7yx#RL(DS#hVzk>~p{EPI-OKDgJ_F0!AQ-#ump!`eHh;ARE&w9;ZpN1stuM0Z7*Xcjr1UoOUj<MzpHLN)To2r)0)y($|ftanJ^)1K*XjS+2w%SxE7+|n3``HH`}CGr!F7wt?JIX8gFv0F&=#_x-a+^8rc=Ne{-V3>K^-xW3f&<L=a`65iy)JtzB*CGdqbAsBr=0qIN#M7+KHS}9Frk9T~<V5tT>tZG{cI;^c$p?(;;<4FzB!zrW;YY1S7w!Ih$YJ?F5l&lV?rhP=)-|jhoG2P=JcY$|u`Mv4Rq&M39iRFmK4Lgpb<*7LI>N2_C@a=!rAMo@)D_fw$f`M{gGYg=)OLrNzHUx_AxOfDu}fO-R5^jhX$uuz{G@UaGP&8HIynR_g#TKZs`mEzFCX4N|9JwkaF@6O)b;w^w@+NQGUx2z=JDp}L73;^s?I5q<Hns9uLnM2XrJhMTGJl4_Z^YM#n~-wzQKUSyfx02pIxEBzPGBm8{(m1v>zW!Vz<Y3G~e<51HGn$(`R33QB!P4mNg%PkZ9lnS}-qjundN{En>N>GYJ(!?o+$Lp6w8TL9$&(x>!|Px#y(~X$42<h2GQ<fo*4#OEdP<(O5H&hYa4rU|uj_fY^Ay8)Z8vdY{G=>J_~hJuzn;fk0{5E=0O9lJO^c`;n%>-0n6|vqwAeowxwRFCGct3JkL?3}Om8jE%DNBan^9A?_pinth~s`OH2svrokA(@AEZdYG9+&9rd_&OaKVwtq)4XrLzn^mqB}!X8eOpG16zC~|j*xZH8<Max5k_e~u1ZUJsPGC8JWi0&{39$Xu+ekyi@9_aC|ED%~{;8Aqx1!xhUSI}@|z(_*cFDC#4V&1&Lt561Ct<tQbs|Ds<@E62trcM(uKG6A}g3hweg~EdtPGE%h=sHx)H?g#6U?m^By)&1tNR#S-Y_~8>5+bS!%z^QS8b}?dNP`U$o<gSP{sV3c$}-gew`l7bfrKAfmHY^)kxnPuJ5@##P~D&k-8l$+NN%zxPk4$qyncW610E7yzxZkLBYt>6zhD0_`7pfqQ<)p}jA2;YKirvJm+aU@&K{k)Yg||#=(N;aEJGhy6?JJHfe%B|(k^>*jK>cMxXr1<>(OD3zZQJK20zjr8!p{;0%LdSLFNa;0<))OKfhSeaCZZ*d2=~kJbcUM%DcLElwE5W`2~ju*wKNJP<_{wg=d{0p7UxiOftSW%BXhaLvpG+9Q=Imz&HvCn#+iR>uR{vHZAL@@wb}@<4o-<{PNO$@c`4v_~cE*`;1IzKDX$|Yl(81)g{x>CquPpR)y7{J6#trlMd|YT+?FH1YMhwC$$jC%=_Gf@QvigGz+RV7m(`PtI?9*C-wX?b-jYc4n4{tO71Fa>8K6G3lY5kw`lc2!`AIn6G6t3PrG+41F?bVtz$4BWmt@CxlA)iox$<_OZMiT6C_i!Ob|pI1kr9&O096oB9E_M!mxNkI_;cCU_m+yB?k?bys5Xy%TpYIEM)ChzrK3^AHgcH6sk~_c0Cna$kZU#XC8)OPzBQkhq&om8ztwc7@aj?1%fUgY=%3s=E?{GuEcDk!App9vkaIMeO<JePpd_UwND8wK){hi%DAej9%1P-_Ovk^)3yavNF^(linmnu)r6<#=O|lFjZ8cqq<VX*Abk~j?qXwGUrr<Hb7iFuF`FxpWHkexy^A2UR1Avkg&IIl-`>-;PJT(CcRqQtuy;e;61r;Gc-}+H!CO4bn;Wm@Wef;SFOcVaiO?eGeT@^rO{g5^3ZMjl#yVHlBYO7lv8?f;K4bIpm+xks`CW3I<F{$v%;~Jf-}ylS0j!~nd2PYazm3SwP{CYaq+@UIE?V4<*P|14o5pHp6y$F{C-@x#T0q4|k4AG)9Z|HmmJ{)0jDWtiT7kG6Bfyvb!T+O{6{cVYAUn=Y&470DHlrU<Nbx6JgIlX|G`S3L4&Dn5`AYqvMlEUt{`P^z^35iQ*$twcSR?>?c<XW00tfc^Eoh!_D;_{T*3I#p{mL)Bq<FnrWoh2}F9kh-wR^PsnE4p5)Fv@r<&fWbYXFOZ%_v7X7joI#Laa+X<eu<<Pk^#hlv{gQ=3sBxq0R)wH$|awf0g=D!usxiIY8KFX$**c0}R(2^IsE8K5C~iyVxz$fHV2F!A^42H7`8t*mpemFHlPZ1QY-O00;m803iUeejrDDApih*WdHym0001GcyMKMX>Db0baO9sVRK)0H!o#&VQh6_bZKvHE^v9(U3+um#*zOYp8{d35>iW14|`woDR=AjI@yZ7yXCc0m6CZZ5EzoMKn#qBD6XQa`x^K4?vvcFyJvvGgVH*wx~khGA~27ho}PZsU@{twUR6<C2FpbGumAGu?GGof-@kr;etPzN>VJ6u;f)`pk)H)|t|C9)25WU3CRI_Y+%L+YjEgc3i<#$fSAV6GMDGg!ySNBmrKvjkOOPa07^i5d)03)<lj0=Y0NO0uGvp{JgOg&P>U6&?X60>p;cdz?D=tn>)^WM1ma|ZAPln<7GvEJnt}B2k^QzqVap9L6<^S~V?fZAWUU_i}9Me$wVNj~I&SO>ZpnCa>3QK>hQUX)>aq0uEpR1M1m9)>e1_Qj7u6r-kH!8>c{iVVj{YWM85(TA7_I_0W^4oVGe|+`sHBl_pWgwt+n#e?f+q?Ha`metK`ppmC>-T?q^8*0~saws1Tmc(QIZITjF8shZgbk8HS%e8z)9Ki&(xS>TotG+_`Y1-PEZ2GkVgz_eed<>3x!bb93r&~^6@+CKB!0HhrC#Sjw!uRFi1Jcjt2F+EH72aT`|+>-PUq3W7JudcsBX)z6TSRn;eQB9FhhDAsXV>`j}&pbu96@}I}cdg#F3I!`LHiH!1fEDF!dQb<zr3OSk1(Oal7Xr(1#$rCYpnSUaksV<yal8L&4PAf)TMCShx`QVesXiq2?8Mg}T4feo&Qq3qpl{-CyTkLq0#!yEM^3RQQ`9%T(&`HZf$O07&2him((|&K_XSS33R%i~;_pmM2Tl<zySg>4~}tk_rq$ysmCR>}2n)bnY)P<9v@bQc)aQK6_ghew9`QFG!FVApSPU3SVKid#q}DoT@c2zM<(B89FJi)c)V!e4G-;BoN|sjV=jB*n$+c5y5`t7YZ<e3WPStnCJoAVzi>NZE&p$uiRj<mueH-#K7?;E@Dy%JIqs8r4OM^*E9mJ+G-VtF%S*Hu*!pQfAa0w%o~kHp0~>N)?X}ERaxa~v4G0p<#u@7^Gw+$C^nEG`}#|vQ~R7N0ThDjDl|eWRgqc+RZ<e=q|K^I!&2*{Zjt2GA`HTffsq9z4P$G;Xd9_0_Zdydl)r`sASF1(N<ggn>3c8)noRwV>eq@CLydb{ZApO@ewrEGxA^$}ryn5vm;UIp71)S*>*1cC!(AhT!mFRYdHds=Zw$c2$G^NffBFn?$DWUWqvh$-)$`M{P(4%O>Dl?yD0+T+x>_w?oT}*Qi{&a@oj-f=ELyz?UoN9(=hXJu%W(Pf{6(a`e0KhP<W0H*{r%nhcYpu>V}Gb;FP=XA;`#E6XDfAnwtW8bEPV0&?CII^<+I==UexmGi*psNR%b6)Uqr$A>6cL$J`bM1Jbn7~^n7(rLwUc)qJZ(^5S*6k$S7}PEc~3rQ5MowB%&{SW@QFwA0QJdhXBQpwW!LHnCt>+N3skjkoyus6F@`!TE+Qw@fG+l&n^G$bd^LXflZ0as#E|NG;(EDgCs|^)0AXu11W|QPDu$v<cv&#hoBf0Y4e%4Fx+D(Nn9`*Wq?5-)^VOp{iWVdA*6WUY5uv6q3A=XDgIF_G=T``Q$JHWBgPFPhzKx%^36?47&dCQ+!K^sS_cKJ!I~b6m`pkv#8RU{8^vRVI-82XQ$-R)dn(&8PD>g9Si;zB2}2rsrQ~;Ap+T7kH!7iyTTR43bs)cMOg@njvbYSnEK7ntBXb?|PfEiCc_d{)S^w6hmQIE)JSOy?iq~m9@zz*iB}NAmQmGI}TXK}7GKnNinleT&p+rf_4xI~~M2s11Wf~da#8H-3yC7jm5iEy1Xs9M8Yq$>+&={j^VqUO{r&=nN2C@QW74e8+u+0pR8?FEbjOnIIg#;U&af_Q6Dhr)G0k35F>}d6CmD=}hoW@cHj=xr37yt?zNWF}y!41uVTJb}nBL1mvFnEAPTEZF=g3GD@i&7a8ugVBoa2s=J5^uG<dRTCsn{i=FHKXoxsliL0Eo$^4<#p?=&Q#>VSp;LGipDIHI5nI?1*o!*Tb=5rG|Yol$sOEYs9077k;SwuRKQOj7c+I}q0Gtb>^*RAjQJ-TdRUX?Wrp7Ff;|&KYcj!kMKsCc@Vd&VZJzImS{1X|3T`vE@W8SqEnQ5F1f3#?6DgL-^CMaD&7Syfr%X|iDHX|M2z{tnKvv+PJ&YU*uY-NS%TfkgC?k?6ox&pG0j9f9()C7xv!P6K7OIk`vJ)f~aF((i37H{YS*^|Z;Re81I|TV!nGQAnc|_t8MTr?GiC5-nMS{2yo-I|dl~QJFd0fKjOOz0WThJqsozx8gt!aAjSd=R5o2@J$;z2^SI%~JAU%(Ia=+Pqhn|%b*pxiuo+>xMo(BRkXJHg|IEWyriNVDu9Rx~%wE)$CME@75|e~O1SEkD*vmLGJ#R>W;^)I(POkSJOSMMfqylwC0h=sI4TY735A%OG3wKImZ;%kc49Q6ng%94p4-T6EwylI~3)j8Y_Ep%!u_1v{;yOogn1)H)};psnCZM+zwaT4CV`KxLZK>{oHYI(S)q`oxIrpj@b<%(mFkB&f?T#Qm!zAlx-+fPAZkvWzM*0JGj&7YX}LJaZW++o+)6wy+ptaPS&sF{`tXCrTD0CGLY##z?U1n)Cyi9GP2?FU6t+>nv78;vm$lm++U=cEVy9%X(xL*ah%zU=ZPvQmZu5+&v6-+*`;ey|pHVE2{$aNYWJ}FiDB&No+~EEth*Q(rbd9Wwa7t2|_U=1pqp4ssR6Gxk-cS6YFwFY61Nn(ag(or(&K0ID=ur0xp@%<nQvFsx{e$Jded1>xc$`f0_r3JceIWyGSDhWz(~*%2LYr%rz;~fuJ&~SX`Cy6UJsOA+|4H$1-&>YihsIf~6v#i0u!Ie9FU{D!557KO{+2alQAzbp?|(VQZQ0X$eDRiV=TfM#K;Bpn0wsoK%P=;mv|XgOBu-Be{UMB~12k^dfd4mXMRs{AiD-Kt$LsX^5?TT(c@sJD4-#pOAG580VH&P2aN3Up3_$L<*%Mme?q0DrWf=q*e>Hrv4|jtjI=CI0D}#+mN=v8naiBMuCS0dEmh{gC>IMfEZ?M7q3&qYk2ai?J`%8XAfPYWxz5Z!jx4^6_8Y|!hp;Nt>TjK#vdPVUQxjv3Yd0y^5rq9)@xXL#4T~LRA_V^7i<Q++DY;X3QQ2e1vClz>w$F%KUe_DZa~>KFY5#jsMk1C7&c!SWu6fK%OghE+DEKtX`XjbcG3-^ks@rwieJ<G<QK+{XzlH=;1~%ml^0eSg}&F`bqb;rqfkThf*(Q#Y@VX42xD3XWF>)divM^B!~l?##g29HvdYD4s+Q~m!?CwT8*l|bdZd>?H1JOd_KeBhf%Ji6OA-_>i=pZu$!W!sQ>{1rgauOx+L9|`K+6l}AP-CmQ$~)YsRwt^gU9(T*4Sc(G@$8<urBl}AlV9y2Y@J9vx5QSEUgukjp7-RdR#$>BQFCr5zv#TQW}%a&tS^PgTKI0xdg1{2NHDcl{#jgtMZsFC7hV8CU@OdB*6^@EbMHQCET4GcB5kUSWB?}P0Y@<d4v>uNsyJA{l0BhvHnI_4_44%|5TOokf>DOXx6Z%fP2-Q;2L5IUasKwC=Xg*SSf_aP%28iIr2yCoB5#z=CNeKtSdn{sFl)`#2@I_+s0&2U`;&D76mJDqZoUBGCw|HMh!r(fN1U&@ZEtNL~Tv!Jt794BX)f1U!^_~b7Fd42><ZfyCG>V!ni1)_#hThwD(16@H0Jn#24h&3zx#e<cI+<E<&SaUKJZopa32O#(zx>zEkk4b$TH&^=S0I9psucreupK3lyo0qMDS`YqZ$OF=8K1QbrViq6X_d<8cV22~{cPWTri~QS|XvG9SfpscuWkIUHvu&v$cXI!9{dFBV|Ga<P!8pZ>wegt9%FHaWzLI<?~e!+(dB!Eo<*q1;^Gp;ioda#6#NSDox6+BCKFk5d9^QeNB4PAhlmwc0kIE<4Sf@mw??7}8Q|rqWrQt|)J1F>rIDt&T0wm#ua#L1!+W^>e$)VNjih#0Ov)w7I*lYbY%VX`#e$Cm}H_;6`%<Rg5R?X;aEW6##^uGxCzN@kE-BM^k?^X;0M|dsbx;cyfG4G`}!#u1@ES29*&WQhGmWV;|NTp-UqBCzV2Hj$6Q`<9O2a%r<#YFs4^6oJLJJI+_mEaB$kKs-w<sMJsUWR=Q!#p^g&N^n2P6r(5GjHiNwgV7H<b?(}+DpQBgNco*F&*TXni#|}qt-mdr1t!%u9ZjCYE-Qvc8^cS-+Cw&~Os_qR_tLq*K9gEzXvG{S_;*Jd*SS`mE^(VYlG+H^ta<rF26m+WQZJBA<|4tq+j6|@!nMZ&z?b=!akpzNMs_w`Rg-E?6=ySlpLUC-+qX4=0J$QQ8EwV0okM092$ewz)&^Ycrspfh15U7?eT`&47%@D0tX{!+>S^YbKcR#u?6ol6>;<RkPs;lEVAX{Ssex0`U?`{;r`a}^Z^T=yvpJ~xXzmRAdP8rRc#r-XV)?9r7K*ZemVK0)sTrTBTs;K3biX9<#N9M=>$a0OqAD*XmC!mr7%G|R)cZ?0&SFxo@yqwu)*1Q@wrZKIEUaLIy?>Z|SK?YU6FwKD;Pu%9~JM73?g=ph$?sQp(HqKSFuwK(bJh4Tw3C^EBtNRY_$~*?j?p+)N*4}yPfJ=1_zJl$K1^}6Tx0w{QSDOBfyIT@Z{L#<pXadJ3gikqc+x5w8qi!Sk%0)Szbctxkw)n`d*9BX{IP+ByZrJT8H!Agi(co9wZ}h+W*4ffrvB&$}w_VXB7mxX#mv1cZlb&`Lr_60wmvtMj`7FVL&k_)lL&pc?Y(afe;MklLFur)vS735zU{QqNbr<Nd0lM!U7jR(Q8%e3RG{@WEw!pexK70D~i)WKsats>qUl-&*6<vqi)L&4JR9tHR_)mUWA*#NTzo?k{v)ODu^$!%p6n+l2aP%UNPL5X3DIHVQq~+qZecQ1?a+Xl;8~$5k#<mIpeKhqONz+9v6vI$J=&ETOqv*83hd1+M|KBJqWfY2z7i;VvVjzDsBOa7iSFwMAeoy@Kd7I(;IL!FmKs1iq4G}stw$;_K7d?@a`B^w_+bj9);2`sSa_OJ9id9mm$AFv39Rhe(7ysVxgquaCc+)?$p2HtW`P-8mVgUX)b0a5f6^H<W1@GXF*l09`ZIq$BDfJWeymi)1ggzxKia!39v$&d|e?mZVm~#dQuoBMv-K9aX)ne}TiAp$fmuBn*jcn&Ie}r?WKOsxq7d77f-gLlaO+wPA${~vUOVqHGu#9U$s{seonA)}#Vf8Q`CXAg(NBK$@cQ7XG{5Jv^BIbh~gd$GWq{(iq;#)q&0=*kvi%&vYla8J^zKJwq@4Bt~UzzP`-i9yy!0yW;*eFnf0{F%=*o3j!?<eocw4q1xoVzCK$DO8L1tPn$fzt{+1tPoM-H@~k?UiprDcBuC$H^rMlseJ^HcsULAa;5wBp}QRKZQ3=$Ixn56<(--l(2!LXbGbGuo|b4)kWYpk)6PBL&H|w*N4@FcY>IflW%>^C6L`{uvT9KvlVyPpx24*3?y-rrEUNOQJuS|d@`cp1xxSt>N`PM5GRJ=9IHo6Z|vvPSGQ2rlJl$ia|Zu%+e_i-C7*~PwPMmM9Y4YpOs>x6T@6w9g}$^S_N|Xa*V><60|_w9IG{j1i;)G2Li>nPJ1k1=<9!e4#5F?}g^ryGV%V|2QL^Ywp6h(P8i}1Vr|%5m{B2xp>F~n^VlM;e-pOqZ#~RG|gd%wzb_piEXxZel7L9{xy0mj$osLIGTy4Xkmb+-v$2!!ULVjb<CDlpf?{p9mJ(D<$%Vh7T0UZ~#hkj4+=gXE$S{3HlU!2<mqi?^VBfkXiod%tNFkl>0u7{6jp)H-a<kP~SPg-r4D(9ni1*t?ipa@rV6VRbz8t{aH(fH0`R7ii4s{r!IE$Pf1z=U#WmQe8h@8hD7Gki1@%IIt-fvTNro5*G+Ks$^a^O;@r{-Jl?m(SmBX_t=^<(6cRCcGPNGv57zs8U?i2M5Vl_W!|$59dv5D{Y_E_{bTBrn`tE^s$E($vdBS66|&R#Ox!qb`^#iw-fDjl(;=1ke5r_h(9b=J=rv3meY<f>9M7zquVytdiCy`f$+*<$hKG3$zqt9Vp8@A7`w9`w>pTHyzICyOcu<)CbP}@vxDv5fz)GLCjpIpBZXV{I+$vyY1)YW1$z||Vil?+4w%n@m~+|{I-%?)!)apw8Wfc-EIPEKOz*WYkWHYVUVhp`Hw&_mRLOt?R0D=mBV5MB(c^3~BXuE!%a~#<9-H^Hq@eQ%kQPhM2Ms4R4Vc%ua@rb^QxGx={!C^F+|x1VKA=c0M^(8xeleQR;a}R{PX;0`PEK9&iRKv{Egwt6wvX*36<S*{jWsta4CoTTtC3?R=KVp59)M)KU5%(<G#`p`xT#2YOKhy`eKBaH2Mp#<Dd|^ZJ{pYgXn^14_oIG{)y#p|blhMYN2B><ZvBuBLRSQE-XoEy{hB`fI!K#ZJ}>=Rc+HSQ^r<z2mHUo}+xlTTwS{UHL{XzH4s;k+3RkvUf9B5j3-$C}9r-v7n%0JJi-Y87Pu-o|#XmV8Cdy#n0UmKNwfrK;&IdtEydIA%_`q}{a7|p>Lvq+p9#pr^d`#B7shbR{od@jZ|B==)#8lBjq@|~|#fv$}AD_p4Er|DCPde)FtA=(P2GJanO18$mmVc-l9&i-?|H#ne40lk5T3WOh)wM-RBCmNb#_N;L_-qm`x|2A(KE1`*iqG!49k$5vJjR`zpc(hL>#{u9%)*;`gK=RG6VmftTz;!}yBgV`#NjVQCKt8OQC!{?{ITA+hrs0O;_P&OU+3%lO>Ko~*>#3xyMSXrgohg)HOZ`8Zx@c&7R3(oVdBsTHHto4*z<0I2}B0t5I*Z6tuFl$B29|mM*n=Aa&&aP3-Wa_6pZj#)@iwu?MxZY8QHJPzii6C9Nj;%h@+oak<?MKUw=<2jt!4}vZt(&BMTb1T#CxBN!H5?fU<2Kx#E5&;FEgQ7k50s{ft!isN?@O?d{S(*t{nBW&PAm9=J5_WIXAkN6ErYYe>YnWqSu{3Mg?TUzHA|M2Ebsmh<E4L)Q)M#WMrT_E)vH-sf9(|L!cPdw)kC<N+R8KhA!Y+UL(2>#wt%^<zyql@H4}TP%ri$}?X+qCRa>tcJBN>jc_CmF@Lxo9**?u3!DNa(4%yZ@))$nm6F>Yfhc_8(2RIEr<XEEqGh5-XB0aJPtH-<3KAk1}H7-x|}}X3pBZXM#9~xIIudCfh`F6HX5|7wqqXGuoM<$#rX<Bg+Vp_d1uzFK{=L6_zv~rJe=Tb_`ega)^5N><9QCovreYd>t{zO+9lbMx&6sMoeZ^#(MnpJaMsYwf>SrN<-i8`bavY5D|^)o%TOleD4l!fiz+qT5H^;F?jA{_GPlQ2BSZUs+S61_dS`R2kJsx_SM`&tCQ`6l@SPzH$(sH|wnj9WbZzS-=eSh5E;phF48&Z`R&g2yNixnyKQFI--p!wk|NY|XkN@*u|9w2?*U8b(%TcXECtX|}a@oi3Y*?+V%QgXwZ?G=A0BenMpvOU~)rzuvO2+EK-f5Zp#(LN<lDmfw<nw&j2{0d4&-tXVRj=1-U;7ePNSd<>Z4<+Ws=pa3hYr|?^;DH$;X3NmLv0+41{xh5-3_|Qj6WYf9jqo@#Gf>A6T3B;M>gOb>;#@fpT_N}k#CW$zyEPReI)piu^TLk!+X1(te$C4jKPc0RW2lLNkWI)*6W<EEKt^Ig<7Cb>58#*>`Q^{XQ1od0MT1)*x&mYvPc|eTMq7LEIX#Dy^lf+eRu{nOV>Gj%hDZYHEDF)-aG7rzSEYgt)usqA8FIuT%zEsZC38-ehW$j&|V4@x$$Wv>782Cq|k91$UIsZ=vsjJFi!{!*S=Z+9X0=8|N7z-S6VHgzM5|DX&&ZNQ1$^J6oeA#>Td`HY|Hm@<}+wv9-LKAI7UspNe<A|tuUEMu@`rJO*A~vy(1xwTi~K?3J)O)8<%!LxIRO30nVy<9<Y1^9z)+wb$9~aav3Cj2cYLtxbtb(ede@=U5-wpA$MSfI03F-25k};#{d-*_5^CC9U?~!3>ZPzE%;NT&Qu0`oCo=d{+$fCpRQNpxd*+vNOe3D7nfGmXHxZ(%JssmyVZX5A&~*?4Vl)yTDiRRPalzEclxar>m8lsSiC+z>7-ZcjR#qeuUPW=rX`<mS~4jKtJdxHHHdC9wP>#4fm>AU=u;#(A*{ynTW_1cp$mMPn{QTeUX;glnG$LL6Z*I=IFTEAfDqrGvG>P(R1S?Odbv>({_={?9;sW37-c;B$;=E!{cLh|I-lkAfnTV`qbX%EjDEK#E2*#TVn4Dm%4n48kuc1DY|n2=H}}*fnHqu%BFWDPEYBP@+Jj3Ck=Pb7g~nd9#zPiQh-=b9GNqO;Cq-rrl3=NlRyCKH+M6JcgS51Xr&f~apEj&~f0Jn^S4L6Cr4nO0vPP;(OU{Om$_&hQO1Y%eJgjWVic7@ZGJFFO-&2TcZ&bRZhI0ONknbBzdE#V;%G^q7O`Z9~o`Q81f%K@IL0T<Y!23s!w!AgyF)`NerD=BzwY^&>OGaOI30}>r&U*(03M-LDeE1%Q$!|UsV6vtsmX7AbMUtl8HD5tt&-BQBXYdH2E6A#s+UMJLnBxw(z>UCTf{*BeUN2d0Iw|Xdxiq5FY@o;_o)86Re0&J8;eY;I93e17%YUSL!V#e*8Ct#l^Sk#yym|fV<C}@ipnFbeK{@eJ$>QA}S(rlPcQu;$Pdd5tQ8v(t(KVgY(SR-%dAOn#Z;l0*+0-xTE(H9ph<anUnF(-2#XyKef<$~ocFZSvKqZXLho@x*wLiw+CxFV9*F4wB3#)zTE3$U`)M=k~+jqRPqa#7pb<67bT@0X8W$!hk19=R;Ol`topmjYT%97b<r(Tdpr98N<%Na_~{Ue7YM|Q55W>fmal(y7Czq;)pv@$=m;?{t-dHLYH3>Vv8mZ=|Y5A#8%q|B!tsYd&NKjo<N>lpSV)rfE6V4Z5ZLz2IIm^Sv++>vLFZ~0Zq7tks9i?ik%9p@G`r`i}3`t(Dd+Fk%qU8f@UB84VVZ(fEBoygj7)zS0;+b710E`Yis!q023;#)eEMk>x|Km0OYP-Pgw>{Zj%I1!yz{S;@%P}(W{9o}G~K(O{H$UZ2mx8TnGPQpP!dGe@%G%BBtq;3PcmjOCg{Kf*yv|6dW_B9Inimg6<DqE?d#zYsH6t2O<QqBFI%heQ^XTjbjXEmCjsAED*CL~T>OihUl%a_B!mP$SAB&GtDaSJKx@3%f~on&K=>2yc=7WYu7GaQVle^VHXYD-4mL;I;j%fX|ucSlDIhW1)4d5y1hArrnbuk&fmXBUPzuEFjeLo=#ThBmUWpbK7Udl4OsWMA0Z4-LV32mKwl#ohd#4m6mp6`a*VsYv`4I;s6Y!!}2BYUJ$4v`G<8EU1Z@j9gZ`FxS*BmRgq>A;<)EI;Bcqvo9WVXhZN*Oq=YB&vbnyH&lzhiMFSPdYKAJ-UKGmIX^u$su`}!inbC$W#UnCAZIV1N-dGh_Brjjz>xe8axrhFpa>Ck(H*{T&FL|NJzZ;774@`<o<*?D=x!$e?f0+#{N|%M%KHj7hsKjb_g4F`bo<=HUH}XTbOEJYA`65Kq3JqXp0a48f)|cY#IwT|`0UN`%@pPuTVs*^lr9>QHWZ0d6EhI-FXR-i-BbrU&CQL1)(K&K!mPn-FQ<4FNpd0np}Hj;`M^niq_<8X@xu83=AWIO+T*RKcx@Jj>K;toDZXx%;r!{6zz_kQ{-owkr)LwRK#k+vWyI!eZI|yJ82f8FJ8N!dHlc=lXR7s_N#vYhr(uSMP3R2b)Sg46RTt|@cgp0DJ)T63$heFAw1(-(^|&_`XTV8slfNA<$?$DuEpG6DvYH$)6@vR`X#Caao0_W(Ifc0x66^fVVMYNhp0ACQy!!lLSyM<%kSZHuJY<X8p8NNFgo93e-6&&(`SxZK4wvhJMDZ2hZ=9fE{ej3*(OsDJ<=v?{RNZy=n`6$Gc7KQ_Z13YRq_p$7cRQ7OLATD_68CTkeKN_X?xxM*OvMNHX<w)wY2xe}k&*y=LKE1<Y@PcMt;#Wr&0lsu7Zh_goS2>-PEH*d<v?=pp+Sdoea*nbx#h=3uk)0wO$;n)I3K=+#vnyre;o(42e6rg!42m}_fH?%34L(e_~9*LXWQk`EXGgGP3L}X*X``?<hPy4Xs?u^-_$<?eZ<?(^I%`Jz1c>Cvep~WmJW+D0Q66@8Kq2%@tC~shSR71IsO-w`%GP;4cqu<UrxN<wqT?Bn(>DZqw<hfZ3(;*iJfz_-WRDhy7k9c{3ew}wPiAJ(Tq~O!EIbzo|+S%JWF!Ex&Pd&Q`7q~>6fcBC)Y~_til2EA!SGF$+03<(=6Wog-=w3Vo2#`K<CH~xrAFWR0J0Tl7c#^Ux2`|yqvl<r*nI%F^BiS%S4y=>g~*Y6tO`0m~^TAm_8{3haZ0Le@(eE+LhZ>={0<d8y#D7k5akSzTPS1s{rV;LS4~ibp^fDhg}=f3gf|&$QRVXH?Zyl81wvvtN*}n@@WOIoWT!N>pD)g|4#YFwHC+m4FO}&ZwzoXaH0<lJn43r4CH_4T0Khe7+w=R<_%w*4e~|P-zG!H%c~2_YR+d^{6D}aaNe6EhfMXEx-E$-(T2N2jnO*RZ@k3IR^76xo6Jg05w^Yf$+%ep`XL{R9N$KnM0v0?7PqRo(`XonD+|MxbZ_J)7RLIu3fKo9%a~LDYJ3Ka8&8mONaw&6g+n#-LpzHyJny>EgOlUnkrs`g#m@f1YOH#X%L<R1tqJ2rqIY$ZM0YAT@p@C&ZsJyLjW?ZQs$ZNIH9Z%G`8fL}Jnm*_I9)^(H@P^W+@!yGYQs6(&|!DCt%?1bGYZspz%?B;QP84FW4^+ML4L*mPar1xIgxnCevjYEJt;otMn6qj$&B7G)8)Zn)caphO9KQH000080000X0RE0lpH~O~0E89*03HAU0A+Y^WpZh4Wo~qHFLYsZUv@VyWq5RDa&BR4E^v8`SW9!{wh_MTS0K!V6LYDR9Vd>QIcyYFd8^jDTV6XCSz$pWID$nIGyqB(tMuRfH6DBnNA~KVA%RAtpI>(~Ns`OQzLBN0j90w3LK{{q%~V}ukvB{{Vy(l&uY=Hg*78o)!dSz2SFtzW|L5`tW=f@Hx66|xIXS7dYFSa#gB`Re3MSiLX)y0PWx18AGbblutyG-_`$pcxm3rt(t5jp+g8u<UK3Lg|t13Pz<1g&Fy#-ag?IT_zi58y3?t5V(!6*DS;yrJL>3J!F*p{0@51o~*$ho$%=A|8vya@}JS}UC~ZdEJG;uoX3jG0^h<=6ixYT1Z2r;1x1nKX)5MKq~9++^%tXj$)zXxc)yd?#S0lS<UixT4_gjnHYY)i0v7g;vUDPz+m@$ykL$vUA4G*xzWws6m&)Y~QO+Y|c*@{@|Po-ZaeK3iwpgi5(O7ycwX0W)ITdG6`+r%AF*Czkm0U83CcJEOAN6%Ue<9Cytryq)j)th*t;%wrkW4PH?@0wK{i2BX+#pGcQQaJ`H`}?CDB-%Pj=(G3S_d(F?4Wk8lwk_vYd^4IaCDRXXLIkZQp3N;>gxL|5LnT;Cy-Dv?J$y@=l6#CY$8CabZni;aaPH#r%sxfkii(ap>m$!AM{K%KXDm4uhL7M(TQAN3$IZ|g$cxmSN=^iO0zEx;2_%u|-Q+&ZP;Ne=I%Xvf*Ppg7=S<dy9D!R8cM!44H&bEVb8RU)h8+GRK1$WbAJu03H5r*$dlH|#H4_iTi9?;Dbfcl*>4(jO%gwxFl1rO+ED4MT#1sU=!^JY)afvWbM=&R)O%ySsn33^8zpV;myYnTSgxmMJ$~?%eF5XD2$LVS#%QQHXUyKxG6zLvYKAtla~31nb2N)*z_ZVd3q6^JHY~tvk!?=}t0vHojdgWO*XPp0m-!8B5}&ak&Jv!cEEb^Wx;p7Y5NPs&q{1JF(w3aA^hXu%bP8|2_ZB$m80J<QIGl!P(0Ql3gtjkjTP*?^QLuC_%Qw)|^%g0xt(+MH7-PO$q`t7hFX&Uy_B}{LQVZhDNMe`gPBg*r)VwGs=(?8Va3@N8IdbQ~%>>{>m;0bdW-bfoAW30R`fgcRNv0W?$@CFMH8QIGf55;aAV41{}1=LJ1F=d?sPuH9e#pt4!YLp>V*Kw*1k^-^BKtvwwZLNhk8Tf;kAwyTa@{)$LnTJit?b!tU&=v)BIw!_i|ebDDWF5IH>E9@eUf7o94&F$yjrS8a{eGb}l!Fl;<BIXu`*0c?0J+^uGPXxhu|qn)2>h1Dnq>my;~kbWxq>m##<U?trcvIW7PJ4^XDmtVHBfrstBa0PPvI{#+c4P_0y3j(TbvP-5-tNP+@`_=1t>hxONOEpA5&dY}?KrPA*@RUMZ<)178%-auP%j*3ESjxbsg#PnVF?our>(uT5&Ou7`$Jh!tr2du}{>`GG<+4j>5#hPkRHK_?0z_S+ki0XcmObg<ugQ19l7V71Nsrc%nCjlA`2%b%3X(saScYiz&IkV|k8Xr&fG<L|G`aZh&D$S<7s-V}pF<DdIsCx;E}3cEa$8bS(;_#5>k?J5PX7M{KF{Yrn=jI<pR4@(iw#ar_h+nb_|9x`=KG)C{rL9X+xM4~(NWMmQ`9DEgqGI*(+cCN%!gPne7TwWr3#gWsvH`LY5?j|Yo!HxWCIkBCYhC%Db{8tyMlJtgdhMyn>u0homNAio^8ffVQ9#8PvhB`Yz69Bi86{36IP}ePf9ho9jfA?q|CA>8TBZ9=L*VM3!4UPpB)%ecRKL2(2bh7C6jr!)9z+f`DoPz$=dEq;L6qn2OxEXVvb2iwpaL*z2K|onHgL+Ss;!L74vi7UFJnxDT+C5sd}pa$=GwC4*n)OcG$Um8=K|Olum7J=0fZS^{ooxJ+C^HqsglU)#0F52%;EJn9I7b6h;gTD&tI&8aq_`j6&*GA!vNkxS1xOV0G-S-Eh)FQQu+%;%d1=xH>Mme~sNG>Ec)NaRQ1nPiz_w_uVMuy7a@|@PCA0#~Q=SIItRvv!NYaUr7DJk!x~x@$o0B2h_!%`pFE9hcYysGzjKeYGYSD<2WU6kBxZgOKM|wLxDq^y$uxMHTX(B5UR#gR|zd7*Q;UXWutAd*hAC7V0t{9B94!pDP58|{>$1}ge$#kk$QJm35o00gH$i<g`RX>HOFqZn8i%fAPMBW`JO^68Yp94%n+s-6-&&!DR6<BC8{qw4bShB&4H&@h4W>Z7n(bI2hInTZaGU$d_66SnMICqWZP1o)XfE6ei|9vtAWVlOGF6Z3TcPFKVw(dlR7!?d(l<IDCRaLr5pD=nQt<b-jCMp38+w_I!m7?2ry#Qyb5@kEKh#!OiPwXy*8K)poVt|M0Tx)`z{bHLW7J+XQdz5u=Fn5ZduYh_*~ZPSPduKS=SDK-x~4JG(2F0p}A@t$Fd`*wd9@M9!;#6Dw`Z8Y+ouEm`9E(<e>SFHS(W%Rzd~e&A<M9w%St+oU%H3@-BLQ^+Z0x4>{@ih#jVHNI?@n0FKKqD{qcAS~IvlL})YgK5l(sL?C_q@bNOAWo`)U81M(fcgUf07Q@e{(Rm2NRUVzY$aF(Gv855+dExnUz5~q7nJmieNV05<k`My8<^aN$WbX57;jIhW7;^_yK82jf&Ljuh4aOqh)KKFwHm~DH+taBjLqd<A;l4$cL^WV`ph4QA-bv$u(z__1Y4^PN<lq9+(G)?VtEHCL4tg*ay=1q|$?s500|XQR000O8001EX`No1L><j<^R4D)e9RL6TWq5F9a%pX4Zgg`mbYXK}b~i6=Wprq7WOFWXdCgkubK5o&|E|9RWj|EvQHt!e(;27A)Jf$eZER-}r!zf|hXaw2gfm|uLECaJ|NHF&eCy%6Pv;K_1uS+Ki`^e~mvkKGLy;*Fs|PbS$*qdTvdDD;|0S6)YbBOh@~Gl*nyICJ5Lc(4#N97KJ)|nrk&4Y^F!<yrnaYI(HkmR-7OOzKnx9>MoC~8OAe1nuT#*-Pnq)>Sb!cF9ymCdFCBDk@B+EsT4RRT!A&gIim`i^xbetAOL`7~ypmIOc3q>k@Iz6A`(kL}T#(`L=7=?gX)^alVkR=Pb(4jUPu}F$I0QJ%|+=#V8P2$7|lMt^o$&J+l)|VQHK<5VJ6$VT(8C(FZia|}Ga}gzhUTPrLu}DMdD^T}cgge89X3Zo;s6>!}Sx}ayX%LzD>&d`zoWTIh8VS!^7HAvKD@kMw^b)1!mL`x!`k@5%@`@-Av~S#c$_2xdDg#RqNwenVW|N{?cKqt^4|DJJ+xhE1-(9?M#W@%SW(NbkY)_pG1pGDC9t;M*Rq}@H6tk<WP@@42iF0MvN$@FEzFi!g2(TeN-)VK9p9oW=p}K}aS4<|88=7Kot4XbsoWQXqA&tl|UCR@(3=?U@OdL<9G`vuzF*=?6Ort9qMXg~b?$G{3gGr=Tt>IsoN1(zs`7sUOg0ULIN0zlO)8-(v0FN?>pM<n87<b;DemuXtc=ylw?+&j@Vgn(^^ZynXNvwbj|Es36a(R2&s<;H_LErUstc~Xla}_Q}Y<aT&k{l05Cpv)Os(;d9;7I-JdU_+i11&E<zI)@;Oz<z0I#=Q^85U}uWl1)4PDMmV4Va-Yu3|W93PW5CzA3cHA;vR^T4e5wno5N#9@cV4D3ymyb9i(>ht@3qIe)=A5z7dIgzy58qKNgELcI_LthMHz{eU|DK-d<5Gio1DqcpbD;@Z-hRI|89@f?e<Eovu*n7vbu+aAq*8TJN^Tx}Wh;v|b?s2^3(odrNrGV6{;`hlP$S^P`+X2)z#Ar%7g7EM`(XYI60){g0l>lT7<2aa;-e2#D9<i5IhzBY{gUG~lCA4C~k)loR_=WkA5|LwS~I6EZSsrZy0!)!1#8f<F$=r!nKD0Ao#ETKpQBK)7A2OsQjf){)sJi!M-5PSg4a5_G20~UdMqVpw2wHmH~l`$EGhD)-$p|QiJUy=4(wJ~PbHf1}AQ1(>(I%5mtmhi_>i^hUTP4+Y@vO}BhP<)e3T|{jH7UocH2CRQQb&qbgcm^3*{0hQLcmqqfe$!dt$RQ1gm5b<3D)72eg*b_0P>BL*k5L<*+%t9$LUzN*PAEAXxlf4RgAq(o-1>zEaV}r|F@Ecmd#S<u$%hPH+Az>JNMXJd8WTz8yc2-?;JX(f|L-8s5(pa}_NRx&>mou)+B5PKK)`uz7n9QxZb{0nLd;$5%}i{510aMf-E6AVqNx*S7GnIek&D#gIpN>t_m)R@OU*gVrg3Xk+Ef0i5cbkY2(FKBwj~cQ#W9m2gW6w4b&g~b4^!oO5BRI6J?GO*1bHqlM8MQC^f=YYhqO3^qTssDb8natr1Mu$NT>(t8^D<WDvSW`iXt-#swvp&Qm=}Pb9y{QBXds&N=_m;hF>OF4KQwMD2U)Tcg14uiscF%q&Dl2b2E43_J+z(+%p9auMQEMy%~QlN!XUwI@i^zdh0`HK7~cBbA?*sOu}inRrWh8FXo2iCXd(Kcu-BTgZ`+dw|Kah{MfMu?Il?Imd%fv%Pj5FH#JA8c!CuySq79J%%D)&web1gh2uRde{#@a{s)mi?_Ib@<k_?`Qa2IZXWO%*a*q`{?<GGj^?Mq5Wx)o>?_E$CZpVTf3kM7X-eLohJG4J+flJ8HEq*aov{jOh`(V<l(|8oCwAoD5yxW&2ha%4oYP|nbZMtc+6;_J>T5;rLn)kDoS(L;W3*JJ)-r@8QVccbk2nLLExJt%V?re*v`~{SQcPc9z5!74Zm=^Qtv-uHB_yCT1g?Bx4u`7oyTO-blT&OT-@I*jW$6+%`cMq{Ss@NPIWK*8HcdztnZNNxjvdW7{J)}c?as^N1qiuZ>%8<yH0Kx0+4e#)77XRQBAJ`0;$!J90LbD}fbec!q6^71Y=U<Ac)wEGNu+YZs)=LS_If_b-kPbxH2YXam;<PZozId4F_4KA&#q-1nowiQw={DvzS8X!>og3-j+Ep&x!_jOyD*v~a4qF}4JcS0=CTBJH8BY@L{gbk5#f>C~xnEIx&be`|-=8!>azKvVCam204BG>lycgFCY5cW&m-uo~gfiRQSi$061E<1p2N9cTvT9{UGQXYhkzcU0U!A@YGJ_SMhxV~<&<ko#h8r+$B(+TfAUqaekOg*j0<%fMCOY=RBEWte@{5YlXl_wp)(Z<DQ3(P~9Z3+U?gfK3rQ>O%;fW4LU5~R5&yOeNM*E)H9hk&1<k+!q<6P$AjJFo)JjZ=w#%gR}l{wg$#@a_TD!tM!?7CzFwazVTZc1Swe68^C+llxJX9e*$w2Ve&JK%8&%Cv=Eb4t^S2n~zC?_z+1$Ehot9x%d^@Nqhc!2_`ibzBy%Fs5^l8hRknOA{TNCSv!a>OH$jRcza(+dPW9$hqAG#XUBzbP4U<>^^~G&-z=%e{OcyowRC_uVt!mYp^4F?w09H7N|SzmvsDN2O)ZCNr%r*FXvCgkT)7OOa(dt6p?0t1FQv=+Q{3feTWTXi$6e{GF9xQ@sHEpYdBr(*DXhrDMB*SQ%h$ge#9NpDGv8&(J2>e)^&Z?IW^0pSzOhJ_`UdOZ%)pm1Q;Yx;JE}p)LT8l*BiOY?y7z!<{v6v!8{rcPN~6eS%A@(GScDZha48D<x=}v#fDHH<3h<mBug?yjQ*-Ld7E5>l3#oDjJ4NP^(1AFFgM&*_g3WNu9@q~<1uL!KlWg^9tHLPdd2y+g8W<img=Dg6+$<S`e*C0V@a?reXA0pS-ac6HILh7G_oifH0c@Td${rdNmLF_0cj)@kk9Bvm71iR6GC}`{tZ_F+;cYRvUdty8*r68gR7%HP?j~bfXJXUZ0h0i`WX|DKF^QKD4kV6F`Ag9eaL-m=tn8+x!&wvOE+t_ZAQ2In2GVxAqcvtIMEedHJg~lcgVu&={kJcL5;+-tw=K12M3c?eU~;N;UMHV2f8m?q6+_>;%F_pDvI|7Zm;VSrf|`QDF*Nv=73(FEKMXI8m8o|Z+rv~9g?I+vfCJ)va@HDZOQwv)D8pynvF}g)NQVx?XJSq@9m;vi?*KO7oguEH$k~R%C&N+sqKn}7Bn)dwp^d;7Q}1Tt9!8yF9y4wyW?f)(3!+WcRAI#&x0h=F(yzITKFMAPut_rP-8m{M$%Cc)t0XJM?G|~vyNvl2ZXa)_7=QIRcr%xhL#<-xA9}GTD5}gUoRW?Dv@fgJU3U|DwX0hzsi1^|9m;<K3vJbeGW&)S-FR$!m#S}f##tta!xv>iQ9a&?c%ots(Otaf`|AUXKzJwY1xk{<Fb_sRu_SS{Rfx<z^%Vd?%@dHjaX!0L%dt=Rfb(m<rlDNThzQKd010tNl^Geg@p^<I9S1=qNB#8<4a0`A+J?btI+old_I=Kx(5_6Z>tsrKxC9k1Y>Y*s#%0MrJ|H9f!|_4PQtRey0&GIr1k@t2C6Fa0@LY*4q$_VK4jr9QgWjq4}zV_6-ZiaXajIrhLF+jE{z5|U&-;7n=iSlmu35E1R|oeH`KjXSRkn56aBKJOYmRTD4pd`DOMUOIJh805jY5-fA!MU`Xb;@-A!t~bnPpFAwbM3@yrAF*x*sgC8Gh>PkZ3bo)5Ki?uGhR4LjRwr!V*=$*Alkc+9khh8oIR*$}^hv<Xkw_&mHdNg6d0Nb2e2AgpWsu1#lkXmfJf71+F}AwHAklJg1L4JKhMigPc_RqSuPvbG}KgbA8}XSIDfvl1+bEeah5UfJdw_S-A_+lFWKvDX+irsA&r>SA2xMYc~_3ZSA`3V)sC_$?Y50a#l&dIbg2?K1!@!2WuLL<tnv)$6TPKPa35i3kFW5`}N*tF8KzB*>p%;KGeK|NQ9+Ta=1&XOO64sKK(L^8!a;O=bXmz_u;)sFa!Q>}XQ^ap^Sr8{Wm`*}L~w^N&vTaoo(lLb3d24u34?f&FY;+;mc+X71R0p;pHX_f}^7`BnSorbWBj5GaJkXXU3sqNJn|Dwwuh+@&oha8bjBGwf7AY?Iy#Olzdxc+Y6RfBxX{+p8WesXe2A`;+B2K+cZEmB}j~-SM)wTdmsoDYrfN4^T@31QY-O00;m803iULwve)w1^@tU5&!@m0001GcyMKMX>Db0baO9sVRK)0H!pB<WpH6~Wn?aJd8JovZ`(Ey{_bDFsUIe-w%QHYFr+ZBB<s>_=^6yvfWa^XS~}TWBvB=)ByN!Zz9Wx%^Gn)c1`^ohz45ub=g1%kt`*c=0fUFS5UDV%fHhCIuToy!an0@o+-t_GjES;r3||$%N>PawH|Yj<b;Apm12>HV9S1=$ndC~AEJ^alt|bX7)>0X?QAxv%kd>ZH{Mv@=O(9m@@2V;5Ekn0eO=hkyuUh@flCPl^G*YD-@5*qUsHQTaggAp#W};fRN(z6gEYSejzHkjHh1y2cieGmtH=vuskWNe>c$sj*@becgoy#g0>#6&lD#QR1CrK-{k|}5{Rk8+If>GOqFr{sRIgzV>AT>^p)-e}6DJ{B?B?^I;CH~a6(@8X$OftyPu*+B?EFct(Khc!AwWQ`|GiCkpPWBEqWhn*EzL_%b#ey_{J&8`7$$Mcogn2A$sKVZIWD~SOodQVIMZoXb+^@!H`wo-|gPV)9AQ~JOMfT4f4vh@Mj@`Lg&WmEj)7$xQ2Mo`+;fuiAYlVWPvqnxm-i++DX}}<k=H_#;w58+GXw3|K+7l%NE;{PVB-CPEa?B`No6~+gSn8EEstv0t9G`+z=L0vST8jrLtnOpQOE~EwrFBZ*)FU$gLp2bwTq<UTec3QkQ2?<pQ0g#x8W>&#S8p#)&;L$7oc%g`7c9}yjt5Ev1@nS^xVgMwWuvjDrD0kWP?-W-JN?)QWE;*hk>{Xr6+<xM4qRCip0<DSXh!kW#LgaxBvOgBsE}~>oMX1gv&fx8#}6@lM0N^yl&9T^Y>dG&>WdhZLP+LpSB1@Xbz;6kfUqbY@V|vCj0Tk>G_X^xu?4Vl-ddx5UD;0_yq6giGtaROE5buF$2cC6^Q+729bCdk7vxhE2c5}il&7<+)AQdBC5W22!iIBw*%kv<;U|d3r9n`-8qdTUFpKk+t7E<4-+li>Sk*DtTq(YF;KxR;wgwQCDBi#WX?ajoWm^Fih-i?cbB-L8;QFB7dZrEsgftz6F=YYm%@nK=+u7Z>o?pDDo-fhkF)9sxFIVf4*0aqwWa6Ve*fx~yq%fdP!I00~|6eX<e+GkMYx8w;`uoox-X{Tj>2#Neio1_*;babE%k$tDBG5;%!*JckXVUdwQbwW8j%3bdZ#vm8WKTFApQF6f896Xn9EtBEGb$OtYf8l)J2L^<4NiO@JtTq40B#JXyj!PaIMcX0#xZC|74AHa;0p%z<Bj@X*i49Civ2SkYqZk}#oMnt5@Qes@_{P}JEpPu*$S{$J;C6`4H@87s$~7fvJLVUa!<#7-DExNsoOqbUpAk2%-FD!+)_R2VUq6AP+mxG!kyAYo4i7Lc~+k?(V6(19+b7Q;MkF|lCmHLUTu&0X^H~H2cktW5`ChtPwiFBR#0*a+YkZSNNf$8WlsVx*ts2Y5~ns&e8E#-W&?sK1n=8b-58b&rA^Fk02eL)b4HdhKd4p8J6!bEgwaelwY#L|qA;Ln086y0?(IzKl50hAs2G{b;%nX2QrwPE@)4&LADhovu)+XuZG9SeFoAm!*t6KZfi&*9&Kq0|slWVKdd7k8Pojr`gRgMUP2JvRZ=gMdvp|F~j?J_1$qsQ*Ootd|FYp0BTceZz8^Dh9fmgrdyi6uP(H%xq?wXWBQsSKfC+hGC;2&OzdyR$NE5L7KxlxqbUGvjTQ{5)E*XKX_62$k~+X`Py`26GeoP!5!qxMl}pu(Filj!;l%cSd(6s3S|gvQvmvkdTIkl8C-A=g+^?)m3t2TlR3;5kDB28Tm;_QxX);U=dqY@D4wYPafe?AGWh_B7z{rV<|;JaZvadR*Eo@Y59k_C_*=)n~%)2|U09PAt5_SrNb&DBy)0!WP45TsrOm=)x*2+EKBjsXP)qc5iYjv3ux-&)%8_#qMp{3}B3O;&E6#5(^TxgVMPgiFF_?LVxkM$B(QqsZn)j7Q_Ox9Fk3cyYa7l2XF_jHe9)|uz&G6`r6K0d@W`=8dlr@MGXaxCRZLPwIKZyZ|CJer^iz^#&Tgl`rA-K>JijUZ^MNL0IO^1Lc=dMu-4%+ibko&?P*)(DE3LzVJkZIE_agOx+NELuYC9#L9ei3d?Qnf&epM<Ec^QG;kDRyl^HE}o&bjnel(G&jgk()5redR;g8*($q_mmNiL2w^75IU{TEP60|XQR000O8001EX3E5{&Fbx0ztSkTk8~^|SWq5F9a%pX4Zgg`mbYXK}b~i6_WpHnDbS`jt&01}b<F*n0?q9*GU+k-`eMzopcB24?yG?UIdKV-?i*{iMv_#v=N}?iCyX!9Yzjuc3da-ZsP@vrpUWptIhr{8_GebpD^h;K-S{B<PBciJ++7?VKiEL>t8rnh^X^N6dB3Q{XS@4?BIwuV+T9%WdqB{l+v^)qV;wXw{vu(>OlBU~Eb}dU&plNt3fvDz^A_Fm-nYK*a+sA_2r;5tGW|HyJ%qc9FoNZ}W%Dl*=rfGn>EH<|PCmd`!)m_yb2o<Dmj98M2wC!NI6^m&LtEQQBys6|>k)^*0UN6I!(hQE-JuN%61UqCtBk-4^Hi~+e-m$|{wb0$JWjiWan$ZRaq!p8GkqO<cwU=gPCnRgr4d+sT#+&7AF`La`jdm;k&E_r3c$<sGl^R)W38;~nM%{*gEdy=W<SEKHEHAnu=Sva^(03=`C8~MKJK1znM9(u#FF=hXF)KJJvlGFQf)o+o{04giG_~5Gh!nLI>-K!@NLeWuh*evrC{D?0$5V@=S^trIEOXWpLGQt~3<X$0Eh#XEjeyOpK<%KGoa|W5TD2Le0LnTSG5K}R4kV+s;U;8JmY~X}fsu0G@@}^W9W2SFlSqb|WT2!?S%^K%_j?C#XB#TBJt;vEb#{nndJ(h{FlRz;y~g=Nykl}6Ig!$_NYT>MP4=G0=L@gbEpL%NP^Ie9+B6akg}9BBz^UnWyw{)NYBzFvxT59{!TgfnF*t}Y{cuUJblr&yT1?Dm8Aw{fc(Coc(dZ&D#<**8;N=Z+tgu9}SUL)8Q)vUIXwSPgeV}cXeD&w^bMmTZX5`FF4=dIW$}m6hnk|PDy5%G&seW8|D&3%_H1j9hl2bXao48`Mwj+We&Fk}YQq6uOu*>i$m27TkmgIG48z8vp=%20r5}I#bf5}YWy#DN&EPt)xRQa6*Rc<;4%lD_m2&5FK;>V<D#z3?Z`s@2*R=CQ56{C0HzJS>w-5ewnNkt#$&bCM0$Ba8*nQ<Xgk@1El5n3XItBj*ZYg)DiU-T#aeofx~`2HudWfVgsL|Y}SGNmOrE#B&&nnTQ$T?6xAtYP>H5gD{w9-{F+m}#x6>d#0p=9S<B8F|?Al9|A*U7d~{;u<0g0T<Y@Gd1s-mx^G(G;`V`tKyA5xh7`qz`-`SD=Ky!29ZnkqWcI!_#7Os)C|XZ^}$UK^<EGE=F0U!gxd2wS}(OHY+5ckpa#=0^1BtNBErB(tEOa8VDsykHVvzDWZPSzpY^FXzn%Ju=!a#$*GY0k?Bjr9C>Ah+e|@4?uL2yvA3{@m?z#855yDTVxGj_#1S(Z16PjyP#)V{+n49&lE#^=NFWnXZ=U`u4++UJcOQO&BB|h9UG*Ix7B|Ov_OCX3{nq3A2_{$!qLI#9y1Zy#r?`c_f8Su5(sRbs890V99o)1b&p4|bZ<xbkJWbsV#d~4eZ1OpYK(wsaLa!+n9RzKk4(5_)m)emfJ?$i*In@2!dZ~z)!7Jzhh=*t1y)#&KS1|S8%fIY|nfFLh00ufMXg(pMFQ_B|B2_#^dw7vsm2)c#U$p@~{!LNG|ENAzu<P9!L$&SJzzMw=DyBgs&stQ?~pt1VB12xM7*>!NJYspw_H5ej2M8k0*U{hev=40hAkR=kHlW&uW&B4}-04LG{Jo(=N$FiHY<?TG07*YW&B^z)-L{Ce$E!EH;bbWiiyjaK4A{6pM&kYuW50|H=^VXs1?OYIBa3!cz!`o#eL&P%$Y>xS!FsKb&1W6JS{qW}JzkU4h{$DrmBIR)v&G2YyQsXoNM6{uBWC{>GAvJrHb3CJ=P<{n=cI&ScL$uxqjwr@er%uNc?B|0~3at+w6<Gr1+#umtI7*xu8gS-$^!A50?|&}P*{LUtUQq_dy~B8B_uOcDabm;zPU}qvX%T1!az(q3koPF)3>l3e5QOk>dab$>sg9~Tw0811a=f(|?f|}M$DMLVTkMovU_x|riA%rvO8tJVeqZD7_ZR9{^?t8<zyCV&UKNc2yjFkxHl|p$qwS;WdqiKn4PF!L@Njvxym*^FaOEgVhFHOF%1@Vpi(`-I;gzh^?I}3BZP8ZROS^pk#6HCsRs7tr>ZGLACa0*WD>72vt<}eRd8A7WyH=`itg=Zm*>=olq*IePz%W>oS(%D1h2Zs<o8bT0Xaeyj4M%Cyy8N7#lA<>v$FxjV&f!e<A7R{3Ap*X``MyiYY9j8e$OT(nT=sePutRFFgL0v#?2CLG$n6@nZoH|DslvQ@(H8`9hvccl4oZZcYsNo?B|<rpzF<9KbTZ)pF&rR{5187l?~ZTeh)rMYNzqS8uISsb@F!&N51ADIWzsK3Cr1M~G<ko3dqi+94UIm3;ej4(Gg!l}*p%#;g$-<cq}P*1JEAtiOj*}JK<$rp5vm{NCXc*}hJHQk+be~+)?t3*vzkZ%;;kr_3ndjR7vNX-mqEH2Svi2tJ_fyB0O<EG0fjwq&~G0{+)-%=qEQx?5g<;^$-Pie;@V~rM4KmCmbN-fV`~raX~hAcSUB(U1CtG0Ye=tooM{F=#W}-xLTo%)lK1aIMT{L7E_u{AHKGGlt?IILBY4?CJhnFP+@izW*H|OGfL%w3XQ~?5-|(WAy^V3zUbl6yOK1nqbW|A!i^0|?T(TE30=}$mz&@1qBEvAb5k9@E%fmI^6y%I=w?f&oNp6TuZVWPn|MvxZP$dSBHy(-_fc9B0LCQ*6R7RR-lbAB(mX@sE$$jdN_}tl#w=|!IzGEV53y=nE#N1Yh4iZv%jW{PSP)x+GAgnHp6Fq@-5gF|SEisei+t?;$0FMM#&7Z=Ozp5ErkpcL(!abYQi<7==q+86hg@N7;VS!Bvu^@j))SI2hyD{K|9ePSf=T#IEDqb;&BIb%x4>Ap(nH=2UHm!^n{Wd{v#BxmZi{CrgDzJ-PZLf~bps78a3FhnlBe-eLZV1RgzS#tHQmYBUm}<C%-t}!{?B@X|(HWRt>|}R<sK$=cT;CXh&4q^EmEaf|xd&vXb&61FBrO!FDc3xKY`5p@aE*rgc`@x%kKf|ldSy)TTlZ3B(|D@>>s|fnSpy~iSEf}ocz$evP=%k7HK8+NkKjTf0aE^h;LhrzgD!PiR|QVK&a|uhgcI}*#d%cgQo34CqOC}P(NuV6WolFES(kCINzR$}F#hUk=rvW9NV%ViWqVo|V(1bd+5(iVS?Xh$gH2`<Ldeost*+gvmsk4VmKG%zoRhgR9`@L%Wh7q5ux}JLl?F}jdJ^YjREp%{{QP{_hgwgd)%Xi=o9HO<K6dUzTez;cx`WoazF`*0uPwR;m6WFZPQ8XjAiVTCS>Z7b9$bVzvth*Wt(!xtxO`C&+76y7Lw|1FdD%}!PnDrnb+=#DU7C_}P~Ek{*~0GCs9Z!;!AZdZ%S{!be!ThXo45ar3^Y(BsqjShUA&$b$62<s0lFWF>cW#U`9Rd+$nk};8GvgHZ<K?@>6{(t>&{Ce+ryB;g`(srmKdcfZoiK&N$<8elAN$dygdoRC4PwjjVFYA3P4;}YokWHX&DvD3BggD1B->PQ-$lX#?zbNX|ysIE>d@gF-GEJH=mxo@p##-hxXvdt~>OVhu28I_jw{?!_%Q7&F553ps^S22NpFlpxy@)$G+VdThNQGXmTW4)^v|vaH9Vm8EhQ&$aPPXE8pFB<zvFds70${(&~;|-^&ug<7Rlm{yxtRUuKWqF0}<rQ#^k{TQBnk|JORz=B6o++T9=#^*-!4^~kq<o`y~Cmh=i48X_Q1{j?lncQg$35cC|047iZ6D;Qz%6heEz5Oez;LURkT-1k>k4t4b^1G-utl7*kIg!tD0G`$n*5yR)xb3juTRbjrh1F}A1iQayE`!V<$kEjm^FzYR9E5<cE3lx&jYE$ew-U(L*i3gjhJ?)wGvc7m?&{xhs4R%{r(rM-zo+vaQTq<>dRq|rHW$OFH9ph+|hqJn?7$zMG-n$#zmJ6|Rm`BwY#%l;~K3dk{7Zey<WT$Eb8`y@5^)7j@i7K6tJuCASY#LD;p<zEJyP#9Za>I$%7Qw69pZ~mYuAuT6o&-emF}6X7(uS}YQkRb0e*B~cl=!7th=?bLSOaxKoW+{z1}vt3^I<V+c!I2R4cbOn*LMY9{spZzP!C#qG!uc_clNJ(Lfa1`dpKb9*;aq9sV~WT%RcrESC8n}=T}2cC%$3ok&Zt?3Mv!eF&V93zX~4zBC)9S{-dedsQj>nu1ws=?;y+hhiaJXAxpE>m~uhH7rN4zM8&S=t-V>&EMoO<&u9*##~p}L=ElqFUT18kH3kWyg{Dp=UXyDXFiFDiX&EG=1XRUykUxn!xn2Dw>V1NvPr~d!P)h>@6aWAK2mk;8Aplxqux7?4008lv000{R003opaAk67ZDnqBb1!sZb6<8hFLQKtWO*)ddBuHebK6Fi=y(1K45qf0upvtJWF~v*aI5^D)Hs&Qai;DTi-m#+P{arXICw}_RQm7tJh~g*jRpxi$zFd*M4%t1PoMWW-SB<?b6U5%_$p7mO;**-Syi@0;@y7l)lHk+dv{s0^{(=AeUlZ}-lq$1y-n9QwHHI9vPix6pMLSGwrH|lI`e&hGTBt+&WoZ=+q6|0MP9bsmsJDpi?WHEtSsuuL_AC4CSK=pU8l8t^sl-s<lm)wT5sDX%jGYei>Fzad7iFmP&rSMZsIm?l5E{@n|<7D^Xy8#{{%k;kWE}i67;xgvQ4~h)Hu_L2mfA*$8W2utU`JeH{~u{M+hiCSjR<KWa~JOwsF1XXZ1FI{@1@nn=DVmNzk8eT^5_{TF&(|V)`{bnoK5y5%1@;u3=rT+T=QIKBqCDJ(<4!x=#1F++a@A0K#8-`y6naCSIjBvTn+K=oRU0T6wXzi2;fP=8CUVudKZKCfo1fLHae`?Q^kf6Tq8Smu<C9qbx}Q%I1E$ugZS`dZVf=8!;pP=`9G0=4*P3>#gDS!z2HM*^;b+Hf441Wra8OH6-|<=cixyX$5~5O?{59Zol`%M6y3_A9k$)QGkId0dU?88_cfr@@nconeFfWK<ljFS=LQ8#bIVunga#ircqPUI6<%iV70xXW`Tmeh<7Q0tp@e;Jid+3cV&|1=YaR?7LXc=pZF?*|L$UhB^>BPt>S9EJ<w<e6r91IKvQY4hDX);I?wj<$7wyU+AJU4N$hEPxEqi!&{j?CJk+3WV%Sypth)&L*?yVc0sue-=QR`Nz}pPwE^zCkBy9-Rp_XxbjiluQvr7Nn0tqqYff$Le&aUD*oi$&ZBh7$d1rv^F9Ph3Gg<U(HxQRBA#B3rIB#y6`7ONCwY2iKi)W@H5Utk~Z1rJOaG^O{!qhRZOmG0vzowCrG>*6N#o`fC|LB04=wP^?<bqkzUFQ)!WWYAampV#=GH;TJJV7vyYzuzLlymx7{-g=$<M1@hdjYlL<wIH2AZ^XsD_xhJNFL^E!IfR3r7zSO?$>5(Z-@N@0{ru_e>z_j}DesEBjFSi`FWY2k63T@|+{-=U6(AGfi1%_(*{R6)?jER>l(xA1^fvnC^9PuJU+zIPq7VfES$C7K%XJxHpGi+ZGQwv-F?CiqKo%@2XCU=JeYSO4O#{_wwz~nLHpMZ4e#CC1U!h}E-q25OyHAS*P#Rs`HzaE>=xMeQ+h|Hpq@mEO_vsqtn1;_;)}*_7YRgOLf-gwmLZ_vmxRu$Az(BdY&_tvDPf)PW8^cnj_61GD|9I>wz0FX^`73yY)ADrK#$WZu4SG*t&0tdyBr`%3tj7~7V+P`<nFjE`sLp;(y+19)4EW2n9$phrah$`lXb7a_2%%x^L9jpd{24a$1NR8V<~&_^yZEacMfmR`9DJn%JJ{&Vn+lNa<pOHt8-*Fzg4DYR+;euz*;Hvt0trR*EXnGdsI7tOS!2%d^+ekNbH!*C(f~i$sPLX$Jpb;=lfRglui^}V`ple1MphqN8~>AHAT(_%v}1}iO<=k!lr#_1fw=;pt7qx|na`eW9=-RkfJ3U3!K?T2Iwf*%wkdPYIq^MnHEE2{yMn>tH_Airf%+Na&AeX#_}I(i>KZJqj?H0URGr>cy1^Nx<N^q1{=|*-eU%lBx%Q3!QwBC5NKZ1k_a(IOK#~>pCavb)gSlt{*>i&Ig(rn1OAs7{Z8MNAup|&Y$;>w?VUxG@mc`D*gpbe@?cyTaq%{gRQ|?YpZuf&u7%|O#1V4cW0+FlrhF_{H{4F>t{kqQEgr$24iasvZNRG=@D6L8uT?kfcixQ+J#o3V-8?u%dwpw?%ms29YqFFz~&8nw72pEl^bDH!ku$R7F61Js}a-2qtggUnMIS&+Az=OG-R$z;wUa_nKTCX9;Z{sQhb1L-ODo5e0^pabS3*p>TnSJJh@j@l8kpoal`v?>MvGGZ}J&5TaXNaEMr@gs`1=I_o+sUw;ruEF?Jrb*CUE63km8-B{oY*cmn{1t-rHWl7QPw){A9}bT**))D`i6%=ld7&vYitcgLaBG?!w2ez#GL*B@uojOOp;-B_8|7Z-sc%eYmj4)@XWN|gtG(eG?iEif?zp+_I&loqcU<xWG(upNC&94fR3$k(nSx9(A*i~l!ao){>!M_mPPkxXiZ%&#Q(b&aht%D>k_2Cgd)k}>!JifwdUucw#c4=rUWgpxUkc%+9Cl4#I*y`CJH$bzW^MWEv7nNyv~6Ib+T_8gNFPX#P2Q!BO9I9GfhF9-9FRJe<A+nmb_g0(w_$6l#;QNrb<F;BXxU%rv30f`8L|R=5B;?rw%XBUi)RWfz=RxBI!0mYjmYG)HytA#~^C`Je!UXtu?BEIjRW5qv~X2w>JH$?)E9FML=B~q%{UTniZ%$fo`ij#;N_bxG6wlzVHawWUJO7L&?a5t%|SmtloNU0h>^GasXPAAOoSF#Oi>GEVgFq3hXeF(t;Xo)hvD=<X|j%LGtQj{Mmcykt<$g#0+F2fZYKy`u+u3Hk&rj?=dn|=C>eep)uMg=L*EEX5%7F#DMWNEDNO}Mw~<l5RJMz#{Lux%)lCADC*gmrMMtxb7#YTWY-h`M9;qgS^a<`Jwmt6E-{>89!+?-ut2a2)99|OFiaUSha<5F#dyTgq+by#f_j`+=@3D>%-!F2UTUW^1dx&~Y^Ia%6$=Ct;qXbjGW7CtP5wzY*=R>IbatTu2U`pKPsplBm=&6#>)Ppw=9?K^l`GO2E642r9)SSC?$|1xVK*Xjm?#A|{|3nx+MkOJBuUFbZO!bK^w$k(7=0$dJ=;&ZoZ6bNf!37i`x{o4WC@L(6(p`$PYC;omMWJe11x7}7|A^ombUOU;v9FB41Pq-`%l00){ZvMO)BECnyLvM`Xsbuu(rT#(?bJ{8la~Qf5#1J3XU=1^KbI|c*TkL03$pChBn~&)PftgD7$EaR)9cQ>d@NxcPJH$gybKYB2F(%NQ9nu(^;Vg*^3=#HbKcL@*-8bPP09jJM3oFohR1Wm&9~q<2@p(n4nY9?mBRs5rTfqhv8gGcvsS4m>iAyOsqJIxkD_8ey12uphSZvc3|3m5Q3)VD1`SzhZ{$D;V1)spcM#-?%^CLU7E7=>K(8V`(1ML?$Ydf3*@l}ZR80xL#lt~z0T7ZbnX@CSkPPoF*T=eYK%v%w`HjZ3iK(R+&OLsZFg6SXA<wS6X`U6#c}3t)4Qz9!srb+_c7uDHk&UAf=yJd)r@IYQ(SBWqQ@!6H7r9X=Zv60Q)U=ywJ$EZCG`odAM>)v7wJ|lV@!W`B^QDTh3^?U#0uI5l#~OX!g3{uWzHA?@+CAM!rt`9o~30opcqFgFllwk`579qi_&k@_eB%t%K`hJ88%5Vpd@vt0z(B4$n`}5(9K2u_hi@zn)Ym*P@g9311BArRWPw^CyUz9tq40PyFgP}&(h*Ht8m1RAHMt}dh_zjOOD-u>T+}tc3zP;Hxm)gYHlc7Fe9$Wz%i(r=|$)mz@Bo}?xADaQSsm3MSuV0Rdo68-G?7PzFqjbXZhvx%a1?5yZrpq+s{9{Tl~-Ehs)^YFJCTy{P^>immfYX{7no@tF^%KFMoQwFhJXG&eoV%_BG@1%jM_Se|NM6^scx4<}~x`%a8AVd>_5}@w3^d&B05`@4=j?eZ!)Z39Zf_(>55WUys5%9BlqWd&5$0uG{f8f9R}WJHtrj=$vX4r=={Fg$zxYHnoXOiyJ}jC|US~?4m^1RoU+A=;|KhCcAwTfpED7NnMq9<aFS(S_t*47F;qiMznH<DY&0arzD^s3QX+Y$w|?131d&ikV|N>^#9$a75Lk$<+D|=lJPirW9-M0V5HHb(BmIn>RpU2)laaJE34PQdeu~+G9;jM&ufM$*;ZKa%to(4O1$32S@o7*l@&7;``!y0VtKJzvV|%{52&ZOxS!IB#B;PaxsQdEo-O`Qn8#@dTox&K>Xg#;a4i&aSCu8+uB{snOgSod0Us&P)-c=a%h#7YaZiN{Z1z_;Lu5o<B}{|{py`$=Bx3Hu7tm;3YK^C9_67a7d#Hk_3*LD78!%-dXr;VNS#pBj{#?j-*w-j4z&D7db((KN&zU{eOF$MF#nP=3=q3wY7NWcPQt$05Dna9k9!4BrkAj&9WlRHMgw)IDnps1ms^Gs}&eEIH?%2dg=+KjV{9Or1$MDV6YL|dlm}dn|q4#?#15~^CYa|T7XqOc&=4YOGf4eY;k!IkcOOSM!QAGyM*Cd9WZ}K#`Mjy$@AV?^I5M(-wYIM|AlkWm56>E(MO{Sk`qr>~}g^3B5bLT8z>e*4gXhw5*3;{3@wSJ+(L0H@btdMa2N{ZM7JifHTAVA5*PifDwQ!7ML&f*-rrQZ5PQ5ovDiIxTJM||Xt$U*}3M3LCR?smYa%W8hy|N0tNe4u+(0tcXA5}kz|fwd+omOR{DlA7rJ0jqF?XJjqeC8<${eE<gysM~71OPg((s4$v9awHy4Y!8xfJX^O(Jj?0`V90XR$J1{J4)`R7aA6`9jCF`;P`{uEfO5a;ZQ14tiEkMdz!EsrScAFP>wSwdpaT|YGD^UaMgONNVJn*~t)~}(zCKOb&4gPT#m_D-D0*rsoR{$27r`E!jY|svF8sRzNslPpp)2!)_x%Nk4&mH(dgfcADX&Ga1h@$S@c!%K{VNgSrJw}{cpoG*kLM3PV9u43jS2Ag*{c~aIciBW$Rddi)+4ng#h`ByNNkBRoJ2I(WzC_aIG`1Q*v0>Xv3~IW>l?`A3x<hG;L;tiej=93+yua&KMun1<qr0g04~zIw0hxv`A!xX0E}dbrmzIyj0~;zL=G?rI1vLz3DA&n#HKZ3={o6r2KsH2eWeOolFhaX>U4+B%M`;jk)R+^p@6XeUe}4{Yp@Hzo-XT*#9UrpXHC7(N!``Jt{-6My-ItjS)WoYSc)NC;@j`7ss#T!EmGwdXa&QwxF(}m2c~OlRk`>sFfmIsdL>3e3-&RwdW%<92c~n`lz&-GI31{58*RU&>+jB&=>VDibXRJ%>}TQZLF(*!2R0{muok7gvHIMXc^P3VtA~1SwWrhq*+Z^*thY9N$cgBCfwDBVn_dLH*1~Rwy#(2VNymr2STF-IHAaPT__N#Rem~aZV`V6sIoST~v$Mau+N?2=h!$2;=I}?+ZM%TB-@q{nT8<m=K@k|6KoyjBMwi@7o2*y`!w^gU6GVAL48uha{LrIZpP{s;flVKrm>niCd~f&B%oFJAGFGO&jU)SXlb3hyEsNWx+{Kuhh&N4IMfI+{v2l!Tko-Waw5i@MbdCoI&3=`##V_wbdgv+(H>|=O5;VKUP4Q?fZ`;Bm&QPJHciuh*+u{WtjHrO3GB7#hVF?$NeELiD>AUFDUva)9tI_&N{K1`kRDb>1d!J!R?QWMf=leWvU=R6dgUV2^w{daJ3btNn$m^SIEi0ig{ro~34P@>Opu0YoW>b#C0D3Wfe78*ttb5Djd#c?<jSkK)STZGy4zvTdDsaQh5bjH3&sVd3xu5#iROf*}`yn|i!SOzFb-f<6(Ry+nM(Alp5Oe*!JGKm#^XC_qOC&-&Z6Q55rf)6cgINn78c={(o1^EgsVI;=m>nS+H==@UY?f#MmQ%Xx!DWYyV6O3;5`$D#a3+|K1I|Xz2j9rh+1y$Si4=OiRwIJD_2{x8;`VFLgt$rX=V+L<X|IqOUWNRMf><uaI+zp_)~865KKNI8hGAsVd-(rdc|`?(F_NiIXEQKO>_0kq*K+716%uhckW*M==+>x#I|R`8A_omXo36Y0MO-%=G#y<;OnvI}ZWLERX@rN6?ibJg=GssZD8q?5PaQ*3D&&MJPd_tps8*kn^;Do3a^CJ<DH|>=)ErW+m{nV57YwEas^UUXi^Jf^B1o+k@UA91Urw)z2)dQJ;SsT4Hv~O2VdD->8u0+J3xr`n*TUAPA3sP0{fLvE6yh*R>63boX9jc20@2>`c;;&jUS!OP6{FFrRtGnvi)U)7V`R~VH98e{h69YJZ3@YTg~m^+ia~clg$Suh%c4rM+)&;f8#x^7rBSCeLJ+b<LZNH1vL1cw2Nm>fsT~0H1BSh==sU&^zmTJs35KW^P6;&N>{3w}(blZoW8Q%(YvWqj9wxAS3Q@*pXMB*XnfjmKztS_pN8?}heX-tx<p5rNM?l&`-C;Jvx`3$-ThQwG4ckPY?!h1|$W#K059{{<-Qa~o*Q<N8(5P;&0HSqT*Sc*x(@VL41SDx`C&pksQ$j&-1ZW-s18gzZOlovor+~Jkj@p9AE1{&NV{P7<T)RVc%=RsjdMKxgt17;)Q5jov_XQDiKa=x2Of+g@Nfgg$Qrt?$aO7|=OM9$SS(z!LxXKSqmOS3qcqm|-HMKYna51|Gy$hp?z?l;}#%CS6#gZ7S{oNx$t8!*~9gbEtL<<<;x5_0LEp8kZdHrscLkwG)7e}YKNI)`C4Y=W6Itq;Y_58S6nY=cI?t*aFJzFqzW_PiVSn@v|SdOAQuB<ym_e2AO=fb#A?=2sLnq3@NBoW!ug=+*RNSQ!IZ1mVe(ySfpO7EinV$eGlv~4!0xuY$|skZ%C6k8A&7D)@TBLdxTRQyP__IApHS&&|-3RtSi2hxc4Szb01+qz-;MW*&&`ru2^!8V<IsGG2BLhpI7(oSzMtLwx8LaH;q44FYo(WLS}(aFA!^E|DnTsdAhEqLOfFJ#GM0nQ?3cF4te;enM})2WV16gY!`DrJ@P?67eZ^8emc@m+_3#(g9Q01Z{y(y!T`AYabETd4}-1b;bBSaNw6YeEdbPz@0V6P(3eIv4;2b;C3$>Wk^~=idRcpMMtwGc1pq_L)H~Xomlp;wZt4N?M)g0H-kCcLQex!30Km^q(@N{GY2w?}3s9{y>K;&llh)CIj#5rYdjJh@0I3k*9|n-epO%T|5g%oDFUCb4QJF+)EPELKrQ;m~4N9uzK%_M>s|DeijmQ(NKYReh!G?*MU{LRwOBSZL~pHIbXCp3`3`)r@574_SrtJMGKgAN<(=6p=N+*{K!4490aTKrvvUe5!7s7T>C-aF0t6%;WA2HWhfI*7OO~(9+;bk_njUe0zegW&_A*S0vR&avtCyYvtcGhfEZ!f&JOTR-K|s`m~mLrCvFSjY_bUBiIR`{g-4N9k1!$a{FshToK;kJUGB0vbuC8^nH)mJIsI3b<k^amds4KT^F@pz3S=XaIF7xt&WdjQPZuENYCOv6eJD}7dFj`5-I5c)o#x(yHn{6FMbE6$iUL##%+vri)Gh04{Qv_JN9SU5Jh^-D#5{mjFU(U<7=^@Z2l(;_Kh%Vfq&GUOQ)Sq$>w9`CKLOR7Xubs37Le(4QUZV9(utsmYc4{S>EqwO)tPZ9X_4PbkJ!?T`Dl~Im_XD{w1_9sRonoA%t5#?L=JHW%2O7LOw4UVU!&dUX+IUi9xMI9eyXS(rZkOCC6N}zU77*Ie#kt-R2$pDiOUG?DxBt?Rq^C#%yn?0r6`IU8q*li*-YKPVI21zW6WDB_OWLqXs(QUr<LS)2z;>9C~a~t1Ff=|E7d}$`3(N$BX=+%FFYFPJwCN0TCN5Xdi4K4B;8B0M4ZH|d#OkvBPEhpINIhly3lyfj5tr&Crqq#rGqUTAX@s;Z*g4M&<{sX`!Wsd4;~%^XDr&86yl5-B|PX)yqx09$1*fC{UP^}4-j`}0E{|rxB8M(baNKqtZYkBo7xS4#ftJ(rzu$yYDEzIU}|Mwx_QEGf>058;4E>I#@$qkU)3|v&zSE?iS`Tw<uw^kpU`tbw)QJLeS@!Ap2?%U-eTcNoyGF*8FK-?Lmog93|!$()UiMmGO7>?b_t0u_*S048EX8)g(L%l{KNd6B`(>=$CLX31I2-Y0l!HKKWbWsZAIZRj`gZBI&{FJz8G?+?H5vK4vhS$B{V_3M|B1ULPS=Ux|@gJib_Tf&)J8$gkoHxXl~jXEm<I|^2XP70TcR6ltNB(umtMHvgLHE4wbh&6lC!#ztqlas=FyHFc%#QVIi9~sN*zeH?4W7Qt#!wPD9M@3JGD)rQg9NP98IcBy7EY9jr@VHaelJB|zm!orKqGa{A8aVlgO!&8XAd9lA40b(i!@o=eeC891=TW&8toGTmXU{~otw;7*|vY0cU_C;j#suR^gg9e>mDfu4kx#|qr$ln5)^6yzw$B1p{ZI8+$WsGsfzYpQ#5Xa4N{Br09$EIiq`B$mKl-nD$bi;r7)NZRK=zNu&4KmLho4ctxWV~r*ad8{N%F~v}e>Jf@1pjyMzOBo+;CU*t)!c3>};f|g_#D64E=Y|_U<MvZ}WhR(AD~hr)beUn`v|y++LWPzX<+nh6)l*%F_YPqkv2xWJlar3e*rQOEgQmn17(9W6dYVQILfyz)`sDhEFr~i=-1mS{BS0;Uu;kI_VwR_e##<e_6wC}E+H2@p&GW%)zk24ya8re$G)=e)LilNJ(}iuD<;5xt4v(oF#yQ%Kj>?sd6e&uSkLxLNya%nMQ3H)@rO~hHB@YfeA2qI^0cEs0Rxv18>Ba%>KRlVNdUMB5kEFl)p?b8oYG;6+&=MTdA7)sb+IKO-1Nc%m3K(7ysdDQiQ&u=y9^MGqtV`8tUG1Y%tAj)7IWRfIFhb-dvV_rL7&R+bTndT_DD1;O-NeD+7&@r{;%R*|T@Ic4>rwV90Ka{h2Mz7PhaV>6*-BZ>Br^|Z-i_<XiVR?uwWxT!7>j-ffqYO7iu6@9SsLOY#)xQzndfmv?sq*f_raCu7eg3fv~d{JW9H@>PXI-)6mV?Mk)2@wWw27f*s;km3hlW?M6TAuBf3r(e%db04<mGh6cZ|ga(2g5N1rVQhs5GoVRIe4)sxa_LfNt1%on4G@DAS>!((9vJ2<MmMHM10X>vp%JN2wl&@^5x(lOx9s*=k$tw$_a2e+fj()x%d>>1TgCq*Wxq=h`rNH|<wF-%JXB@(WGf_6r=&Frw|;XzfTe%giX`AslxnB{0oN=tQ&H@{@)KRl_&(J<3-#5e;_N9P=j>i;&&@SChd_IO7%yRHDm_X}m1^KXA#XBE_57~V-2>mC8baWu&49hsz@k@uSCO1V6hFW6^2s0aoPM^S~<ls9QX*C@qxSx9Sg6gJr89A}qHwLGddir{fA1s-S-(P()fdg1pQB3}21jw3w~&<;+`Fdt3*2VIPwxKF<}WP#yg$ar$trK3iYS2HYZo0Le5LWz=AaIRo1Hn>KN#vYG@!&gZhrE~u+AmLsuiK<7a4yadI^Fqk#bJVV=|LZ1=6R+HWB2D>%x0ueSwri|Io%z4BxD$~=XMkzw-_bP0aK{tU6N7^gRYkuIhNC&(!R}&D>0=Cg^A4G#eSz=y!0YKZbsm$g8SD&f*_7)t@4Cw;Ck%2h6*SnZ3p_>#vqgtsM;m(1vSxfgC6uu%OOBJF21kgyiV95r5uAdwPRJwC(H|#t=fETS7sdMiST*U0kt|hCAh~q>$pf3FKsT=W{DfiSU0F4Fnm^JgsE2^j<DJv~aa8M@pcsp}|2>Gtq5O}AXP?G5;_3+gMuCPI`_(DSqypf`2)xg~4Z<rthP~AR<0q~I)&~c|TFXdf2j`-sw@VkEgf+dWUX5iv{>Bk)5Kl5*w1LJO8!J`LKMvx@vDO=B2ge(3ojt=nXY{}6|DH>GvSP=7Po6xS)SjG_xp#YVCyFzL-Yo^<7*)<fq>;OGBX75P=<pSGHqzrsV?O@gt~}myn#WgZ?u$I-$z-$?{a1Wbo%%Uy93!{RXRySu<G}A%nmk>2&qj7N>pMnuyaMCoW^y?EfP%q0&Q1DlJ;!&obF+@OeFykMujmurfj^=Tolpe0=c9rXS_pQ2ihubVJ#LdyPGTD4knNa%rwbY%ug9qHb9DYgbH7jVOAEm)qEBW;(c|2EfCrC$c>2Hx3RHhbVL@?Cs#jTvrTm_P#;gu^R4E=E;m_VDarrvcf#g<=|2dZ_aIA0UU0(j=W%Y<b)fYjHyBWIgh1PiOaf4T3dD0i0jVT7m>Wn`6xF$jf^$XN4+valf6WM#jnj=|&f-Q&A#eGRAZzhy==>pq)4Z-1Af7^x~F8}L?9ebbq>c@&6i2RX{EV2D~{vgZO9_6PvjI$jlW_##E1NLcw#JUKq;p-B{7KX_S8yDWvtUNQS%y`tH;xSX53aaG+$BgnP*C?t!Q9t6~xJ}%37qGRq{tE;Vk+@)3`wir_E#x2lCemJBxCOF2I>j}DbZTm@r)J$EOut{nUDiK5J{IE9=Xd;dTUGQCK5=I!e`jW!VQn|oI^x-_GmdA<GrHPwE=)Xwg4RmiWV%neB?BNy@|UMw8T~=8Iu$kU5au0vL46t&sQ}2|yxGop3nDglx~W~O-hpcK)uBVx$BI4=6z}#bAPoflNkb6u3*%-!aR;>z-=xSQ>1R4;AOsVBsze^Cn?C3h>jElxfx&D1q1Z!x7*Ulv(txfYcHk<2m;7f}lG4p-xh3XL6YJUq2_#OZ+8wg%ysT5_S?ZGxicO5eqdj=kWkuqg!?&(RRm0+9=EoLk_v+a8;AJB-#z}DGLK3R8?Jkza8?P{#RF%2BBtl-X2b-$yztgbn`#yf$B>^>_mwUWy!;7zSz7d;PA8QjZ76|0kjYSAn!>O+ht8KIIucc33E673*B;k(AZIVchIhp9`Fl^U98s61%s>$t7S%|Jr7uKYrFfbTw-jCVnr=;lCw)H?*Lzcf}$wH+H@XS#3>ch*||L{XO&%y`Pu5Sn?yws}Cl;Ix$#OIeE-(3ErLm_UesM2d^$mw^0VxXwbg6#F>$1iXH@0aNE<A?tlI_XXyo|FCf7xk4mba$gO+KZ`w`T589zAXBfi9Qj&9SCPC9Lct*iv8iX&}88lJv@fgm39-uTY2@p#%pIdZ%3ba&2qsXQK9e}@Fso(tTC?9Q04E4BQE5sUzpCw8R`qVzfkAvtftTZ8n-y$7oPMcg*UyJeII(y@N$@Ev%lCMM#Wnj%T`skTDQ`7!cpGQHj1L;sC{&HD_xAHO*xuYnIGzlTxfqE+dor|I7_f_v`)Nvi-Bq)7x)@4gSVoX<}v#6puza)2P3N9U9#2@dt41&cGEwrZdxun(VEbO!zI&v;mIq1fD2934UL0G-KT0(y9#@E`vb?|-J9w(disPc0ho-!n=e_~btm8eRyrDb+UY(1C<5!#i32aaL*x$B*eFr8Q9<t-Il4fwE!-F0SlZrPv5WppFszGmEf~3y?+WV=Tvr{s4%zb!58Zvgli>qMe44xahJnN2;+uY~7mc`WO~*1mbHI+Q-@6O=(D|ytb1L0<OC2p!j0*-vO}t~cRc}gO!varkH)#d9@*Ssu_}c}dfx6fM>`eJ=()W6_x55f_p8VAu4}ur=3oP)d0w+@LX>V8LP1M@OhW#5t)V<EuaZF?VluDt?k%9dA>f^`xx5+}$J+$t0QmYRS^X7=lcZ^-*+30fpvKe<<vpY`b+5>7nv;hVfd?n3%)XvHeR39GI+>$kjq{6u;^hP_<Apd$gR_E~uc$Q%?c83|7Z`W)l&<8c!7zyqs6U89BE<+Yh4}djf=HmCnM_-uUb+fN!Uz~btyORN?L%7#ug89`;`yaZs+97JItA38UyDVrR1No;#H}k>91A%!C5WTFLY!k1WBZZItOBBbyriG_AKwM6qydV!DrrZL5ibDn?2kIC5n6a*0(ohn1Ydj{Bm_1GqGt3Ud5<!SI*WP}6xXpiK#>dn*NxIJ>F<|GN3ZZY{qEO<`+~C{ZCy)==kr|)*TMp=SU6DIa;!4H-8|ilC!tX#VMP$mla8vgYWByzekld!tz4R7PL08-JhtbhRcMP<>-{UR!_~l*<R|*_v#Wj6U{a)U-k6Qsd@#(n&Yc@=ChNjTK^B=`|a_!SWEq)bFwQ!RtTRX%+dSl(}(H>9SWNtQMi+PN8bdzdPMxej0v#UH+LVQ5~;4Qif|0CiN<2xAG23!Ir*MJQAzz!~?D)D<n<rU>W6Ht|T-mYa3ye;s0UfM^NG*V2%vhsB1{Y+o0$M?u8ah270Go&4E_U=-;KC<!f-r_Ww`Nw4YVK@1Ru{-tOULXOyr7yj{yRf9lYMepfCMu$yYBxYo;%)3Mbr(s4{+Cs}AH2UD(g$Oz;Q6alVs81pW(o`CX<Xx%qwq`4;G8DuI;Ts__=q}vXPMo+(W+sbZ6r~3)Hr%QWE?pi-ErQ$%XM`xXmFp|Eg#%ydOhG7n1Od^$bIsM3$WiRyj+4yTLD;~2fP4kM$U-#4A<pc2eqCL-7h?{^@;p)xhHMI3!<o7$2IEXqiQ-zyc6uPGP;CHm4zX{FZVD4kMfSXr1SqP54lV^X*)$^6-5<z28pj<7)F(Bu8&tb3&xz*qWJVD$F?K!HA?`TG#@5(u}~^FoVMic5ZN^q+Y(bS&2yoH7}omusyay9yB?5=Bj&QRb(>a6Np}37eJvn;qz2x>isGh`ADocS#kYFwPMpkE-#^nc-1e<}T(z&&toJ@TdS-lW)&3G%x=E{wj-f1FpI!3e1Zz)kQ|DKizLUq(&ba`*v*n%5|BA}1D!7CSg(<j(u7>a4pS{N>34K!z*jeT*edm2@*b*3HeRG9(XNs>4_FMO^Q&eAjia>>CJFL_I>*_8mNRY5qa{liY#%u0L^kvrtHb-yPqb`_Q$qL4xaaS&;czxGxXCMQ&ajjE=8QAhIzh}fvxq@wmZpr16DRL~5cDKh{#rV$tjBL8a^I**Fw*L=MO9KQH000080000X01WDgK_U?V0HZ$u04D$d0A+Y^WpZh4Wo~qHFLYsZUv@Vyb98lNd0%8{VP|e{b98BAb1rasy&G$j+qUt$eg(^ZAsvdobEc0`s>~#@ow{}7Ox$!jqtS365|Y@X2!<ffr+EC|ySo5LfDfH6N$U?j3S8_i7K?ph>HGfs6Dtajm8ED{lZ$dGQ&I8Et9YIA3P(KFW_bf3nYYg$c($x+4i8>l?u(X}Dfg;c?6`~<i?7;7oB&Pgr3I7n`A$4|si<o%D^X?$D?L_oujVkC0Tm{3@_jCW4l9U8UdoE64cis`i$!zfXPN=*!JCX%9Pu=s<R?}lp?hBE`#dGyJyCniOIa%cS6jgnyq##vEcRZrLh?n;4+sSmcunv%|EbM^J3v>lnl-#QYlIvbsamz{Tx$CFKpNg0dClX6@B53zz7{7hN%n2i);vkP{8Wj$vE(Ti3qAH9DM|&OilTre;#)Hz<9pT?O_rxkH=%4#)fq)pR%+h&Z{8aCHz#)Bnta1#Qnw(D6OS1zY|qlBL(E`D@-#u>k^Q5HKv1P9L3BlOWbzn!^2lEN;g88aFZdX723wZp<-y?H3($`&QM!;6;*d9TJj>D|J@5N&<OeQc#gRRtX6a?cS<Vinkg&_$YJ!Ca|K60cVW3Q}?sHPAh(<{{4MUxBs&{JZM14+3nbly;PvV{@X2W5QLTlbu1xtA%*@<_r65f#vHJALJ6)m%u_|1{!^;`asr)I<B6K{?p>z1K@CcATzXBq<d1{FlLQy}_xvgeFcYsZ=t_Q(w9YX29YH(mnAEXhS1?DJZJKqYKdnT0D_#=}kIoo~D~C{cW35BaG*DY(t*#n!;$bEv>?0sK~j=cNVyV%3AgvjPYC)di~n9&iUh>fzQ~0?p8SW=769a=uaUtcTQCxA0ay-fXQ|YUuO+?X9M`>oFK+5J*2eIP}=()_a4c<{;FvWB<7qFz++Ky);9im+w6__GJgg!H9S_@r{xDiOIX5Kk5+}hWv1Z9r%!E%~5XvZEtl>MJgI0ie3j`G9GUH3q}G#B|n@%O_egOEhDn=?N|O*>GU29lTpmT28g^OmyM>6|3G5}n%lLa2I^QJc%Yv5_4ns@yZ?Yak^>~}YLwu3F;8~mr(kUx>2Dup;98=ZrsPwo?1}cSDiUO{c%Z?<n91f`@n8pg5b6y);hBPi-wV1s=u%l4IBsZ?i8ITlqwQ??LDV^i=RzIz$a@xfaAxo*u5aQ$EcE)pPrT#}ukkpcG4n*pmyKA$AFp6LUVJuoleqro=<$cD$Ww5>%@JN6MUlnmb%-;tF%pEZQOZDw@ORHK8XcCghQ$Zo1U~rLN=R6(0`()9s^kl%oP6s;<binLe}aG$P=c^q&=pu~@N-@T7H)u}r4z2u+G#DAGq9rN5~K00B_}o%>JA!|sha>^_B)VCkBZO<a0zA0yFp~O-wEW=+81{&AP<}YvOpcQ5=$=AT71Yr95>+z7-;nyy|K@u;TjCMzukJjTX*Ylkc6ud&e4j<?1G(yJqPt4ockRJTA%lBS)v7?&gj#Qfwg9J&CbC_VR`_|1=!B;=eP~Sp;!>bmusV}V;)JlVpd_a6$JZ2uqFV&Ffw{#vxpMVBp~L?52f<LT2nwtx>4r@slzAf95+E2nLk>gNhw7;K@mib+b+R~@~q!r;yLo}_<3Eh(=KCP-tg1P3zqm7JVQhS-~#l6ae3ArjWmXc>)>y4bqiOwvH_pNnRZ+_qa#`_M*5?@`8XOOBh&Mg5YzrjOKuw6VH04T-!pTx#vx|dm=Vk;SMy!TOo%RKSr!nFGq9Gh2-e{-bS*x~Oq+EGJ`*c;3}n-0XACJI#Q+PuMk}L*4o9Cf5Fujx5%?4!`U-sp$wAsEYo=fzpp}3F+z!UL5IFc2CEuMCpYGr}z<3S3;`*O;3w9lBqHGdzM?dwc>DQT5tU(jdZ@VJ<LDY8}AB}lZ3(@#nqzsu_4H3C5r{|sjN+2wKadZ2p7=wMqn4%^{3-LD4z!xzb?Mwzk4&s`#ENS=yoX9bcDo;R>^a;ZVg_@>a0KnQoaiCuIrWM}zYE6fetjcQWLN@AM2_ZRJU9u4{qRas<t24}iRGUqumKbVd$PARMR4bq`Wijnn3`T2Zz*5Ji!RWXJ8c{yM9N_-7mL%#I7%B`ZYe*)P6OX)){>$Y*zIy+6I9?w=MIQY6`rW_#AOCps{yqKr)UgZtihQElt_J!7ry;#dc5Q|}u0LLOI0G^|;E71UVn$wKYq(Cw5+D))_I|!43V`n<05HzdJNQC~yB4#9kM^e0mf3q}q3w-G?;joy)27z<Sh%(rTPjuPSPx8Wzp|A#SgC6Sea$k!>8GjpaAE<#x62;GfDr2@%Qxc3FENaaz{uU_`~!r%6ceVbQX5dfn(8mEtBm*qPP%-U+=P=Gk3G&rNV%~vY{#xZ`{s{U@T3%A5H`&zK{!Tff^65x43?!i1zka3McIgtMx-2tuEUv{x>W^mrd!N~qfM1}$-ZWU9KcnyWXlWyKyVDq-f-@o6~&IFcQ9>I$OS*J^qh2HmZ(3uTHD17^s85CQW+{1#RA?3^X{ZqZ3yQuF+D;!Qw+j!V?))u7&Zj745H{CzWv(oiT;{o#8NnTZOScZBOOMDC7`j^%BI%Q#e^~u)eXJgQb0^`-d||1bxxzQeZMX5O7UTMfbd__Rt4shQv`6&qYLrSDCNQlgN$p0bUFpIMmnn)&XDdcvzu;?Y#s`atkq^5SZ>3W{oaivx8rpZ94^g7X%i~k<FI!8<EO<XJgdcYUWb_{`JK%)hudGAc}=jnK}n3Vp)KILPD4ji$GUV?20ES=j5EOr<l`C^9r4(dVbm{GjEN>Z14J<;u5#^v^X{8>mW2E{^~bHeXLSxv3OBP0gWT<epk$<!DhXkdCw+_pmeUDj#uNJZ9AGu!39xa1>tsb{aAM{oUY}qvdR5%|SWjM>le)CRfaU#*fzaHH3hT_g3)4~tq{x`UvPTKNg4OpH<Ejd&71;0t)zO9}`4~%<2@wbgk&E6f$Mu=Awmq)d?}ouwROet^Vj(N$&Y__>cuabR6V%w;4C8qw^$Gtydp6;xW^&-`wCP&CAUOAefuasflO?b3dDc_G8k5Nb9#87&77x`;2hY7vwRTFB#rX@bEe9|ZT|#Efa>&BxXjzJzn7B4E_i2d=S%E$!y^oX%I-OPPT&s0bDl|!3Yi@rR11TLaOA*KATbp1K-*m_kO_}>>;5aAog_EOL>Krh>nq|j6O=T6skMu5}7@ERKCvYBO-A>d0sbxjpsAL<{-g#NLp&Cds>7|499|+daq-?!s6fgBrNB9JBJnqpancvJtCFpaJt;MA!Z6=n36FdBm%+AXOk7Y@Owbl^={XCjZ_a&}oV_Hw8!i%f^KPmYv?3D7)0uL@PF2J1t2*kI>V&aTxnbsHMC%Q`aED?1~FadlxKb$Wg6_6_43atSv<!zt8cmhkmLM^mG7*<@S??Bv`aWG>&zCWKA&9I<|g*v*fgG6qxM6%Z51r8kIvnWJ@&$B3}MN@OEj*d%8F$~YMN5N)v!6=$Sg$WwhPcKb4`qU#*$<$NKxn-egwSB1bi!6}BaS`eYyo4nIB|M2tH8s$}hyuFeYwQz67>^q$)jg4Q^*On@+5Q4cHr;~p+t@Hnur!;KdF=emjv)%fARx;#xxU8AJ+3VCOZ$x+{Y<bi3rls5sgQlMn%U9+i8-_ke8vmjP-z7V92a&FVzh@`u5YFp>QaRsr5gGbI6<(Z;uS?Odhx4AqpHnl{&rN|nygn{gdjc9_wjBS=>pyGB1$ZaVbXCyR<rZBv%(wOwWo@2+J0UZpo;I#*N9Mq&IxgX`u|&uJu@0-GDhh=onVi4Wc>yeGz<AG>w~J>;=1LL&6~54eJz2N=irf3>SiX3S;Ggm;b&e8nH2dQAC~RM^XAJ7aa(1RWm~hZ?vh}wt_bGu1)N#@R5j-{)^V_PNzKCbU!1gWi(+lT7sVue_=KtuVT$@qh&*kdjZHOzn5fxU6|1VHidAGNsc2qe=Z6`YM$|P(uH7ccV1}G-L5&ud0^s>8kZ4Xca|l($Jbqh4BqdIX<UDg@YgvJvq47&M1!mbsz4};sZ5Xn8wXp`e9u)+;8tX<!dfU;JZ7<bK-BMef!a?5#@f^HgM{G%VP1Ime_>sPcQj|@<E7)bb*ER58-S}bT4aRQ$uxn{FT^iHHOO4lDb{SyZjA{zOs=Zv=x7vNnb+0dW>&Eis^7|iNy@X}pb-LjYwHx+Ft^;@Ha11+&LaVRBs%uB8{wkgNc8p;h==tcXeuA?RcfR-8bhVMVT@36IaqiydTwFyosc~I$B-d&=(csFQ+Wr%2lgFKQ+sXUI3~tkj2Y;nkkFjY3?sjl7df+8GUR5%8xbEK!rtk-zw$6-76nQ$tti=OBS~KHDB-9HMs+trH>X-x}hnM^)an%VRr|*!d{Ce#Tf;xkUa9o?V&C$wfwz={acQF;>SY0FTWyA1pSvTp3Qw5<jE=u<%Io7~ZQmfOb>%O0yV}C`m>_D|0F?L`;H>SQRI6nFt5gUK^&gw!{v&maneNIE9qc$g(I9OeCTiq$TA7Yy9s6KM!&KNdZ7;d#%^ydh1*&FLFY|t5~b{*U=7;QMCqHg5IX+PVNxvPiE3>S0G7>L11b>FL&IvV=D_aj6|*p^~U>rx*-by<L40pUkBsw)7M>#MhKJ*@o87!0>ZL^AKyt;VLl5~1&T8W4S>J^*5-rN!%!@6I4q(HsG}I>oW8SAFBN;mqwJQy#$fa4S9L+(W7%GSdU!VQYgyrVIi~dF`Piv)apkaXS`STBrR=$2QcAh;G_C#(Q?hlUdjL`gZJEU%xP}mFC;ul2~z+Gak6%lf|S>zGsTI8-<Tuc`KL32FFyJ4?dxI0{`6)Z@vXT>%aT@Z_B^p!`Jlx>u;Ao`eA=FlxO2=iNhL*3iP}Q$mIRha;o>uISXo4!KxS!H@9xy5x1p;>HKFN4AKd34X!lejNj*}bsy^NB_4-$uaKh{YvVWKj$=9mM|HFMbI*n>StXBx-00MYRhOiIjX@9#xo#yN1^~26BR>xm+_BDyu5A~(v7GBRT1?;RS3>6c;}dnvodmjD;lQRTCb*?QYGkuA_+0g&b$lA209%ZLoBaN3oE|9lBF{i(5>E>vUK?&A4`Q_nIR{l<-^Mo=0`%U2J+Wi^+O|TwyA8PWCqxv@+-rZFuQjT}s(aNyfa<8?2ULqo1(N<4zsGw4^qqKk-zTadpq_U{Tl>>^KFwX0b0$I7_&rZz$sQI9(x2-`=fDw1$Jy|D_O4mGU;wG`PFqKaYR$v(?e$&l882L0H%^;eXCC#M5)NU5Vc=q1cQhA8r{W0yKau&*T(>Bq!w4S@cJu7IDA$f<sSo-IA+42;s%HAteAYM<DdCihVdALq(~8hmPIwkQ5{#`b1z<y|nD3ieQJnDu0F<oXrH;?KjA<xNkXSM`d7{DT<?_vU|HAbP5*I7HOnfLM223Y)OY}wmU6^b^u=lxykjd&yiKei={_5@fSBSU7ybpHK?i#kgd*0)>(o6lV^of=Ep37zm=bLX6x)Mp7l1rJ0Iyu1dRC{yxWp#&E*Akzz9scpEW6dO2!Sj(9*;2F(rp<Xa)jl>b8-CZ`GWu{may}>@e0+}G(&qbc{66qxEGT~Ib!b@)`87xfezCOR_9x^PKbjZ)HN~|t`Q@z1M&^d*{{>J>0|XQR000O8001EXAbqr*r40Z8S1AAh8vp<RWq5F9a%pX4Zgg`mbYXK}b~i6}X=QhFE^v9J8f|miw)wk$1*Uxnl~R(EcGsO!_2#5b>Ybb9T&G>nm*YVsByph#l^`v<S@zrayZ|65N_N)hE)fMjZ+P&$!^7e5>~>q^X<m!VSI@;wu682TrKzhXt#e%pS!UwJ#ra#YQhKB6>P{?mC9Y0?7B}Bc27{Nc#_z<UXeuM>wGuDiU0jHKBUg$LQ@z=0lh=yhR?0L*E#ADo_)X-c$uot-)T+`=nN0>SLcXQCQ59n3e<%}+w9;GnDs{Dy1uuaZrdgOeuN!0_OQAN~`c7=5`7jw=sFh6b#Mu*3$=!INkkNQs=~Nj5m<W`YE0F`D$a0`krgghVP(fYEw4Mxx!{K1Ctn@}C$+D@NN+k*GaI32tSeCk$D6Sa{?Ak`wYlhEcEz?381Hw53ljUh0x33tozT1*440KxFd1_^|+1>$~D7Osy_WZ)ZouirugTX&MwFp4|Q04rpYSd^zOX3$eQBdqMU(E&rKA=?t0?!1fTFk|;mS%kO?RdFr#y3xf1md*dVZ7RsRF*f=Fqp|#8<|wN|Cv}8TGjx2GCBE%iO9y7T$Tx(Xqn1dwJ={#P6%Y7t4vj%OQn{DO6w;lR*X~^z|m}FsyqzX=-A+Ys!VmJvT>7__1_sDk2Haws@m#;PQ{~@N`fWT3`bUy5HtYhX}eE6I|cvcsVv~s5jayzICwDP8c64Ll0-%o%Mt4mKlxJN%EX%vm;@NN_}JH4EQSSHB`9q-{M6FGPbG7s#4oaF)LB*ODjL2kKa_e`_Kk8l3b`Xc6nv7KC5os>!bxyCBXX6!Me&_Dc}Ri~eAAd(EWlxj2YGj?%M$ie=QrxJMCzGDD_HN9_-_8Ne0?k@B}JW{T*Qj4u05u}>|$?({D^@Os6KpDKQ~)+NtIcfHZ#$ZjomXoIFcQ?q7HeIvTw}dFZ+h<K2C`}!zPrjQ9GQ`16%66#n14?T9zx=A<_qa8(9KL16%sN$ty4jSlddl6;<3}geufVm8cO)DK_eK<?f$N;PNE<5qE#+INp>Xg9`H+;ZdH#h#!at2jTs^)(M8^eGjh`ME%mnZhi?-7ZDbDDJ#}^g#XPTOi#*8R+YSqg;~ojM6S94X4k+!279aDkHq-<0F2o{k^%Yi3{6;Cvf{_vUs>6ZDr+S(p+QS|0b;}M>NVKGS{E564x^Vfftm@XE2K!oA*KN*Y3K(@$SMbQ)6#@QL`{>B5&aBzzOUKk@Fx`LV}SihWEv8jb@_B{vp<JC@sVMMdwes<c*GaPTWDP*HlEbDkej@WtPZbFV)5ktXd;Uux=%0&mgpj+0}cWhEG0O&Mq5B0N|w`;=@W5-SjI}^&X%N-2E~dH7zz%(y_>(3g;DObeY9CNMUl`p*-hwi#`$<c|Ho~Dv}uo59`o11lo&sd6XOAyPQYd6lu80jHsB%7Ylipes^dIEqU7VIbDDy(4R|ejaIMLQxQ6PNh(S;p98UnK3y$~rgxLpw90~XR^F0T10tXSV>>&t@HINn|uC&I*q~~2ye8Q|EJk-d7a}kx^JRJmzH=y*79ye6~3_&N)-(3E1c9~qBy}USkag|_n??E2~?SQ!?um|||;{3(w)ft0+a+~VrE0<pe57!M|RgB{-US@fN+uX?82yzo0Q>yYJ@@Mgw6**!A44M^K!JLG3M4!WIYe_+rAw`ec?M6?5o@M%#0EXS)0VFZ~*POogR)5=H<t<%-yUwjfuKD{t!*slxT?Wf%7SZkq_6{C1CwFl^o>mGpkf4p5`<=VNfRps1{{5A4Ckmuol?N5N_>Xw4O9jCAWgQ-~&NVn&9Ln(C=MCR9i8q($ucmLF|8xA~^!bldYRn)cV&4Uszfjd9XmtwMUv@Vxbov2~l!Nqy9pKlzUW=dRCo`cJ{{}P0L`iO6$n--j97#y`vH(lmHuc0KJz;SyB7qo&K}4ivsSt^ryFm=2-bjJg*%OeE3uK~J>Tw1R&h6k5wy*#aK2pgocoFO&s5)+PY4H=5gwAo|H1+4A<uEh}SM9K7XoJ?R(&Y+0gpU!JxF^nxbgHUA%KosUz)I#}wNCW5`7OYA6F9q;?{(m1Sb($!8Cz^l(ATr4??=J$B>XZKs7`LZ6VJ^ea-592Kl?PU|BLM2qj7Vn*r#~iecv|w36)~=0}McS)WFLUR(Oc8Ty+|k@5As;x!4j5RMp9GM-zAW$y?VBKl{udiw#sKJ^l6(T{N8|g<hl5hv)D`(EJ!5sg>KOJ#~6zd!6t~7z%^jhT#G-P?#8lBt>y#IP1mEJsS!lXn_3u5N6Lfj$vbrIDo&m+XL92Y!L6qfTN-OSQv*G?L*Ft%16*ERf9*WFo-Wy8eg9N>U`JMODqk~(CkP6YW8P%B3PgLHpQ{C;n@A8VUE~9^~zCGSNRIOz5p&;Rph0zM+>2_k~@w`BH~F{As4FPtwSwW)*8ABjirR(zzWX`eXGm}qVK62pBc5n)`)vfSzzR{Q4FNlOD<s7vK3h9G_?s8-L-<&5IPo)_+qzKZSdxL`bz_{o$T>WT{_?-{u!Z>qv&L<^3@t!GWxa4vwA)90mz^HgH>cFOt(SWwhugqYvK0e<&pa2fO}ZWPTHB0QFRS=6#11I4h~~cLRC!dLySGGY>Il0*~d0dmSY?KJo;p%>IjiNUZ*|Brp(l`OjXM<rrM+hqXY+JWL;Mw)B&n)L+yUtl*<mw{(q0{DuUgVd+!c<h=Ss`mDDLz6wr0QBv*Z8?Cg0AT{d(<U}aO0;>e&H<1D(J%GvJd^z{#R@=Wx8%F$SjxT?T$Ar5S?xA<oA_1`8>c~_7M>>GnROPE`tn;OiTlT?B$+|6SeB<2Y*kD#ao=f(NZ3VMSq2am=1ika%=5-fu}n&uwHj^0hf_0pho=&;MTc~R&cZdT${<eBZE0a4gt6OvK}YDhKp<97j>Z>vh8RzN<h#11RLcD|lzyCe2_+5xNun+St}Ygt-+uihLF0s}b&xL^<h>??f>YJ&AI4JfZ1ucOu7m&TN8wE*;9^xz>MHQ@7J6yPd|M-pJPG{G9Vpm1RYbJ>7Q1&CY&Cocy+lN7E5APWO~<^IAg(z&B}WgXoYU}z-D&_maZT=D|IO`e~>cnAO5DEa^`Zy`X80_7~|`Dil_5Fp&%+&KtT2w-Cr&q!-p8lh2q96G;FRJM9}#M_^Tp51QQDk-y*segRbA^K=0UVnN90n7ODus7@dfy=SUDAKWKgDkH1#D3q|KLLNv;QX!kGauau_nz@N8B#3Rwm>2@E+3yPe}X{)7^*A9J^^G%BhiRhKXz}X4B1U9_mc{Uun1H84*UhzgEZc{z_oLez(rbL#qBY46m{5*V(&9^_t47P@#yx4Bh}f(ak@R<dVO|!dG_<wUJrm#G=Tyx^ykm=yC4exS?43(U+96nalC3FO+0+f;$6-c&27T)0$F`dnPw{;0j&8ZIoWl5$_fH)T(HRaT?b<t<YL0{+<lhEy}-2H=IQ%9?XPsnOL=}PGsBtn4^`=B!e)28N2+wzqzcLv$l>O0gHa_<5dm)mCIK-hXtq#_+d4pJs6D7X2ozLF#U>7l7P_wWW?G>tupbcaZZDw4We17IqKKVtii{ZuI1d~+>Fu;oOK>cs+c|MxGEk>aP~tnMOCaB-Ft@z!YPVFoWEa!d+72=qJ4kt$Dc$-y4rq?LS7*G7xrid7D#if`F`bI1klmTlI3z-^x};d22>`|J*_I9c<iQRObR#NGEDyJ*(SdP7fN}8<{4u#ETPP<~L=y7G4?<Gm!JusfJB&f3Ix+8v7CaVP2G|Jdr9J<#J%6wBzzFO+dmdeX=ID3rsm8=Wd?lPgASe5)8{_xnMt%qt@lBW`J{pB^hatF>gSH3%4x@X2;BV|_2mXj67<u>$N$R>0dc7Q{MZO(#%P@9z1OAOhfalA<^$w6a#bD>cfwCUK_zW1lcUwtr?{4q#Qo{ZYK}cjKaiqrVYzMm+Uo6giW(#0Qs{x3=+74h`yFpq>?8FyfsdEUgNo-q7$GUgp$tO1oFZb|F?Qz*Q6gbO-t}oGcp;ytHW4F3hxuOMzn!Hk_s${LJWB0mew{ZSrWI%rn1wTOA70p&^AqA5Wg-&Ucn1C_`omB?LfPD^)aNLOgY7N~G*_5XFIwWx%>ku(u5L5c4q2$zE&=WYoKtw;U-5^Kum3I!IIbSq{53G1zHZJlH%3dZTM|XPwQm4AgIE!qd3uezTYQO*T1FGYf&kh8Xm2a>XczR!ZMsqcnFYBFd)2J-=+MvB2%`L|*N50}FBuEQb26SK9o0+C`b!Gdy*+j0L<G*904Ps(82`bHlcCJB*R&S8R6dau82X!-6usvYl0_EL7CJVgDp=t)l(mPIi?Uf!AhHBVWHZ(rv_vZJi=JzA~4E!*?(mMljQ|746EWd$<li^~=Eo8lR^JYYYNQ?dzAu<DTpeNRl`|Z5KL?=-7LR@zw@E(xiDS979*m;CqfWV5pMm{qe?1NnpxtYre78~bpSo1pVqqL#UJ;8fOAx?)*_gw!6P)h>@6aWAK2mk;8Apn2nMLMAX007Yd000~S0044jadl~OWo>0{baO9bZ){;=En#h9WpXZbcyw)zK?=e!6h!wuMZk3uq==w&p~a;Xyg*5l9~=114@pYR?M+>~8HV98@4$wT9eIU}taaMea(mt%FP1b#h%7|14&ln2HxQtP&lP0`tXeU%e&`35@@k4`Qc${dT`SJw)XX`DbdmDy^1Mz$(<0#$kW#hDAITRofJsI`PzjPWw0R_DB=~W+j~h%QzWIK5{5$ecTJFgMSH6gC#^wu9O9KQH000080000X05h{7UQ_`900II402=@R0CHt<b!l>CZDnqBb1!3WY++(8VqtS-E_8TwEl@j8!!Quu{VOan*Ti|0R>%W_5u^+-Fgos~vFg}vd}#^4o^!%@-+iyU2mS^pYc#@%w+jxz^Ge8qgTL7+_z^rl^UpAuf!oBbPpF+EZi1*MQn`iqVdAgt*Q1$bDFwqx`$X6XQ&OrHRU=A<;R4?Nu23OlhKBikw*K5}1G{1FGG1m}8v_oq_X^cQvZQ-J@F{HaS~M(CG4KYW1+x)F!QvD!baS6$$?M4cJjO*SDayU5S!<mOcN()r!9pa1+b*tGX9?kjIMggICKz%Gj^fIUG4JrAR3(no2H^)g7uLq|K<gM17AzUNWyxH5)EHerXS+>SvZ(z@uZE?lSWGior_^3Zs?sjsFCXuDqC?mO*BKJ7+$xjtdKz=V{S!GyZ7w>0j@EVdU69C19YjS`-{se9nwxx2O^?Dk{@?P{2ufi00l=+~Bo6xnP)h>@6aWAK2mk;8Api=OHlV%$00001001EX0044jadl~OWo>0{baO9bZ){;=EpTOWV`Xr3X>V>Wba-@i%uQ>=AQXk+y?>=t?^UC5Kxmm&p#z2Px-))6A<@v7#r*qby6LLB^Pa_XEdARp_g?&N?MMZcK#srfx#zZ7%PCOgq?dKI7N~hdEVXU>BNauGdb(SyP5{YV`cv+G+b@irJ0BDEZDnnX)fIMqfBT^4{o*iyMlD3pi0F+6lnJ<0$|N%w966$lLj31lvum}~`766Rvs;_W5<g#Wzjo4~Trgh7Nhgk{4Z*+wQ7Xa50mqakH3Uw608mQ<1QY-O00;m803iV7Ql)QT0001O0000N0001VVRK({b#8QNZDlWCUukY>bYEXCaCr^M!3~2j5Cp)#t5{tD7I{Z$A`Msu7IVU!k<TRUC-WT<dGq!>S3+%Lr!B-iwCYg$6M9~)54D1^tK-Vckqcq`u%ouTI?lGT;!7?2@F*f7r1WpsW+~wXnc^c(5`Iui0|XQR000O8001EXJNEu}77qXbhcy5I7ytkObYXK}a&>NWX>DaMVRCe7W?^G=b1rasy<2UM+_n+^zQ2OimrBW=&I#HgSa*vWt_!qjoFGoq50`_im8e_#w2~rHcV1ubzxNrw=v{kjJE(!gD~TM=4Cm#UA?<nIn?s!EYxeSoe_xbEeq<G|rI3mj31?L;4nm1LE?HLAtmai+rcEMtIcNMmMv4P3lnhs^Z#l1++H)2sG@j)=z2P-03;puv?|v^?UM6wQQXy|c_7h(A7dn6}XjaWG7jn<#s^IrnE-C9&GEp#bI5aAzWs`kV+)AP#q?@*nB`ZpL;yDAKs;rYe=2T^^!j<QFt5sH)2Np$HqnesWk^RWxq9|3YL|Mqy%Dziu6(@NtCD^d9u)sbSJNNibmWBI$h}GWxE~Qzg!aEFii~P_VqWGc8M9$l1XwXbj$4V48ZqAG1xMttQ6}?-nRw>U|l*C0@2q@9Ni}QwWXi&f|{z89mRt$ec*5^JHGHM^qoB$H1(3N6BibARw$~Rxu!~hQr9r~l+vh9FiEK`vvOUkV{r9=YvDAgQFFPK#IlQXTvy~!9B7e{}@@-2sYKrqmDi6=4=L%f@+*N%nc>?er!s;<l0_g<DoCT^NqZ#s5QuSXPMO^sA+$B8c9NiP^n@PJ{P{yqq7XUoEx(PPU<NKeua9+#!{Ra52o8#b?kslu*iR~jsGy-YmI%UE^$O?C={$V34G{#aE;O+V9HtEbmxabX6tqB-o~t1Qvp!bA$67ugY{dqKc!chwjUyb~lix#NuBQ1P1W$`XM`6v{C>Re>JAn%l_C`*$ImikyX~EE(*ob&m(KXe(+MMMZpo=Rbl`<4>(}Bl735Pk}?jjpc;KglTiAq~9_*_Q+t}ev;rTBCFc|R72wKJ+ESLS=Msvduwmayp7kJ#vv2pQ!Eo9wttUv*_(<mUEV{AqW@tk48^Lagt74~l>7MWKmN&|(?k#=Fe9M)UZb*$Z@eG~_xyb-Zn#vwTh+{uD4_Wi^ft5-`>zfFYZIy?DL#M`>@s4>AJ`Y{t7l)kzlOXS+88zB&KTCcP<u^RFBF7j#S7o7cOFb!VoH(byfYti6urDo9*Wm~9v^mTys;z0n#ZZ%F&MxaJ#Q2{3`1d4Ap~+WIOuE9exIr(!Ui8i677zP%NfN`NZl~~RjYvBnx$zhG|<V}z71hFaI^4mn~K^u57xNgg$N)~d8;4Il$6V)78ML_t#eLU$Mq4EUH$<bNx%K7=9zfE&AjkIFGF7P7{V~I%(l0%A*<?Rrz5>3r3|xFZ$^Jf?|xxp=#Slk`<fyV=^~i>oaGG+dH9JIlgA?Q-55wxlzz~S!U&+0JVvZ(#a^>cl?Z9@qvIgIV2&14t5J6IM&^KFO_7V@wl|`04n)HmfZc!XP1EfkIcK-~meACE5HV+clzqv(UkY#Pt5ZJm8Aj^zUe56W#PlE992O%+H${p7dx#5>!NxD4T@9QxWm4su75olR581$CKjc)7tn;pfFmWCd{#ul-)vyuM^TKzfD8|x)Q75YbqUAj~Ht|1^y$y>Rw6GU7aupaeS>wm1AjG8~f>ziUVBHR*yd^;JfW)leT0o$6Qz1iP)g8}E-~zx)2wMWz*2BD`C3Wk*u0FhslRa-2`I+D@GGa<Z3&37=5vOU*0jyF=u@w?(UB|#76)d0q^Ue3K*@3H=9E60WWU4)Pr73>fA{L?+V*oY-ZcnO*Mg$<cFAnhyw_ysq&F=(pAmI;y*7~<eBFNzhuXO5^Tqy!SYj(p+EKlBZCjiFp4EqQ+DhY!Sdeyv|7_fWCp}LgvttN<wf#v8WiAxZK;;jUW_cW<NCV%_Z<g-d7w>dW{?d{vpX=`Pp7=md;C?tw}$@7eCwp4v9?l~z92<f|vp!7SKem*CLw#$=U+NCx6n|jM2ca8Eerc6*|$L6J-ceVNY8V{NGV4fetqWaKNLDeXKAVvoY98W!%pf_gYp!Ai7S{QNN4RK~QGYDGMux+dVvKV73udPz7*`oTYR{e<TMUh)0iafGj%he1v?}N_%H1b4v2e0pV+JoY-c9<ezzAbxr93*^7U26wF-I5N550?gyql&^z)+Ggz!bf^Tzr9N>LHqd2FYWV?=)6oi3K(YeoL-$FD9T{10{KK8XX=i2KOt;Uq5RSh{VJ+oCegQ0J7EapGfsI9hu<6bwb6%s7uwv36mf(E?Jz#98?uZUY<J{}#JdKono@^p$+M`)+8Wcp;cDm?;JqCqq}Dwd-zfX$X=FHD>)&l?+_sYk<0Wm^miZI;hm*H`5Z}JWHa|QDlc^NQIzB>z_kKV$S{2xLvojIzrrgjU+F7C)@n4&GVa!kkM$^{T5xZ!ft_)1&X>8zDU{UO!mnGgT&0zNJj3;&CS!Y_ECKb&2nmpkz!z4Y+miK~HW!2=dF88!gQQ2>6(gc9;hCg>%3Xoq8a-sJG$NkW(BmTfZ_efqy_6RtGNuC53huGUNPsQ@WY|X{pSKfL!479vK+kl8q$LNfKfio#&G}zhckm|^KyE%{3;CVHS(AOruP#<d7WEmgr!3d2RbxeTQ+5mtp2sJ48zFAFGZOfYlHGe{x^C_i{7uua#Dsm=;&gT?6*1=_b$3H}bD-Gha70jA_@kP5SQ@1)%3Y}LDol_FcmPH>|8l5bUCMA-CHIxWukd<zVT~VX4hh5n4+2NG)5Xot?v+=1f=eUVzxTgi5Fc9o?UdE{{+NKCsn@L<Jd(=r&6}E9kH=Br%^zGoW!-lFf1ym5Gd{q05$wELwS*KnXwF36L?Kp5z=;#fd55#S#n#NQjITOA%vxMeCe`ev5=`jKm9fUl@Dna3@+Y^u8tazg}jR;pd@LvCROi{i$gCC2d#@V6`29yV#4VXk8CS`T>Pp4ypX!RuR=W=#5*)k2LtsNN5Wc1^lg*$-d6xb%3>mHh;QgNuE8CX9a|LVP~FyR1rH&+QvwO$zDkn#MeD@dF8jq0%lAcP%Y`uqgH+v592E}LBaOP@it1^634#GGH~wo^T7QW)=@bdg1Bewqpv@SICqe!oJ<24ZiL1mQeuaz_vNRflAN6n);6A5ki^q=waz5we`e)FDB%fv{K?>?+$Gp~J3?MlRa3H4q1|t7CXV0eTvJ(g~f~$BH%7q&Nsga0umpZpx)YiE7Cd(LYeVUp5jGx{>xsjk@kh-lT-MGfXDxio&+SCtBCEbW$OcxK16#7kS<~=EaW3wVp0`p7v^*rmAuQdb^`O`ID+iv3E^T@cKF4fJ?naD~l_16;@<)RA^b$r+AmtbGRhalU!X&Y*NCz2FBA}0_1g+G$ms6pyB1XH_2wyIcVp&_Cfn1SQz**C`Hd~!`h(iaBmj|>yR~VElg**9-T{doj{k&bHhW*End$(E8V4e3f!9W+!=7|=wjf2>xE$s<W41=!>b5qPN;>{j1j$L*$$>)5h?y&`5L6cP`7P<q~Pep{)~bo^M2%6BX!ZL8-@rKbU<C}*dzNU_%Pg#$5Yh^2TB-n)o&&{0oSbrBcpNgY|aEMXW9aCZ#XY9y<j?tyc$ETExWoNjixe8s!b1-O(*rbGiH(tG5u*M6lTY26xD0PnHTg;W5f~mvlV7f0Vk2%@C0Kp%^fR!xs2%V_XwDbyn6-&XBh&PKEu+8TyYfr)5H##%)3Ektn|j8#Wo}veVxZEJ50Wve3^jLR{rvS2rr59Jn(?~C%3XWU_53S=;+>g3sX#AdlC9{@W-?kW?Bujlx;E$z)ZXHEaAr`Fw_b$#ekOC1spivDKqJ_qw3J$f<C~&ayJK(UU`!L+Rz0URN&wc)tldWW3b!^s?F?SP}L-+adp1+^9fSpKT6XzW%DUOUsl0L@0#-tnY%|PgOzQOJx{7wHFDk#u1#0BU=Ln-ONZ;6B=8<TL0SU);Ki(VhLo+3*eGY}t6|{N14aU`*)uxY`sx!{Bt&|&w=<sNu(D{-B?cS8=Qvsvv8B;#(rQ%dK3uuYxu&KlZNW!KwK!7mdfWrCJ=ASd8oNiR2J)>5+rz27MOstYY<=Q)oUG|=&Hifdn*_6UOHVlzNJCTrl5&c_)5o$;Nq+{)=+MSXa}wFpgUj{S98eTc;lLhD5_&wLg|1?n`>e}}832)a$ktqQ!P5&Gb@4!_sUiI@D(J3LxDyN7A0GSC<TYDPJ~lH-&ToAq&dG5KAt|9t8Yf3Bwb9ORrJc^YUFZ%?HK@<67ls-CyBhi4waK;%b6gx*n03|0p6gRU;yf$u@4&M)>F3R>SKmgjUjOZ+j?JYGL-_x>7cEdUdY$Vl4iW04foP!7Bsk7=jwWr*9-f-HxDeR%FbJCj^G!ps<=kC5NavWRyK=t?uVY4ZYePekQRwa%Gx24_6?Kr0B(r@o(@nW;2+Syb0NHJ7Jtt#7oh-cXsWVdEUh4Vn)nV#ow6_GG>t07wyCYa#Gmvd5OqI8yV*0|w2wMdGv-X8re0Yp;BGoz)gYNTZ=haS`_VZ}A-6=D@*Lbk+XClCphV6)tobs%^=Dp}_-DPmw3fl309)HdfIJdrsr0;|GaOct<8P6EEG91d@j3M@CbSXaNXwUr|jj+4BvAa0>>Vo=a4o$WcEsPw^wV4@nhlzSL%o)gL&N}8AGdg0lHQDvlBL<mosmzYgy(D<zKG@9Nx8LkC;S&jckjULej5GKsbB~72rR?*P$~Zhr8=qZgh7tzWu%}Ll#B;5kK}Ml~MT7i7)Y6qXEJkNB^Ufr%PB|aNxpb`mwW?tSwK|RjiITD@RM+YO=A2v$9~a7FT|`pQogSkRI909m4dgTQ03r3QaXl$;5t(i;+gm};*q^tCB>fmM^3?sNe-!#qp+)XhDFq!ccpm{Ny5+8nWY?`mb!Yv+)xJ!78j*nSJaSgj-37?yd42M=)2e>zZiIRLQ<h#vqPvdf=!Cn~+k*#Y8m9VWfLc3slha9u)Njl`5E@?_)%`N_uZgzAbk~;e_#C8DGuU=CpUh4q1+fSYe{BR|1NvtT`aYK7$B^so(BM+7UCc7wswx<<-ZnHRhD={q{{v7<0|XQR000O8001EXs>wksRv7>Q)>Z%j6#xJLbYXK}a&>NWX>DaMVRLh3baO6ndCgqwlH0hI{;#J%m_L$7qIkb3(NuXIPu5lAjLY^+s?yUXg8<lKT1_e>)$N{6|J?`JN8Bgba{!Pcb!&O-l-;hfBoa6P&h<OMAq~TLljXVKV^WGuMU#~vY$@3Fy2$dH@Zv6C6_i)xE~}TtrY70Vx+v=`zam+lic;iEkTqr3^h#9bVDMVhl+&7$J6e%jQD$i-IH6ZG%d48yOF@b>%~(cP<fh<a1*5KuDyxe!6V<d7D?uwDME)+at7Sb%i}Hrn6``dd72ISsP!k{5D<Dy??#HAm2wl~p%xNvi`yZx1tmwz-oBQ`g$(Do_YoIDS=S4lBjAdNpHC?Ul+m#7!lJ_tz%IfS^kiWk9nUtHn&TfDYG$)%%;H)cLWWB6#_Ny!x=D;wF!5}S*8?soWn|f1<#X>WO&>V)NbynonV4$xpX|-HsN&B;|*CtzKFrDVoKdT~dz896Eje|n3hVUEsIVNvlus6W8kNC6cQHthUwcOO%s{I8MX};IuW}RlMwk58z`MxPvFpw$AvM6`1t!Y{9wkyTIHlk`N|CTAja?<tvTIz?o^*jeLy#Su0ZjH$+5Eg#Z%rhw{Qwq$l+V=XwOPtA=(7L$E*aDd|Mu`-8#^`E+>>LdKNPgI?P={`_3b_rO=L9FBbhz5wRAUr9=-Mq#=?2trx&l?HkYQSVL2Y0E^p}@EF5bO)`Qja!lZOF;KLcibOO2WSDKDY2E;0@mja9Z;(xT2N5X6R)Tif&}VPcHC&DFfHz2`+XcS4IX$4R-*JZ>k6&r{op1IMwffH59BuGj64E2*IKMSBVV9(upphB>_vI1nrHG`lj9VI5j+Q;;9FLj9{|)9G?^bp@OOh6|Gw(_^aw%dCpkt(cy>jOi28sFu|61Dr`}+YT2|<OtWcLz~)`pIY2agP6x65K(AZ)U#|aPOaGUQZMnXkh|2g!zdPB>>Q+?{f4}-?ZEEeeH?!ZE3ZTj%8GWp_(r&T{*1N^7M|}~ekf=_BjBm!@X&RF#HE}kz;lO&G+<FmX_AIvNPWj<aq4hCaMLhOB7Ta&j`unIUEkN}rx!93Of2}bCq*j1IA)T4RZjn&xovq4dYST+1};lo+q2@3$1$goO<g|<xbM+8q@39v=(|OQ<-30BGfJHR7#={66FiUOFbG`F_oFY%dgmKvz4HmIXO>7}Pb6+i{V<{)3tT@4Q%5iwBo<|9Ks^sE!DfyhQbsMFMp2sL6)r+YoM64{dyW$Zj&FZ4*5i82Ea&3mSNAQSI<JnlM<K8pe0C@pcn@$kejIQsjX4W#EAjw%xn3-iD7D>$M~P3<C}2+D1r|6O*FM2=n0WA6ie9Y+qRmBJT#LL~+|cz`XY?mbY0Kl#jw9DiQzxJ-0`K7p76~2$3!RACuFr)7&ILS*MN>acJj;qg(DTGjEg|A0Ji%kn{VWd4WdNnpl$!H3IeW%bwj7Ox?*}yHp67sC1_=-qP7?XxKazyUQR>66o*UB0a%`FcTO%RDG_fNablp42QE<|Z`$d@gtssxHC)Dx8sb$-q&jK#|Ftq|Zj_ov!LR&;Ww>-;XZUn=paS(ATXaow&JlAI2Vqxf6=?MmhUVN0hm+Gs&VzdzaFpe+Q!1HVhsn~jQo7IyQEw97`@-DWty@~4v#{fMOHu$&Lb{O>;2e%i|z;{^~3y2RQvLeQ4!npl)0sWK9Yc=8A^_-9FeLJxM<529D25!teYPlh}BMu<nwvsS(lSJ4&4I|f%SqKrshdcl@kunI>)+qynfb&@~7&M?PAor_^&$O`r0E0hea$9b5$9E&JCZOp!ocUhBArP=Mc6`X6B5-zYlth9-^tBT&4jAVyxW71#od7WDDah&C!50NQ-v*b@nN`P`1(pk{DJ+2L!sX22UYbC}Md%JVA9B(JxGGs53OjNVJ7Hepx(*<EJF-q;9LRd@7#v<?pX<{S;KNam_Kt#+Nnvebnn-^Ai%t?+iDQl|jKaj_fFK}8^kP4B60lhAI8g-i=RUQ30L@nDd|gL&Y*nq`&t1m({ws)LYJuL+IF7(urHpbArxOCg1rzhV6fl>Y*Z>M>Y_T9rEXxJa2O;Pmv*PHK|MLC#b4*GpZADX21B5Kf>T5FetjTQI>bhYRKoD>}E>erdU_UXy04HJzAkiQJv>H2h4B0T4o8zZ8a1)?x02dtqC#P_ijV{o#{K)$@g2R{N@$ZWX7NtQT91&CI3hE}{5rM@KbA6yGAWS5luo*)L#$YN=BDe+63;1n2l#5eN%k%tiA}#xJEdE_-SrULAgU-4^KtpbG50W%sD~nT}x=9+vfdz0n3S-}K0Pw^y=Lz*=5eAMM3gMpIT=Tz7TK2UW`}9n@D8{pCU~ykWspWxcCQjmtSfnBKzyTyqn54cF@Hh=D-=axu0l*1D3Mwoh(+E!G47TmY!8eeU{RcREzDw{TZX-OhQW3zH&;!SdthA#zv0O{UZjgd%#v#C_lv~seV+;IsV1uLZ0I#25upc|0!`<5gkeOM3tiL8lPfnuaSVO@ef`vX4kl}ccL9^JhKzCDDxNhu)5f>2Iq7?Um!5#R%ojS440Bl=sL{FK$<J!Izor?tTi&e3U1h>#`v#LwFy00@<O=wk#x|-ZZrfXWi3(Iwu4?!EOFi4U(34AXB4srlWwul*r>@JLil{zfK{XL%tz9T3Yb3zmFc!0uLdUC_h4r15#p4Ne@tX^&s5dY1zv(BknXM2=9Z4P>;YUNYp*lkoz?uv57&GkV+ch&NV_K(LTb5kThWl_<a<MK&ED*(<hPXN3G%)=~;$6$=Y3tT7i?I=xwzufsuaRrc@09nU{e8qLa@;MJvo4Y&(j|mx|@7OMnz`4edGttPlB2Zb*J>iBtV9qIVd0;e=;{;EOtE6|P64|B_CHN}<i?x*AizO{`u;za(X?`_*E&2BD`RnPq+?n3VVu5P9W%(I$J=OTAP%!~r)B@bYQJH!c<kfy?`?2k^nA14&Lr`dl{+2@{huKjga52oGp$qI|9#|6YEhaREh?ahKn)OTee)3-v*Ezz29kHm)dc6U^$xQm&rUYbuCCpsZGTT<*zM6jSs-{oOsuQ5T9RK#uZ`?QpcjP=R?Z?63^-urx>c^K0(8R^tm%qGv_v)v=k@+Xs?BLB$Z(jDgf?YxENn$$`K7}lZI$(Sdm6E`TUFN$1pmisX5@RqJKyE}9m{TpllToZq8KQt^q^ipinS3v6L1uEgO#>LBR~jNajAWB6CBt@=O?xw|7BqnXxv9l))ZQ*iSu)YKTGO(wx_4=QKg5e9%gKXbjEP~&|4ieNKoBO)y*>psz-YFe2F)r#exa+4khLDe)c85a6_chWgfwam#)cn%C?Ab>J;txU8aN8xz*!ly_UD%wqZIX~%q5?P&vw_M1%#bJKul!odWgk3GhL%2KCWf4(v0wo)fXVNF;>D{O40Q%k$;oF6}f0+DQSTyk(!BBR|}k$ByP&Jeg~{>DytOWQ}eCM`|ZNBJS`d`SOV0MuPo1+%W@-A)^-i`pk^s3m)>nCa#nC~$#Y{<r<2GSZM*kWsi9|HiF#->i#A4NK5tpjYm5at+YZt$Wn-K#Omu-)PIS1zlBGpn^*c3J+m0PA>qrvG$m`-G6Tsu)Pw#Y*(N;k^J=cIFx{828(sFMw#iEge;t04_X`jlAD0P6JAYiE!^6E47q5NHQ8%cvm@I&7QHp~cC;2pSm!~LTj*JW`9ptqW}C07@yX_sT!=JHS%SRNMNso~~CBQIoWv8iVyg@xkv*itv-lJ6IcvL#G0DT)=`{DH12Ah+X~48wRUstxEYZCN(f1c*ieV<}-(<s}RAWGJeZvQ{w7a=MZhY7QjD_dmWPSw(UIM8J$>y&^Zsf4_hK2Cq|wf)$(;IZ#4hv;sW8S{7J)h(%v%_7O|<DpOK>mBPdTj3BY>Dlgy$c?%dsjOFNP>X{xNsh48Cf`o$;tYrjKAqDhbL;S=dLP_uiGDTIORkC!DwuNY|Wy`7EEs-Wlc`B7xGNCaTQWYCWWt6OOi~CRH!-p>0=kLpn`0#<qj0aXtQdI?qMR6wZUrjr?hb6;REP8`cB{^W(HKAz@3x)^}koFp-F3NJV#-bu6i&Ds1P{~Hgh#Xu3w8Pv8TbYe;wVIdMbl$5LG~tK*GK4R!%T`TI#D9&~)|b%MU^e`O=QduH0;@M+X%)HIfG{yiC4z{X1OTlR%hs#^{PxGn*d4e6-odbcm}~fB$n_$bVaMvbP3O9)YN_v`I?4IV_}6L}#z5N%O$+|n_+vJ+jmIaWp3xLMc8YDWx-tVXpkBYz%Rp_$QNe()^O!)`x~i8riQ%xpPh;&^#w74YV*(D7`*0Ed9~0LZb>;zFk#wXv0SBa8XxLE-6IDto->^)L3u_Y7NqYJ4@SsM0e0((YM0S1ce#Cv1L&J=#`1;sd#8y1o1Cc)y8+BrccCj}vdGXK0I?2nonv&EH&<$BzjL@iWUq~u1@?2cWpsr_v4jVBP27LU!y^2bW*~-l|&&r|tQAsPrNS{FnDz4?vk&Z8VpgHJ=!rSowxPx*VDM|UvlzbYLiXV^4-C?YeyrY277_32lQy4?~+Iq7F506fGj1l3oC*Ol?Y^i8i&;(ty3K?|{MNZ}3iMDthzhHH`sM%D@>0pdIL9aXaVSU39P6@vBW(28)P=U4W35cgNNlZ_MMwvjsrxno4`D$m1Xcug`$<dRM41C#eMQ;*LXS%gSHA9a)Bevx<A7C4aVeF``UYEI9D}E!<(OwH!g<M@{8T(`!9vq>?vnz~7b2;2nd{n<X{L_5z1FfNAGmb2oSClQGxz?!B9(EcwlWn@lX?4cL&LEa;PvDKZx79%%lUt;#?r(FEJ`+x0&rUUWe{G+FcAwl;l+K&V9a~yWFd9#851wVTfOXc|?5W&TE&F(W*#)r&TD9PHwufAgZugA_+ZYBcfs758GweFSHe-+yM!(*H3d+Lv^OS9d(nIzQv7^eZ6%$Fc*QJLVZ<V_BSX(E$jW`+a6FjV1qbCRbBYEC|hbkO5z_0@Yl@p{ZWX;`D<XuE2J-3E}N^e!1+RMFC<Dpf^M^iVOJfQ4MEx_TZNgHL7T$vgyYG&&`TAB^8d?LxiW^HJaJr=EL$)CNR>Pp+WlNc%hByX!Fch#qtHA)YN6ng1lckq)r;=!!%ykx60vJ5s|twBw!G+sQ64VvZcgtYIa^mj&S+HUeymS1b%)b&f<<?foviUMu<jLeL@lnX=D+)JujutsZq-bU?&U{W#EFy|EWi!`l7-I4IliVlcUS-90BgT}#%8^8v&*aCbohz4S+EF>3#H&1DGq5T02(L@hD!GmA{|CXR0qO97dX!e}o&j9`>&#wSLp?`k={M|&JZ1WAvAeYmp6w^cAT6TXfO?&cE>mJgO&va$={edozhoZMy!r_>$r+;C{G|;$&b)<$&q~;H*-{ZvC>!}}x+U7v|g85c`4kh~2wCDPBA6ZD~vqJ%XN(iKXkDI)Fu0JI#g1mKI<kb-zf-1B@i8Rejb>(1V6uE9cCyo^yCW(JKe_u%5?o!w5QyVrfRI7>NCql^4QHnC%kd!*v$I;f3@k%yi_$&WzG?D*L@&AAS`fLP)^`>}~(D4Z&V3l*~7yww)P+l~z%3`w~+TixU8cr!yKV9iP-*$R=QsdvL{RtHV@mWehxj7m)7%V=m+2`;{dWTmkRy#tsjxVG2Pu3Aq$CU8c`f)$DO_y5lB=>XllpfUQ;}OM?xZzgjT8HN(t*UXFDxIK-v2_H>e;RiuQjGJ`jn)7@DdFgOAC<7&mD%z1<KWuAo8xZ4Q(ZLqbhW9L!{f~Wyrk8A&W25U#8<^|bht-jV|q8@DL?R|Y<5JJAd?+(yMex4@gL^W68p_ZpQd{!d277tgv}Z}NR<i*`u%l)#tJn<<tN(dNw;L2>2glsXQ{QL-JpyE9)QwK<gd$34&l5rZ}}*5RcEUeZi<w0PnMhH?`4r+sm(*p$OH76t&~#S6u0=Ccu_A!d57$jE{EI|n-wS5LagOR&b@s0x2{M{vz2)$0PkXoW9aP|#yEiDZnCO^ar@gaPy3#~G!Cc938r#npr+)B#@>PcJ8IiQ`EojA(ocELO>lY$`<x{q4s3S%_tx6nfd&53Jb~X?1sOhcQuBu0v+<+S+L7^e@^u2!k9Qy7?mX#kqF--evvjvWslR|N!X}tZu-#>674E*%9GDJw$Gzv)6@8gm=nX$+0#20MxSkyxvX-Co>l~C|qWz76pJ2`RkTaqmzXz{*EBEmmI4#pMeI~I_?=8p{?-%%9gp#7EkQyAR&PmZWw5hdkRG@O3Y@gVts5mgDNjJVDHu-aVsIx~^Cau*9tQ9EcL+9Bu-EmAJ?cz`R%8t>bVBiQy<Z;4-9`Ui6NHrr5Ae=||*vj-#`NR>C5C3nk>vJ`E-VM2B3F9t)%hqaGTCyd0)%~+H4$RJSnYbK1XuHAVff6Ruym&w!haEgtqaPR20-BRB+tI=M*)w<Bq$Q{uS=Pd`EPeI`mR?`r94^}vu$O8%Ad#!hikA9tNTL$B0nn~Ozfs-app?gYzXFu^#oHI!Et$<xOJ}{!K6U5PXt&L|+tp=J;7YnZb={ms3GJIVz#~I2*BnqAy6XFhO<gTcJPO}(2Pj=!4qAAoUms!qBU}G~+gn=V(nF}ju>S5MW;5!#i9eC<u(#2KXh=33k?(s_8$1;hjOS`>G)htz1@zAEn`L%fqYMx&OSaeRqLw>5TQFJ)k?%EYVm#k+qCuB(j|age&VP6Jt*!LfsSS~lquB{FOVDw!9%E|kptc+74b}D|2nNsLRtqrLl6?O?v4dXQ>nxYcgD_#fTfF+=_4^@q-*Y^&9SR?h!>sYP!`qjC?fm-7tGAD{!SBXpX)74?pZGc%aGYo|P2ILMEO0hGPCV{ou)X>BI^ZHuV6$@(#0K9*gD;=aI;&M>e<J;HaSW4k-WOcIO^pvZ?N}z6wn}ga6{;><^TiG0rv0KWhG^=ebJx}|TvUv%!Dsc{?djzq%4?iPFL~Z$wQ96KC2e0%@n~yu-_5t=_sdB1q*T-IJHCE!Znm&x2LKq{H>0anaR=+$$ZQ+|mp)oNklnQAZmNQT;ZW@^DtYy*Yr2f<pMW+uCJlF<c4vd*b_Brnx;<*<+d%nH6(x)^+#hbVV@{(zztyY=W!tWKcF^KLGch_H&_=DqXGX8xWHie+;(!@_;t_6pQU}TpuCzuZi?F6BX?YJO*}O^#=vDZAcmRcqH2XL=OdS9x2BvuByf>Y}anH23H{3?c!wwSDb}sifx^r&#H2`U@H}&CNRUJxYXOxZXkmb-`9vtm=I+AFsH%|fNUf*NCBKhaAL#3Gl+1(mByd2s1<Yq;?QPH$Lx>3=|)LJ37#@9bh#v`TG0nX!oQDw@+N$<S3l#Xx%nrrC3ZP{zsBBRDBbSI?KuC|Rv$oL!>XwE_<D#I4P?c-ncmFK|TXP;O&wk`qqo>>akFLZ1nahpN3E@my?L8o2`SN*A*b5>*xDnRhnVDja3Fwa|v1CU-dC1Fdk-5mY!p*aos@Ij_kvI0tjCPhvbL3G{@CQl3@PtyP&d}A&q5>uoZBkB+pIcx%PNdYKhi4^!6m`EmHfY(&f>LKoy8B@tJ<c+{hy#Z4Sd?rQbOTcS<=^f;QSYZW@0T@w$YpDz9^r8R>kY8T+E3Gt6m$+XBa?r#zEC=hUAk_CUyA(gPme{!Selv;M)y*a_OS~@I$?NT*A%bW`t?H`W=Eh6vXHy6)Be_O@-Pfo!<w{LWZuL~<ugt@zyv0-M^?*u46ylI%+Ui^s$(hbv8Ov@dE}l_6wk1Bj+tOCr+j~pdt_1Z28g>?bfh{fv2N1iIn|g8=>sP0W=2X(S*gisV*{Kp~lwz)Ae*pX8+*EyHt_i7U1}=AsWE#x1mpI^^i|z9RmpkF_aG~oi=?huqbh#fWwo90{a~K~gx^etqt|;khivz?ETQ!XkW*unKc(Hl&Wxq+Ukh1$@d?Xl4i7j+-*zj;ZhV|&pBlwLdlwGw=rhWNbSeIS45BA4ZTC4O4mGRj#BrwK-G@!>hVf9!$L6cSsEP4Al96b)Yr8QeJpxQ&NOxu3C$e{7DShm)$p;52mY91kz)s5;0G*Dk2wg1esRPAEJd>ew}&LQn==OvEArqKvm$*DpU`T4kaGOy>Yhn_Tp>8Bps`S${7(@4ip*;TIjSnzCn%nsM-H^=-t0d8A;C##;7rzaJ~(}0$-Eqhc9c=6`v|NEc+m9HF?(Ln9J{4RGdk192B?bm0Wy^2;UowAO8ZP`iQj!u<*?5M8h`}X+1eQc!}W5)wE{;#Xlm!p1ep$cE6Rb^Y5UNlO0X=Z@2D$R?>!T$qLO9KQH000080000X074`FOjiW}0I&`K02lxO0CZt<UvhPBbZKp6FJW|aWo~q7Z*DGddBs;tZ`?KzzUx;^)r;1Z729nuHdMfU1PJN`0qo=w1O!*&uGp1GnWXG&gZ%dnhkAO~KIBpzY-xtWd42QPIF8>a>$ob!X{`%h!|k8%uYs4mb;7_#8*r6?Rw^Y*xHDW?q&31=_$B-KSGcKp!?P&5^M{Q7M-)zKm4eZZn>~nTEy_}=EpSyrEw;Sa!`~mlb!{!ciiRsE3k%!`Fv4+(TC(V!hTm`Aynl6j{r1(i)@!tiJ13OC8f1f_4Sw+OX`#`j+(;CJKX_50nfIMQ-L+6#SHXGIS%Dhr8pIcAoi}DD_Vx;tR;Tv9^SX<wQ#;$6;60WGPq(!!q{Cp8(#Zz_4Y#{2isLwnHbytV*rs!x5sZ1Ow85iF>0p?RqMqjVt-!CS6dPc+Y&pXSK5|ng5#VD*y-De<l4U6r!)!QV18H=ddENOttwdn7+Df9!($2};I)>h{2VQq##?YN>JIBWRZ2p?pwr661Sm8rAC!U<_(8w(svbrR~;Epy$bwJ3J$~K*%?E7?Ccui+0PHqO946`L_o7Dsy`oN4Z`W&O!w)LI`2r=}OA&d}Gq7@vydx(Pe&7o6(aP{%b3avJBo3)rwgBkT2Tnj(sa6TRx7{NLJ;Ni}wz-5l3TD}!#nXTbDlB?la=4{EEr|k)6k+~)Bz|SRnOy80`C2*ju=xxniQKfLtus~8^kT0FdY{gqaCqo_OFu6{#xo5u8lZDi3Jd!;~@t7=?keKQkRtb)dhj0JRO4-ar2`CsNk({S+5g6ewM&{6v-S-<ZM*90rBJo3`9{yLO2u64-+I8kjWC<(ZEqzY_tU&RE_v!3&Nd2eNi_dgdp}h&x-$jP>1&PI&1dt+`T*ybQKLwvw6MWJ>b;oKtnNFmmwscV>tD)8p9l=gZ&P)XyG=+XtaF-+p9+IwA3K-7(VhZSx%I8JTQIy`6!q{Le)}%)PIT4O{d&i;-vG#_}fIRlHAa(=+Tcf+yE|E4#jq!g`?nk(*Fi^I_rdo8)2Tdrxu5m<aRqtWF$JT;_4rgF}8pdS`ooaF5;Ss4e!U$XpuwxWEp>Qx0DtKqH!{<P=Vo1*xC#Q2Jp=ky&wa?+EI1Oi4Q>KQ55>gSWmoIhDNhQs*05SLtC;K3PGi1|~@wW4qZtHKE4Mp!tW^wfqSZ3P!@-4e!)^4W)C%pRM;^el|olREh9DL7kC^66QP)R@ApBo#c(JdOv+Wag#o5<3#5SBdrOJ_}B^eenPZ5InuGDNVi`eQw5=wb3-5iXujNFDlIP=Qqua?=+?zwTHu3nS}cl+e&m2UJ~xbtZ+A33->W)_O!(j~d*%ZXYF@2pnq7JwR{5&d{%~fZPhL!Cj84t>5w<h1^z-mIB=VaJi-3kE2x>fD*`z%x`l&B71|Vv0&Hx8q@qo^u~T`i@(h_1un6sKl;4>WsEk<-5~yeOp>K#17)vpnZ$M4Z$__cZma8Yn=N$H%6eR7y=sH?C=IT&@xkCOc5C0g^WJtWI!12^wVaAus=+?K@yL!>p$P>k`Wu!rG_U4+v=iMOZqJBfFQbF{LA}JweExl2)x4g)>4OEM$j}NFn^e477AOjj0vTfkM#wUTZ;Cg5x5l+Crd*F<i<cKnDY?U4`$$HHXA^%=wrl;8iTIfuBbD&=B`dyN4L^cHW&`++<Z79-gCHK+Ed6vG**z_KMsV!T`B*MTj!qXZ<3K+}vbO>YrSzdn7QaW*>4w$Q1P!%;?KM=x;sBxRB^RF{IRKH49kd5Dx__IuoPqQoP)h>@6aWAK2mk;8App9j=#D25005;i000;O004Ahb6;|GZggpFWiMiHZ*+5Xa$#^TaCyyI>vP*UlK;-X0?YkCazY`No#Zmg@v1zDv*Q{k@g+{Gw!A79L_!kB6v-h-*@{>H`*q_%f_ivzwO4oN$=JlJ(P;e8FTi!(&%CP3v`w;E&WUJeP)fKEgcs|qE{hc}nm452YhII@S2=@5^0v&`%oC(28$K(`J95YCg6Gt6ZkD{SZ3Xnr8tBKGY|Y8_rdgH+DT{nV=4DL;Z`z8mA|+`ifb@ov^E`)X1ZQ=!bdoaVgg@}4ZIDnw7+JA`EqH23ozd}$%_+Ihnk96rvZ8=qHzklEGnU-(BBj7wCJ0-XSt>NgBbcjUb@M!1Rb}0fq%7unmH^9=kXdK4WW^#Y7C=gHlC6LR>bS1!IP<z(kvN{WO<VIgCYlZ_KnSqtvJj4A>gq+sYQatUcTpDRyA+C2#hPWF%?z1qD4H%}**00;mAZ}UYBoZ%Lt#PLoZ%|W)J?g{k{D+UNy3V<$P$*vOD2{f5N4+@U&Zq*=Z<sz_nTioeT=_deExFz=_84Viz24uSFZE*n_YcEU(?f5*ZF+$pI<LOU%Z35M~A?V+tjQOu)Ql@3wJ`?7xe5UJ#|CfTEbpr-*p>Mw%P>jATJVLWjx`k`IMfRR#<OS&f2-6dHRz6Y+8zTRc)|(Oiy3Y7pD3;%k%PH)}PQ9J)t6R?y&ZZ{!GUmU;Lq&<>gFs^_rg0suT?l+B%Mt^0_7R*n*rNOcVvz{f+ag!%SA>S;0w_mrbLfBT-1o7PcFRHGq4Tm(a!Fo!pl-z*0~|7Lu&teAERTj->HRLG`2|`EY&p%lX^iE<V1C-1Ei44K#ys-4=dQuHcZQA^GX2yL-T#(EL?(2F$w8h)eK)m;PR6g}-8zFPa)qp@s!PNalGfmeEb!@<5JMWGI|gn1r=CPytrl@TfonoBO^KMfUPcGzVu*mOQzWvs$d1g|%zHF3UzQMX3aa3M<m*g5EE)Wa+!@+Eue>bE}&ghCiUj1e{77*YhQ(j$vb}w()QKVIgrxBR#^Mh6FL9<U)b_tn!17E=;F8QqW`>2;p{eIwes=Zjq;u^rvUqb7P6ApH;D606;(&Mg-SQo)e%l0;Sr(#H>>Am}Ld4H_*j&hF(zTs@|><185@MmGTFWzA%V=LQME0(KM*RpP_iFtb(<<ngLe|4D{)OH}I`%0Z_nB6g~DL*t*jJ;a-kB7eB_vUJd?$4x|=;2m83Am3yQ`uI6x4x56VdM_|El&JoZskQqr_9Uo!@b+FwC?_lErf$n!aB;ss-9QS}MU*RNp3iuYf{}NgIAYtB>de&5(1^Sw0U_?K+MU$=gMO~M*?|v=#L&cK@Y<9aQ9nQNm<qlO!&#6`rup+cyah8Gt%9@S80@oqL>feECjAtm6COxir4F(1+vpxkN66`FG3Ah-KZc?=f<OTfM?sEN&<*ngaj#{+>P8TOIh=50fgEXsnY$IvGeVYx{ip&aVzBQ-2=6|$V%~5}f=LwpS5&9amd1&-m0X}yY&@(5=QC3x8VEiXY_G2aCflwd&ofH6!6+=tdFfmd?LBWsKV{3=qO1fasYv?mrJsNbskjyEV-@#bfvd$#Fl)?RZ*nuSp&v^}wOAQ_5ZVr$(*+ZCYvDXZJ9?_qo1zYin?dJGc9Y_}xzi`{71h@9;I#B_g9l@(^#w}R0JWE-_2R2X%Csdr1pZ35Py=S25LXww@7<q_3f+wI6DH5<P2p;qUv<IjDfNL1`pLE3=1~apeD9^4Gp>zYKv(oU&U%@E{Ulr*p^Y7hOH3tuOv23Qqd#2lmCSQ*9pf|6fGyI-F&!J|0=8=EuN_|2DM@@i?&=mYJ;Xisx2y%VGn_zGJ3_VLw-sJI)ZzjEWLqR3_P^Gp9K5%z@Phsr({N`6L)P-L^eYk+a)w}rW@|Vx&pZ^}~=C@zpoyXrUzg+%ubrHY2_;&gB;tN#wUXvGWw?SyHnF+y_4{a6{XT`ikEf+=pSvWokLc7&5>MD{8&P$dGAKO8tuW!dwBgK>_IjcddRp9-Bq>Vxo9?Y3EEz2Xh{{1~QGwsR3%)B*BYdVo>2#K$%LQ@ksI3ACI5~e(TAQ9^nLP>hjSZn$3P#ierP+pYfBIne2<G3A{S2!HJY8<WoX!uOlr|`SlBxM49%N(h#Sc4uhuC&_VmxtigkW*0io*}28o8auUIgfhTsIOSn`?{CH9TuK>5~*Mu07I;Szp7{G#j)Ad7<ll4zySDQUuj8n#{o(-gHmiEv7`s*yjz#q7THX#nYHQko_WFT3BbyeTG0#ldk#pb8J;Ah)Tb978(`>J!iQDIIwJ7{!0$b;{QyX@#lj1ZcGReb_%Y`TmTbt?75i{bx|og_RFL!QOMtixM{PV&2(W>zCKfPd+>aY*x!6!~P<!U8811wH*B`L;8q3`R`~LJj%>4zt^kU&#)iCS}uQO2D-}z0c+XGnnppS!ok~z(&0z5G6UI$cEFodmH28J<%t=PyMMy)&=bHri!WB`SvO&QHZtOo*uL8o|syZ$;zU}(P&kMRs5MLAU_NLzHnsJBDdI7v7GusS}qP-V9b*nfXLeg&3>!AvAAn~_QaVXB}(7fSK}N2I904yJ`a4qt??!?V*c<?Afrk%vURFl{y!*B$4e37he&-KA$@o)uZceN`+9I&X^vXJI*2OvJ!MteJ>ACj37_{12p-uvUPUlLs-NAR!FCto2yXCm%n2LGEYe0~eYxpeMj-yS+W3A^bbVf74Kx#`teK9j*(>CuO}yf&LuQ*ZX}B9NDrGv{(sW@)0UxA$=K8(6k)*|Bn;LhKAS_$r4}y$lO8T86t4Q1tM^$$sri8S-xqqL@L*EhN?@j6>8r+t56?jtCiHTN0jUjYT5ryHEXniwYW~cR-LWkKt8YO8GUgAz(3MO>#hn3rQQ?%it<$SIBg`o$)1u9)z(k;&l%V}W$j_Zv?aFep3$b><d~M-27zNzK#I$``xZkrOd~y_qT1Rj1KnK7I0S)vxf_5S$YYr97zEVAKpJ;S5etR^r8{cf-I`>lM)|G7I!nGLQ@BHU=-W%m0Co33yd#qX#4So|-f92Y1*v_NXi|u7BWvx~m)94Ac3#(Z`<I({pT6GoJwVRUu{4+JrR1!FXiV2gfm1azhJ<dHh%vrC>T`@X(dF-BxUN+Pjiwyz^#Cur1_tQrF3n)OWI^c6nauFS<(&nA!O;5})NqWV@)gT4q)5sXpcT1oGatRyQK14}-zSY+xhJXyy!BMz_W15U3OCR@1E2c{gfUtsD$vBFzP;*1rpaN|a~S`=&2!R)D9ART+c3X%2U1jXx=-a5oMF=B!m_jdBy+H<U<7>-T|*xCc-;o{(Qvu_NAaixu<hp^MA23_pe3a<h01Bel(6uF+YyEcDtrX9f>#GNvYhT*5S6(Y<}nULlL)1dq#0Wjr9jOW12kl<)2F8|Qx1w1Eo;6ZNzEZ<B~0!B=I8(c81P}-mkvA<^lg+zRtYVVHGPJ9k39olnd^X%v4Y{PK<^S{D$0B<-JbolNZ;BK{!xHi8X{7;oG4QHn?p(j4j;-z!DpvY0aoXbfY=YEm8^`C91lE0V<|JUIu|o{tIgT?WTce$=y7-j;~Vn9?PPp9)zDIE7-;nLRDSolPP}N?C0r#72Hh}kqnW1&N!8g>G5yhyTd{+j0BHCfL770oO9KG*!X4zicgES8ZIIF(a@U<bNzfUjC-s85RJDYkTJeGObyIJK+2VuAn7!2KFgvq*n5;xkpPw-wNAlK|3Q)xR<v$=X;@gVJzB-|ArMrKvO1invGlt~T7x@tk7pnC0-pU3#P74E3uaw*y;86z{=u0$9*qAOuq2Y2MliaKH@WArhHQ=ecp|Nirj&7UG>EE{Nkhp7gmE?{_8rI=%KC#EnR8bf0e4agEek^Inr(?HE;daMfnFdVcI{Q}$LyrfTaQ@^H;7Ib{V%a$Wt{vMW-NY!8G5@niwS?WQ{oKxrMv)5Zk9CIWA(+NAIRIgo1zREiNp^XUYF4EDVbs?79(T*8sl?>@^R&D#(8f_%s`P%#dhz^znLj_k>eM>k|M&agj_UsrQB8+R>rhVqujEt3i}OqJ_UiK56GUfB<eLW&oHnU~RCkLQZC?u{1SP+;Sq}DAy=1V6;aSzM8YBssP$O(suDQ$^aF*h=GAl@1)I4Ws#qqajKq;59Y*CxZS!^rs%L`P{4=bRU!gQ?Bp<zIs<9BiU_i(!#`NYlI6Sr!|Huc!X>M=z*l1m$@HD?FM?}Ok6fv|IZGNduxaR^o92U_ZBlk~5eB7L7!bN-z_ajV^*I;nN_y{#IyqkgRz(7yV!X0x7?^n|tH?uGKWb9l9r<4!scYv*h#i9x##nuo30KTKm2Tti32|G>5GM{Vu+Y-pPYo;O>~c->vt?0dAm1O2H|DJzv2s!gs=;tWmeA1xd=^qwDpp#)gkg!<(aSYSNBE6~;Q%BgdpJLB0q&HsOO<-XT`<e*5AKA|$7THIk)lD+lmS5JV3%^OnWRU|)U?B#pBh;fVJdp?Wvg%`Xr`6Jb7m^nJ5|3+VWT^FDwZ((jdlgXXxPGj%Dddvkh$2cX1!G0H|NinFKz8*wx85~T~X?(OD0{kNxmAwI!fC~-M!S_P~&WVJLQQ)@1?hvE}f!8n2-p%t<QP_Vw&gw7eZrzVpH^0U2&Tr1$DQX>183XUHt1#E<$R3sVS_uE%N)GEOEE=}JKM8vZI>yt79`DTEsx9vDfr<Kv8#e3j$feAC`rWu^IPku#(?Y&YaT#nbz9cl-7FGp>77zo2o?d8rD^_H4G@<=7oZeLTaTRR-IxAab&WiudV30UVRmvp@q_1ikx)?|LtB>2xoiOxd?4t|vVpP3d3-INIl$-_R$qy$&-1l|)ofqWe)%EVZq<ZOyfdZgq7#|F<w`1!tDJ`9jY%S9e3+Z+gJG`qKco6>MHlSI<SA&>92c}xihMFC0%8){$M5t^00P}16_xgW{Gz&Wqt^PzTyP`2WO=cT>eySydQ-VCmDiZ=?C{+6boM0s^r3F;PmEbwIB-KkxZac<b5xR2zP)HrVKu)*QAUKE#IZPuvApah9BzF!L;=Q|N^WAf)Y|27CJ`}Jp!ti<yeD;K>fZPi2jmgv#cN=XWBTbp&Vep>JYTsuIe;{mu%64yWl`hb+#c*#lO~96JmiettM}!1S5ZBpXwb3!0367iGJ1=qv{uC2)soVP9H#6!GT0Sa~%xtx+{xwPem3-!4s?ccwx2Ky8fHuG&%1pk*Mnh1`(+P&6oX^qip@TO?kIo>Qj|aw_Cg4yj)dTNg`qxC(yBGnsg{wD3v|i-p%y)mH-@)C`$lUH^p^m;H0>ilTen>yV-PL1**F5GdObjF4+4<-25LkYn(K8bzk<jAPc3h1X4h{DMM^V>l>yCb#(N*#dcMVH5@44xDy=YM_Udt9g<?12?{r5Od%Os8iYb0f98Z$k_2Zszje;41smKiKH^6dv`fNcHdHj>p(62eV#L!$k$)XtC|7~t4Gq4Mbr1v4e$N4gQ_vd;1z^d*Flk6n7*c7!$DeM$VuUeb2M{7haAN3K+?aRbZH<M2%|re)P|`3DIIzf+U)r83MW-BZiB>|XZS{@P%S@!Dji{0ubIhfh-##RJKEIZ#D&@>3JV&w=ZVo+B_E$CzD>W0|~)V?<gUyA$nxAncs~0#Hi>1QY-O00;m803iT|slQyi2LJ%k7XSbi0001VVRK({b#8QNZDlWGZ*FF3XD)DgomgFy+qMpU_pe}eA8L1Ot?i~sW|N(1n<kw;+-9d~`;zHsXo<4fl}MGOyxHXDzxRODmu+pld;PL200cqc00bCC(d!RYAyet*%nBoUAwTj>A@JdxR9nk(ZJ65$mK*(1D7Jsb#D@x!rBIF+tTNi^R2Oj+MUx4HN|q#f?P?>Egvqkf#xbsxcHBv=>}1l=GVXX<aBGEaNq<|df{Kc}T_HE^`fGenCX??6YvS(zX|U|-cT<bm#FN<1LJ8wHe5F+`x0e%!U&%itO57$+-w0(dnN*HlvDNdX=Q>eVv<Q}8vb@mTGvkJ-tE3ty-vmYfW4~%8z@tB&nd!vxvMQi%qjdomndn)S5vjB=?~++(z=`r|^)zaDx~yEs_(|B*$c;b{q=iHHR>VOt)oP!VIuo62QSd!a_N%}*YMWFV!M*DQ{PIO9+)if;D@3*+;b5)E5xGiRmq;aDl1!~A@>%d4{bbIF5+8bzy<{r}h#<ghi4QO0y~SR_PyKRl5tml5KY3A$*T!fwjgFL+wRH?<g&ESxy?7BE-_fOF)8%~iR~*ew?CRVNxMhtUeMrfg<r!-Zxb%zOkwT;aGvc3G8eu!jhH#15A5hfofAxkJ_@>0!8u<9GiC6nOHQmFAA-v>852OAFp7rvq?NJbH9p#nyuqY+AoGc#mS%*XRRg1w#`v?`m2!J-LQ%X2G#+&Qa)0q9hGMVSXpiuOHX9^y%j5Ebckqyen85i|n_NI;k`O&8Tp=@1Ig@Gl%{rWQt8@#}IY+Z^z=zh10u-k>H)xuUdPA*f*E5S>9V*X=_)jL;bcf&dbTv=%y{FWH4Ln-i#LZ`gwX_V0yr~)K;;kaGwpDk{+xe3Nt2c!G?Qs~=2CtFb6J5CD!2TnL33Ztt5<M-gtSscfIL24QzLZ0sUsCqqrIcKXmTL-aiSAH)~N->0op2?*svMGuWuM3ytJayXKT^%oKc9+|dC!XGy;$^&i;<<Oa&<VVe4|(fQr*UO*MerM0?FBa6rMtw+k7E3IbhbHrGH1_&Dcq6MN#Ta4C|&n>EfR=<iRLV_S;d<K6dy!dyJ7L?wQE8Euec=C80aqaRysR!eaJ$)qibyaWWe%(?O4yd+s_DRWY1U!w<=!Q9af3GFn2BQ+lGny=Msv0w`jS3x>~MR!EeAK4#BH2k@b-X$^`O@yWW7+avA)O{f#RCdPJ~R?$E$g!!R5OCob@hO2t6{*Npv#>02O%@d^cHk8F0gn(>X*MePLTQ%c>+mM#`3wjmEB-zr>pGCkb$*>@qrhzY?7T~I9&o;zU@Tf*h1;Xp<3z>}tIK!1;fGrF+j+}@gd1ahf((Xsp$)1V2Kft8e8$z0ebu}|BPPSAs6P({W>)1szyvDmf{_lco23bPl$Yc=8(UT)wr$mkmGxA!RtY{=~o8=|I%2_-n;K+4GB3Dc+|KC?>%%0sJb(w<XxKd!m)MblryXf-|^6!lO-=Z7?2XVM)LJxorBDS82QOl@E@LiSiWrVmhZ%<}j+Cblsj&_AEKC3D=0Hwx+0ULc=($9x`ZyTWzS`7GQgsCYj9^bf%!I=5}q>S;R|QZo-AqDF?4#9!F{tM(@ATc;39VsvhJbcgW6e}8=a+r{<U>({J?tXFyUb*t~|7wpyBS8rh}l`nKP=ra{nQQ)@V8&RMVaggDg9f%vD(p|~T&Dn0JJ=N9esrtJ$4wHgE;pmj`=+nND#=3<;HO#`nnjN4Lt;dbXVs;%U>5ePhFl~OKpR=OkuuC+J9frtY<QI-gkO`6UJt3l82!+ga)1Eq;(fI<sqn*PNwc{uOg}`mfsNU2{{!<Izz8rz5vtiHykR_-l(_{LKhMjYJ5A}E~Q!aC4MmIV5=!A}gdASo@Hx4*$GG=Un3_ENp&SuuTuE*^Zvc7Ba$}s#x)mVOY%}Sr(zFVQ){d<AWaMG5Yx1?9rW;y9<Umkp#drc!TTG9X)AkcS#e#AVPKD$QnK@;dNYR?Oqu}x9?LX!z6!XV>o^R{l1ZPYghT_pka@gwH%I~r?&3^)LzR67v)gtNzu>(%lF-DOf8WZXk@?YrKI$IXv-)$;MyUoRIwQDwAHzJP)NZp-{D4m}%%x-n?Bnqqm*)>R+x!`6`K>vKUJIyGFpeTQkOG?cK#H;K*6^F_p+_MZ<z*2uCB_ADp!=v9q3Q?p|4M_x7#T^?rkLGusDuMZ>u;DJyRjQfNJ69HAWQJKTx;6QBl(A4tA%o%NU=M^_PyGO>c<Qi%1{=YYzF-52EOtqqj*d4IyFL~QO0w`SetXY3L=C{oyx9H$gcidLu$hA=i124C|U|9dXX9Xa33Mqkt%CxX^wx<&g6#H))rAZ<%J6_)*K)~~2bWhW$r=w<n`h4JW^xg{R21cj7*!*O8KMJ{5{(h*Krr!Ru6BcOLAF}j#zv(yM^b)<n^t5&(?)-CL)MRkDkwo}sa0&ZX^yDIqj!ps>Gzz_8mnSd^_Ckit^ppOyC(Vunj+nvpCNB2}8G}j|;!6+Sr?>$t?twth<1sC<Q+g1e`Zci;JWCuvI-SjClYaqFO9KQH000080000X0CuwjYe*OX00vM1022TJ0CZt<UvhPBbZKp6FJxhKVJ>ia%{*&!+cvh}^(#<z?v%{RlI<jISE=rf6T4gAX%bJ9c4k+rp(zrwSW~2y58J9%|NWi^KmsIXyWPDrHy>gNI5+?f4$d2L9OuU@yp@r7_wn6FQIu7*6=Al@^0E*?8i{OE##tI9LOzr-#}5mchO2du-->mR#!Fe0#b_|N%H<}=0|XXvA>Lj6e3Yfhb|fxS0Uc!=R!NW#MVg6?%tJ|N#X5^*g79%#ltEg?K?zGKv%!YYNfFeOh%6UTcAq9$5Q$}+$YLbk>4gGh!rNHLWd`3EaaNU^sw@WbAqE7vop>GGND)U8<|?;1b1usuPGMva=2-!_l0^Kol6fqPaUo>{n{ymzFj(f<TKN95Dyv-jK499w+<-LAO4@cY7^t>ii!5dMFiXQK&tbL^4=nV!&#PPp(N&fu7Y{P5$}Hy*o1k1J@j`=L!B>qomkd&FH*tESA<ok6P+Y><1dBut#rt4`y=uK9r2H}n%X(gSAmHDbYQD(xEFV%+P-g2m^l|N-hJ;(fAkES^3=)486f15kR>A4(uYEj_;lR_g7KLo4y!v=`;s12;%jKubj~|BhA^LYf<Yg?Qp(ui-^nt;0Sd#x+ycrAz$d3NovriZP&p-Vrrovg3<))a7$HmK0unv9$p1dzcVYVLAk>bZNi-IzMnMYZE<1mJ^4<9~$KKl$4e^Q8S#D5ab*f>J-vk%LO)4L!!JwEwn1gq*n`Sx$0FFpa%)7Pi|pO3%sU%xsTd?`LImvI=wRr&N%6mS@cJi;p=vc<15EWawm#pmE=B+kOJLZBbcfP%=1K~iLJ%3+pAz>mPtz>C{ach>jkuaP3lSBpgue*Nn7_}i}|`D*$4G<dm;qTn==ub07T=nT#;zCZi<$It$|k3W3;>9hao^6#|Topa_)#LMHMaNfd)(^vTE9W~*nbNF=f`glKR=JelRynlP~6Amt`O(L0#mX1lf-EEvkdw*AKlDOOx(U5l>28G*89dOfy>kw%jm<!13cr^rKV1O_n0Z||f%Z5-E0O<{go8ahZBQ)|Z$YYQl6IhRCl03K{-NfapT2uu{n=CEy3L>|hW$}3)++^t(5sh~ibMDn*oiR&}F)|hX=Wnu&^ml5ieQRf`nU|h}oXOU>a2SI^4?hMGaC?DZxZgc6cqEqsMF3wg7w`Q9<ZmwiMVH<aNB=Dn5L>e-4$B#I4DqVZ=M$PW&+ZE#z-&(6?qRDmiqLo?SZnHnn9d^LzNu4{%cE}{PXvWn!Qv9hIP~h%YQ2zvJ^?X^TaeHI;-r)=Ll4D?*Nnh_@e<^CNdrc(vUuaR07zN@)#J31R!5m{yO6LM*bCBOgo2@P5wT~1$p=&!;+G()Bnd6t3^ev23*tiB0?u7Jmw1umNc_*Ij~{*n(I$3Csr_Wfo8JR<NOS?pzMaN}W<nPh3o;DO#Cz`C@3jWQeXc9`umHuDDseFBE@jXnPB8+ZF4G9-=1x#rRi09xVkMu&V26)}eC?=-c#7C5PD|cIhq3-RE1~c;k5O?CFPK+{S9V|D8l{U<8aZ<q#F;l3SxtkA$YCIL161V92gd2R7O`4q)gTzcJf%%oTtPqR0<>}$wpk1;nHO8bw0NP94;(VE8iA>{e|cWiJW-`FSRS}ic>~hIU<6dJX|R?J9k9j6rl-h}OE5i>WF3^@%FUgBM7xvI{Xa&(Zf-y>X}sOtrlqSpXOp8>b5DFRHE8gAcD{6ePUXV}cwL#9s2kMw^&D^u^v5r(E5L-FH8am^xv&2rPfU!p4O_j4!G8vfgBpbA8!Ig~u2;2pT0@kr0fQV=5V)r{2;l$#`rGQekE3#hOXB0?uk#krO2#*<vVl_H+h9djfko|Cc|u#*Stic@@xAO#d7`*lNfc0IU{S-(C?>UmiIQVJ6fZ!mfNeYxBuI)7q(mcONNXV+v&NU=2e1|e8Z|lEt-)OeC`BYuq=94sLtDUg`02x6(O^vE8qSDK3S>kS*gSS*uCxHd64qvjilW)mt=vwNV7-U}<HRSeyI~u%Zbz2yIxR$Nq+hWKLgW?)IN<26eeBJ#0EO`M;m;umBpia^-vcnuVBh8(>sx69^bJ%Xo;vMC6(^BTrcUkFF;NT~Z=&4`RvV3$Q5>0PS!pRBYJH(R1<f#MFCU+NeR6zy!abmLld9keEJg!J*K5J}n>@P%HU^<RLB$0yAF@<YFQ}rpgk4l<4)RpryjIQ&`HFrK|0dh}psD^vXe?~#25xyjU~ZqwBFtkD)8z^sRy>vw9DSoyYgNj84_@Q<xs<<wI2@C`iyF81HN$1S9lsl&k8`<(W?04;jFTbxr(hP)9z;<sK)Q%*DbB&WMts-T#z|aXv(`W(3vfUuO##h_<@y>B!soWi!OK)MYCgbmeCKgso(EguYRj8_Y;vuOH8>)m_fTom<~LQo$qMQ5l{^z6sQMca?!@XQ!wikH1%5%=<E4fpTyIMDzJF!C5TGlqP8#F~3JT;V@bOm}OeF%5%6k=`zz))qDaR&~ttc}Qf;C;%fGF~Kd7pD8vH-TplVHQsDklr<m@2(Z!6hBSH~<h~62xol0s5y31qJ{#e*-I!d4!+65$H7_I64`wHv;!QOYWr0+KYEPCEjp*Ju%SNCRgIZhzDf?8#a5C4YdRUmoB%UIz~T^d^mF7$aC8xd4qJ@$g^_0k?tMZ%^psWGU)_)Ezl^w6DP`tNyzr#b&u$4p|)Zuc~<a^9S(L4EmUonn6dy}A1}88cqA_4yV|0Gi$IJo=z?ndMw!?r;=8GNfAK5foWZu{k1gPgE({N<&1=A!N?@|Bd6VoJ4eO6?`j9mkdPLAmO<nJ%XZmI;b$56y^VvVEI0rT0H9`#8ZmJa+9%yq#yn20jgmiL^=x!gMVUT_&j@hENY*anlqpQEDC$DKoQbQkEZKbL$G?$z7CfoEiem3Qdn2>7-gBkXMAxS<b8ncH&0edB|(`9^Pp?^aAf<9e=NYy6Uk&0s;-JMuKeQMbuR>NleYqT323BATQ1BQ+pdYOxiu|_b7X@V%)1T+Td6Z5sw#}(<uHe5>?YRFczF{)I8&{u*G>7<0;8x{Vg2n`WgL>Vy~*a4u9V=&-2q^gks{dBz7Brb(X4E!J;lQ;yV4U-E{y-5sbVEQe3lXV-(KnGVqG?&pEy&@9O6jT%|nM6lfRjL&*-_*g~Y1_tQj$$5*;97=dmLnL6^|5ej!^g2fM(bPn>|#I~#Mbn4UdbV=DL=cVui7iWfwp|>qw7qPU(Yc{jwEI@B1iGk9x)=6g(+={;Etq)5gW0GPz0%z4`tU!_e}Z{qR|aV8%NW}F<d&J76#}VASP9%4wxXh>1lVbW13nShB};Y&7duaH=^p%^^R@ZTEQzOE81|&jI<Z-VWz3=%tydG>!KZ>>Ny}pz;3nSQ5)KV$Q>6f)ksU2Vvd3(L{CDaZu|VXAtFT~UbVOfs|0Qv1;@whxWE`PILO><I>*Wxsx5Y3GY45GLna1;)4hc)>o}1qmz4|#zm7kVFYBq;YI0~0%Ii7HZDT+!!cgZ<MXlc=YauAw5U+AE5sKQun6@ymn>l5>lDwp8-x7P!LF;pn*^7sx1`pY-m*|0JRW`_(GsiQ>ZZ#owR_lh}8IqJ-OUIcO+5wBI6)d_;i;A)ca63U3#y2teL^_JbcDTfd<!>@o7ljp=+6R$?!~FGTW)ovCbC#9aa>?g(Jg_fR&$`XHTrp!X^w1PJW(wCtBmI+_PC+)8O7~A~0KM{=Q~O0u7yrb7Bh5fCMy!I&#lEAyIzBo-Vq?(pW)nk|&kYB$6{D=hi&n@K{ELdNSm7WHCP?U})waEug70|dutI~q49vhThVU&Bv3v1?5t$LMVcW#e+5{SuP_U3J`@@yE{2Hz~jviUKbhNM`b(J&%*h~tN&>PY|8j+-^Xz)iXSVt0e!r~NfNGq15EiJrggt;B-(soow`W^`T6*_$8TQMT0nAWemMk`Kgl}KbZ)!;-at(yKteE`M2oinu+wIz$6DYjJ@t{-Zmqx%RuX$2q-#w66t&5;1RAp*mKBZW&r4|Z|QF0}xo+cujzc8)o^0h_dVLU|2`P@0<A>o7X;(39%FId^R;GIc2jy=`c$6dl3nMl+%Xf9eQTD}VBg5LhI^MuV+lf&sXW49Q0WNbgoMyrq(aTrMS;pP`J5$a(QbEQ2_qx&jQ;Xlq-?GJqLNFr7z-;}>?wLWv|JEvaeXp+GYnX5Bs%F0GnUmL3Z^jQ=(R2ReUYNLGnT&d6WlIbw1w=9Jc?FYf^zDo5mWWS3$a4Q(2}wAVV#fd*W#fGt<0GU95}uBHUS<T42`a^aU56Y4oxj!QZhT9PcnV1rH+CkYf+7y)?8IInjJ`zK1-vvm@woh0f(6O~%v$V%8&!NDZdN5{{zOC4o<jybx)#P`}WE2i#Xq<ret<&U}ygvc}KfKeop`Z(9fmUN&ZtdriLI0s+(?z-N824y>U)|0JK{CN-zL$5kV^Bh>-1|14`4lZ%HP6RQ(>x7Tji^rQBMBmS$Rg;H{;<ZfKAKx|WXeXbE8)uq7Nt;uoL@RKcwT6$W=f*@)xqSEICoVrS;!#6;$zE@kpQVS+iSL@^omwU=gFuv;g7MzIw(oiH<JPxL0_R*^VwSs9N5D=i%3dI)yF+7z5qo>4nFqI>TV|7!<N3a^pez$$M{2E5i!ZR)rLjX=(F?{qv`r*#)Y{Bv^VT=~G?&duZ7xI-W9!saTM#lKmcWo0=j})%VI;cNLsxkgRUrnY*Bc-cn$(!f8%J?)lV*T2G^73^swmU<?&@c;Y3c%NHB>Vpu@fgRdl`rOaL=YSSBFk{-JVWihBDrudNcMoLQ&w8EA`+yBgUsplR+)N8gLzfq3fwI_Jn0%Q+IJz7466j!#_7Bqp_&4OuDJSF+uTM(+5{!Ln$&5m`N#%WsMyj%+nM9HK`y)ng39=2k6o&DSINrlvNmX&9^%FfM`258>%-{j~x6KJ6$R=v_h=TeqbiFaRp6L3ECRUZaQ3fLN-oQW<vI)VzRk~)tq&1`EX5-wM(<?L(gg<G56YZ%$n@4wVFZW!I>V1aqdI4R<Fk`Aub&v8Hwd4HNPE_-v(Zkq%3Pxg)XOzaYyd|j})*n{S7p*7(&y{mU;gdSz{x>&&J%rq<P=yx;mDbvgIpS5cgSrD?yj3C4bA^K${yl`$AJ@lVgp>qh7|9Dz?sa-Kv$rP0ZM2!D(tN?XI~|g{4%Q`t26v&oaRK|MFM|LpW}QBftLhF$EQM6HL35Ww=5#X%yeZQH2TJHyGd|d?(-j^{qk^<l&0aiG}iSG%3Kv5ua!^;`+Ld9$jD84iT&tuAy>8u!x=Hj5q_tRZ;3<TPwxRhlhD~K0V<sK}KS2Ng;|VT!BzjX?dM7_v^^y^|c2un+?U2fT~nknM;ao0}UvjE=r7ogI=Pt@P8xEOlk6GW+*J_!aLMi=SH+^GEcju+P0qQtFRuniWk|R*N8Yo@Mw{>jQ}s+w7=}y23$w4r+tOeHk7^`Q~hB$=%y7^Avy}{CO2fP`N%XQELW6VTspP2#9B_YjOBWwk#Hj^1oaV^x?5zu@5U5cF*J2T{YI1OY{$d6I_&8qbAXb4fEF#ACEjEi+Hv0p;Q8ALJx#<O`{_Vx%NX-48_Jm9NL?|}l!?Ryq{^Bk={k5;+KumKgNRLYP?UR*8YqI$HoVBU>H5C4WkcbBIA!+eST;7}DK{Rs)TWjFK0=@IP;kJIxv7|TdC8v2v;#V8Zm2(eZZVu9^+eJ6Kn0t2adzG60QStEpw_;p+HKfFLJU4ly7!T1ZAzMZrfbW@dS)Zj#)TZc7yNhJXv+C2R7Yx?s8ZbC2$NB$(Pc5`Jx6Cu<5C$MJLMuBn9AGtWmJ_T)>L);(c>Cu{{UUlI^WxfQq^^&yX=&Bd*CDPm)~|m3b!-*_c~*Q(-b6rPZbAcdM$eWk+k^tDt*Vkw0Vq5Z2j(}xDL<*O%B&{(tTDbUk>9GGpB3A=-h4(j82-)jcZ7Ps8L*X_RWm?C<}jT)C6|B4dZ_LctML{woiKtb4=rXYJSnBuXfsx;@0Y`RfuIepPIxoBS&GUdo_kfs;;mip6YINj*v}h3d-zsLP<p^)T4l&tfVoS{h&vO(wi`T@2Yi<t|fYS!kHOwpL6`ZzM?H3#k@yUwAOY}+NEoCy<Zy;+g_eNAqt&y{R~u((v@oqS>8*yQYWO!U!DR9n}7xQtmJ86oV7P$hH||xO44Hn6LE~P`}t5yH<N6}>|cuSL4ekn*zo~}M-{=BZ^VsEr7DJ@Qh4p-fOU+<el!|b;qC7GQFk3YXgW^2ClC~xT$66H7}oGG6u$lXOM#ET0dk3QL&Y0)aj4gQ8di%_wy)_RT*5E@OT0d&?1Ru#;x~+9GFtCIJX_;}Yx9db0Ins{+KzDT$2$X+8OY{S3j<}*A4vJNw#EnAdWYgSK}N02c=VAuN82&bnx&NlQ<Sn0AF!1>C=88fkAF<v%?1ZdGpl|6M26Me=}aLwP-SJt?$AoGcVPxMxk=T|s4A*{X6R3BC{@aps}$T@Y{9?Q8dTYyCR2br&>YB%7xqGi;>C;JeeMTOn$p&J*k<Z(aL_icm(#U%A4v(+qr9GBW}4s-;J(lQZIp$MKC1cW`u5ahX}Xjy<@A81tlH0h`K6RmH#RY(<Yh|rS^`Di)($;^n-knp<*725kk_{&Se76qgAG~;K_c!~uuTIYy*6U#jig@}*aSJ5&DoNQsFENDc2RLE{nFSl#zuxwljyVsR+#nGAs)04I{8}HstzKz(q&3vi&5j&y#{(6qiPeOoNgwp%O139pBaghxxL^g-&#*PH5#z-rDH$yb`B*T3=yf$%QU1NE;6D^*%FuduP=|qMkq-LD~;78u)J2*oOp4j6PK6@Gy~ZKo7LMPeKouV)pZF7c-K_0{0<k%K&y7(*9S-|pwH?J-4%SsXf~D{u7iBbQ)H!FWZ5kS`@sTG^dS2n%u+CzW2_iau5d0F;2K2=-(uU1;0sk(sL_5^D8v#gtvC7!Ua#nDJ6?@oX~nLZ8hxzar#gKcYZvLa<(?suYKhY+9Mct9J@=tqMAoKvUB=Y{G!J^xBfpz-H=ay)ibtLkA5#3*8zI{FzxV8YUSN5p2TI~PWK|hRyW3C|d(&akZ_m|Ol@20fQ~ySc@;U}O4r4z^e=*eDLha@Yb$n-+$eSSCI=bc(xO5pm)V<E5nT!uvz77&BsHk6&onZuqe(XU#%VDjLmn5so+9Ky)R59`x=M|<7@I_hkP#cYQxCUHR#ZjLsDdP=XOTb;wQM?gRJ(4CQY<jeQ?=})2GEj10Y{G5I@tV=O(d)P*Sja@Zk<*TX8r8}Yn7;wh6nqwzf+G7KgH*OhNUVD3sXc(}w*q($^;f+~Oh_7B&2*jGfce%MiZ-bgav`ExbG6Eg&^9)TSk>D&RJFDdWWA6|O*%Z|QT~PgRij#0>-d~@ym@oY1E}Zx?tt-}to^XQ^R%oF!2DeTq)H2w_eY{lot^59s<J4u$Lp7Dw8SwT+M!@I-%c*TBk5<}R}bqpso<;Q>gXypyAID8jhv=ZnPBiYbBcd!t8rbCr(y)08x&eX<Fipzc$llhCrN6+V{c9j_8J9I<Q`_mHmMF>F`eO80?!-t{qo0Ce#DdS!>xDZt@Rv&u0iQIg2KlKoeyrGvvJ@FT+gp1s82mGkNjK$mnyqEhh!tJ3+Ojf-{yN{F7<_Y2*n+hqVD!>hvz^YmRbLNPeug%F`oxMQl-?YK7Yht+x5_)y>Dv;=JFqVzh^zc%QMs#K0c1isJ?@eb9UV`z0<3G12(Shk_la;(D#!Lpn)zpTLgL*|0l;AbsQCa){atLP(bRp5wWO~>ey~w7RIkuY|Ukv&rd};TRM$vxXX|#gw0<;=@!2_?q4R9@nom;mGAw<<zGti^iQRzWV{jG8(Fo+wwO>7kEics*q{2f6pjrO(TtD=7&ZZhyKoU-QPWrRFFL;$gpX5DFySw_(Db95JgYYDiRXDezo3B+nFjw0P)h>@6aWAK2mk;8ApjE=mONMt005mO000*N004Ahb6;|GZggpFWiMo9V{c?>Zf7oVd9_+wliRixe%G%+`H+$cMejP(Oh&OXO%vPkG_gIlC!JQK;Xov`VwNHp0^Exe=ePG<01}k6>rFe^hh2&Q&i(QoXrm~4yOW(N#io-@u@mY>6}_!=lfq+LS4Cxok$ZS*E`)4KQ8lG%75r+fXjM1wRH3_4ym|RNo6TNoJf*s756iL^+ui^vvDMwaucfGr*z{Fxfr!<@?%?xYmwl}=@l&O)jQ~kRD~n5od}dehZoZd4>rMa*)unJmHKNuRmF2xaAGQK5-6&C3#&*?)q({+2r$B^iFE3O!d$&`<s9F`44qO5q;J)S4>PmID0@(?@1t!w+F(8X9MWb%O3Rbn*EQ+GpY}@I*$n$M)`%dLKY;JXD!90z&64{&C%)Kk6l|?O$QO3X8Nwce~jsN(w(T#t$U^u33Z(Fpud;hJhYq_aaDt?qLE)4Xo?uwn0C~G^tt?J;Mxdoqgx~bl)*=$y-tzdWMMyWE!Ukmg_%IfA-nTox#J6*!#0yGAb|I&kP=vgd96eVK*H6B^a1pKt}w$`$Q71Sat`+aL-TEU-HL}ceoa*pigSLacR=ZSfeBHh?CPXg|a@j4x-sm7qSr75av`ApVEjcZ1=vXfv7vy7t@I7Evm;pK#WcM4q9Ama>8m2YmXGO_bSW_I%N-@b^QHkoP)U8*=qva-5R#>UBN@#PwbZmJ6~z$ao<F+LYho`^3#2eXKC$8R?Kd*JnKc#eVdd(|x8b-hYv^h$iEkkzY}f`)Cnmp6H%u5$`fL;eB#o_~2pt5)qHv?T`KLTo`?TFl&{Zu9o^>q{o|n|O(-vZOz>Sn8be8^~?6u3-t~qa~#cO9B1>L2M7ZRunE-maSoD{5VZq9F)UOFER6mkhHqc^$<bdfAi+YS1(`u=jnG5Z7?VupEU}6-V|!!b-;1t3q~f{A)xcT)yC#kQ`tO+sH(TgVsKBj#iZY6Q=5t>;v6c9fNT|K@PW3F?ZgwX+IXBIM^;9OpJd&ur(LJJIC{}s$-44Yq^|Z*B7qta$fD$k`CwtHpUiCZBC<UD7|XZ50ymH~Q2GSuT-oYMh4FBTw>LlOS~cR==!d7zzxnpxm?D8EK72%g$twOt--t%;p?Weo)PE$0(plgPo-`=*AOuw04t!iezsaHr&AYS4CCW41UCXX~3-wj8u-pa~TmiFp;8)$bU^I3i^yX(M_PkRy7j{Q+`7t(}k8hOeYggie1gN3Gtmqd|TCd0Og+^mAe|ZN4`r5_5f^u9#To<}Q0+g68a*@xel~&2ie6RMpyX7jya^n)W=)BN9)ZgLl(6K4ST$$fijReL}>!HlCrBZNGb%F1p>h>-62YBTw&$JM^41!+w^hOr8zNP*egxXiac70P&uY-bRi=8smDWNwt5Eb7)6W}e_Z8FYO7baj4qq-~D+(-=Hc?o?^IhgPWjAd2#h);6M&w{WbLeTIb7+!z=m{k<4A}G?^wzr{-ftCatz)|zdNBA4sIFfqZ>5be#zqGfakaYnFg|r9DnySREe?0&gP?pyab+~S$z%D3B=8Z*vxqE}B45A*iz=dj1BsiY2Cvuk?zDtkk^-j0{{DJoTi;R$lE3c1NTpeU1Kw)J1Sy6+(;smRJbMROQmQaIT4G;>Ns6Rsy7$^R$YenV~s5(2wK7NR>dx<%XwT0RW(HlC;$GBrPBn|<Iu@13L8E#NzPI(mnj-OI7=ajFSZF0zHT5J8=u-yKI`r;uU)RG?_ojP@|K0zJ*b9I{ZwLwa1?vr9VsB{=k@#Ihvqjag;p4>&e#devU-Ob|loh;f@EF2W$SFT*;#ZGIZVxP?}WdP5mJbfrvl0Y^zt236_V%zt}l1w(Yaarw`^K*Ev>o`CH+|r(5kV8v)t)Y6JXl>E6uo6Mz*dnTT!@zX~*c57v6wDTiMpIVMK@>NMprs|Du!qi@7$B3~1)vizw#Q7aFjZt@*4F5)#hu<p;fd2`nb7%PEdma7>JDSzgzi_=0%=&;C@6Vgr>?R*EkY$_s8mRbiX#g-Q#b{o6<-F^!$c0=0#hWA*857<ai_0SaUR6Va(GHsXY0vnkW0}1VGDwx_f<Q!Q5bk*KCDMvKP0K@5UA#RniD95T}S60t=tr}!CAH`Q}5P&YaNyjKnt(p^a%@YzFX1k74CAA5I3a7I1zFK;9@+!c0iw3Pd9mY5&@+HE{JOQ?WgGpXo2XRhw~_%(HwY;mY@qhXA>&I49@X1j*nR+6#RlWfCa#l2p-+gPb2ui)nSK!uo3ZQL1XuKuqqzy>F3uCB9UbNbqE%*Boi4~UV&>=%rn4hk)nZMrf9oLce%m62S+pU!is%wEG5^z$9GH$KBN55<`HB9RQsy|Rl)C>c+~(b0FW!|IpfY$^bO2MC8);|Gl-Byp;`+-c!C}vkq7lh%+7W~%<k;};V_NEVPPadvTxLpo&BT$+(sj2P|Wt&9v}PJh(Z!SM6cewcpiCqI8yHs;OR}v_be-Ni%A_5w)~o!Bn+rEpk_;`5y=x-FBr}igl8ZuW8#;%d;$l<`#@@#-CO1J9TWni)7^qeP!~P`Dqw54LK%kx<1$tUS{t;>=%a~~<UT#$@-T*>=`N5AxSfNTOuv&?oCZ*P#uV^Mmg=f1)KnXON6TX=f-Pe)<eKo@FUEXg$$t{S7EO=HAMbaDYnFsm#7ESStTP)cTeWqfWCX?$(iD9uqs0MFe3>DDw|O|EVCEPn1}hfaI!)hO*<n~L>7FYCgKT-20oafnhyQY{HvxpAoL>6Mn)zZdDffXvI|Z`Z7^8xMDuJGn_{v(~$`E3s)492xB?sMV;BUeU6a>AC<Mq{{gkiZNek&Ac&A!1$^bN_lKGJqzc9&@nM$OYE7IW`{Fu>R$WRGnhS~8~W7-qzSOO#}sO$JmP+yafOG~z9p0n8Q`!Acmq;cXoP4zmjcd#^feygVU6ys}ECgpy$9Hy@`!BAue(hI({mG*vxXi%+?9_-nF(jip{afZ)ju+5`F0jlyshb|Uv357_cRiXH*M35+o>+^L(l=>^g+L6z_~78pSHWH9luszpWsg)UTlNBB&JnU5qrmHykD3hPq*dUP9wS+n3|aU=E$zoxkHZW0R-^NY_Q`~BPX5uGjJLfDrM38|WXPg&y1$K=e1@p5`n1P<h7@c=^aopeLWr}WliQR0xg5|Vs49#JQ^I*bHk#|LVbds=a~ULW7`Btan2J?Qe@!Ru)LJaL{Q?;606ihb6mrwf8RG!hx=4`4BhnKco_O~W3a#-A-k{JUq6P^iaYQPwS93r`#{0zE39YYZ*O#}!Yci}hq&oSejf)PRS;^L0F3q8pB&qU3dTIck2YVYtMj^rG%NN$<rWW543L<MlA*(`Sbl(}Uu_9cq53fsqI{fkPgPVU?`p*N=$u40&iWNb34F4{A0^^@$k#6=wQ$Gr1sH%uXgD^~34*MmK2oSVCFF?*{n06aaIX)zrw7V;i|T&J_VeEW)ZxkYBgsKsw0(8!HeBJ&KDXwD2VktqJBjUEmI?0Z4KA2P@to)R_)2$H*obLfzGn<eu_$j8l&0BToxU6r&VhtoZQyUPPj5*_yk+9t2_7PlAhfK*}i95V~2>_Hi<}mstWYztI(lQ`_9-LO7%bw+nE7wcyk1@qU0`_<}r&l#Wc3;5$hdTjPrZ92R_g$>vB;VQLH<2c{vwbb)$%Ti%JzxfGqHVX5>GcjP*JoByCY3@}5X%AAzA$+=feFEYdticJM*YAcvJ_@ZFENf4i=qX0`E<FoP0g>Q9@U&#9}^dT22{I`Ot!2cW|QN#|Y>5i_M8XU5qr)`X_Y>&Z<UfXs`iQtCa57Mre`hhVRB-8t*$A?TGQ%n|E=kzf0rwk<19<%;a2NBai=<c*7V(Zyajb@mK3Wbr75Z3ymC~x4E$rk_1<k*zIo@{ji0yNkTJPMp3z$yBJS=-8DhhKSM?Knx6^n4%~oC3nBgHF`n|6ft3<U{pfyva@FB;b6oHsDHj{u6QJ^~J%h?F1)I=gaAw34RHZ;?8j4vUko|b3`f>#WnonO?MBZ2ZQO+y-a$CK_1djeNT21lJi2?N&WW%J~C$i1yD-^1QY-O00;m803iU2hm8Xv2><}58~^|r0001VVRK({b#8QNZDlWTWpZO>aCB*JZZ2?n%~@@a8@CbuzQ2OleCR6GZO-R5A5;j4oz_7BySdtFi$XXM)QXar+g)zR-Ln+K|Gm$U`~G6rXg^ed&nI$bI5V7iW`-Ar;m>Euv{5IOi<G^5{o_4T`IF2`ZLUn@2llU@UjOvX8@*SleD~rFD^wv<l}q+sm1kyTWyL;~ER&@;h*HEs@V83ZQt6zP@=~r@l1h<R#hM*tDHDkOhQu7?QB-MZ-4X5aN0Fv`kvuXhGm)1nv4O#FDrfRiq$*Km%zkXOOT}KMmgzjbf+2-)=ySr!_D`%l%PY&}lQay}`1$&^_{WRi*&q>xQ?4*ND<JM(nne1gL?mYkjf{i~4ziH=k(W##Ng6v=*70fRagBp841?eZA&m25RaQoF&Qw-tQ^KZPmx5&4nuc7Hq{3RruZh~FAgEXJDl6b~%krXTi-jrGQ6y!{WzXX7kKghmZX*bS?>ZswXV03q(!#%+fZ>w~`?!JBI6nwu#MLdT+?FCwq@C>S^<`$&Cfdv_Hy6KQ2bGk8Tf^ltM`0t}%W>tkppTL8RL(xM3-k#|zi>l$c#P@ySlIGf$Y`&1x>^ruTeTa8cHvW8&ZLIJeQBs+&N0=Ze113P@dh_c{@W+HK6>lFb5{Ysob{n~2VEz30Tv~xU9%^VR<h?9@2++0PPX+VW6yy(8mQ?Rcix8&>)E}|XA1=0Jqw$)A4a>4qTT8W0Xd1n3R#@VP3s5FRjx|TU0|$PA<8qX{+iI?NhNX{Ce=Y~ZbRUP&W{QSh{fCZuIH*c6956pdAyjUs%Y0-3YA9W_NYxJO`Ak7L3M959AdRuyRWabX>jXBZr@9z^3zZdvdM__N}$xpA!z)kHtAtVnK`iJR%C6~ii~IaAX82NWlM8ivv(r9H<FS@n<2l%&GS;&xCLU~U}tLqaJmJ;Ki8W<=Zzx+0(lq3<k47JE}n!+39O@@GhJv2WhJYx>MzD<GYm184LzJRRZghOez^ZT6Qu)3d4W}M8DLc=%6bg`9{c!S2;;EO1NA|_&yhI?IFZmFh*;V<9Gtgmzq!owk+Jq)x~M>gG&|j5%lQ?1i_#>Wa$R}Rm0`cu*X&CV)u@}gM3Hk-nZc<}-*~7-R~5JO?vFXo<$Hwo?CxqoT5Voy%@SGB#48=&kKfoq3;c}mL|r}%@6II{5*!@JV3z6xu|?Ixa=MP~&!&eli#od4#lb__OL_BYjb^lRamo*btXm(m4qTso`#FV&O4BDUb>tm6TBoFnR!gQ{b*f=so6r$*G^(|vz3^eRniKsMqxPv#Im~2^^Nn*=-Aa^aWsw^w;Ou+fS$QC4A~703>EW^}0HmlbJ*n@iqDZflP!zz<2@DdG$3iQ#B;rXcWRPHIbwJ#+qcXO<DLkA|sx?^_PtEj~852_S=3ePbO!UL6>N&xYQ?kGlEm$}C_lqjGuS9Blk+i#g7`vSCa61EqyYLAW0fM8>!fo&94o;g}7$`Z=W$AwX*?q_zSt_zggwQ>-&_O-xtNZ77YXGam32H{!S)@loAGDc-d0qDdh^#A|%EU>!3fG~z2E74bZU3rs?78YYTwE%5T|02v!yvRF!n!&gxpQK#OBsc4e|Q<LI%CdKoz7^+->U<<_ARl;XeA4|+uS{LNA)0X<ewoPoMg${Z~o|Ex}g9~&1FaiGHyVL4juuHK!~OcbLrY^kUoq%!<rv0Tl@QZfreYuf<%HeB)AU$8|}lc-{%J<PPqmiCgJF4Kdo>#jqrL3;eHB%7A}3p9P%WRD{qFP1*)$2HLYs5G9twgg=UD(brl)(xAASif9*M77?h3iu4Yne(fVgI+ZDzh)1r|H4!zgym<I?DR_EJL!U@+dqQYS?)vfP%wwaa{q{D9#z#2t^cwbS9O=f~2!|j^==R8hb$MXl!Elv;v>f6r{PW$O$+%I0q@B*44bK6Vi-?Lq)_rIVG2_L3e?P>zLFkZ7p_5b5LAeR$<w>)*ZiE4zSsN0U?k3reJw$hgestN<C&LV;%C%qobF4xO7PKqj8#cm>whMM?sWXwcoeuBx_#HnVE`BFP%YbHDtq19|???NPP0h<oor*!xLp}X2^J9;^|JfPJy?h)aFpWW#X3lhnn9Ctu1w3qi`5+INAx?uobQKB?XZUKFuFei*sAX~e*SWP65PS_@mrn%$cF+17c!u7(Yu9mhkaiI}qX}f_$o@aK6hDzzuo}tiPl3kssW1ra`I-mnL!tBj#81~BPFxSxWxaPE%!7;*!3TSlmj9_3sL_v?6uz<nbOB}+*m&8}U$Uqbpzh<4-K1_70Y`bN*{}X`DHTQ1>|If(mGrGAKkIoY!e>Y!_zfBwIpwdBJgeI!z_~wm;F00G)Un`NSQjM40<EbBYRDP6(z|ytF<Jp{Y7Isn2`e(VR;=(>6?~$Er#v6bpMH_sc%^L{Y;@{49pc@Qf87@wdFs`ezsKy=rd{zxCk3~UmrjgT#27vy9+?20%O~dEXB7z~{?a@1NKACV(@))%TPP@WgNb^N)IWmYai_)h(Ylg$wd^EyMPLA3u;?9jlevQOM+3oEr&ZNkr)oMwGEA&nPbx34&PsTFau}V|NHR5PaMK1QamvU~xxNbtj1<tyO{Py@&H?AZp@Mip0TMr(m<TInl<aMM!QTL{=LGye{P_k2cv+q^v-g}kr2>pYppTy97;@VI0X0I!HfWE>LDc}j=>)-hcY5cSm1OXXUv>xKrjg$&qn0s$(zb-s7)@J(uQnim8j|Gz_x|VkGQs@0MdG8J_EXoAU9bmudbi8?b)~jKm8<FYpd8Bc$ILdJUrvVxQy1x-bQ^jKLb~s=4g$-W5=0(l%4Tg`ox#M#)hkFC^Pe0_7`rW7R2}<bU{`0&3zU!S@f@Wan^&gn{496EuVqpQuK~T|M44blCWO6eTC2{>45D~#@g66b3--3lM{%1Owu>XiXx8Hl4o@G~AHn*+;yUEi{aV^h4zNO0d(Qka5rWKk=-~DrXszD8^qqNSWCip{7GasKVJMb#(WYY{^OZyo78&FFF1QY-O00;m803iUiKS&oE4gdh%H2?q_0001VVRK({b#8QNZDlWTWpZ<Ab97~HV`VOIdCgjFZ{x-h{;prKVLwE=Nc3IW1_ciaa7~*wXqp;Lu0UfL1Vt__#uTY?N&9?;|9fY4_LW@9^0fu(4|gItJ3Bis&&=#@6h+^+c~-Ogw!5vG8&*~|7wo?3_w0^$Rax<3&35h36ue;fUDd<azGX+=iAwam>6z%;F1z7z6h(_g*|i6jrsdEN9ZyqM9gc0+v#e>_KI^Nt5sQU-_lsy7{ks)3@|gAey4vYsm++`b#D3_jT0iw!hmx8{emJ_%A1it!u5s4&Rhi|z(2SQ027m9=>pyl~+pXoRAC4ct$?`p?r@wPC)cyCoxZ&!1*0+Z$Pf>Pyk!MZYRG`6hpNT!a6Z`DT-+q;%1nb3Wu~-zmWIIq{&C??5Gr{|%XooK6Y1g*>+R>(bu}UNxQ0QgHk6FiA+i+H+oi;@!Zduml`|6G}(K6CB`_tun)^Rii*eAPdtAh1a^T7VlAzjC_LP!Hi@lH(5wo>V?c&jzN8Vf0T75vD-HJbbZMn1|{_}63yp168&8L|ucePyW2wv%sI)v#rxG_*NnO;y5@VpMjtW>IwjHxnCj%UpUBeIrg!^4~h!@fu|D?Xt9T5^=|CU;wI7>ov2InMAY#Da$Bt^LFznc|OJ9bEH7{al=tnW93~KzG{-3S4SkI5iy2UDu%MGJ}e_w_Nvq*`~xr)%N2x7p&6K$iBVbAlr8W@7IEEzb(fkJZJqY~L%&>E4FmtBdC=oym~R<X&2i{)z1yv0O^+Bwu>(Z)krTNRSwLhjSMXnopUg-{T2cl$Y9sl`_XIRsb~W`3&mpW#*9+||MFpz`Uw<Hv(VimEOGMsw0A#5HIo5e1M<^2D3y{E*6CucZffM3OIhQZ2kz*BiH+8#PMz5lkujf*ZjVnQ6u$m}VekPv)OWyMiY!5R%M#k^ukD7iRZ`A*sejJ;d=xM?eR!)3Ti7}{8pyz?hX-Ic{5C2HD67@Bd#y@ey%B*|%gI*<irH};jy;Y^cqg1L%MU)}KNYdeDW5~`&NFKLf=B3LBmUkb?8P(flVivLns!IT>^I|D&I&wTmz&I<4<>aHq#C7l+3`RX}Mc*m=$2!c@zV)DT73Ki?8vrTp`1aeZ7J+H3C5j5%6d}U}PggCLG;fa&skZ1ybrhOw#brxe71j`RSg?+8jtJly)~6TdEF>ZxKtNpFF9-!m7zZ3O1u=;e%WDU+ot&ysY%QhKP;G&OQmIOGI+vz|+2aJYri`h67ci!3BpxW{NiY%Mrz_IPwZUdhgkhZRL|YF%gfasUjN&*#|1%Vju6tn}u*)jJ{X%0Iy>CQTa;C&L*<41f3d&GK+2v43wfBvL{>aHYGU69g?+9jaXWW|k{-gYcw4of}TBm%YNiCW4YC;nqZUH8jSjIHH*#6WFFquqfX?rUlSBta2pql{$1cC!U?0AQU(2))fNiV&$sR3RKP9(8kj(92FUG2p*IZP6(tbPD~73WF>U?2ZfQt>@1M5R4UQcLd&QQ?#-3?Vw&`VP1uZx^UsqZlH8&H!}2#DmqMqmV9R@_=*Q!3@@EWK>0=$(HJcEET>0O%X6a_8&<geM@@=ltvw_MB0S}mVTu&fSr(>FuLE7-?c*zC^pp(^(8Te8?tP7zbrK?8nR};VXxSm*I(-YeA!w7Vjt@)=Qd?({9cU!uc?oI<KPCKX&Sx(QCP-Geh)Ry2WZ=V{_YYwp9Ug7D_C2yJ>X@>3xU5oerS7WF0$rkfC_1Yjv<q(Or@r0P<@mDYnvttT_9E3OrhgiOuLH!Rs=eW34Bbe+fm^aX)%SWWTs^=!h%INl~yatBNozPXlM@}M5yI&nLndf*q=a%37wC_#>?jOvw{^>Y3MTe(Q}&UI5f9?3`&pIfe&e)?Lr?jkOPcdq>u@6e(bxp0n!`jZd2aej7B^q(-=~Hb^LR6{1)R!x#?_?7FY}P4UY6lxEkX%RGi?s<Ko;yS)J)Ze{Kh%10SDj0Ga^KI<gvtlav>r7#IZ5TjjS<6s(o>7nlG1_4wOUkni>6El4q=6JxmlPrYD?2MLIp2_vvp!$maDKzAjhhGcF?NN0lohe~;7@p6%$D&iOmmpZ_`<hNECQf#%*t4Rgy!^s%vP8Y^utno_UR|a!_XdA9@FF@(T^(<14+7&ECrn^3-j&>^$Z==Kg;I|PN8!A-%4~Yks&YZ4{a#KT!7~G6-NGE0M%gPnEOmVr`TE%^93Kle&EmuaR#Bx%S7)844bsFu@rmGE=ZzYuo?nW9bGvHMu?9q|)Ny!y1Q6|0+P^w?P)N&(L^)aAUZ}(9?SI`{xRsW#;a+cC>iQq+GlM>(hK04YHG?S^Cqu?L>Jw2QVLXgDXNCJvNZ3cu3D}xHxq3(s+vRV0FP8_D2d7{aVVlOveWf@a7Ih17}0r1t|mU6IPA*fGRkzR>hS9BfngV|F4J3E;bq#GJz3wEf)A?x$KTh}XFyBp-OHZ%0_Z3i`*V?E7l?Fg;NbXytV7~)_{;**M^t&+rA*6XAl63_Nd!D&gRrKQ*Okx^-9G~e?>7Rfk_pflPi&&i?Sz7?ChH+Jg_aB}R8$j6!z)PihAZ<T=;qh4*suy<Kqq1RahlR(UDtt*Rm?Y-QJwL7d*WGa`X0JMbYI#4G&nj)BlIFRIe8zoTHD#Fh~qr)i#X%$ypyPx6Q<V|Wd1aThhxh{>pweJIFvoE}(uEoKch=Q($h7olsw}$WmdAg$K?a=fumt63F4ZHzu2SX7i#HJQX7Mf4{sFlV#k_ozJoX4FWYkRDj^7e47dCy~eXdxX(o>ah(tIeBh^c0@ois3*SQPb8LBux9|1(UOVCFD+j(Q-jME2se0@A=e?2|}EmO%qKCo5a_8N=D(8%tW0tK&vs43)h>OgiOa_ht;sjc653eZq+Y|mO-+3U=1|vit$^P4~IeROwt)1l$Is9Evp2ptA=NCw~KZdY@7Z++MDwbbX1_uWk&_67AVgISLIm_iB6acw-*>U#0q<p8Q>i?!j4}MX>2l7m{5Dhh}t>y4fw|E(H8}x2>`9rW-sgkfL?bnqlcz`Q55fZ-8MIPdJmZg;-|wjuVY6d6Em~WB2xKwB$|N`_BD+<ErhRvOM7oXl}Zv^vc1}x>LJ|ennCwg*Nq@}iT~v-EW8<@UXE=doy*9MoC~)J_0}h#lK@y<PFA*IZ&p{YJv1|1W>M{&to*Dt_8G)_2IX5>a+wYl{=H>`pnCwM9d$l@u7VI-w&yK=<yN;aE?G>DaCK;&EU#gxnp<vhuu@O4d88>nSI?hcC$E23JZ&Dyo9H)jy-MDEb^YYPige{L5}ADux0|-~Yi0x*)iiN-#A7LIn-R3A<1yqv3TSgO<`yh=%GH=$Rtm|)LB}v5i-bhxZ<KikE5<&%WlppP9TZ4_-#ATAbY|_lW=_{Jwf<s}!wiC1H`do@OoIEpJep&W%lZn62UZ;pL!a&7*AbHj9?FB?zzBJ(h%T-`TGtm>=)zG<JIn7FRv-{cD!Ck>;D>C5FC+_g%O510AOHMrqjy1Y0|w0lI*Qp}X>Q5w&=$PD_{%?7yZeRb3WpjF<DlH?Dkz;l@*-w`!^|uXJL2Z}QC%89QWbkete3h_*|lUTG$vJ_rc1%=(m#froKe17v$7tbT)>$hwr{?2`$8BV%h6ytty=dO(5H0nWNlAe*$L3U<WNQq^$LRV<QI72>sJH*K(8zql?EgB;rc)~6C>J5p<s)JQGEcW+>leq;($dUcUN)D&5ut{i|MhvH``3N=QBN{#(&?4;dqoc7u*YV<jgcvG_hGz0KTv3N&6rk=Re^)cMn%mIKG7tY@e*R;$p~AA$}b1gYI$Q+Pu=~<hqf_RSS_-RGZ_TVsNKTQcW*8j8cZj8%^?{=U0HH&2%kA37neHAdqHO68vCwgyw=d<S^M;cEzY+l@z!(yk+7l-REwPjV93`Sv-GgJZX(boI=OGGmWyloK&~VpLEdcR4D4q2$L--<(Ymue5kMYG-r?VKlNDOn%Te5F?+;+(ikr}%6IG$z9@2PMEC;9S#!{(F$q7Io1);L87)0`vOj^N(?|T1Ls17b<_=K*YgYtJ0QHb84lwzNlm77RGj|NkP0hJ`1)iJJ^e?|aaOxs~Vo5{8bDzFHFghr+=q0bh$vAd*z-&Gz+u@rD;+xxrW^;RlM6&jEa0p+fFW9xav`9nFr*PmIz)=7n%2y`Yff{YH!%I(H_8{<BuFvo?i+d<Gqme<foDN;REW6=p7}2@8Wt`@csR305a7ClBruQx-seM0}bj7~jPO5<bXZdwG(I=JqT7g7_^s8j-6z-&<+&j8@3q^}Mr3iQrUGtUq4bAOam%qNaVT-kyo8NRL)X(^-X-RETn_lxi>q;yj)pi7SbGKx@GGTg0)=GA63cWugg$8vetFhS(`s`&{9U1HS<9_GLtSaN33$r72t>*bWdvVsU(yf8v-_adf_bB@G-JF>UPMXG?o||bb$6v7@LAkZrp#c_dE&o>H7siU-eiK{>33B_axdC9S`gn0hd*fRT6x316_laGUX7HI?YM)mg2F($@n^~UR+jE;6_TJypLWPNbC?4YI#0_Z|8FXer52p)jllxqL>*-*tgP@O-=~OF_xTc>WO5I%`dJK@dIWRGD`-B$D-0hf=E!WO!!p&iv*F5V0$o~ycO9KQH000080000X07*V?(LDnI0G|i|02lxO0CZt<UvhPBbZKp6FK}{iW@&6`Zf7oVd8JiLkJ~m7zUx;IJV}79Lvu6WLy+B~hhVcnoA$6+2oyE8S&NU7l(TBE|Gh(zmMPmo3)B~Do}Bq+_>ofj2kU#{g#^M>M?QUh_(&u$H&~FMyd&R)KM;5`z2L$VH!6OxT8Y1z7g~{452|BUfU7B`MX|G5lBU@W9uqVT5z=VuF}2cuif~1dbn&kjZ7spGj(K*rn4bBaw(=~yp*2=>a2Q;4u*<4|vV)W0uzk`Bs!c)gm!iD3{7^*AE=n~YI#vtUuoDwKYkMfGsqK`72uDI9qAhXWCT&$a&#ZR=iIRpCv`R;XUzYi{t}0aw>Lkz*AZ;X5Vh7HbKdjd<eq?)aWJ|ukPa1^dtR+H`az33-VRUt+I0=VfFVvPP7f=Q3mhMLg1uwOXbLhb`54lzi(u}#{S@gaBmTQI{59&uR`0^AX<swt>j|)4z`L@?>t}`RoTzjqiMJjDT@zdIj{xEl#kHd}^AqVp%eY_&7x4T!L5CMd&OkUNT1Kv28d+S<1N7pCqe-78+(OPZ){p_${#Z9Fa;>gh9vhFoM06#W*@MiGkBJ&H~3r|;sV&K{^>mUffJbrwjuPYMZ-2P?<Sm7RbVC^<b`tbPs)7Q_R=X7DetuSS5Ikj7U`E}}Nm6!7gxdOW9C{5<-ZU^^U?5seL(Y);BKeZcM#!J)*wsum4X4mV#2C(CLZ-#Bm-J~H#Ygl&Xx7?jpN*{nB4v5W7+6z`UrXOOrbgdTB0?msO2^6<Sfy*ra6Y&|)m)A)bDaTzDfi;Bc;4N?l-v2$68Cym7hU4xc7v*^)8E$iG+k<aX0M=%}dgT0aN?u(Bx|$ihqup|X-*3>Rib<fG6jcZ5b_7&{)OKvz(Sy4<dwfp%Cfe=<ab{R65b+U1UQ&!?e+G2Vl2F=St;74(WGvt_rn1s$Vx7(WM%7Vs-n_EP2`u?6+|$R>G0{YDOrO_;-c0Y#CXFPWHjO4{)#-jTaW4W`(?{ljSLFg?eg={WE`pTEDVzBw=QZp;{DcpemqR}n(PF7MW)~ME8fz9FgeJIW(deL5(;{ylB(ul68I-DLhZy=^`~y%+0|XQR000O8001EX$FhJbVGRHPsw@Bi761SMbYXK}a&>NWX>DaMa&>NWX>DaLaCyZVYj51R@%#P?p8BCuqA0en?T0M{oa;E(;PS%7NegTj1TArQRqJ)4w6+uEzjubDBvPVmw*^|m*j<ss;c(tFl;?RLN>1{Mr935X-?A^~Br8SB%7ll6-({lB$~`Hnl;<SnyR6LGtSSXzWeU@Eo+VieD|Wo$u!LN*<c622h&<0*t#(aS5K4Dl+clh0k`;B;w6M0US|;(VR(dR{%9h`?d3N28s4p0Zx;D!du9~$680isxgvS9xTQ$i+?-{YS<t6UWg9_f#VUTJvHE_z4D#c@J*w<Ba%bN5P(8yP-RgyCy$lIJ9*<aMU^@_kB9PvD5b;}#VnyLeav<GhRtjpUB7{_u_H@s$zJh5aP;3}Bp)pb?!XeH5|=LlF8bt{0O1n1l2Ko0B(NPp`9P0q`Gdx*$8CT>6i4JRzmUmRJUrGF-6MGje-@=_wJS&AomSyw^|j5It01~)*7;ylR&uC8u*8j)YV5P^p!*F3Mvy#QXyvm^k)o2Wmnq7$?LWQvMFsV{iG3zd3=<XK37RW0o|rFpf_S`lA09S;=G@DJBSN~2=6)ShzT07%!cK@qB^IB0^ZuC2S)*i*3vu&5cehZ>~v=mTW7Tc%>fXW>*oHtUeQIE9b(b^sfQAI}`+1wC+<`f636mRYN6*m@)tpXd$0ry%H#ivbEG*)^r5tiIYWlo-_K=41>1kvOoLZ(f{j1I4nRq_sqY;Zmf{4+eIEp6s$b*X&ekNwXACX9e4HVX$^IwOH9;1UfBfp51WYT$r}^HIEg<byeko$#R7?D$}eWaZG-+IsH5Gp~fA+2C|~)T6UdtB<i0$X?gl$#~C^p;7BJpxy{;xw4VYrLfTL?LG~s^f<d{-o>v8LoBNkiG%qEIU=|I?=Uy9wZ418_1VmvYlujf^_Yi>>L*I!OtX#>PcjrG}(yI@@T)wAo-oLv1`^s7hC*Pw%wvBIS2w>S|`;d5TSAt{;TI4ll%GG!wdChWR+S8yAEQ8~JK`ps#nyT?<5Ymn>SaL9!rX@|%iVJWe;3>OW`BT7GN>1yoffL~TK$Q!I>;oS^+aRz;n(7vHf3vmORcTD16PgI=@V;iX|Dd9cMwO`G0l0mI33X=1>L4ns@+)sDp9$J_b<UM10(+R@1VIdFqp~Y_?g!+Dn4GSiOne%vCA(i+%GrP?&+x$fjwNk=Prz2xNi}zC;Ira_gS~SWOQ#7@82tq}cz-wq9C8lHi))DTmNx?RleQ1%jpQ^{9Gz<y2s2`J4dKsyT4mX={W3Frxr3J3W-STd2BR8;!~#$UdrnTlIR-(vh!&hid=fs9Tfx)-A*%Y{;B?wYo@}Lt!T>5Ip8ETeCOpyLUQaAd`Xo?+q60TShW8gLypyiP;0<!p$-tv#CV3_X?+Xqnx$%+k$x%}s7y$A;s9h*OEfBXW_uYCVG{e?s+qrXx5q``P9&1qbv~nMUgFjo(X&tEP<C(%05{iaDH-k~BRlda`Nr3<gp@@Mi7Zk)w$E3_HEYkrQ`;YwU{n8^c8J|=XUo<+K3`sEV9wP<wDM1wvG+RTr^k@MIVmy*>K~2($ShCiF06#?&3=+~^+H)%DplO!0li6UJn8@??X_-#Qo|b&D^oaF`PCoirA=9)+^A4Mz+LH{Xt+7Pf)SeS-{&YKWGBd5VXO)I)6za*vk3NYUO%!T@a?fp2O~M`{Gj#4fZXI={_^spoEB8<SYfaSr|4&q{&z=#@z=em^Q62n$tdy&t#_}`LKOxBhJ8xE~kU8J8<emyvKoZgTI`gDWRkgC};WAH3%|ZvtrOyx?FmtS$Y@cE7s@Frzhro|Ph7pleHMn|+47z~EG&3Y6KVntc7s0Zc9#Tz6%t`^v3*HD2V98ZwDa+&`@%(uhf0x~5r82yMl6X2FDcV*#Ch7wjHCxp+KwT_LZi@i{c}YAD(xWXC9zG~kQGGuKE>ZIYGZd?&k8qUYkw73hr+w~XE24*Z5$Y?JYKnt&;1{IBiZqAFL9e@<&$m_MYGWk9$B)3GHIfaHCgOKMp@R3-*9hn4pektr(Bvpo7$Yn1GoGhPnkb9hm|8^H3(oFF{Mv!h6*_cUWw}*QhbBk3;l!k6XOJO}Lo&EgkZ;0;sEi3q=ZDc}<Ily)Ux7SW|3|1leqcik)JeCgf7>7vYNe>H5gXh*gHx4laNQ&VNVE6j>t=`Zysp6sR^_V!X$elo*ZB$r1gJy7p8X)0dULf(R;CXvLI8V>)$;N^j_%CsL$FjMSbnI|L48o^{Un!ZVtWuFnK?QgcD$rZ+4{tLTEA!Jw0a+P-nG@$AlcPZ>S^0Iyt;gS{`u_{ee;f9zJH}^zi<iS)7AN}R}aHr0r~O#>f*0cu)sx}Zs$NK?@;OVBTms=Hu#)0OI^B975;oeb_QQp%?0ZO%iq2;Nih2*Sg7lQ3q%oRg2Lh~N3ZQqN@i81XkW2Vr+8Nor<xAR&jSPCl4C)^$i?SZ=QxZjBZBcv&c`~xehJYF>Ke$vE<S#qlqtc=t_x8acc5VHk7tlGw2+5`mGUGLQa2CT{@_iQ483GHVp&Gl;O`FwYi_^{P{SU82HD6hBH7`j?e533#}~mC%iyXd)4KvCzDPz8?5JX=1j<%vH5*T=fCot$2DDtpHbF7wE5n%JjEaIDB^#jfzLnliw2iNvlR~Jby=@Rdja=nN5OFYf;;_&%s2O>ayvwE)^Q#>gb1b_{3$(j~tn}kf?fth%u;vtR;>2MmQgYNx&81$Q{>cVINhWni3tmv5X75TIl{*7{tmAK<x7N?!=b5KqJMgq6L=>SJw3>FLX3TPV52N2VH2Dgo4{B9l6gCD_{{>v@#EV+8j#%0^YW&Tj|H5(yl9>#1GYLCUz=x>;$J?|NtYe|&=p<(5SvCtE*y+371TXj%<!`|vfCdGwietR@5I%Nw1_}*xQMNQB9$5dF*5?OPFRNzJ!>UU%n+bXrr}{c-vK8bpz7CtZL?jN~ogYi}3F*E8%)a9Pw3vZVZ8Ln5-!0^L25pP!-g3&fnCrO)XW<Le3_CATMZR{#(=2gzV<)Ru$4%a(mL4+90dqL?)ux`zxbwi?k0uDp=jJ{{KXjCWjg_CG$5U^b+Wg;A6$ikq3I0e9gXe>cjG0%ZpVVb#)qHC2bFO*NawtWhG#AcH2%Lsld=5%Sm4eA<D}NdIDs3z)C!4r37^Ba+^FEh1Wp%5T4fOZR*W}xP|GSY^7{q`cIe0m{1T^nsB8O|7bS)R~oAM;jEwJZK#V}KZBcQUGvl$Mz1=8w!-%2Ttx=9a#Phi;d<ycceY?2Kydw1_2r3YV9+n<aE8&KE_HIfi0OEjI*rD{`}WGUDOGK)D=J8Ww~eBq=F3?0hdw|@HA?A5L3P<YW+bx)q)b+3T<8)OMejtzcW?W3!|eUOdm0rJd3r_W-O=;KUCFdk0gm~!CJ&{eZE4XOKKiYwxk&|-qhcGwc&HgjL+SOw4n-1ZTBa@Pa$T@RKM`GNr`UfP!e{S~1gy-$>vuH^E=CrM}x5jY|r+5_Gh_FCIteGyiUFOKxkgmr1wE|w5@L0q=JZ)lP8zE5-oq{fchn%Rlv0TU)~m2v<6WRKm?Mcyn5D_tE(3+17VsFK}*8tJGaG093gVdatadNC4P)qSV<o0nEpWjBzS>Z}~c5{@r&(4?vAe&V~-Xo*qj?6>cRX_l%r;_8^ln-cZlYuX$Z1q46Sgax+pdGd6+cRNaNYhJMqY(j)jy8!6#15K_`d2ME}%IxUsE>fAmxAH2J%*a4tBxFWIrq9a8ZcE_G5#*~v;Igm{GrgXuYJ)oM5hvEEmixn+o2;g;6g`3xFstokRJsX#dk9fSShb?j`ICK(z42?aCU~!?ptdu#h6+DTGDG8asl{kqpp8ncyO+kuwahG!8NZF?t(vQ&C^#f*#=UOA?r6zxdq$o8!RTaym%+{{C{27CotQ1on%4878)H|4d7kIJhg9Oo$>j-p-JxH8d8(WDnBnBS025RxhxB^`*ez@D{=RCkcii{vAIX(|;rF+&Kz!1!@S@4Uyfe0PUUa#$;wSa}?4nn`m|^HYh&<(Yc<7VukpAc?)q9MIZ2Ni2!x%4hSlw;>lT4Wh6{T7YsBl$8yQ;aNO}W>mD0Fco9M}Al+m~4@cf5Zw)C%8k!_-pdDK<{WD<z7wf|VT@BI=!wAA##N5l!d%p?4ru2Wmh@nS7JVg%Ir$MX{s>?px~3QqbSdlcv!iSC79OaRVB>M|$us&AqqsuJPQat;R~F#pP(axNO$=y#OrjSbi=8+(>{MH^9Vu*pKcd58XMM0z`p9JXR8mH&UkC@Cob&by`qaw|Xh@L|u*Pm<rf}CYIhqW`X)vA6R?_UJswHdCTIH=#TUL=OI*IC)#iBJ$t4$o&^8?Z*3~sa7Rt%M;Kkk?B;91c@4jU)xQ8xO9KQH000080000X0RH-LcNGEv06YW$01*HH0CZ(@baO9bZ*FFEWpi{caCvQ0%WkAF5WMph7ALb}P%@chrInC2ij)(%MY&o%xS8?d2eQpP_UqTiOd!$vfN*tHxx3xY^Zc1O1yT}=AW74}$1_oeqa>ZNM58|&+iyCHNQCxyu;7K!q&;HWyDHD~EE8%l)|27t(fQlX8`~bU&Ki||$&72AB~(f{K{Xv!NzjK84#?(h46*Q4aFaC=dNH~ahca!N7=AVO1x5$5dCRh_mEatC7y9}kT*#7YUA-ATN-Tb5BrJ|yEb9!GQ-`uE$sjH$-5ic<Fh(LL^&F`bf~H)knr`k(ve_2JC$%%9^~Xq^Gz&Eld&D9BzS|KO#Q?sIf~g}_3!c)XEoZZ$B$L4+vFJyuXNB|7_CXVE%GvD)xvL%*U6%b$7g2TM5~P^LqNl#nxy<CHK#w4qL*(@5LprdX8XOW_wVQsXaxyLKnG=7p-rbj<@KJD%I^hr71&`xm6><R{Q}k)KMD?iHtdwv?>+0>-Ab5{D&Kf%+a94lR_IPC{kR#6Fm0H1t4aSK{jB1aZ3w;Qb$ESs~H;#GJhvNmWSJmJeUzrZT7yK^a>!wvovC$zDiiKOUo}rFY^a|49n&w9x##}Ka2L>F5(4#Lji*A(ub_L&uoF<o|!caRcR*Ly0ou<JLnVu)#)m!)vIixXI^9S275rG_DOCvObrMr@Zn<RSyuAWl6iKkgi9@#%oO9KQH000080000X0DAQ7X8r{L08$hH02crN0CZ(@baO9sWpi|2VRCe7W?^G=b1rasy;xgs+cp$__pcDRhf0N`H9^+`1@chz0u<P~q3HHv1Oz1=*{ms26)7if;QzjJsEZ}pPSUiSpDgNJzjLMX$`nB<+)BwLkVR=K7yN~fv-ZD(qRgd&?zz^|Ik0Xv%Zbwo%d1*DSwJi*Cv%ZH+sO%M0sdaM$M34jRC9myvHbQ%q$_BiK7p;3`yH|+wC_`)jh3lUd?oB^diKQ{J)cFh*(_DU+FrqDG|PsafiXS45f-9Xo<D;;u;BhKLklWz59vpbF}Z0fQEh|E-er`~9=??oq|Q{^FdDN}#yI34z{0VdSc?MC#!j^ZQN1<S`flxUszB(v3`d;S@xAvl1BBacc?Dn4D9c5W$YpJ6{HHUHWVu8LfNzAV!5-Czs6pl4eXWHG`!|Qchqo+YuY$$-Jh-@;2Tb6Dca`0+Xcx`zW+iGQL_QJ{l8_P89?EO!Rzt%+32D*tf5-u!H3Rz(Qvp?_2J&2{5|KewxiZ3qObbm`-lygBwtSG1bM&T%n(d|mQa*h#D6XQ=*t!58kJJ>U0tXqFMNvDENQJ?hMx_fX=F%-IOa_X;t6(FoH2O>siW>>nmX0?sJ*dK^D|$p<q)=?v3tK}YM;2lUNb&pv$qR5RlM(&<=frLS=;`I2t?>XkhxHitX+H0bj3z8G7Kf>_5cSScEfAQbXkSx*{`m*n%>(O%tL<e=5gp6P_d;0+Te&@UBs#iBZ7NEKSdrq_L1vaQ8^j(1g5?X@cX>B3>vL5@*a<ml*FkOd97<4<c=81kj*xNeiYnqXvjV*r(D-V@D0&1_qy3VcApY}1TpoaB5d0E+$S9-^feOkxQCJ;B7XO$JC+c!EHYT<C5VO&?Q3^al&8bwmefqS^mSJ5Qm7Q%~9JU6d)s~ku9kH1>A+d+!&_v5OfIe_C%+%OHjFoyx4Wpn5mT^2RYj@DW@2%582N6RA#%05d1~x%$4Vacw%v!cDrmnP5u_-a8Oe8D{gbh~M;}ncxYtG{93a3z*Jh{bQ^R)(@VlO(Mvr=raKV$D1Xl7D7YzlSho{Vx6n`rh3M~@UPNgC9xcRsqOj+Dfb?bXFgWs-1jDP(69%d@9(B5}|h7M54pf$gr%6l@JQ?u6BQ8y@KTw6-S8gYgy*QmY~7IXYmu)E2iB%(tA|wJb}_!v8VnOeL36(*Qf*>Qc^U5_gE#cw3o}Rpf-HB0NeZQV93rX$G21<`U;#^zbF*QT<^8#yA>X^J;@&X;G6IwrGwUak|F5X*RhM38C5m_v6yuxiVOUAO*j9HCh{Q$7=+8@T?Dh4VW^iNA13;|Bya1;pmE?-xQh`7??YjgcBn?ZqGY3#~Cv6NW=BnrWm+19~)-FV|j6UsP|lla=vScxT6_0>(DVC!Z7-2*?TvN2S{F5MlabY(T!U{Me#cc=@{0wC;51uWa)_(&#o5z<hb%FKS;L}C(e?x=RbpE<b7+^D3N+gr`4{@Z+Lmb%H|qy-t!vm;H6C0s<|Jm0ZPkpI~Ujj#I(rqGBLKYy;<wJD7Q3!b=kj$n-t3~_~0qLhp!H6ff_trjPZKpMMxBZUeO2;N{w2hw;G}DF6(0u3SrlFKs~y2aJv#0zrE;NQQv{^AeT3&1`n5D3c4+|SDJnBL{*8cC21r38Q$!@2wldWBwQ?>;n2l3nr^7iuJBLh@HTgu7^5b)Q|3Ly9mF;QzCmSd7$W}#4a^gqyD=Csu4B!v*_cQLphN<Sgf1{6{Z7wvL^2WS8V`N_?xLS0X1s#$*KVo`$mT(pMF?G^XbH`oqo+jADJyZpw}rvI2^^Y*!^>H;KwowC^#^w}#ZL92^Ms<(xN?;1gZ+qIFWj4nJGF;oloq&A;~4p&E;m8^c9=zI0p`GA0_=`OWcMTAD-LY1B8O2b-!lKb2G7oSeIlELKJi3{2}B=ZD6s8nlx#7oVZt&&)i*5SM<dp!&TJ<0fOGns#d&af88FTZAvNc$;p;!R(J40Xe*sWS0|XQR000O8001EX<G^H~%Lo7fmLC8B6aWAKbY*jNb1!sdb97%}b8}^Mb1ras)mhzh+qe;b*I&WG^npwzO3LJ%spmA)YvTKuxn47g+lQ(u1|mTTF-h<M$g<<{|K44Ir1&X28Q(*dhgjt9Vn2Vo3yG|d%6NOG_o>)U5+yTlZ%h$tu5P*VgnR!truom;*B?Ie-w#}ypIIKKTur^}z2Ynu`3_%xvMiI5<+GN#lUFc<!Yf%H;A=%>cvd7L<&Mx)(kfCXz9K&4!WcN+5m(>BOhz{q0~{6A!oLV7yA;keRfpV&jE77ak+8_<itmT=!Ie@H%wk4nBBCb~TFhGW6Q@7%;Pdx?yubRDe)@3r?$gvhutXFs*X}_qk9jItOu<DW5j=K}3&jhjICVnZGtClC`JGXq5CtQqlVCELL@CgCKOo&Tn63TV0r>SD1Sq()?0>bC88>?wyQ6vH0oOl^clg4OQe9Hw2fl@o%ni?txAL|m=7y_G<U$(}&7wVzZVD-KGfTvsIVk?0oOrLi<zms2pj=*P5V78z-qdm%r_^s7JWUakhG4X~%r$>F1sj%BrpA)~bam}(o+h2QjZ=9{70Y)V4oAXp4|&Fw4tL!6<RiWz!5FU&SyAvj_U#!V26@_*8}zwll0X#TK)?42Kj34(BP7&CDnOqV3D(O^^=$^<7MoyLNYYhgkc_9AdyC$yzD$vLwT<3t<&nAA<x=sPl}#=OrPqe`Pq4x?_>L;B3z=)~zg}G6F(s4L?Db+XsEVT*KJ+Bng>u-*_#y9HBF<cxr=x@k+1^8ec@OT~$Js;5bH7aEz?*r?$9e0~<A|_e@}#!4Qor(iXZB?3X`}qkc}|aDMBGW5_D4vLh=<`(3BxP7>(Z%^P`ye@<&I`tYqsMy)u-MorgnOustTyoNedCm{;?Wr5x!5Buiu6X__qX07s0fu@}MQd^@Cc5Z{T!pRwwc9Kbk;B-eNn78P8<Cx>g51Bq`J&!T9Z0Tfh}KA|crJF6Z-Fv&)C>L5d>eKa$Bf&Kl0tdw}{esIA294+c9zN0;_tQ7WN61`YWPknEQctKq#1c{b$OVWnc~9&D_-Ep&|X;d`NYWTd)p9Anv(lvu&otNpf|FlQO>Q0qMd;&O<<Z?C_*aBNlf0>IwIVh6y~RJ-!ldCkKp<t#rGZB7mCRpdF~T-{nD2K_3gM*$%@7|L>K_Ta~yo1;`Wv_!`4VGi3GX3ss6$(X0%R^+56>oxE;!dP~jOMeEGwEP`3uzjM<V;!}sT-bRjzVH<SK+BJ98pVwgT(6d=$BlH62itpV8CClV{(o(<d^js9bM^ayr9LVUVo6Yo)jmK;OZGTt-z`(;vNJp_MR7U|(R<oyfoVsvIQ&!kgq34Mq$Yt=)xdD)2f>;+XOYdc@Aj7Ox`_|X^N_*N2#8-|g>B0KvaadVEo450+0x--Bo&r<>7q{h19*DSI_!`KLx|_sk?wZ;4(_A9rLKo9&Y*rJW>x($*eQC|R#C5)mz$B|YjTG;Fy}D$-^4-<bh`0i0Hc8z*#6gWMwKk<%jIG@FjzIVf#Ip*{E21ghev~l@iFC~QgYd<!OBtNBTJQ0m)>vzwrI2XJ0Sc`=KP#MC|RI-qG+qn<MbIr<rAnggg)mj@1xMO(CBMW=?pqOG39*|T8&C*A=Y}f+`wW0g=)pEAbnvu=wQWBNq_~r$N^|TGqiKeVVerQ=b9!WXDP+0;5mKK3Tv0<2?m^=ADPf#T5F^3ZfQ>^z|*Z~Jn&`E$pSaki#G!V&kXoTum>B&j|?^!7hrxTa6fO%=_5Y+7fTQPc@C<kQ+~&?0w%afX7eItP!iBf0IKoeZsfshmbsdXFvwJB=pMis!2_1Su3w%JF(YLDVzbd;{9L?xMS1dbFlj?-1E{y8GKU0z;JGh6U_g->zxH0ZOXarN9z=f2QW4KykstuU%ie$s>jv9G)GQ0lSS}KL9<iKmIRp|djeFB`!6OJ-yS0$={syqT-*X70grp*dmjw1f5eb&gQG<b+1KL}mAy<%?VTrw23{YXRX&_mDZQDaeY@;Y1oScZAYoODw>6GJ;@lBobE1O{NX=lJY5j?gb!d_p7v51V{yiv3fh3OR5Amx_ZHe6JB0YPuyC;F|zO@z`GE0o;9XsB1%`h(8s38uwQ$Z<sDf_jxu>9-~#2Db@w4<m1RR`XfVxEr*Y4T>}_Qb)9?q@Pis6t}L6pmDqvO*V~pgMxmO<`;;*Q70{+V(ktPAF@I>jC4K&Io)ywP8h=hc)y6kqOrCkyipABOG6u0{4)>?;4o9sUfl976yr^C2*NtuxV-od|1F9iv9XujFee;I03P+UJgA7blgK@Dn2Tl-mc#yRyMmq{86<qU#DJKG$Ik8OS!VcamUAAD39z0((>nLw?Wv$PqU8bD^NRx}b;h$TIB~Te)qC@teG*ID>f_*ew9)$mTFlQG9gK|yFB7aVoO2zqf}?l0tl1<-hL6VdF>U|;5p>|C7nif4c(&6@eVdNyl*@gViu}e_i(PdAg<-SJwiJ=c)k?qnjg$&XdZ|(oc_Kl6gZCGB;*(P0PHiAH*a)K*PTI^EycJ!@<?F@Uw{N}gzxS3GXJd=}xQlZW&GDY~_>AD5o4&s_oG+NIDPH74rf%JgzHHw2|KS#4$GD@_#xyWA!=!}`SL6L6h0aEgT<rD+ubDGuY${gk)ZlFjPA_l>3c?GOMkf!!0l?E$zuRRj>-sckLAQD!84UYa+!@o5O88gG?1=W*PB+v5Lxe|{g4`^&xjeUCe&joTXQ8igE#;B_t{1bvuvxNsxOnsUCvtW}{RzfoP^}VzGt&ld_s{X*G*dF*6t{6#^cs`E?K~Qm9@6>AqiPsy$5>B{n={65#$4{8sXxHFDIHS_Mm;T+CMeV$4JqJ};@(KeGkV&GYJ$$XIGKpVqxN!$Qu{L+r5H|1%U{q=zhyIfum3+#O9KQH000080000X0A*}f9{UFX0HPKE022TJ0CZ(@baO9sWpi|2VRd9_bS`jtwOCt^+qM#Z_pe~t7dem>d$Zj(!O=m}c6XbiCk@i{p~o-;TB2=cB~c}*-F1Wf_cOzbF19yG7dQs?B5L@}a4z2rO(}gdC%3E=1v6X|v*U#2re!6OjS-djn-`>EW=GhTiCP<?*xjrV_q-%)rg<r9t`kb>WU^7RB7nm3l4%WCqH3gq4ZJB?&L{ryy_U6;Mc7iTgWMZ<bT7=l>9_w5e@p(FRa`d+6L6Y6*lWU*kt*ML8~A<0m8f`abYhs!Zof@fTL=>n?L)@uB5On`O-yoGHB9kL*R0VyX|y*3_Nul9FYqLn^+s$1!ZokCVn)dNN4p-A-%VTW-NW%_|2ORZwPd&Kty`N+Ce~f#ov8Qk<PERE<LbqPz-KLWW^gGJ1xd-mF7ol>nO$sHF<hLFC1HjB>#6=X#0b|5z9Ctbv$D*x2qL(NiQ)HV>XhO~anq{$(DRX~o7R8^oi1@(GBjm1?ZMZ~<h!ikxh%L>0Gnd+<jKt)Q(JwkX-Nr9-b+zO6>B1GR7{k-ON=aqHqmrCB^#-5iHI7pR^YbpT>>N<aSs@Lf|_kS;b@vi6(}#^WL1-4o_U=i66q)iQD=|eH3~*Fjmc|S^Vnu+lw})H@+_MsifdWk@@Sf1_CU5R7b}Z29qLLNNQcIN%92&<f|0*kuJ*B2^@8-O-E#M66FM=x)>1{wMNGbn$?R-CkI8vVE>_cC(#19&Xx_T?1ec{A=Q^x_u@$SKyfe*t5iQQVt?~{6)Ih1iNic<$s~CRlJ9NFM3x1EzSFGOh=xh)|%LTgA1Oi1AJ+o$>gXcP1((b=$AAV4qu%_X#=K~cL+j7Vy`oigp3AIjvXZjP@xe{v*1SOCfKr*2pd!%4`K<i%M=fmj9`pj7k$j9&O``_Nax`ww-EPg=dhhJttcS7U|nI{(yi2izc{qn~@vlBa>J+R~IdiKXy^7*mk$Kxehal?>>@odx=2YRp&n+?Y_MFxMGv-*~4BG>PEZpf|RcbXocee9l`n>_`lL$2%n43w*(SOGs%QkoQsIO>Y82i|tfnI4j8WMq@wr1OEat7jW+ot_=8Tkg^6z=Mu?G*qolqqBH`#0;7Sy0SIPVH=QGL^<FpIuzsgJa4Uav_ZLIR9Bv@d|}7lBk@7?47jOc>ITUHDAXvhDx#Vpr&1`ccd7-5D^_X_<oWOpZ!c~M1{11_hEdptq=);mu32XdkY>Kmx|W{R;Nx3v``wPIO4d#c;-X#nde-duGA7Q_n0Wn@F@rvIcu0>PZk#rL4(L<HBYZMO{=fKi3LQcAut_f#lR{?hIB7VQ;Z=jLp*Ye-GACl=4@{ukI8@ub&;D@=*cP514iV93{L*`tJSEg_rVbHjB{wAj@RDi+MrpbtDRk`yUlw#I2W>mbcRasYQXg2?f-6jY*oJIv-&?l!pkY$W>>G`lb3(PpQ{3`I&a>WybhV_N6G01bFy`+gPzH{7hg^79AHZ~#*dyU~D2tseynmMTa*ow-xxn97=X5nhDtR3RG#ev0wpHlT(t5(B<xpff@3eH@>f2y}3Azy+J0gh=)epL&x4|JS`+WeuRnU3&Az)v!L$?ju=p&u)um^h=^l92>FeK&B(yvl$+>23>d`g=|IkFGy;v~j+1F5A!=Nu!CqU>iQU)~9fzy_F03mfrLZiUfZl1+M=^U1a0Hi5ESD%Cc24>UB;ob5niw&pt+AhON5ugy=e9MItn-#2LbqZv*ru@yCpRNgF`RsMis$qJTy%Yd4@iD@aI8kS)e0oRYl1Xs%_g8mHZSN)vrPY+}sv_!5|4)Snd(vg`#I>}2IZ=-&g>L$L}O<Jg**KLJ$%a3*V>|^ylax)<R3kFOl7u(xd4o12eK0G1h$Q=iG`1zV#os*j1LJqgu-c2DR+@_hV%RR9dl7zg1T#<KmIO`1At~i0#5ed1IZCSu`#qd5+8?L}F00<HHLkzhSK-G8~mLR!>b0<fL!H-Mpt5r7w1Jk+-)5*zhwwU7-?|uh}MDJL`m$L<o-34BM)9#IZJVKftUZJ`~o49*Sm)41ffzZG-*5x$~vNS%CLk_tR3gRTy9#3`DxvYrwkRMwV6A+=u-={8>l5mBygpktLvLCC6n@)&tY(v8Qr~S0E@Sri{_X|lWS)rq-KSeM-!PgY;Zg^#eYcJd`|6lDDD5<bB*vT$4)Y?fN^nb<Q+iIZ?2p{w*+Ye87n84N0t1LS|h%uB#Dj)l>3*^iNGPsl9L)mDckO5bh5(nroj{=C*kYVL!P*~x;wEP{NqUXmF@)soUMkxKLG43GkfX`sFHR4h8?ZHTY`YOU&Bk2RToJCB|MiZ*$pttzc(SpsNJ)BPihX|?3OPh^8q%KFTBTu|LoWugicoG|=OZf6F3)2pAe%3WDUyYUxBcB){h|3S*zCQleWVwJjJ|@qTXY=!m@4o$J)v>FS+W<|N#l`gFYV5hs;+;MJ0=Tcq4-&sORdA=c&&w9Gqy@h#a`oof8S%%23s!50ZD-hRybaLk!XA(1cA|)qt<5;+k=fOo#RaSbbMLTKf!7MM-upgDE*FWvg&lmw8CmDWULhudG%}ZEN`HR&_CLS9e)ZqWpZpEw1kt~=fraE8Q#}+wtHoNB!tBEVha~?GtZ<_cSnxkkO9KQH000080000X05I#4x_1Zw0NEA*02crN0CZ(@baO9sWpi|2VsCGBb98cHa4v9p#aLf&<2Dk1&!^z=UgWJKZ?fs42+jv&Hz^u4w>38I0);OSXo<FYkw_g$Iq@#~-8(~4GUdp*>%CkaVu_j=a(?rlqN<g)M46M7Ru!q4-IQWUgkPS)>q&SvyFg=iH8(BAJ^YsZ6RWtXS<X*d>Uw9nv4Me^OgGYsil<AZtg)Kat$<pql_+^83Jw%@Hzr$Fih^05ak&**$qGK4oSYPVMNnqE+;L@vFjmNQBg}@IEU%j^Zwi(%S!6~vI_Fu~D6v(Yp_-%169Ru44n(F-(<IB>QL<60xl!eoC!@4x8n!Ttvj;~Gl7ag0&<TZgI(EXw<VRK-K62#iBjoFde6E{7&5Ypy3GVTiDe@kyE-1jdQIbB8>6FlS>Dk-#>^+UC)<|WQ&NqSRJiSQIdL$*6i6hhe-;K~5fi(1CgyiRr@JZNBuwbg0Fx(_xSlRF!t(6YA;onQW>`NFT%dy-xtaR4tSe+M`BuEu2^s{u!wGm2YLax-}^85keNZyUf+1q$`e?apn>V(cWoD@7S!CD3Rs7eMtEG^fy=J1~gLuL*-hZ*&7YwkR7I_Pl;o`XO0ys>Oq@-cBH?%B!ngHc%i$krS%1n4!BW~Fom8Uzbmr0>2@&*>0nFjQ*lQrLukM@NgdJ+npNtr*}{WcfyvMaIz*nK0Q_7y-v~j$R{b#`2=`@iD2CeB?WL&fz6zHE7jB9m2X<f-Q1x%!yl84cHJtNjjkE0#IC)V!g2sEt@+zZm#4%CEb(GHDrM$078;V8PqffFk$DiBqcL;blD@nb<&()U6Pe}wvFb*Y@8J#4{4@%XuC=n1JT{Gj{H#(8bl?@RaF51WAf(B;}g?s6M0D+J)Q)@*ui_+&_fE(xuC`tfPuH&9-MS`JG*&>1q||n<$iws%dhi}2|TgU@-Sm2emPeKpPr40-+k(>5cG^}m>O=GwHhLAD5<8=?(h*p5%KX(TdFl!we<`#A+*q9%O{0kYYC<jxtT=%Kk00q0w+C|7dq`e`?EMxg4^L?r}RHsG}sdWCC;Vyb{mXIIsqd>s1=i?kdu%MAvAqDg>yRHrtr`URKO;|HAo5W*=?0>F{SnBfCe%@D>BQUZQ>H5H$+Sn0t7_)aB>{52tuJ;z(X9gKGg*vvkiD)3FS%14cEeAG-Ia9FkZMk61M-3%qV0jByZ0JZbK_oG$p^BfB|e>DhsB}JXIQ~!YdteS+HS3|1rD!aP{%$n@`tZtNsq7qfE2k=0E-NX?8XL3GIvy+M4}1&)s!)cl+V<)#u-`j<GR%0%v}IaSHAPF8>Pq$6v3nvM;yyw;%3qvg?~Kw;ylrQN~+(ERY65<$tIxLk`?!;{(_1d3R43I_68KJ(|MlpQQIG|4(r3{Uxn;`1IfK*0@+4p&Tl^vZ?A_sJCb7`TO)@u--<_8$1t+Z>gTR_R&t+dfj7G0Ij#%BKBXmQQw8*n))xJq)*+UTFWh%5LaO{R+z*q<;y+_f&K?7(`Z2>C!r6$?ynWJh(p3F97n|wp2Ls`j4z_%2>W12co}bdO^Q^q<ZG7iva)0q%d7%y#8<8}r+B7TE7bR}32%E5P@Bt~*8;i*_rcE(QBT#9UNu(DTj67Jhr4y}0ehwE#vFvIUrg<@r4dp3gz4DF7QN;s*J8<u-3Z(dd@Lnb1iLGR^#hW)Axmzc>m>{h$9W2k33)><d|09%i|2!VVK~|O9eYZq_u*}ZZ_qpeI6AfXyDzOmg0oDqaQV}Y#@#1K%e>6|j*h}VMig6>LxnK_%k<)h{W28fBJ<x$Q3*SB-FhS#yj!hHzUAfgd>pzi{Qp;i1*H~JLOEGA(AMP}o<G(Kib`8&4Dn77gp#L?$TviC4!}WsV%4%)rA}Js9{^Y1hKWBM4hgYh<U2xBr#0rwXcvFF0_K?<64mYy{#GpQZ<`vn#nF&E$f0xY1hgZ9fftS?u;+0pI%dq*0oMr_lg4o9+D(fs<Zx(%okkD{vfNE^@{6GDl9k^x_3B}Tb^?ZIhP{=JQs*-;m>CSxPTi=dALR3g*)tDs`L~JrJD|@Bn5}D`3sB1KV1gz3n=hgUI4H)I@f<+ozCOe`1W?7HyU1i&<A|d<09mxD!%aE`l<M5#r!d?@E4#8@yTK;vq@QS%>7|`bV?`OS$kIMW`6p_%nYLmDb8MH0m{pa?GfeJd>~qh*fA35@%sz1a;gm&PhdJ7z4st}x6L54qKBYJi+FOi$%}UoC^4X`m`On$))%?meLXe^PvlD3P=y($pGhz7oEdm4dqEcBB7JKn!CUqkpZTeNz=!M2%ksRwyXHI)&p`|!gPnc0~-*yl*1dt((WLp?wu(t^fSO<w68fd%|ClOKad~mAj5rQ-c7dzAa^~;9g<q^pI_Pe)*(nP|XhHXefon8o`d&r>KqLEsfm!QTLjj)Q*_=W<0cW`vH-rv$()jNky2V*N_OnRqpmq$_d#*;4E4^W9?*GpdE#(7-S2;8@QE-;Y7D@HE~PL-$NR;SxDx~~Nk!vVhHPnu=z2$Dq<Z6F#TTMR+<5E1*M0;N~OfANq<(R5(JZif^<;w6*Ur0vNewH|oGwwiz{G|uA~GLhVnlH=<HrI(^8xJ=&QXg_$P;3i*k<+38+;3nREVs#pRf$?r7z@58Old3V+MNPPMgK}t}lzt>E$8nuNYoR&L?46`nrhl{qz*|7fU|)rEetYoIt4t}3A`A@!7y(zel;XnT@}XCVqTG~tpfH~Y+<laJ+y@c>8a*Td-&-v)?vo|WVF-o+K7{hXjl$qX!jD>g<r1>Ptb>&HE+P+4dh!QQO9KQH000080000X08PUhVPFUV0Jj|g01^NI0CZ(@baO9sWpi|2WMOn+E^v9hS>2D@HWGjLUm@_}AQg&s*Ucrlphl7GwnczL8x*}KpFz+PWiu;_DoJJI8vfsJhSV2j*}Jz*>xW&5<P3-N@tYY+>?)~D@JY*ZmMggnDrUBYSO>yQe1z9o^KAI8%0<E3_qr6uaBXIe(lAX`T^h0D2~$SoEH%32^R5~AK`E)?fEl?HDaCEQm*SLF#tdt@gtGvC>sk~UO=T+Sj+G(@5p-;43du6czZkAcRyc-kqLLM-V#hY{H)|GGidRf=#|U?E!-@ls0qJM1M85BKa$JB#G@H%Rf@vMx0qt)nMA&L1!Pi@+dGyxu$ao%TZvIn+niqL9h(CNyP_I%kwGS4Zj)6lncqbH3ja2(E^2wBx1}M-%q0c8N*^ZA^v$B+i86iszM2BNHn@bhwst_g)KsA9VgD@mUF;UL1Z(fr)P^{eW@Fos!ZlWj-Lh_jv{?s*2&S7?NGW<DJ!FEe;7=MS85&RL6k+&cQ2h-<*t9W`9Dv074VR23l{rb1pH}mY6e5y9&4C#Z&%ofProSENpvDq3}@alBI@ni&b*YA$H0~LC!R<rT?dr@Xq%Oqu$wFfAYh*BF?rd*Se4z2rO2!PRuQz)v)IblEsW`*J9>QqLNyDb-spx^QB<mqvrrN?qLX}r#ejlEF*CbpW_W+Qkb?}C(>gdCl+rgSu|EBx_7^Gl)vXTFh2@J7-`AaF<0xCbsF$JN(4?%A1v@5YZT8tUot(&lh`+MMa~>h)TFK@Y%CjDv6EV7_{U0<e!wJGw8iPL+H>5w6d#Jf59-hSf0zfgyU3y9$VwwZiCtXNY)o=p#K?*n)5euqG#523aLUj<L0Obh2)Yj>|EB>lMhs2UIQou-JAW??5+maJl#J9~4h+;r~nce~0hKS(gC}XX;Kql$Vo0k`|nmbp>$WQQC}W$>QLfZ|)zM+UNlT#Z9fs$-EfOE_<*;dJyzP4INm}J{e576lR0V<ZZeb5g+|m-`b|m_9F;S<rYc+YSq5zE#NO=btVj`0BVQD+@EUo){jO*(?Dh~WO~n0fw+;Tnb4o4C=GoOW=kDsU9TaJQ&wV`kruVa6OQevxULJ+da$#dCrVGcSi0nS4bj>U(@M$Dyu^I4I5e)0SjNAAL#D#KKyk@&aF7n{GwYHFerOXDiZYkW4s`(GA{&vo0P;OdVrV*n!DQ~_WR(bTM;%7TQ4OIrSLR0ssO%pII4{BdSHVBr<1IYe)Yv>X!*_$>_ulZtQ3gByrfMGArPc~S`Q4%;ouv5T_nH-9!AnP`_zqlHlp9zM+I|zf0F_S?{w&Rh(p5B|z61(7+3)~?<L*Rpw|(<8+@WU-zzOE)E6T9dbQhmwwFLZFMa%2e`Evl9CHcMPYLEMoI}VU6);us<p<OAEkn@VO)OHmZ7OD)eFnV;@R1;TScwdodk_#w=)a88%(1kT$=PYF|Hnp<xj>oWkKm!zfR~4+fXd2IbT9&SqGW#)9oQtE=Sej!j-TawYvQ9WH$`~&+2efb;sZ@Q3kU_%$eFC&F{1cS!JbMa2AhX5mUSnvYq1t|BLUSE@eblH_9CLfqbF)ptr1ODWN6CYNfTspJr;r?u|1HT^mE-kTaWc;fQF7`!FzY~QrmThxPxX!!1s$Xki?-PQu{D(aif@EA&<&mF)Qw}V9yfO3`ApNS&AxK7>BXnmtTa3Zhw_(a5L&i(bs@R15bG}i{cOO1dhxR?c|47m*4eCEnXuKz0M#wh_{pH7w!sUe2vZ4)GO8RJ2WokGMf0-nUWT<olvv=iUKL#ic!voGD;dTfAiS*`@f{PC<+^O!pVcw$PJ4-64}7#^iHO`Y^o}vfwe`CTstG3@>IxkX4Dq6kHdm(pA4-|Xv#i{Q))H8bMhQR+TWeX=hKDY%+tL}%*NQzlVgj{T4bdA#XF;vIJ;8hfjTOD;yrMe*4pBk)!Kijo39hZ7if}GFj>f4H9`9hR5{kZ9A71_VN3|upH}d35gjU9jjZ>2mArsvGa{EhDq`sur?eAv(?WKDJG$0ax(2#|>;sL`p_xp3p3%2H1CX+q*@5IeQ{JgWx|E{xTa`X7vO8NNFR#VW`<->{lyFu>;(X_B$1XYM_%@zbQ2_$h=yikxN<c>U247M2%P(a&KaDIWnzcUP%C<xLlL4B(W5!-G8h>hu=Zh4W(+6>Y5Pr7}7%h4H{IfM4bbLHZcV@*)6pF}-ryJyq~Pp~rEt+^s&Y|54VgM;gL*=ATJr1piTTfCE`YrcgZ-XT=eiFO5A=-JL4Y9OMlx?9qje<3p{3fh);qAVbTy5KZM$+$F_Xr4zDqv`+%Ch7yy76>rymk51rds-NssfD0fZnQA)*-pL|*u8dtvILloZZHQav?m^B#AtTmat?Z=rFSca7c=vzpc-%=U;8}6U!SEm-*X*|*iQ)}ZH&)EVYBE?Xl>gjMWL}{6+UA)oZKrsL-gs77vb{I$9}WDMS$Zf?sGP1j;Au?t{RR00x?doX((5cTmfooXv7{9&F|eF*L=F=={<l)Y5?YGCU(5kU8AvOddp8#4@IwkBKw)@<A;B@U#33bHnTxf$}1eiIjfL~4oTmO3cnMD<fAAGtV?wOUK`j*DhhC@Ugws5Qu2q8{QA>-@U-wv9K3>LM9}e~M6=J__eysyfSQ}gvrn<d&=x(jc=!or__@Th&r_=X{vH8w5~y#Un7kcowvCLH3SRV<&HwqzCiPb@|8{4cRM5YkprFQOXCe=%{iH>y{Zd0I-X~Goe5vU^EaE6M4*vsCO9KQH000080000X0GzOH{`LX@06GT%02u%P0CZ(@baO9sWpi|2WNBe%Zf|pRX=8IPaCwbYO^@0z5WVMDSUEXT6qRMQP^3Lo>ZymSSA;AR4_S8;JGC7a7V+OZwi7Vj4>dwbJehfK{AR|;uGhv9>A*OxTbpGvTGY_>N5U;peYP{YBmJ?~wGK{X^y#>L_AdSd^NTFH!l4HvJ5bJ6j$5{0RIummz@5};3jX%QrTGB|S!?xSn013VG9h`Rv~{w+PzSmaGjR122*#DtF2FOXtmCQ%8$msg8I(iZ3gK($(<FXgEWf}Gt;@1Zz>YYuF2xKqQZ-CkrWG)!!-Ffq;83@N0LyBlUxW>|K|L$|qFBqf&{|roGJ;ocK*&0<KfWQR!Np&yuU^4uOX;#C%c3M(l)Nb4`98<*AGvU2_>r#Hvy$NN9)FvX{Ihx}$wP$L;1Yv&t>$iicZ-A@wRE1?PI+&1uPw?RH<+);I<L9QOENw5_BYfn-<)Gu5$;z(Eh?~10Hr{T`xp+5Hb)jLvjv*ATYc~hF?Zbad_qlPA3l%0`z&_x-@Is*Hq_ZWy+!)^7HOFvMZGbXyD5uJxW^{;ntayI*ABThQkiO6lGALlno2NR;;Yb2elBlDjTUJ|ti`z}Yl`#Vt@A4|(h4}_xa0UGO-SUMg6y;jpNLdb{u}a`R+WCiHcQ%uqKXeiy6=&_1w~Kq(!9rVE<BBo84X1`v*9G_b$e`35L|V*C2NDCek>D%X@NCzkF)#)%<>Mxi89TqKZ<Gx0&8-ew?!N_Gz?O?WpU%Vo~^Kf@rE*xSseJ7hwt_4<c;$eUp7&QYg*`gIZP|Sd<9TH%@&n#hO0Q%6K7G3;ds8?j+ED@mDbV5=%FuSy;w=Y(yMtn@T*|jUO_9v_s~*p1enMCdp?rHOj8Mw@RVEm3ck0!Hhw;0E@+7_=Md$|Ej0dQAO8Xrp`$uwvitG|e=LQ+1b8JpW2_SLCUD4>F0tW5r7qE9pN$}uaK|2e6Zum!J?VT0;UQl|e-^Z6@MqvH9b?N&xr^*CP)h>@6aWAK2mk;8Apmez%TV(N003zj000;O004Alb98erbY*jNUuAY-Y;|FDX>V>WaCxm+ZExc?68`RA!ScSyIIU{u+Rg3)2AZY;dbn#+>}?Ai3`1Zf+GZn*DoG{lUF5&t3|}P5mh*yM5I7b!GvvHIGs93yFK;ey$Xa|ht>T)nx+HSF7KLCH`Ss@4EAnS6)SXU=s7u~(_^1s~kPWZ7Vn)cC6td(QrF1e8yGANQ{?M}S9-6)3+Ds;EC3l2>ZnzRVkfAffbbf!H@q1RaPLStFCIo&iw@j#i^UtD?b?R1RA?@mwW0A_Ov&}~F4KqA1Sc8J{9XCo8+P|*Erp_DL@SN59fh)fr)O4>J#Y<5jw|m>bKD^9}s?~<8{8mb%jbe>^s|%^PWlSg2$z<|_(=zjROyGUP7j!4fw&HX;DfyaIPM2i*!HQ5EDC>cm$%rV+%7em**)@5?%r*_;yfjg3!|DP&xFBEc3jR>BV;ipN2eSN1MG1euJ9|ut|9(DuT%|p20T$dyB^-UXjz6AZo)TKK9fz+J{0LsAkLf7kyNI%sn-gSj&myt{4G|N@XJd6OjSZ5~!=aYB*Dvif*}J9r@#as{zk7<|e0Id}_U!o7kM{F#<O2M>=8A)B-eA~+2D~J<qSzM|Cqjc8H^P8?!^v7Ia{2ScKR=S&){tL*`_CtGee=l+67{-5BH1dUdBYSw1h2p8?L)%ZAT949LpPj2%F3OnF@7>D&8Nv<h?Up+!|sP(z924eEYLt?kXx=r$q5TK$><BLABnSJL(mMR273^*<wQxj%jhYmbU8%61*JpY<d9-#v-x?3;6<m|gAxEJhJQ8*u4HAqYjo0)Pp8MSf>~cCDo}e#0S=BZE<@nVdxPhtP)Vm*j|ag@da*>KpU+kuMlAea%Y_(&n1UdaTkGYJ^c#o|K*FtJ8ey``wWv33#gzEsb`2FG8RCv#hDh@30k=RjFYXej9MhTuu=`g_+TcP-S7fmu??x#KR?AuxaHa^B9Q`b22)hR4H!^>as?-U++={A%#3K*P6kALS9fD40bh?~dX}u)_q74x-encQkPuZ~55JO$A=koRsUKnjn38e+jdJpeC0_3~4GNP%V{Fr^xfx9=Utu0{CZauRriA|*oJbwy`NC<T0A38ii{TQm6(G*aEmy6lJ0annGy2FLjc)NZ#LTFUW56J#uH}UV+tkQ=B^AR}~)x!H&fPEp{i&=K|{RnS|wE@op^(ZmYVA(zhn!%<%mqy6WVv*Eal_IPJ`PSDPGvFL&f;}R}031iWQ6TL^p`;Hdkqe!Jh@3g4bBNBay+Q0h3Y$vSo9Ch?2HyyAhq0o!*n}NRve<hu8eO3Q1Fn<JShB<urcV575SAU!(*bcvrbi=TOxQP&=(u~d^mliVeq{;}eiplf?;5l3f;(q`Y3L9z;Tp52S9Hs&sx7d<qPqMZ>KS<Yb$R`13%vmRf>$_ejaau8FhGwDQ1u!cTal~jsT>TJea>KTl56NxPLO@!5T_nQ;+Px(fQai<B3tqY0X5WjzKRI|)8uzB6Teal+I+G@3|~;|66)>wWb2<SR|Al4>$_S$)VZsYh{`*mvBpK19o5)16IJiPv@5fI>ygv`i|_4Oqr)9K`u+R&As2&JU#9jgfPL5Nk&Ek}ViO#949PO?O4=GqQ|WpUXbGHM!jQ3J>Mp}((wglHcjb_}IVr_ytpJ?#Rj+UJ5_&CDY_$}3L2FgQW?s^18j*bpB#4Hm$^OfA1_5UqLe?~#DQ)TY0FeX2>zmQ#wgwzyYV_UcVlBTpSYP2F*M-Z$YzXH8_`T!2(S5U2gjT76?Ts5@b2lwJGkPnMV8)5}noo^W^Jn7trkETZYZS_sT-4xgQ9|_U2G9w5i&9#6Dus3y%x&M;ZDK=E_rqs_T@qkP@6@lfVz<1)wi@k{hA_Yda8T;}ht9kgTHGGA>Jc}1cfgFXQcLIjmvQ{&{CSG}=y(`i{!oEsHNf0{UQfL0DY=K{mepoKSqXTg@u{K%<V->v)L`a>hdO##$8Xj(RuOC}VNmwJLwTJtp%W>JR<UC5kkBteBov?vrfakci%@0;OY8Zt0FI&l9=p;Gn!9Gi2T%7Rh<F(uRlMGq?Er_5?pOeTQ=B9RPJ3Lv2KE4<?YCz(8cd6@<DV^v#a!&}Ec!rkhe(gfum&F-wUaw2B>rBgyAOg4Gz_pg$A&0e7rDOQUXS^Lbkq&p0(w-U<|#B(bty_$*{#vs&UhPuGO>wV&TKTDt$ITQ{tU)SUY7?}7(9*DBHtFq%^ZGA(`Pw-8|b0@6lZL`@U2tyubugsou4?fZLME(YCuew)W9#NZUd7MH>SV_0BrPP{{77FH9sFDr<1#+C9e$2J5K0HpjhWYrc2E?^D)js<duAI6w%sNY_?Cag$5|ZsULgnK+w~8Z9P&tnzzT|aQXjbn3o|!Db}`GGchE*<_OCVZ|1APIbO5<Pc6>u2%4|i9!8C2_`aO2R;Mg7fEiea=_Kzj$D>;WoG*-k*~Eg3b5h*AYT2p+G&qEuE}HxF@ZWf!ipyuN??MX)u(p**La%1{jNY`Dt*hf&6e6n&C@{k^NU+g`LZ0WadmPUgffxibuqp#;8u#*|!b!WI!F@z8qpQ#HTfnH12;)6=8W+F(c=eH7-u!;`(cTS&{q0zTHQL`ppcU;u0$|MlA5cpJ1QY-O00;m803iT`+YQ^$1^@t~6951f0001VWpi|MFLY&dbYE_7bY)_1Z)+}adBs?5Z`(Ey{_bBva33lUvX&I<Py`S#CsCWAiIdq*hs`qxTB2>W5-E_B)2!%!-|<DVoWyC`epo&v67Pk_yXPLBl9!c`8d0TtxqIY`rI00U->W;#K)RHoL|TYKMT<HwGLnjdE(jNzErj@pytb8fl^X?;u_E6vdDmzj4<`vZxw^bOpZ0paj4hGbhgb(p63LV(HcXLKuHj`#v>?m;6Eho^Wv&BVR-~eO9ime3kL(Vf({=x-hv3I=@j(f0-)%fbxyTod@ETsb%Bo&eQlw0&t;yHVThg5Otbr6lqu9C)2!vWNLi&+EM$nHcDjC;mb~v}{KcS&a*N_pUCFROG%azRm%aCHAPBkm;BJ_^23h6Zn(=|&!qNPY{1v-Kw$!Oudz0LIi(uC}Wwax_(QO?t%&T_tjC-A_GPzwpR3#y<ToH>`G){%A7$z-IMrdrFu3d|4QpA9%%`jM^}F<J8k!eMJ!&~lMco1qXn=@Eir>RK}5O#`>ycUFD-JA-g@PR(H;vrWO#F9^3A=hYow;AaG{?o>qOYJ)>G!yaVtj0b}zT!hb(;i+4Lj8wH*Q{bc%Mt|N3&(uC}j~iOl3}Q5hiwZ$A)wKc?(2QvINe6wW5AlI;rND%n<u=Zg6~zoIQOlI!In-L(B~w7V#|)cS1)_ZEmCbLW?<%7o#|Wn)W4IG0rzKO6WfR}$FLsH2VrkIC5N<X4URtr#pfK(<MP!jeTw=EI&hX0RPO{AaFsoRb$T`&dHj7==&EeyzteFceg1qLHPaai@8RBW8nBj3UNrvN-GtUq1U`<$_E%xBY5rAYM<LGmz{t$mj@e#n7szD|54Yb$5C3>(q?4T8!=4e2C)a9v;jx*e9xrdH|Apiw}&tJM$Yx?rlZzhQDe$){GS-gJL5q;^hV#}iL7=%K}Ef7oumEju7GteR>6dy$Jn`CuW<f#Q<;M3B;)Z#zD6r-QfuLpo~cN(9{_<m{|MGxMV1LO9xh-~9$d8z}uhgWQ*u$E_#@1QaDU0}C0O)ztPW#FFkjC~4Q)H7a}Om=M}dvJduOW>{(qP7vXZLn2xK+}=MXkhi-_8~H*GT@7s*wWg-NC%&7AYX8Q$Au3$qE*Fs*7f7`J_uk{(>eGbjwACHc#|Eh<FtS$bWzX^<Q7h5j_Pc8)GW|}?MWe^&(ssDFMJ<1gzk7O10GD*_{AQZ>%XRFSEIAx<ZO~8r!hX*r->O?1`&*Jaw&KTHyPfe>F*e_i>Y98bu&Im$o25#-SBO)&-i+L{(d-3$nnkS^ulOQ&z?H7tU0OH(5`_yH}1dqx+XmC>UH{T#kd7qMjCb`)HFf?R8L?MS_@8YbNEwhe+9ZZU{8ZBp4(uHodz>^f5|kH|LgtWnI+gp1CZLj7=Rf)+<$O^Jd@CyQ}3}c#Cvf*N_K3zQ{&@i*8u0lZ$N7;@E*O%)ZozREemnGR}VU4;`C-v$-@x6obNm3&4|1?9}O?aIJq32k4}@ZDTG3AXrb61qv6f;icHRKrl(hbj+}aXPd=8&KW@&)2^k(99FW%sZ;!ve!SQekiJzSRoixXI>I^4W?~^fPhD@(6;9+EX_S)IKj1Hhzes>=Pk;ky@9ks4?CT@9qPX&8ta+t$9^EIGSHIhq_)dicvQivw7R<KbMR`343<NGUpZhxga{guM?*|~LWBg5-9fB>(ASQRX?B7X(g=O~kAW{>Yj7%Yogt?eyb^HS7Y$8PF9a=SGb2DW;*-*pLur1SDIr!|KH<tx6&mt+M4N^88`8-R1o%D{}hr9GGF-knp0Zu`9?@O>=Q6zMyN=m%NH&wNb)9eL>T)+PEDv9=!YnQ7}v7HnzmE|*xKf8uKsJV3Zd_5R(=^nBl_*Cfwm^nRjdgTwhYsSh(^)Iu{uI;VC@`8o;vSODxEdUjTKxS4OeAxdFM2KT`5ke%WV)jSzpmMgj_SoSRu&O7u{QfWRbZQW)5FSI1gXx`z>5M<~gnfJFvT~df7BNcoH7x`+feI22?zKpg%K@9tve=Qh&pb3ZT2gM4J-?6vLV8&7JM~nYNJGIMCwpDlf#dfkGN>gb++w64ph|_-0e^$d#2}cjCS!a;mVHYQCWs>HKp|Q(jBBhW4ZWG;nj3HM6>^Ew&64jL*|66Z6u!?=2A>qp<eBzl#w7bW4Rtr%(yv4?UXe$ugGHl7iOlAImRgsNV1>Imn_*w^EjGu^kn}^4M?X(81`M0)TF!Lj-@%5k)O%;6>5Cc@BMjPP$A5cpJ1QY-O00;m803iTuMdViU1ONc%4gdfj0001VWpi|MFLY&dbYE_7bY)_1Z);y<Wo&77Wpa5gaCxOy-*4MC5PsKR!SPAT6nQDyZU|st?UrGE2s&)t9%dK>Ezvd`Nfb!RNgCvT-<?RxqT?jZ>KB{j-FH9u-5o8PPH9V4+=#cY6X{l4Znm{tg{4&C*Jq>JWUW<0I&M*f_=7)S8H7x~>a;3_F=6M6?AEdtNmzHcRv4Q&B&$?yidI>%QtFCTqLw?MZv%3g5cs@*_typc<>Te$hfn#i!8%@E@r_^zY~0?s1+c7CO(U&4&|pnz!Hnv4DS~RAWJ!|z?3^jq^2YXBB$Zf`ST4=Z9g0$x%!g1@HrQn>D*#*7I}j>5u0?Cj;%w=SmO^%@F(}RmEi5;u4!gii(m_tHgJ>acq_HO0!;g~&43sZw#VeBrX)Sogthljh27v?(^uOU(Vv2&FQneP{1FXB2H{zV6-Y;Bb4sC_GNHellde|ni^)4*vkZ8KRk5ILMq_qdPkp^4&sO&T9E`=^9{(1!Gaz5lLA6AL;8Wq6jJ*kRvE6OXTdfW9j9dQ<P1IRg{TI^2YDvZFz?-%cWrOTX@*VWuXW5VX2bno!9!dNZ4)H`%6dEUVz<Vqc)TX=-W!$}5eA=kXR71aSFnKZzED|91UV1g_eZ!6~6YuL54TL_!=og+fBXUNEDVw;X(aV9>5@nG)?rZdtS1i3K+YtwC!`P>KL?g4WSwsX=6ZNN9UUVsV4kljIq=FxVgp`7VqMI0^2boBL@IUtkOL&~#k>aa(Bz%h)#13oyL>ZidNdMj?gplp!5GpJU!`*CJ$U)>hnEzJtJ9wc}g*@w;*W1ya_1TUHjbV5x>CinII9Id~O+WUOP^o}VWb|5j6e(LJrW$uk0RdT(?@6%Ik7pJPNZ=difa)Hu^mTS?bHRLNzCp1HDsY*aDH?>-&^rV1!LbEJGl(RwIa=j5qtiyZIeQV)5XV8(>a)Y-xkuR>bw8FJeDxqa*-FHxx1iD|j<?RN3)~e*SFBj^3RW;URpz@?KM=W^I?6bIGA0UnWbrEq;tzf!9M?lYk(S&wdf_F$S7s;dSooZ%Lq~U}B){et>6y0+{Opu1qtG?+>I*H`4I=`*b@s2(4lSTJ<=ua1*f9e!w%U`{Dn|fkQhhbK1#Z4tQ5J+4@V{;jA?l|L?p6Ag37^rsG{?}aAhONXJGgj8B6-;T?0QqZXV6b%mtL3vt`n@i<(3;645WmG%KfUo@hccLxl%7$1t9Y5k4@>eL@DVmeZ3LmOXhyytPtVSsK8`H^+$%Jf1`_%YuX}N!wbD3^Y`LAg(P8?AMq~FVaiYd}cXBeu8aaO+9KX*ah2!(;Q4u%0;`5Q3`|XG0;HaGm75C51hU+VQIYRe@F8##^I8X{ZsyDu%MKKem<lR7Oh;{~u2HJpM{t3h2@&YGL9ydPFi&U5R>K%D_z&|ZMi_#iqHHRm-;q~zT!2brdY{d}KGp~-E1kYbaMmJaBOse8NX6D03Uu+e(9QId!gcJWxr+a$&e;38Svil9Ev%`WsTmvP!37p(h%Wt?=hW8nK&IfDGu4r-SAoCqw0exYI)E!?$p)&49;5{A3E#Dl@$z3pVe%Nyco6{yUAuV+>^nsFhFVdT+#l_(M-4(EKYJ2TU3~R4?|ALzRyES|*URtp}Yx&o|94Euz8w&0}P)h>@6aWAK2mk;8ApqG=_yJo6003nc000>P004Alb98erbY*jNUvOn|b7^yQWo~0-E^v9RSW9o?HW0r1R}k!B<%J`6z1toF_z?5~6fG7x<kTP_Xo<ENOQcFtPP~EtdxtMsl4ZwkHy1}D=fQ90n;B}UWQi(j)&*Z6URF{WbPtcC@PD;4Oq=*5>ayA)s*$KhODk!pCaM;Omn@~q@FmTRj(YCGi{F%z>Ue*}l;+wnk+CR|j1c_Y#|87X`t}RW)@<V5EAT`rMhcd%nDXzcdQot_?z~fs=7ft(u@ZF9f@nc2y_RM&8jnV!oGp<(6wzy{a+Z_KnjpGXTfSwQEWys2Y)e8#P854)a3qpRrRvIXDH2mw#3DAn^&T|dqB)9dXo_&IKpl2_mRxDOe0;K`EwL>>;Ymy31TBjid|_LrcJtZzpewCvle}SG(iOwE=+l0J-f&6v1|CF}vK>plOi*ri6`L=lEXHobk`~2+W_Hu>w9ssSTH~yzQ3|FLtMkivg1kmAOtoeZxq^uV0%4Ma_vxyTiv)j64OMtN9;5jjeR49IWm4pq;4$mq5iBZ2cY1<KW@t_g1wCG2x?){WIv&RDj|Ui@5Z4|Fw-JtM^MZ?2>Sc&CMc6+zEfR0`(PQ#78HU+gbcqFS3HEvhki*vWMRCK(!g4KH;DwSKCa~XN)?8f#)#m}IUO23J3a5&44eC3j3@O~$=^;4VclZ+d7db&m&X-H3z?>yDkt))JWb0aN`b^<D?Br$#r+LFV<Cgg@PHxpR7}mNpA0Wsv$tI2NaZ6XKAT3mW#FLCVBEvJjLm4$G-k%rqWMCFCE9BQ%j^VX7)YN(&7RLiMDG8-%!0mCntmW?=eRk2&L$*hyX(r36V20(VnbGX^2h)k8Y6lE>*On0S9SgJl)d_g!>@P}+)rr%$Gbvhi*%$ej4B#G^7|wR{F@4sM595|}kCGv;@%KDc)cIaiTMRt#&AA7BqXJ?<Bs2)`aL7IAYHMyS@2(?ZY*c65c3Un=#iq=k(dPGDq|UdgKu!z|37;i%kA+V9GzmioIjcRFt48}3DbRDoTN?JH73%i<;xxe7z!Q>77*TA@74s37g`HA#T^QYu_5UlWz~+E|t7QDg${O?P%F^hls6oz+7FRR5HJf-%tr<irzTyJFbrhc*o=ZENg2|)_pqG98@hNqarLPI&$-uhp2FVzHycQy~-*r`c;R2X#E5qpyb6(TQq`eaT0N;B+SWLTUHUcm~pGBu5>Rz_sic`DG-xCwPr}a^^<MLxn4E*HW{QO-S?jSKP?7I$<i&|&vIL8gTb(&sK<KB0}+Dz=%(8}4Jby^aP)SB5*Ia`Bh91=CL6;jrQz)VV(ys#^!H1>z)N0sW5%i*FHIj9K+Cr#|42Y!9zb4S)HW4#NRe~DWg(sGk?6=s_x2w4=o%A1l3zO*IYA3qEQ#*P&q-r_EN3qA4G)QaK#ha*)ktojBTG+%}qKCI1Rr19<B{-hERIpMEgpVIWXS~ZH}?BW)0qKju`Q|~5Z+MtVkC2VTpTcc_g-HaFk*xd*P8y5|}UE)ox2Hd&xcY~Y^(Xy)pHwgk%h(31fT4Vnh!G3+~qu3vYPwF~Cl)XY)_r~eVakIX}!?al_imJ~*vSF;!mTm=MV&!Utu@_7JjShXs2rQGj{Ui#rxtO5QFgP<TbM%FX8d>s;SZF5pDm?q?XICqFB(@)<5~))rwj8bk?v{$b{`K%D`StGM4kC@G0w%UsfAx*spD%rA4yhiF51vmeXiA0{x=6qAsuh;j*ye{jz2Q}Y?-i@4a=6{d7Xfj$=F#7O+@p{Z_ESPF=$7(=E{d4qKcELo4Z5OSz^_>|1m~}5f!s%@95pT>(7uKrv!()3q;~=y4bM;^<)*GuKM%=@AMxdI>BsHR>2+;*;gL*&U<ZSyoWRLISkE7T)nQPus^@4(s)t<oh+$1ZH>-9wdY2$RaW(=A4q4EeH5%KHR!*Px6gtdZaDQ<r15NYow58j2VkOS%e%~I;T4@__>erGfEcIrh;|<dx52k^6(@YkH^yOctu+Xmrm10{iYdh;&&ZV{%q}IUW?UVlwq;`kgJlX<ckJ_kY$nU4aI2H<)m41Q79_s;1{J4%ZH#+rI<`Fw$!CU)d7wm6}pTay36I=0)NAXOfe*sWS0|XQR000O8001EX*slW(oCW{@x)lHb761SMbY*jNb1!sdb97&Da%FH~a%E&LaCxm*>u=jO5dZGKg3vHP>S!&`bxmPl?Vj1PHpm`djDVme%I2ciMB1@i^nc&+B~iBJxNC#N3CX+T-S3`E5Cng7_J{<_St5B=qPk&%H7h2NTrf1FaKcN5=Bz51Y&M7{O(g_^#iB}4QkAl)auEkXFdAh|RiKngn&ean5W$POY9wk{ozsMk>~cZHBIh$hE;n^Y_y_#X*&A9gQ7Z<=Q7(%bt`?}Q4cG1UjpMm4=)4tOZzL1atz^|C3wupNOOaJg!J78A?{BVeHTP{(B}|B_u_mx}%^F^?Qi@noK~~RWV{gkEju3p!mOM|1!vO2O1#OC|<nLHY9C-G`h#<2K;VB+PgI9qEYTB^0f0yOX7?sP$iL|6{7jBfvqJPAdfrqYJf&W{(fP<1kNdf%<va~8-DIPG&*_<XDQqBH>Kp~3B0t-@_Whp^uoU)`!c{%U=o>War1Y_wgF({I-TF0CvzioH!m{{g=m&g8g3^zGl(L1v@8jW<+p<BLYx!t_dP(W<{C76z9n#d*1wdXZL#_^!?OHfu)4)L$pg^D#5Nyy=C0$c-Y2qwMTW3y{0QkEe?c*!LpA%s8{89mgOTE^NZq;^pUvUWo;Y*{oL1k~hxAQoUYs6|*hXTcRZKl^|YBxFr2!GiJmLh2KYf=8apg&}LDMV*e$HmxH@S}Wn{3WzM5((cDuPNigJcoxMvnd(Zg@D!snkU09zBwn2HwGuf};*FIFBta;Pn$%P-tndv+YcRU3V_v2l><z<<7Z^Q1jUs)nIe|bU600bnO+z;wB)}=4$~=VO8a+ej=u32V`3*)JSX{aV9DRj0ky^s`+2v(~(fBlqsZb>+1QAPK%5Tk$=2NK%*E(&{*C^1c#C16jI?ZB1S1i=_nAC&>G3LBuP#jxhFq&0W-qB27T%BJ6V3$`HFYbZi&t}O+9?(jnbpzH60IUMirLu;yLlvWE7_p+3o5@c!SL=-C`3zujYn6DD6ASu|bkLv^W~|GMye<T`Od~meV4>QyfOR=M*-789Ck6Lb(>GPg?y>JseRCL|U~~b<!A6!>RGyzjd#FwZ2;K*_OgSVV_}!l$u;*!4NG82<5k`LLOyRUwMLUBmdv~!0w_CMHbbDK^EfjoSP$;oLH93h-E<7u2-PPVxA3eoQs5k{$UAN=LB_w3dLktvDb#JH<VbDT@(bUF?3dH+J$C|D#4J&~@C7=h%crIB(sx0gJ8F51babrO*EZq$@@b>6R;M+r22X<?|FUo|s6X->_`@@6AxE{p%jW!H^JPjOl+A1Kh^GIko0J?M%*we8y(lV8m0RXuA-QF@s)06wHz7M4PhJY}do^F$pVeuVNdxfjl9&)|Le;_`m4tp=>rp|n8I&O`{o{D&%y7zuamJq`b3T86s;BUK2JF*}LS@8Dtuj5|=W0jf|-K-X~f1y2xJ&E{$K&yEtG5TbY&ctn*2`yJJi5y|2@42TeZJp5*kB^d0c2)Jf@&&r1kGdMDfo#TL_JwhtCrkJN+|;~)Ql4dM9@cXfRiaZ<@u8lpX~`({(#oSQ!*o)PdAWiyvVtdp%S8m$qypIm%u&J$2NEM*3!g#SJ;Xj3Wp&U`Jy;^!56uL$u5+Hih^1xfQf2F=OQEhxfi1k*SF3D=QbiVAdis2*UiG`?U_Wjv_HhMCYd%np^x4?h4V7Bi#MW|R9gW??%MucM9{3kqJcjC>J{+4ZU!X;?VRPtTR-hMjJ3`wK9xQsj3ynvIFZ4qA<LjMY=eqi)A;NuB@UBgM_UuX3;gr<!yClS|;?v;u&8zQ!b0x%tO3LTzVKH&f;8-ju46JG(rNYd)nh?h9YNnca7)9}dtqq?YQ5yQ_KhHqg(93OT3WMtm6};Lq3ga4S%6b3M;XU6%J%{uxfcJ$#bZk9q95`ON@YZPJT~N1lkHxYGBQ%+yZl%f9TQ!lUVcS<$3;wp}?fWa-eWivrceiehA~S;iU!V7a$@jg}I(o9?A64UbE@;`iAiI0@=7+bxlO9EMsK_3n!jOHRLwlRu@bI<&MS^E{cw~}#&(lNL9@fyDD7EgcfA)<!29rH(y~bpN%o?J5rH(klh3!?39Bx=jVtbuypSykO$jNzkU{(XYH!Q2=ku&lUEkj@NRL6L-b{{U;wjzH@5)Fs$PBn2FOaoI!g8Q~|n<^1~&ZPRdY#K_T{{T=+0|XQR000O8001EX5csr-@dW??j1vF=6aWAKbY*jNb1!sdb97&FWpHnDbS`jtrC4om+cpsX?q5Ohlbxxa?9Odcz{83zTMKL*&;kQC!w_hRwz*2AMpALJBL98IH;J-kw`ubsk;FUR@t)_7cflCH&l%;KZCQ>KEorWq;E3=7X;v}5Lz3<!RSLeU1TkLFn!+!xk=UY26tqO5(RHKM6k|LZv8onQqxxP`rAMQ!6cwT$Ybsd<N@}W!%I=;|CADH++8NHajFXZn4Te-xRYbD?Xo3!sX_`3&x1ZTVHsEwcr@7!;whJ4**G+N%%6=M+Mg`rXoYdw-<BICNC=zs)pj_NhNp^IZPHz&lC8eUvS(+f@pgAVIz#FSTBxA*LmY|aIo!*=8lNTcdKa%Q3a`ef*m>(R(@rzb7=~n-3sJuVBswg!pY4pjt3m?7T-AXNm7P%;44ZnW*?)N|5zWMw1XB<iFD2MZzqhtmkBdADKmoyd<n}1Bm!<nrm_ZSJ;*u*mSLnNSOLx=W2PfkQ%y?*)TosnT&kL@D*5v7x32^OrN?}t3fiKaUNu?!%BczBG(%qLo8UYt>5Sc--h;~<T$(4uFaa4s~+A89ROSR4_ugOG(;bFXgw!4uB#K2v*OqoFeJ*2G!0{=!39k{ZGaIhGXya?IQ01uNij+)^wzR?rfyTKj>__;&W>HIAI`7UsA|@N^y+o(_sXJtsaJ6944~9M<Q4eeIo9O{u}x<4hj=V{}$BElPOcxwSX3>O?6D+3R*$VYdn7#2PK(_{QvdNbYn?rvdv~@y&ha(gz&-B*kGB(rRtUa6nrj?O(>bEdvY+uODKYz>RLdm5K8@a-asOzK^l&QzKgwC>o+e9K;6+(<rJkvQeUJpzJ~6%pnV`4FQ>>ENv>H^F5Fz)Pe!d98={8#0c~2n-0_75gv`@33A`G-E4A!2_@Z;{NCVtbrl`e9JE&<f4lwl<*UE2gYv-i-rr19Nf-or{3j_JdMl-nQ2$L-mp))GTK1p#y>iBr5bo7%y*_`47-dp}Z+8HkJ_v^=++!u&20SmD0&w4$5=EfA8I<Wq|FtA?M`czDr2^6$0&X!*Ap(7*d(b}XrB`%0j&Z|*Xa$~3PyO~OG`|SmDtK^#S&Qq4S(8<K;dxkH2^r4v@dX6F#Q5`@p7n@J?ZzEPdoXd4K|f)TGB;2rWOkgq0A9)Rd-wqGKxxRJl;lv-TOR{Xn$hC3>G+wnGkcuNt=6MQ-V?v1`0FbDIVW33hCCgOe)4GtP2GnnM3X>qH0|9O-<b>qW2+}7T@p02zqh3zdW<>YCRqNB@h!w^(gDRdXtScbTx~YFD;Gf8P=JL(9?`t9$FjCOGGA5<+BPF92t#=^nE%e)H(EO-|N4@#h{hX#OOyZ$2eLAiDQaR5_loJGGg85&10X;>+Cy`++d~_Fsr~5MU7N*r|8{o;3fy)@*8pVF7ws%X&1t7GJJs-V#y9bSawioDJ+x2e7SHS*PWX<FuO|^(-56o+iFTj}*G{iAdO5hI7il(pF3aWX&{~}6-7LON5A1cp@jB+#&+MGwv}fnEeY!e@y=Y{1AhL2wPoJIB8i1g4GUAUYi^w<EAt@+%uHcc4WTsJu1(<F?+#fs;XejinPxw6k=P-X*KOQQk&|AUjqoKN}_r*{}F?3^V1|cG4=BI~TRFH*CK?rKXq;mSMvooV0RQ#%jCA~kr>{rLPeLJSIc1)e86PpywB*hqtY3S?P=63k9ur+^R?473<XNbPqyj5EqYlTt4PU_3vT;HBJ=??o{=P!H}LMtsvZ4@ZcYPs;ULpyr7omTivtO4FtIygoc5G=uMU2r(g?Y}^RMCiABBD0~rg7>E5_w%eDNe0E+6Q0VaL-N6q5N;J3<5|CV!q7J6k^w?@6vy)ow47Ad(9GBr?9U**z<J{C!NJn@IBw`*e3yl)b4=4=WLv_t5Vv1W$lMY5xUmbInyfqZAx<Q|o3jjUvHu!&bZckuPtQqr90>Kqk|`F6aBpkQ|4>T<1QY-O00;m803iV1bT1H*2LJ#e7XSbi0001VWpi|MFLY&dbYF6HZggpFWiD`ewODPB<F*n0?q5MDU!<#6wSBvHITr$&w71t5hg{Jn1qwqTXo<E7%aTG;`xe}P?+i&<r0w-?g49^pE0XhYULOt>MbQsj-?wsiwU(R{(P+MrOpCT5_d;)p-f|*2t4SxChF9eCXZD}({`k3Odq%{*tNETc+N{r_D4I;xvfY!8>1{1mxS*Aq{1YB06Zg<PYOeI~q+3~TofLoQxD<$|GR+jdd!OyO-nNw*fY!a@v~D*-tCZL|x_YFdN=a9<UWt{cg?^-argjz+ab(|UfX$fJfQ^<pIK?$R4AK=Vcf6@i<tkpbm1s7DSUe%{`74g{FCEIN)C`%0oUSBm$}M+;Yp{@4w3B=-9tP#jjD+Q~qKfk>og}CGy5C`2@cAA4-YS`rnr~S7NEO2v$jM|<@ij!R>$QqkOqbhymMqLR#>Ls5H9f1%X7RU)+h6yxar58>)yjCeoTcR3lw8fH)0AAN<XurDndBX3+Tx~UmZVMy46Up;@Z+Z68o6sToH)fW<kkgsP1k_cORgwL)0*K-tAvTc9wBHe#YQxsE%@%9$tvE1TWe>Is37*8c;K}n`C#$fS~Lj4gb^8aGg4yDHeBUaI3;7~F<)|PQrm!Hwb=0(M8HtYxROCQ^B-l;tx=hc;&K6gniUgkyNL^I?1#xIZdTY`se)qRu*?Z$&MC2HmR*1F(nSfgLB<=Cc-Fqyp(I}P(f^Ra8B>Z&P0PAfJoYuphN>D|_@Z>E-!t-7f`mKqW&^~M=NObsZCS^cvx5AA%n130fd64vFci`gY8pz1tR4w46p{*7mBp>@pvHLRxy22|N9lZ1q-6OXk~j^t2-F+UJk`!=3>rRbR0nCioEGrsEV`N(&)`5w{MnKq6s6pOsARxpQatfE5*4Q8T5uVG-qFV?{6rsTQIU*wwBZf66ivZLvaR|Oi>cID;2|&A1cb~#z+S2%>lDkYSVA7b#vX!Mbtm{eivALP`Pbh+MGFV^)lBZ7YBy%qK8v-d0lEcES{#bgU@fCB-~Z$4ZxOg{Q0xtvX4i*R7BuU-3Va#cT&HD~hter2E6VC_%kp_PHBT$9Sw78ftm4Rku!TpyF)W+S-v(YXOU(f|a>&Se?LGr)+ELkT;v_vRIPyIndoiU@tejFLQPtLWJWep;c*sH4r-V0bRr4x8^^XHi7Do#BOpag4VIGKq19Hg+{>xmd@X?4N_~Af6jKNgZI(rQNg>$;?=q{h04tDu0J09ly4pjllIGx<H2iov^Y6@TFvw1ija&S2J=8esZaWq*@(~w#gMW)+YC>;+6pXzH3PC7cUM=za{IGnh_NoDyizh$-JUJ3-RHJRz92ivbIMxGXa9}_<fos5SPk4sPb8Wg}}POz%1WTyJgv<MJAPJRFi9if6YB|ddD0Rqu&+17dV`QxYW{_@|*(_@MGRgBO4ftNiZ<q=SzV0Gkj)aOO(-OOwO%%L!!mxt6W{Eona&<-74QPEV;XEay{bPLgfqr^1{3Zhf>P)1mAuWbgNVJP5JOBD;BW^Q2k_S4l*;pm627Z@?Ry^fMs9{vpG&NYJZYUt;TB{pH;yfQu52c(A+kGm~@2SDp~2MAzfEu_*ZFl5<+HKHL;qKjEdRvri8x`ts_+F10(QT0FHFd+I8Ws6~t;fX<c=5u}pYD)L7$JQ9crO($NTq}n$gBTc{%m6AC8!ErPd6(GEthK2R7?(?7>ihuJ1~-KK#cG`#THZ56ayAMUI<*Bl=#^R%oweY!8rgeU9D;&X@n8*u!uKK}+xtrt<|Sr&%>GLYx^Upo*c9QKUv>*}cV?B*Xa^P@!k-CXc4!^Gh5*jiU@RmiFE-(zAswe;airIC1T>P+=*CPmn0<J<gR4e9f*vj4tHJ5Hjpn#8V5+KPRQ=q;$!(9K$`DfnC&4hf?VFvuz;MV#`D&tinLZ5OYc36DA#|I(d$}`yi=FYc>5Pq-k6?^>1w=~T8x-W0BcQXF>)|Yo(4WxS_D#pC3fdn$*qt2AyvEdDZ6pF)(cfy_h;WuI<Jg$*)}<sKqmdeQ$1$TpY86RNI=tuU^Ki|wyB;1d5$L5b7;T~#?|PSlanQkFxu@{n4+a~0ro6T1xh=7kY*|$@Y|E?%N@%eZarr+w1%W7Z2SQT{9dEr(!ZQblh8QYKo`LM3HLgqjN<6}O@x{-reI4;{yv$7k(Qn8thR2;kpgq2_+Ve`l%KGt7Lz0*Yw$@P0c+Z6~@O5wBo&|XB)r`Mk$n1W=T)Iy2x?{fP!@rol<M3h5d!=&%iXZ3-XD1U$dNOoM==iw_hmtTDL#^3h*uxjA1MlR&TR4J@$XJJ6Wo-A#Afp?&geCtMe~*s6w(rU8S^w^wM@HHo82`1mlaph8e03Gi%?r3nKX%-`(E@unUhod>u$_J4vK^%GwO!)%$Iq<zdhSZ|w6wV1fqm&BGF~gq4Jtjp0^azy5NLen#cOXG%Q<HKTNEk$#YbeX%&VH8d(NNmN<Q2V4TQB7zl+KL08mQ<1QY-O00;m803iTp6a9w%4*&pKH~;_@0001VWpi|MFLY&dbYF90X=QF>WiD`ewHy0y+s65K{}lv*0c1*ZdRTVsO~aZjNi%e9(LI1o5QsB*l5U|$hNK+3S^Br{d)$#nijt*vGhkah-hKDIUw3j{_ph;%S*=zoc2xEtN+q(|S*gc*SIX*lPL?~{JdvprZ)E(T$Q8WJOXuAWe|J>nl=9<Fl`;uk*S)w{mHF05L@nY}RF$k8wJq|phFL`_VtJt-x1!!$n1^h)EuI`vIa$H$e|Z0`S@(V`*3GKosg_l3p6a}eH+o0;SjbXsfr=`uMHM~VgozUCEU#)6H%NFF4*a~&s#;{ReDk0Z3I78c6-g330F0HANwgAaC8JWVRi5!{;0BBrrGyWW%2w&FUHb1TLL{OzU#q&}u~KgH2iZbYdQ<b3xDi!iE1Or5>>yNq5=E4Xdzn^Io~2JbT1x>^D5Dh+j|go_0hq$hfLneq?p3O4J$07>_6>Bv%Rk7gt*keB(&N+pPNhi{=UH9msU~Z+<J}rMH=^9;nfeKExd%*Al}Y_ZWvW&pjWlc?<cH|~DN+fK03&vlV!WfRs6HIS^Fhf^JuF2j1+1%qD@jy}ECE3<7<?PU+Zx1x>v&`h+f|_8(Q22bkwW$27pYu}_$kWo|1IO1R%isv9Jp8IWwe$#;9ovPu>dO}hF)AGa^)nds!MghtC8W%9nZhhk_IEnoX_-tKkoNGzJKT5vG#_w%mA{tSd7A(z!`-T{4-uI{Sd%-eqb$o_2<iPhSrU7$rS%w53PIq?(*+L%V?)7`qNwXMN@^g7Imc<)#5&t5pWsRt<K{dSeT&isFiGJHrh1r>wxF8$iOjx{1!YKs>A_p)~jHNS-*7Vb7#>~@@C{(biM9AjXO^h`{_;R-J9;aY3J#x^E9)c-gfqT+kyGE1M_VM=34`EY13}ofrpg&VYMqrt-+<fMfY=L`WU7g{v(szZ{NK8`t?8E!%F}QpRH$cIVQ%kVw@z8AshF677ySKY8fDx;eUrlk*1yo*+KT3uYdf@_usw$*PB1NCvD{0`e=E%XsaVsv=QKH(OtwfEd3<adQ(?Xme*{I5xg$J&W(Pug@Sqx+O16jTMZBrr+Fp4Myi$@^t3nZGkoIueh4_(#<*vMS8Vh%`uh^>u$?RFJe`l_<uwoeuh1Hugw{17MBZUKI*yhl8;}@CK!Gd@nTeGAIr`-ayhtKp3_V3e(H>lA8t+npoZl163C{8*`AOp2Xm*ynv^YMz@{q$Ek)s_EQ*LK_K9fSlC~%4#eq*4b6^kfK@e~r>p1&9^TX})@(@RwT1=GJ^nwMHgPBTgb?F|#QdUR=Nik1N2krm+<{P1?SZ85?z@knFv5$Pym-?<zbAHinkbME6LjvX*O4rkVmACYF@@634tOmMy;Hsfc2`~pEBj+YuQUSdoZRzmc`xg%P41s^{;zjZ!>Q`{PhjO(2M2Xcy%Uo}eJ=Q^N)s;aVe$hH(UR>T_w4N^g?>mB3cd5g)?`Tco*U;)}O>3B%Jr}GDk{!0gTyz$YxJ+!Vra?%TODL;gTC<Ue<;37R&Z=@1%zYB^_9Em9AEdAClWi6tgWSK__GRab{6^MXhIEQX9MNW{SBqygCZ$!41k?kG5X0y|MC~kQ`UONBvI=pJP03p<3J_5#7`Kktu^e)H893bvwSqj^awUpO3)S`<)g711u$R6<pnIjJd#+3xxGc4KE^Icu+>K=K`4m~YsDLa)S27ipEj(S>7_=6}4E$&jrP=l0}36v!BRFrFja0I2nQ?$Y??H30}sQyHHJOm9&o2)c&2wfC)aBgt5<Dz`5q2Q9I#E$U@enK$C%jOgOSuO)7sh<it=jp377xno1gox6DITPN&IAf!xKO6_g8d=48DSHa3`7-DrZBf$NkaoZn&Dvx-*%!I0w93#j<EjoTGec*8j1HZ&m*?b|;f=c~o-V{{v5r1SSwJEuIH>@ARuyU^>TwRmA*B!C(h^yopF~W<=@iPQO9(f&w~$YaXAt6lg&siYgP*{ehT|(p6)tbW*?4@4;EYRd-4X*0fqNO)W#g6t{Q}(+(;0#kwqTZD`r)+a*uLswJ3-YNLJLArFW)#f_F`=^5H1IGYkYmko%Ca|->)m|$r@wi5OCXYlb`*LPv@C_uW@mJuHj;2F<y3%X=j~mK8$s{aL>!rN|5_k&SYO8=55zx069@cYq8yG-;XAe?1~hkQ7x&71HQmS1hOraMXO>uIm7QC)33=P)j7Xn84ee3C|nSQ6fPQUtT50J3>bF1R|CLMls1aZ;?Qu5)CMb7qS67DT9ddH1#i7xqHxR;4w2h{^lg2&gu5?BO;uA~)f-hpwFMqA#tKWfurQTdjn0to$?T)}+pzDLb7+ZU|G%I1-`I*{RX$V*Ob7J8g)_|KQ}cLW&!MGf)zKYU83|#-w&7l+o55Nxrp-$>iV-eKWDIE;IGPG_$8$0<bWAW|U%43oc?EcHEUGa(T-Sy)mCRj@_$9A#JJ}d)J9o8<(etbcWC-)t3Mj9!0;XOu=NER?Hk7d#cJp@6W))CTg|j^kBHgivIdf-dIvKXu!g~cBKImC)&@jLx(S`%Ryd%Sa_S!I`@b;K3jQ8lV0WdaPxg}cDFrj$y!fe}1Trf&h5I8<auV-O8n;3{$HZ|R?8i55YRV`rC5VDx9sqb>TnP~C5?Iv|OZg~$IpCL-Sy!`TP8hnUW1GquykgJxDGG;RsyF_8UMdnnIhzz?r#~t_iE=xrDWGw#Bu?GP&#~yLMEx=hsRVIpRlh@USKA<Wql=$NF!KN7`eR%Fhy&SL)R;v*vz5bV_78vk4ITOldAIc|HZ}eGRSPE4^-t;vw7#Q=WEc22e#GAQmbHmXMa91;feMgU0kH`!(qTTKzFOOuhhLDK9Ro4v<6(u@*+;049kHjk^ky7MKC<!2?T#Ts{<H`(+v5wO09bh%;G8!n24$mGp#Zd_H3S2(eJ$Bf~P1nc}u#fVX#Bv)>M_1P`Z)S(8j^l_J@L0gkz>2sIbE$9DO_eZoKEJ-ra62$?u9ZWE{2Z+mu<GX{%mh^f{&V*0s%+J>T?S)!NNCSH@RA?BjwA-u^B*GJNm5Nq`3~a(wcgn7&FQ0Z(#3LpAL(bKXt7LF)HCd<sw`14?C4zqc2_Tb7W&9{enTnrWHtj-E0i71-l?p9sT;cLQ4$mwu4)l~@D>D@y0egFq@I2ReG!kUnvc10OUIn#=FR)j$X}u+`}P4PowJANfT$tu@pb_n{LkGV-@S4>#<p7Tba8$UsH{sAJ$OGzpvdyTcUIN`Yh(^xJ&QZ$un-u_R8d4~7!!cH=c(V!0Uf`aqC_Wm)9WRy_*=Q-Gcuitcg}k4jRIQl_iJs~QB+K<jTKv3x+_akQfn3;pLKExB(Xbb;CQ)<OQqIW!Gt4ha$k+K5)bsH5nRNvNTaj(840m#Tpl<Fgoc3FEv&_sVIwhz7dBfDEZ$?4Nhma)c(%Y!ZFGFyD_XR-#?n8W7|Cq`mPw4JnwMBh`Uf!-$_~}^l2%~-sHg+_M?eBgl7nhUrbNxEA^kj2tDaX>LS|4R!;B3@_T-%+{*LapQaocR%A~*Wd4iCwXKbSB>hE)u7q5N|Bc3(aad>@_iMA=rQC?{KkEnGyANl(tv9kFScig|YGN4qjy|#flI!=FZ!(l$33!iQ;__j!nM%*SDya#N`r~jZ?Ns)EZz=J&10nV~Y%oC2En^o`^kIcO3P2kL?u*REB17~`JpQe+*xxT_r*APf&x4wOl)?NkT8$eA1h8qV_oT?(C$QeB#eU*oM*~xufimY1Y<yMx>J*L;+zW;&3?+*t2Mfc(nSlIrYR{prh{Id)BamhspB*2ikPbX0D#ARMYdcub8MbI0_&OWK6-taU_1NZ_J`!W6;0`+2-LM@Z6y{m~nN<&m&RzGWKaMEibeUWP*)x(T>b`4xbTo&VHKv*v@yG8+Uvf52}{NePV4`4r0nBL*VfZ5$-)JYTcr7cSY311Y$s!-sSMVNi?vR6wQUm0_FX*1t8TYycmpn2)hnrg0>a>^BP_Fm21tbhdBwe?0fPYWoT1?M5fzT7oXLsx<aek5u#KpMbypag9NEij`4n=|f%7#w#&Oli>I*)67fd~W;WC@<=!?oFvSf~SxjPA7ivMiYr?;qt!fCqDlX1q6F2GM4StDJxUZR{~KJ%LhE3g%vtc#hZheEdcgqp6p_s>o?Un-OiALPJ>63Sl^%;2hPh|%tJ@Z)<f;7>6lX0j&hoLP>zXuafhHg3QDBHz@ZzB9;@82W#0JhCinjfAm*@zuVA1|5Tt#B<cMMjpu1a~cP_d%jZuB!0Po&H^Z;SNlRng=XwzOP?DH3-a6M2X+)c6On|AmL+6grmcZ?SO%fFsmjInE(<ZJlnjY-T?_DWPX=+ax(pg+ek;5mT972<H!!J(zBcQ(1M$KZtbvBL2_S^yU#6YpamNv`Y30U&f8qyP_`X9x!mrwJ4mhd*}!eX%blCmpAF{I+uIkQzn@HobxK8F_riw0_x7H&O|8q28AiZA{L=41tdi=^8X0#UMjY@JzkC&z?nB<7Zk{>h8+p|C8>3npsdnWHqEZ`J=nf2*Lo#C2a?1_8m+oUbL=H-?pz1DR5lnE_0V><XDwlV`q^tv@@(ezYH^4W;J)x3&SPgQ2&m)XR*RiFH>1;L{__B{9horc1K%oZ9#9vhK1W)0p&=xmYKOD!0p!qAp(%D_(0~K|J%U=Pj86jolA^-uRreGUo?t=dC_q%!)6E(Zekaf`k5TcIeD}Bj)5)DLtj7e76!{DItE0lnZs<TBb8M1#nQLF#sKV($R2(rP<+?z8EE*_=sZV{wWMs44}O88HB|gr0FKQUEgU0pv(RKowtO)}$+`Rb``6z)TLtA}wsuJ8-3GAVy)v)A9fJ4ms;4a#9x=C{0X5Dz_*@`StCcK~Km5&-_71&i=kMQMG8cr}Ms(A)zob=1&ZWBFR58tT`qx~^;qW&0{j=fdrhL#tx;ZVFKrwajyyY2?z1KW$Q15ml^d%bR;4@eHUhQ4%GlFOVEjaYP?A{Ng-M;8=x@dvpk<{S#gt+YxIA%Pl@C*tE=VreSW158v_pcFl;M^r$^|8BWW}#^zICF#P?Kzgz+*!13=~qx;`&IsO5*-L~OBj>9YIpxBFB3kZfh>NV!B+Oyn!vuEJEGASz$bV(qrU-j+uX?RoGqX9eW|9g&&L!4SL0gdYlcEpD)UFrIthE}yg)R3b23k<_n8=kka9>-V<^x`+5koxv{fuk0-x(C%bxTsDzU92*%RF>i^W!na~TOZ{})h80|XQR000O8001EXAoaz0H4Xp(+bsY96951JbY*jNb1!sdb97&GbaiBTE^v9ZT5WIRxDo#DUm>U{Af<|8C;N6mU3716uU&L|x7a-t1q_3rCEDRdmQ+zr;tTTM`wU4*lq@H??RGzGY;uMi4(H{Wq2;zLs@nTi75QXc6<e?5^;ITUo-`Am@YPJz`%<W8_zV7K;wQcps^p27Oze2R+m?HeD=#mbp>kgfRhu<FefZd{@L|h0!meELT&;_0E2?&bcOO4|(j=d%A`waz)x;2syHZqgEAm>!HCOEBb-WezRgt!=<||w6nM{xJsCt*~+tXkJS>)gE(v7G;3!YqwH1fU>iIosc)h@5GMw}G+T5cNFzlmH_yax9>J#N{|mqTXAk%*GWTLrfxpM~0G^(c>$sup=7940eV|5X0=j^sULvws_wj~Tz=pUv20GD$M7l=n+<Epqvrs4gcS{)s}dnuen8Ba|3<tKPSgr|may?)iT{Q-ARz?>zEmkvE;_rBbo>7)yATF%~G1ts}1%cXj0PG-aw3iR77CM9qJxb|N$q&_B7>q~3h)5o6qn(vfH~-1yvEoG;r^kts1sbK(8T%S0aB7F8O0fAlV<9?Z?7ub#~HZm}veSqEXaS|#dTl@BdFka@YQad0&c7PBP@dw3!(iMLpW*cX&xr;=18MJgJFXyV9I*Rp&<Csr8$t(WjR_HluKHGUPZ<sGKPgjUx^9n{;Bkrzka7JFrO6>GM0*FJ-GeN|U2pLgc@X(exjA4V-fR^TpKRTTAnNPDw&2{)}n%!j@aZ229_#cgxO7Y>`sd8w|7nu!}x?dLORMhH^&R9n}%!?J=fkcF4561Y4lMyUC0IyDsG#ORHDsyTIFlwhV(UUN+JPK_a7Px3r3YArfTWd$-#4HA_UmCQG+BT!=;@%9@32PLm4=$P9y0u{H~B#oVs70MJOaxFzgwiwLLFCy>7%U7;LP#gr4j)yThI<G3e4|=AamodU<T5Mxj0>`0Qm2ZM>@i3NoEj9=oeLngJ-JZXE8F|5XP%^f^2qQ10ct6LgJMg{fv4oSuPg;eon%4e(xAQ|><{Q6%npp805fD?i1gR$GWhwHsFXH`yz`jiVOGk6j2nB7!o=FEHk|oaJ<|8!!*}*13<NqL3QpuI@>MN=2N&FE89ZZPfhG*x~+3Nw#!$5(Z?LdXyX^m_O>n!fgiJ_teeOf+f)4^A=8B*M`+rcMK*uYGCth3jV=hOK}EKv9lE{ki<;CY_<AJ=*->j;^ZaL*di4oOz4;*LbQ(1``YzZ_dgi4CQ7EjdwJ^4YZg15S*+UM`PaLF<Q6qfY@8LxD>Kq}k3u?GlDH7=#GK2AQjxBTEBdm}`#-R{3FkTgh55N;3hC#3@jcaux{99tqbtKpioHj(*V$KKlxr8#{7CbR(tJVUuJ29~x>-_&fzQ)S(;8TU;~mL7FaN=%m-_rG+?wlR6I3Wk2F5FwF0LB&3RxEEB1Y5pNzMFR2P7Ycqc(<mQU}{0%gCE7ST)6LsF*?MOH}U~DZj!B`mMEJcP07{)e5081R^tr!U9x^E2>V09YCPW0I^fG$O2ZjhiIvKCI(W3(pmyVqz%F+sH6lQ_Osib|x3s|jl+$(egxR;C>gr5$m%-%-5lG~bu`4AYL>2SU=ka)`pweUB&NF>#)xE`T$P3RXOUVUZj80)|up0Q=%jB)i6qN+L*Uc*X1FinV@x-qTP*=p8E_Liz#7I@6R`?@$wF#(|k#nL!TfK#46^WKc-!@Q6YMCgX>M@QW<$kuA<IDRy4Yrb}>;Du!sZ?=!<f<`8Dks?q4^;-?}PBk>(_8mQw2c8cXRv(5@$U$}_ykZl^jd5ZWjy%}#38=mah8Wjv(HSX<44pbkFQ_K7YH&<wsu+Zssd4#e={3w)2S;ki=6P~13e9Ov8q%x_&B#Rw!=wltwKlFzNXT5Ksj*0tC9#{_8=%ynFpbWgl-Z7(uFn0`X;uwaZ*4aX#R3VGh5u*}R;K+Q>wwRZQ)cBqhStb&_Ch=_ruyx9ka!0Tp@mB@9tf6WqzZUzFoXJ}BvEn|-WK;GMUKfneW@=BOW067OdSN}<v=D&I-S$C1fGO0;k3MOD?!TXsQ@@|l|3AL)ZOI6{wUIf_gA3l4RB})}X=`{rs0qzSMwk)^&C~*&<Lb#k1I~zQwi`zI%<<qpdxMuXLpXHMflP<XnvEUB#!4V#qrh{7kF?Q(9|txcnwSXPrivoFR<${Ccb$Q+wn)CGwnFud+M`C-QCHd71kaTc)G!(-2pSp(-b<H@>U%3uNIb9HNh2W~t)OC_9I4pVsN;d4h_;y~Q%VkIdn+Lv1df_%zlHzq?{7bRw$xMLw6vY!Nxg&qdl^8Joljh3Wym#_H%D+X+G87=bu&)Qp*>R=%~R!fJq&DQ+XS$as<;bUxwbekKupTIdcEWKPEcq+OTgA}L9~)7U_fxfs|_yq<mCF6KDkW{E|u`!>hoZ~_f-Yl<-cq0q{F}4ovJ-xM-LZH`)zR<NTts+*VtxeZMFh{D}&{=;)O?-@Q-x~W2S>hb-0X)SJ;VtGvl{Q8msIy{dKOPMaqzEf>vHS2{rAUxEL78yUs`_il!#FfohH2&YFd4nqsZ%YuJt)o2o!V-HzEtaeuj(E(!Qj^N@i4!MWWPpXH0LKMc5`X0E$A%LPz8rzzBCNO(!1>b`^g9+t;Nc2{U34aCN+X!PVH!Y;e9YavR-WZp37?6>Bq6WXJ*8>!?f>j2@`rN5TN?Gkgi;u=w^?VcO(_K_-{YMf=HA6@QNnN)~GhHYYcuU)~=RUhO9wky#UqBGQ0y@B&ybHe(J)6Ae8+#}K6q>+(n6ZY7V=?u;SmV%BcZ*M9v?EPN7YJ`Wlt_p^o&Zs?C=rF~`SA&sL35s$gvLS)<(dAM|z%&f3-l@4IGAs6V8JIuOvHHixmopu0&YCW&@(1lt*WKL)@R7c6TX7%vZmj!!-+s)c-`IrtXojASt#{ntG*mxC2Zqh+A1j9@qIJ#mEw8SrA_vy1E2PK3UlV?Uwd3M-kzz34GVFXgm3)&UM4${VWhpY53-~r&4kPSN?reOV3LS)QELQ^nTiF@DZgOEeo`*;mImb#cdBW>8uOm(C(GbNQeiR0er`E{&ZRBNqyGl6_By~n~C83($RhA9GEFE8KBG6s|OaSj`3ME$D!0O>lRD0eD)qZyXgI=_OxNT9_Cnq>t?oFv49<OsvUNjAx$Yx31u^F#@XP0jfMJ{KvW3}(&)vm5|-OAP*3Ju9Hp@LaRLE=b}_uXvya8QXMSv#+Y^MeN}5X@W6LpzlJ=jPrnJ(4DLDjx9n>Yc{eLVDP0zL@7MVr?XF?a3I3StT6$oW4QrX4(T=b7?|>Zl(Ni`swrgAKJRs!@xh^e)$ItSnXlt`=9^aD#eg|`0|BDtcnPw-tbED9QSLyj35#<EA3vfti@IZj(zEPNM=GY%PQ9G25qup6|8;du&4KSkB8Hfvj(r~E@tM3;Q*g&e{e!4g$7eh(kBv43*g30^9)JT%6*GGQscBb1z-;@4zKJf5Z`|+Z9<YaZ|e)E-tU5KkI72b%q~VuSqJd;M*8ow>*mggroGgI&c?wm!!C;6W3kk!q@&UiR{$|1(|_Air85brb1f<jhIHAYOO~Dbrsc}F@VF8C)(O8`mav3KPqh?xr#M!&qhmGJ?KbJ1x2B@1b|qD34nV$@Kod5lYE7!3uS$xI(8v4<jr#75Jf#D-*m0otO8=;XNyWJ7FL5AgAp4>;eI>hvUXJyWAuX7~HMO%g5M<<NW4|8h3m#)AxwfM8#k;MFe8b3GVH>Qdm;&<0_1H=;x#IbT@Wih=gbgcz5o!L#acx@3jts{jAHDOMt>=DOl)?|cdfEQ_Nq>*WeiL90I`=<lVBm3&imH9=yN}F{zS5n2>SqM`HM)YqCX#O8KwH04OCz$`mFNCyJ$v;(XvaT;I?i^v4h&B5?@hzr@eGJV0msUW=VZ0HsOc4mtx>UjMFF_wGB<I^zR<g9-xU4k5w4{<@)&p2gu)n`Qn(Moa8Py7=E1^0JvAHA!RsopMuFk-3Z_Tp7#)^C^`H?#C5A3+OniTFJ{_e<pc?2ZM}Fa%A-%YGHBL*oRpa?3rH4DY-EC3e1C~+QhUlS~&Uv}6{L4b-b=$CetW1EmE386yyD;wr8a29g0^ijIq8`eGbynQ6VkP#<eM4(SV=J+yYAqFndnp*`K7=}6A~d`AN(9?+*H&PS!2*`aW5&~V4u<8#*VPU7R-mM|dbp%T!YBCnE|s;z;sDogTBA4}JFVq|KHUg7=#?d!s|!|t@P5v-y;olePtO4O@A8yadn!wzu+<8H&-(?_rZwz!7C^stgD6GiEQSCiP?0OZq)et>iAseUrPPm88tfpFC{G|Fi3+h30Lv@%2N<8Xf&0Z{nDmLCcbuG9LkUKoq>O4}nU23$t;zi7=c99SZM8yZ&-ksckI`6g1K`XCnhfRk=vo@#&ZdV#jJ)v?UaSk;H3|m1dNcCW&q8dS1kh|vqWiJKuqJFDMD8CZPnc?Fc7Ej8;BgCgeyKO`OEUaEHtKxfgkAQn)x~=@oeg5Pg{HP+j0<b7=5kn#v@umTu}qX~`!H-RD~dm4{r-^CI^xh0ZLE(5gT_n-qdKEh+u<-OypnSGt`euy`1NDcvv~UW^u@@Ybyc9+7>xH+KQNE}Z&DxATwXAd$qhAnS=)6wuDM28or3y;EGPd3P)h>@6aWAK2mk;8ApnLpW}luH006>00012T004AvZ)|ffVs&Y3WM5-%Y++(wZf|sDVsCG2E^v9xTkCS$Mwb4sr|5xyEa_07WP9Q)9d0cxF}5nYElFi2qj+HuXp)FP06_zinQW@&*FM1RK4G3@zjL}9Y*38tsY=zZDzOCg?ew|d;CbE~naMJ&q^MR>l(NWGQsw35a2`f?G8S20$$6gNy%H*~%ZR3|!z5E8$;9=hTIHEo=W(4%<$IoYaIh%zwFrVmUDc%w0+FnXysSi+VWY50@=P5Z*mvc!2umgHGj=UxRN2QLRi4?uRXs1tJd#S;H<kQQJ(OXg4=loJl_qn0+BF_=%mRQExJI}HB!dHiAD->oO9in^lY3ci?7|LDqMz5J={PvKy0|!-^_rJ<GukDH^7T5Ywu7ZqK%5LTp6tPTyn7&1B^?m{4<I-=h~+}e>m-f!A?|V_d{CF^&|6hip$7f_GO1Se+{dK;B#c&J`gWuGvv7Gh%HqSRT&`tS9gg0dk9^GZ^e#{Juif*z`r@F+_IB+#x1M>{FnRC9pN0m$W<_2;S~L$;dx?WHee1y%Ibout_%2LqIWEh*?0A=k24K$EA~v>cp2WCcQYBJ}I9V)Y>3dy`HJPSL48`rc1AZo?uEX*!&L6UlUhJI8v;D9rWEO8PPd~r3p6JJ;$w9R#q&E<r-P7wmp177(7>8AedB6O+XIz7UQ7a{WEa^TLK3`R3QgpiF3*r5+d%~~HQ+IpAD37IC-2<7$Dv7b-@S6~9-2?Zel1Pzffh64O_YN8iQiq=jP_qs!v%B}81#Gi%=P?$lax-u@vp-*Kp-3_Zik|RzU3bq4PkxAGQHj?8Wz2s<mHp1{yo*&+Wf)0Pr5~Bv%<+pd$*KTHb-Ev!Z!uEV;Lx@F_*~-p-SO@pz@Eeq`&d(ckWo#@fm{MUOufuA`3WB8t-nj;1MR%X%eD6_kzO;IAl12_&-1z}>PlhN?YobOfuZlm*}M^AvSak(iI|P1;{K)h?eG61POfi60WBGpY{f)hjE;|vg#1v*G9g{4^e#6=nujrrX`&h>FREOe<Y_qf#SO0V^eJuj+kgI-nADjF)AZ?6u}Df)^~eTiLV><fB{l$BYt^TG)QclhfnOboH?P}^eewSKEfn9-3+pQP-kS|n3ts$3r;@6}AzDpWb-ba0H`h15xBx*@)hCklBPe{S`miT;ihamJk-aAmL4|86aA%dM3bz)gB?h%!%hOnvB8<vhDZro<S$jTFdjkzVluxkqajcNpeNyHbBM=UOG{z_@C0vqNq<MrZZeH=x4La{+naNZD!>L@5-%y27<!k)Eh6pWzvOG&SZ7L9=&MFb5;77$9E|L#8qp0U;5(%;;X#!RiFvrOPil6r*@Qq!C&qGMHhGI*4>j4@1Y0>8+$k3N-x`NNU&2XxWlPa92lIEq7)@uqmjuiptY(Fo<ELy#42nUM?(kdkzvqiXA8b>bH9tOcGKL9btj`|QpP=y&n8j0ZDYR$3Op_D(>NhxEW;UozcDhms>$}1X%PR9=6Jx%~aGvUPpyaYaK6)LcdA;JiJxW=s_u}JfWftZ7icSM)#BN5iPKQr+2qr+rPz^iqBC;8R07tK4c0C0(yyd&-#%P0khz(TKp-W2@DOQJl3@T@fsJI`@N87rv2CDzBAuANhrIm|XaTpt~tYHECwWMOLX&a4YsUqA0(zvy58*}R7FS46B>%r|5RLkNT~MwY#Kn|U2CWyP2wnah_%8^f|NG*(a<iX^Ksi3pfVB9*FeS*5|5+ItbE3@tRHy0?#V@F!ttS%+oJFo)o>EY8=s4))}|^0|^FQ3{S*T1PZO_yNn6SWjF9VOY>QDSoQMlo`IKC7jiQz6~hba9mg>@W{15)ZkMqFr*EPK?JtKWtQUxkrf6=P5~E6l7+#kM)A<4nRlSUp*YHjt{g%jK@cFou+cX)u*SOYK_<%;bJ;!X0Wl}$s34b`G6Xm43RZHePAcgWvz78k(uA5!7T$HDz%HcWuni8EpkR7u<p<Z$q7o(K0WQGAOO{JuUxZ54YX}H$2s>e1LUN84TeC4H4hKlk^-5MmP6ag^0i=nn@$z~+nVwB&<I7nvxw@K>#2XC>uQPPiB=0vFUta|`lXEyZzgRc@Po*|r<nNxJU0R=)rZ_IJoi!Ox$D_%~TW6Nm33kFRKttj6iCckyeoS)Szu2(6_hAtarz>JaecAJ?lhOI(MOCN{?_VB1<mH{7#|Da~u!(A!=_i8T!Juk$CGi*d3;(Jp=O<7qy3-Oe8@=IONUiMdLhmqJ>zBS!Dt)U+`c_fszqC5&y9|pb=0E&~)WXDsDIa<<`fD&9Pp4;Bm%+u^<;`q71qjD5#5lA_5!tdF=fx|AJ0xx)1xeO(EgLRJ2$d~~&D+Xd-)lTRrA=>-dg8bzp1m_<`?s63$v7Yh2);Uk*&hmatogdCF0`2;KS5Xz;mzw;Vpi5TAtSK@i-(mUcd^o*gI3Qsv#VhG_GWf^^_NR*$_ut=tkp_ucO$~a-xS8*6yXBc(YDp7!GD4<gVY-8n@jN7<u_-Oi_r`?GyA(^v&q%@boc$~+33yX)pT}t@@VLrvrA&K@x|!u@^n10XlQtQ^4&PNzPcU<XBXg)*7DO4ICjj;8Rb!n^PY&&#q05;Pb;mu)f_go^k-!UTERa6PZ_@;hNqY%3+NuwA7DB@X#CgNctT)aj4sc<8Bgg*@_gHa44@U4X82+}dwZoB;N6Z7`$WifQEf1!ZF!O*CIw)}FJ3iVs?{;9Vx7j;d4vn_Ge~C@s%mbLZj9oXK_5)d{>JFKWv1Jn2>~JV0c8kv%UmTQpdx{DC}G*wU<YWcxTi&v{$NHSD{t)2XhzzEMCSGA<lFJ3zMUtb_)8CgmWqqe(`<e|n$AGJ=^0Nyzj|{vYmsnzeSS6zNJQ}V+0nBvkB^=m^T^57cjE~mbar))KQ2$e2G=ynXW2wpL&gsFajx>Tu8gJ1$od=>Iow7-3|IA@>`J-Lv*c$?1_QhT`lS3>TJvDSjBPahC;}U}5o=4l+TkjDX3IQZrqYinBCv5{tmVyY*jmFuVaJ<>_oO|@j?!=AewDCvswJmZ#c;r3H}rRcT0YKaQ4c`apw8A5t##V7?BY6u-Ak648E)IXW`*}e{A&dUlj1uH5Av+-Bb!LwI;<2^H6CkE#5tRF@{QeS6;bT;>SjWO1sfUV?~KA?l$~}~sOv8NA#3kuQj9+QHNC2{KWEn6#)>A>LF{NVY5SwaCqOxD0fdisq&qsXgVFlh9gn4EB~3$EKRABL&)P#<ha2)CA@uAr5X&}nAh#qcIdK9Vpgj`y$8~xw(<W|N+mNrcoa>9JP2MtW+80!8c(tESptj<kc@D>(fDX%O#bA;u<V@uvNgL4sLMwQf#%-eTiZi`Lm^6%NmDV@BhPj^uFRh(+A}<|waDb@;P&O)bFB>h|pwj;+9~y!f$;u%fYJ-0$pWJgY$vvAj#>-V*B`L(n${s^{rt{lALI7N*$<Ug|p13e9&=YTG7v~3`mhXeD=xK4=@v_WyoalOJVe~1b-P!TecBlL6yErL3<U}Ey)+`IA!(fo#@uR^PX>IaJcD8@yzuAcfFiPOI#c?ME+Y`IPJ89i6C22c5UTbgHLH%gfb|RN<#lp?8u!|)*9Mz#chp_ir8!4*;4M?SoWKs|g_N#!MoAw&4^&E`%8t+{14f#eeWv+{@+h;vOhYr=0AFYUP_J9W2s&ZbkeuON5mwlp|lUy}#{DiD_)qxY)TG=&W$q3=-Lp#J^byIRs!v5TM7doQXDR1Jy$5{^FWP3{UBFIQ$0U0?O9RkKoY+FTl@(@}aEN1PeC)%6&&?)e236hRG4-{;h(^#g$Iq=rm<DIJm^e^za^OCqe$?oAEkyjIFjbkjFF}c@m{WO5%vctNq>EkV`Ol3@yo7S}$1OCr{Y`sZRjf&=^(iRZ548wzvo5u3G)t;TC`{$UbGk8MU;uZ-9E=~eY8zlq!bN~3rKmL-{zK)h-49yTztl(e0@Q)EDRrGY9kaP)5{%5?T@EgQDxj7y2ZpY8?@?j+<%oRmP#K4UP1~0M7PvtU<Ht_huOi}h1;w4`A&;RT{qf|iz6QFDoNM|+J5?UTUH0eMRW3UZt`%12bhSeA%riu#~InNnGPPvUFZERvDVPJ6r*_$=%55pol{ig}Z`>^u-EK~GTO?GqWT-^sUI=3E>j|lWotrK#J4-a|b;j*ZQ#KXO{T<7J+uX5N}J%P{@R>Mf$_p*E?!x-)~)NT|OTyNF(-e`WRa%ip3%7ks4|FB9TNR@MNS0~t;Xk-O}0|Wq9gRN7jJkH#}Z~ao@z>F0vr1^+PK#jDchgB5=7F*WQxRKB)5rq^%v7&eqmcRoVJWlnSLm3p$WCA<}+@ubg+yPJFmzv8N1mR&d@AT*tjPWUY>j=|ap?w>c@dM5x0o(`rW8lEzURzDjIkbM~XcluvzGw<T+vxY-1(lF#52e2!X9DPL$D?P@8U`17Vs(OT-;I2@FL2YL(4g*akKXC$j@R?Lw};2?+S7POS9~pA9^vPaVV-EkY)tcvOi0@YZPJ*Wpdip+QLx1**A|siCk3VPv`6V6tWVfVDTh`UzV~@(BV-uVeH_@S4wOR{bYoz8gtywV*U+$GUnin;1?3-sqvLpTeaqF_RVHWD>-vcbNW*tNfmq5Qw<}5<8-VB`Yh0U|;VKg8)DRvgG(RQwRo_6NnE6s&k)bk=#W<4&PJA9|*vt*Jx9(V999)4p!or$WC{R*54#iym+I_FZMO&8Po^>4pt@k*Ah$+f*T(3tpzTF=upBKQV8n!9mjP{+=w<K-SMWk)kb12s>VLlrlys%m3zU*ieeEOnBVP^NhR-8m|tM;O`)9s1t?`LnXF2S#FDMd2$j<C(_>f7<<+20T?z<r&J&d=eqOfQDsg3@M>tG1Wtvu^{s`jGs_Hp;TJ@NmlaSLatct@kE4J)7u>ZtSp41b1m!odPlGhA{1Qt!Z=>!_cWPd)p&}5=iw1JAn=m#@T%{x;=|BNtDj<2AsIt-pu1N5MEPewa3-wRZcHVt_`o|<kbW!`56w@z{!U>n}ij?M8$0GhRSGz$C)jHE<}vQty45P&6f#PrT1mFoZZ*qhrmR%!8*z6iU8S@ZN~majRqkp{J|r0;A{%!4BV0!X740qd(9+Hf;SU5E4#CkBQScNlchELaEeh6?&N9Ry|m*76jArUo}4WalI%@DbOYI%oS(Cat$YPlo=Pf-Vypcrz`g)x{3B;A={wjFZ(t<6N1X~feHz$wt~Yp;#C4}}UD9yio`kN;Ht<sdTkQ)tT^?{b!W(GM%xm0y?LB*gc6_kiYFm`@2EQ=JAcg}1z>J{C@mEJZ558_ebD(AL#-}%V5fskfciv!s6NVK(gc}A9Z&4h730EM_0~Jz{(tr%)XZHbVY0OE}@IDOgk1<cDs_^bqs;ErnCXQrW`JS#W{pyH+KuY7&c?X_rjO6ZWpOc;edAA*!;m7s5P@R@~=?UH0Gkj*04`*$`t!$|~{IA=dwBwhvc4JlDC9iw!F&{LY9zI!t`<H%{8n<1a?e#e9-a^{#VT{l6di;w#^@n)am@}yIA^z|23nxMI2}|@j;O;e48d_%I-?u@S0i|ze{jNp8G!578rG&ke*uB=403OxNygw?ay*7ozoKvOSBDSd_lT)PY!dlr5bY<WJn6#4VMjXHV?eG76{1sI>@`VPPs{=|DX|3GWRny+co+CU~=5p;%U7lZ40Lo&mUg@M8W#ZdWK4)w>-C|0}SJb8Q{D@k>lC0SelNORg?jHM*)I4x<b#kSrJv%ynX$s_SPIMqrs*&qTIfcp#e&&`3T^bJ&aaXMrFpW`I$BrTI$gLps+9o~v_lbOPnl|7HLt9h`9#p?tm3h5fA>tH>pmV1mV@c&9r`GR_YgwAkj>+X4Q1MPuzX?JO(`Upfq^3=P2=QY+2SpL!`eJ0-f_~I^&kL}%O_+1vN<<UmSKAu@B23bn;tp;-5ouU1bt2ss)lF*?Is3_aUF(jVXjNx-nxIZogGolyR7J5174voO{<DIA=J<ef6M%ilNoF70FLZ|-yuy!06gC##ivJTarIsS*Ii{(C$T9rH`T1hO7;yR*7RiT-k)h**|4(}4^OeVr0^wnA?@*IH*f+?roHJyy`JIbKujuwXw|CCA>-bNfLYr7QcJl6G`j9|gDF;m?;DD;BmQ(z0I)J!0z_$)DIQ92~$iY8&X<GA%ZS~a$pTf%TuGgo360mKj`~C#}O?H<7(ONFrnUty4WqJsj4U83YY7}}Vi#vNHM;x~0t0#{CxLm0&7MhkfYP~6K5%_%kW_0pB)AsFXa&bk$)~T+?*$z8MR+UF^uwU8YW|A$a$;_rxb-^HA=5S|LYi=^K&6CtQ=!=uLqqB)_@M=sDlu;$IgtGS38fo^{h*zY=61C!VAWi1Zu4+SuvB{khF0?iCn;XKALuKc+aomiZGrOo>qfL8F_Zeram(_}EX58RJaMtk2%L*64nN_|-%mwXj^sR1INwO%dV+ntZ+{J<$47SZ>ipvT`m~f+;IbK&JxOQrq7bM+_ii1?)smUP|uL-AuQrlUy(KuV<K(6Q;KI&`p;}S{3#;ySb0N3biDg#Gb$D0eK_rL(FSO&(l+iI!#M~ab@w-u{uoqn>)c&33>;#9xqXX`~N4nxR|ZfR5+@tW$4j^9d7x#uJsoo1!KO!GN39W>GH;q$?}!AI2L6G#H-ulF9++x4D4S8=`&-g~Rdv|ps9ci0@=r#PHnYG2;12iXC3l~iZ9vhxUz_wV1+9zXn$5g>MTDZ^NY0{Yv5A2Q5yfm%XHbn@K+ihLU77um7}k(v4lh+KMf$g3k<a6ZJW?>eUw9pKwQyKLHYU@RIc^vw((+yT(r$Cmj6eQm<EHH`Cxq4?|)#M4>FJ&`{sU`~a`{+j>#7ebil`9H#`?ciOx`y%ES#Fjq9GAv~0p7b?i^y@wk(?$3tT>~)LvC@p;l^IcXNFV(Psy=<n)HJuzmFtGPo<8+<Ze};2f_sNg=(bbwTX}#vNEXl;Y(KVg(waWmRz6MlI^Bh|NZI(+{YOPpsGC!*(cv}2EwJqj+qN)5h~zNUY4v62^AC%8CZ;KmgB{b1%wvBc1@*qwgTx!Y@0p%=>KLQ%Ep(?JoEf*3$n>o6-n!_<p9Qu${~Xr{{u*&b({B<s4Ux7fy2=Y*u;n-X^wdP2WcSpZ$%v0|R#q1_84a^pz)hmy$#1NN>BVll-BWB&eYUx2isX*}2Wo`6ld`aFP)%x_D<W*}NN?=X*%rBvn!DO_b>Yj+Uh+pY-avm$<L39AFI-w&?CFXn0rO}~XR9=ub0M2PiLo`=eIMKa-^-d)pokTC-o9COCp}|#qn&SSnmq%!EIEIm(_T;By=kkUm(RW34Y>EEwQx3?2KO%m{TWPBY_d6R!VP+vPUVT)cg{A>pI$cCWlzj`vF9}<PxEp+_^bKUk$A>xasJS>gif>{o%*A4SyMLWnnyZ~Ly~91Ac*rQ2)fQnKaArbG)p?1^5GVakIvE6SlrWmI56}Y71G~>57R<YE)^%pcuW38o7&a{WB5657I5(hfclhrNbkX`Mun=|<9E(Cp6${zJL{%Yy!BLWJ^qf9a<%t|d@LS1bh2+YfdqMP$D>n^HT=Ju9`av3=hG{S<p{#<o2CeD6q+qB@>5gJ8k+A?ZGS|&6=%5TtJe-6WA0~-fX*#4iu#O5U*EDBwHD?x8>M#Bjm@o`yraEClX5)nH8mf^;EDMb(tc`*F=!iTr=eF@i^H!xN6%}k#F@hb^Z7D~PY`hU6$Hf4L0}><&D{t83s6e~1QY-O00;m803iSxBlf|D2><{j8~^|s0001VZ*OdKFK%yiWnyn{YhPq#Y-x67a(OOrd8HZKa@#icU0;FniOfo@>^gShtuo^@PV1Q_@zib~>eX@}60%TJ1Pg$^)a!rmIk=D@B{|+*k4F}OgLA*aQ52oi{DxjLl8g14Yt0J6?rSFbnpK*tMbVT@5pl~T(JMxl{2n0cW>NB-EU027uUHyI(P*@kVokDa+2}^HEF*kf3#kdMDxs<7qEe%g+rx{82Y$7pYE|-uUvlBUsb*12ku#<Ij;3<SOXin<@!Bj6>6&Up<LREmax{8>a&emd@a5B|vr94~(c%8V^6238bZ=46C40L#Dh}!NHDyQNy)E|k7YF<MIeT;T-BC2U`10#7Kb&UgCzn56z|D9>;3qoy@$~cMMY=AcvE3*{uD-oIxyXJy0y00JybsPaRheum4XX58oobcelI+ihOV-=dOa%)eN?atP(Wqcckkc3}L!mQkwO9(F$IgVwvAq{XADCux%`2`nSW8P<DQgm$ulOw+lZv5TDN!sJ6&SwYnl4I)Dl830#%{*Eh!UxQE>N?EVq$lh66Gy;#Vjybny*-XlZi&xjke0epR4Ga>u8Kp-%e^O6==q>P5wcme?QvqPthE<?~0j~OS5}DyOa$xc#>3U$?Mosce4Cf#N8eII+-kmTvI&*&w{m=WlUa$4iIo-SumfVGmTaQ+K*3W=Z@=@_t8|;tcvlQSo|^LB7Oe=W<C27JL*Y7s3J-<GH1svkH8ICv-JYZ!Yg;0UNdd?RGf66_`{bDI_0Y1a(L{y<dxH1a1WxgyNhgYyDTYJjQmE+hMh_&WE`D#$-#0*){WAJ3I&cOyKYJ<2_l5FQI!vAl#GV_pw_M&@rth59L;BC1yf6!Yr~L*NK(mY5qoEY>{5}e4FaSSdz_sCeX#5VwB)j0$XKRZYg+$-bzr=tv$IP+>F>P$h}^bP+I!$!ch<RUU`jz8&DtAh=h=tTk3XMWo_^>cN(jbs_SQNW`bGg+fLA62`woIC8<T=+YMn$v)^0^hxn^r{l%c0oO97$;OrOC{+@fL`=BZh{_x8<nO!mz0)cz*P2By?Ao#}^~g?IzeHgAD(nXG7;QLQE1nu@*Y^zh(-eE&V!JM6&+!l!qVV@8xQBC&#RsIAYVvE-F^EmvStwW9m44`aNaq$_q`@M}QpY--dwbH>AvyG5_>yWL|;BvO!L)-*acX?{{mehWIBSg)Gg?z!Y`K(X36M+5kvA_Xu#cLb>3I)pPK0KnH|3ho$Xz!md1*msF2TfBJ<r?w`iuwYd#3g%E2%q$|Ra?T@%_Q^#T=wAyc_KsT@Yq9$aHo`nCFcu(bC@4eVpctdI$QoI8h3d1=Y$3!Afe#Iaru9FG(gjv?Od(^kVBq&qmnx-c2`$Gp0d@cxUqWGX<WxE`_%2smk!4fm=sN_y=c49No)!<LgQ+~NYgREyfqzgoY3fqYf{0}g##We8R1EQ`W(nkwY)Z}`?bAlD1h^(RDHI@=4WW`l9#nt^Zk#^=MCH1u)L=Wk$7&1$_~yxFo!s<EASkM*q<P1%L+qqXz0S<vem@oL_?{}*g1#zP{h>q^u7wI$l7%Z!y6&yJzAV`F&6tV5oei0*K3*D#v)IQ{LcW>p3MT?)qSZr}WF<?NO<6+e&Y>BJuIOYroxGht9!#Gg>xSrW7&uTALA?cwd<I+gaw(i*5{-LNIvxh3%>qG6-7Jh}6(kH^kX!V0pi&GC7xnT?7j7=X9N1MaEa!<=*w^A6b=RVBSz`VizZaG}-#?j9Rn3~LxiU)i{BDb3-vs)zWNg<RguRPGl7O#J$8+lom`21Ms4GKvG4&`^Gb?R$IfCZ6J%v;t?o8uZLaXIrQ|K$TO2|912QE{P{{Gc;j`n`+m5VK{dV93lvbRTDTK5iP(870{u9f#2PaC5AXJ)FqVKJOGc1#*ifu>A_ihBed*8U!-1T#&m7Opb7fUc?0zS@PLQJO{u-ZXciCO1G^cOc)kpxC^bjx0`)kl)U<XrViZj(RfM-Dd^en27?@D>*mrJjXT6;2Z8J#pa>pZM}DKg`VSJO$_0>^;cUt=Lu$?zQ<sbmEy6n7A6whK_xk4osAKzf;_Z=wcnuaj8Ft8Yg+LoxYR~?Ix!$}<CdpP@xQh*g+cL#ydrzk{exGp_Fu?=W1O)2oUuXyQmm;3n_z7whE~ZTq(hcAXCt3*2P_?Ln@fU(&{MQA1XiQ;B0Kx|=`!9JSPXc-&3yN6*jpyvp&y~WZPER5`s33}CZJy{MVD<^c2msuAt&%YGs7Tv$xz@yW_MS2CwJ#QS3S+lU&8rC$nx%tzcH!xDS=S#WE7Yz4E`v^K|29EK)Cg0)?h}5blW#yJD%s*aM|I!5)+R%LQ~1OZ658itw3|vZFcVj-Jre{O*MF)=uz(|Q$0aB4RQQ51#H{HL{}AaTAK$iJ2(W-6ka=qdTR4ChRsfGv}`OHSOdR6cX`b#$Hyk+R$ecrY>S7tT}Q0#H)k)j!Y*ALNAPJpp<ms4%=J$;RWu%$$myzmQS8)3W-upyB~xwq{xEg9?i!yl%`i#m@D0<XS(xUWZ2f#Odd^ybgM!BvOjBdL7nzLax}oL&{Bjokbb9h((<PsKF9RhzqTkpi^tm4Ke4wK?#)l~bCh=P?8cg@SJb=x-d)UE&BWl}!2frmnh!R5;p;H{mRNTU13e8Zquu}+~9;+-|w?l<ZlVd19&>>9BnGcFV<$Ymf{D0B$e1v>8O8Swqy;Qap@?}x<Y;_aZ5i%W2pL(M{gel+9PMZqI-o)S?3P#*acvo*beMN6^($TB2&rpFa3=ME?9I)3n_?&4M%Ibm`U+IwgYfzkEzNGTOOd5a!yT%L|GL*~%BU&cyUJs%zx9uxaw!qf1EyA5wqWhV-JZ^7{4LK)W_b%sae~^X!n_Q6kQ4s_EbvkixPC4T>_7IOSq#hIV9_44&v0DNWKVLc;=omQE;w&AGl20L0D<=M5!vtE<tZNm!voRnofQ#Aw*t{)ffX$j9K{fuX!pd%+h+JYE$g&ys<33$?4WEvR_J1qJ1KRnq-2!w-Dhf7YH=Iu})Vi6sH?Yr<)d7GTim#6M4&7Ycro@f~j48{R^&|7>b00@3j3Gfw(>aD5HgM0}HW|!%ShD#7i9g$>_~F}w(f4jD_JK4o%~nujmt?D9+$iZzMt=iPO928D0~7!N00;m803iTyd^}mR0{{TX1^@sF00000000000001_fdBvi06|AZPE=DaZDdeO0Rj{Q6aWAK2mk;8Appp#$-;^Y004p(000yK0000000000005+c+5-RpWN%}0FH}KOUsg9?Lr+XWLN0A&P)h*<6ay3h000O8001EX`qaB&M*si-P5=M^7XSbN0000000000q=A+X003opaAk67ZDnqBb1z?CX>MtBUtcb8c~DCM0u%!j000080000X0FCNN_^to|0O9}u03HAU00000000000HlEu5C8yWcyMKMX>Db0baO9sVRK)0H!ojbX>MtBUtcb8c~DCM0u%!j000080000X08GL!jpPyl0HsI(02=@R00000000000HlHZ5C8yWcyMKMX>Db0baO9sVRK)0H!opzWNCCRaCuNm0Rj{Q6aWAK2mk;8Apq7Ho1hE`001!*000>P0000000000005+c86*GzWq5F9a%pX4Zgg`mbYXK}b~i6$Y-uiVc~DCM0u%!j000080000X0D?=fTdEHL0Oc_N03iSX00000000000HlFjEC2vycyMKMX>Db0baO9sVRK)0H!oyqVP|e{b98BAb1rasP)h*<6ay3h000O8001EXv3?*&d?5e;d1U|qAOHXW0000000000q=7{|003opaAk67ZDnqBb1!sZb6<8hFJ*RNY;|FDX>V>WaCuNm0Rj{Q6aWAK2mk;8Apri4OrKW>004v*0015U0000000000005+c|6Kq8Wq5F9a%pX4Zgg`mbYXK}b~i6&cywiQZeeULaCuNm0Rj{Q6aWAK2mk;8AprTtf+y?@002}e0012T0000000000005+cl4bw^Wq5F9a%pX4Zgg`mbYXK}b~i6=Wprq7WOFWXc~DCM0u%!j000080000X0G_sxvXuq^0BsTg03HAU00000000000HlGra{vHkcyMKMX>Db0baO9sVRK)0H!pB<WpH6~Wn?aJc~DCM0u%!j000080000X014S=O)w1r0IVzk02}}S00000000000HlG6dH?`rcyMKMX>Db0baO9sVRK)0H!pH!aBp&SE^v8JO928D0~7!N00;m803iTcW3Xn%CIA5OoB#kD00000000000001_f%Jy}0A+Y^WpZh4Wo~qHFLYsZUv@Vyb98lNc`k5yP)h*<6ay3h000O8001EX4C;tMA`t)pqdx!uCjbBd0000000000q=ECU003opaAk67ZDnqBb1!sZb6<8hFLQKtWO-j?X<=t>Z*z2MV{<NWc~DCM0u%!j000080000X03dy|ouv%`09Pph02=@R00000000000HlFhzyJVccyMKMX>Db0baO9sVRK)0H!pT+Wp{HfaCuNm0Rj{Q6aWAK2mk;8Apn2nMLMAX007Yd000~S0000000000005+cIL-h7a%FLKX>w(4Wo~qHFJo_PVPY*|ZDM6|E_8TwP)h*<6ay3h000O8001EXGqWIGQ~>}00s;U48vp<R0000000000q=6I90044jadl~OWo>0{baO9bZ){;=En;DFWiE7hbWlqH0u%!j000080000X01B5jpuPYA0003103iSX00000000000HlGS&;S5(WpQ<Ba%F90Zgg`mV{dF>Vl8lGa${w1bZKvHE_8TwP)h*<6ay3h000O8001EX<x-_@U;qFBZ2$lO7XSbN0000000000q=B8$004Ahb6;|GZggpFWiMY}X>MtBUtcb8c~DCM0u%!j000080000X06X^ncNPx-0EaaI02lxO00000000000HlF3(f|N-VRK({b#8QNZDlWEa&&2CVPkZ2E^v8JO928D0~7!N00;m803iUX$w4bt82|v*RsaAM00000000000001_fqvlt0CZt<UvhPBbZKp6FJW_YWpr~caCuNm0Rj{Q6aWAK2mk;8Apk-n{!CW|006KK000;O0000000000005+c2m1g3bYXK}a&>NWX>DaMVRUq5ZggpHZZ2?nP)h*<6ay3h000O8001EXx~AxkClLSur7-{i7ytkO0000000000q=A$G0RVJib6;|GZggpFWiMiHZ*+5Xa$#^TaCuNm0Rj{Q6aWAK2mk;8ApnM{zg)Wq007Y!000#L0000000000005+c@eu(4bYXK}a&>NWX>DaMV{dL|X=g5Qc~DCM0u%!j000080000X0CuwjYe*OX00vM1022TJ00000000000HlH6836!vVRK({b#8QNZDlWHVRT_GaCuNm0Rj{Q6aWAK2mk;8ApjE=mONMt005mO000*N0000000000005+cSTX?sbYXK}a&>NWX>DaMWMyM-WNB_^E^v8JO928D0~7!N00;m803iU2hm8Xv2><}58~^|r00000000000001_f#p5{0CZt<UvhPBbZKp6FK}gYV`Xr3X>V>WaCuNm0Rj{Q6aWAK2mk;8Apo^MNEaFo007-J000^Q0000000000005+cJxBonbYXK}a&>NWX>DaMaAk6HX>)XCZewLGaCuNm0Rj{Q6aWAK2mk;8Apl7}Z_zyi005r|000;O0000000000005+ck5&NybYXK}a&>NWX>DaMaB^>EX>4h3XD)DgP)h*<6ay3h000O8001EX$FhJbVGRHPsw@Bi761SM0000000000q=5oj0RVJib6;|GZggpFWiN7dZggpFWiD`eP)h*<6ay3h000O8001EX{`zot6#@VNJOls$5dZ)H0000000000q=A=b0RVJmb98erV{dL|bY*jNE^v8JO928D0~7!N00;m803iT+^z3H-1pok26aWAh00000000000001_f!k>T0CZ(@baO9sWpi|2VRCe7W?^G=b1rasP)h*<6ay3h000O8001EX<G^H~%Lo7fmLC8B6aWAK0000000000q=61_0RVJmb98erbY*jNUtx1|Wpr~caCuNm0Rj{Q6aWAK2mk;8Apm7;S04KZ005#E000vJ0000000000005+c3wZ$mbY*jNb1!sdb97%}b!2IDE^v8JO928D0~7!N00;m803iS{>yo;62mk=t761Sj00000000000001_fj5Bx0CZ(@baO9sWpi|2VsCGBb98cHa4v9pP)h*<6ay3h000O8001EXO~V>tU<d#Jw;ccg5&!@I0000000000q=Dm!0RVJmb98erbY*jNUu0o)VJ>iaP)h*<6ay3h000O8001EXoUm^G_5uI^ItKs%82|tP0000000000q=9pi0RVJmb98erbY*jNUu0=vXKrtEbZKLAE^v8JO928D0~7!N00;m803iTyR?ATH2LJ$R82|tn00000000000001_fuNNE0CZ(@baO9sWpi|2Wp-g~bzyXAZ*DGdc~DCM0u%!j000080000X0EF8O+t3C80HYHC02TlM00000000000HlG*odE!JWpi|MFLY&dbYE_7bY)_1Z)+}ac~DCM0u%!j000080000X0Bl9%R`LV@0Ot+>03QGV00000000000HlG<qyYeQWpi|MFLY&dbYE_7bY)_1Z);y<Wo&77Wpa5gaCuNm0Rj{Q6aWAK2mk;8ApqG=_yJo6003nc000>P0000000000005+c{HXx|bY*jNb1!sdb97&DWpZ<Ab97~HV`VOIc~DCM0u%!j000080000X0NAes4V(r50J;?b02TlM00000000000HlGAumJ#cWpi|MFLY&dbYF0CWpH6~Wn?aJc~DCM0u%!j000080000X01)`JiSY#h0E`m=02BZK00000000000HlFlw*dfjWpi|MFLY&dbYF61aBp&SE^v8JO928D0~7!N00;m803iV1bT1H*2LJ#e7XSbi00000000000001_fq}gN0CZ(@baO9sWpi|2a&>NWX>DaLaCuNm0Rj{Q6aWAK2mk;8ApmC+{f7Pz003Gz000#L0000000000005+cMa2OCbY*jNb1!sdb97&GV`*h>V`VOIc~DCM0u%!j000080000X03h|nc{L6I0NX79022TJ00000000000HlF-)d2u>Wpi|MFLY&dbYF9Hb!2%iaCuNm0Rj{Q6aWAK2mk;8ApnLpW}luH006>00012T0000000000005+c+~WZNbZ>8Lb1!0bX>4R)V{dF>Vqb1=bY)_1Z)+}ac~DCM0u%!j000080000X02(9q!G;L{03;j$02%-Q00000000000HlGn`vCxSZ*OdKFK%yiWnyn{YhPq#Y-x67a(OOrc~DCQ1^@s605AYB06Gl-0CNQb0000'}

def restore_source_package(package, checkout, persistent, repo_url):
    """Notebook bootstrap; stdlib only, before installing scientific packages.

    This function is copied by the notebook generator so upload of the notebook
    alone is sufficient. Its authoritative source remains this Python module.
    Existing baseline files cannot be replaced by the source package.
    """
    import base64
    import hashlib
    import io
    import json
    import os
    import re
    import stat
    import subprocess
    import zipfile
    from pathlib import Path

    if package.get("schema") != "tas-private-source-v1" or any(
        re.fullmatch(r"[a-f0-9]{40}", package.get(key, "")) is None
        for key in ("base_commit", "source_commit")
    ):
        raise ValueError("Invalid versioned source package.")
    data = base64.b85decode(package["archive_b85"])
    if hashlib.sha256(data).hexdigest() != package["sha256"]:
        raise ValueError("Source archive checksum mismatch.")
    checkout, persistent = Path(checkout), Path(persistent)
    base = package["base_commit"]

    def owned_path(name):
        relative = Path(name)
        allowed = len(relative.parts) > 1 and relative.parts[0] in {
            "tas_runtime",
            "experiments",
            "requirements",
            "tests",
            "tools",
        }
        allowed |= name in {"AGENTS.md", "docs/TAS_V7_COLAB.md"}
        return (
            allowed
            and not relative.is_absolute()
            and ".." not in relative.parts
            and (checkout / relative).resolve().is_relative_to(checkout.resolve())
        )

    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        members = archive.infolist()
        if len(members) != len(package["files"]) or {
            item.filename for item in members
        } != set(package["files"]):
            raise ValueError("Source file manifest mismatch.")
        if sum(item.file_size for item in members) > 8 * 1024**2:
            raise ValueError("Source package exceeds its code-only size limit.")
        contents = {}
        for item in members:
            if not owned_path(item.filename) or stat.S_IFMT(
                item.external_attr >> 16
            ) not in {0, stat.S_IFREG}:
                raise ValueError("Unsafe source member.")
            content = archive.read(item)
            if hashlib.sha256(content).hexdigest() != package["files"][item.filename]:
                raise ValueError("Source member checksum mismatch.")
            contents[item.filename] = content
    if not (checkout / ".git").exists():
        subprocess.run(
            ["git", "clone", "--no-checkout", str(repo_url), str(checkout)], check=True
        )
    found = subprocess.run(
        ["git", "cat-file", "-e", base + "^{commit}"], cwd=checkout, capture_output=True
    )
    if found.returncode:
        subprocess.run(["git", "fetch", "origin", base], cwd=checkout, check=True)
    for name in contents:
        baseline = subprocess.run(
            ["git", "cat-file", "-e", base + ":" + name],
            cwd=checkout,
            capture_output=True,
        )
        if baseline.returncode == 0:
            raise ValueError(
                "Source package cannot overwrite a public baseline file: " + name
            )
    subprocess.run(["git", "checkout", "--detach", base], cwd=checkout, check=True)
    actual = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=checkout, text=True
    ).strip()
    if actual != base:
        raise ValueError("Public baseline checkout mismatch.")
    previous = checkout / ".tas-source.json"
    if previous.exists():
        old = json.loads(previous.read_text())
        for name in set(old.get("files", {})) - set(contents):
            baseline = subprocess.run(
                ["git", "cat-file", "-e", base + ":" + name],
                cwd=checkout,
                capture_output=True,
            )
            if not owned_path(name) or baseline.returncode == 0:
                raise ValueError("Unsafe obsolete source path.")
            (checkout / name).unlink(missing_ok=True)

    def save(path, content):
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name + ".installing")
        temporary.write_bytes(content)
        os.replace(temporary, path)

    for name, content in contents.items():
        save(checkout / name, content)
    receipt = {key: value for key, value in package.items() if key != "archive_b85"}
    receipt_bytes = (json.dumps(receipt, indent=2, sort_keys=True) + "\n").encode()
    save(checkout / ".tas-source.json", receipt_bytes)
    stored = (
        persistent
        / "code"
        / ("source-" + package["source_commit"] + "-" + package["sha256"][:16])
    )
    if (
        not stored.with_suffix(".zip").exists()
        or hashlib.sha256(stored.with_suffix(".zip").read_bytes()).hexdigest()
        != package["sha256"]
    ):
        save(stored.with_suffix(".zip"), data)
    save(stored.with_suffix(".json"), receipt_bytes)
    return receipt


## Connect storage and restore the exact code

Drive authorization may prompt once per runtime. No credentials are saved
in configuration or outputs. Source restoration is automatic and verifies
both the archive and every file before installation. It cannot replace any
file from the frozen public baseline. The same caches work in CPU/GPU sessions.


In [ ]:
try:
    import os, sys, json, subprocess, pathlib, shutil, time
    from IPython.display import display, Markdown, HTML
    from google.colab import drive
    drive.mount("/content/drive")
    persistent = pathlib.Path(PERSISTENT_ROOT)
    persistent.mkdir(parents=True, exist_ok=True)
    checkout = pathlib.Path("/content/tas-v7-checkout")
    if (SOURCE_PACKAGE["base_commit"], SOURCE_PACKAGE["source_commit"]) != (BASELINE_COMMIT, RESEARCH_COMMIT):
        raise ValueError("Regenerate the notebook to change its fixed code versions.")
    source_receipt = restore_source_package(SOURCE_PACKAGE, checkout, persistent, REPO_URL)
    actual_commit = source_receipt["source_commit"]
    os.chdir(checkout)
    if str(checkout) not in sys.path:
        sys.path.insert(0, str(checkout))
    # A new idea notebook may reuse this VM with different experiment modules.
    for module_name in list(sys.modules):
        if module_name == "tas_runtime" or module_name.startswith("tas_runtime."):
            del sys.modules[module_name]
    import importlib
    importlib.invalidate_caches()
    print("Public baseline:", BASELINE_COMMIT)
    print("Experiment Python source:", actual_commit, "| verified files:", len(source_receipt["files"]))
except BaseException:
    import traceback
    traceback.print_exc()
    release_runtime()
    raise


## Isolated, pinned environment

Python 3.10.16 and matched PyTorch 2.4.1 / torchvision 0.19.1 CPU or CUDA
12.1 wheels run in subprocesses. The legacy Transformers 4.35.2 stack is
preserved. No notebook-kernel restart, system Torch replacement or CUDA
extension compilation is needed. GPU mode validates its actual CUDA path.


In [ ]:
try:
    query = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True) if shutil.which("nvidia-smi") else None
    has_gpu = query is not None and query.returncode == 0 and bool(query.stdout.strip())
    effective_phase = ("run" if has_gpu else "prepare") if PHASE == "auto" else PHASE
    mode = "gpu" if effective_phase == "run" else "cpu"
    print("Hardware:", query.stdout.strip() if has_gpu else "CPU")
    print("Phase:", effective_phase, "| environment:", mode)
    if effective_phase == "run" and has_gpu and REQUIRE_A100_80GB and ("A100" not in query.stdout or int(query.stdout.strip().splitlines()[0].split(",")[-1].strip().split()[0]) < 70000):
        raise RuntimeError("This configuration requires A100 80GB; hardware check stopped before environment/model setup.")
    if effective_phase == "run" and not has_gpu:
        raise RuntimeError("Select a GPU runtime, or use PHASE=prepare/score on CPU.")
    if effective_phase == "run" and not (persistent / "preparation.json").is_file():
        raise RuntimeError("Run this notebook once in a CPU runtime first. That downloads/persists models and data before A100 allocation.")
    from tas_runtime.bootstrap import ensure_environment, subprocess_environment
    python = ensure_environment(mode, "/content/tas-v7-envs", checkout,
                                diagnostics_dir=persistent / "diagnostics/environment")
    if DATASET == "amber":
        from tas_runtime.bootstrap import ensure_amber
        ensure_amber(python, "/content/tas-v7-envs", checkout)
    env = dict(subprocess_environment(), PYTHONHASHSEED="0", TOKENIZERS_PARALLELISM="false",
               NLTK_DATA=str(persistent / "nlp"), YOLO_CONFIG_DIR=str(pathlib.Path(LOCAL_ROOT) / "ultralytics"))
    configuration = {
        "persistent_root":PERSISTENT_ROOT, "local_root":LOCAL_ROOT, "stage":STAGE,
        "seeds":SEEDS, "methods":METHODS, "batch_sizes":BATCH_SIZES,
        "attention_backends":ATTENTION_BACKENDS, "last_position_logits":LAST_POSITION_LOGITS,
        "max_session_minutes":MAX_SESSION_MINUTES, "require_a100":REQUIRE_A100_80GB,
        "run_confirmation":RUN_CONFIRMATION, "run_controls":RUN_CONTROLS,
        "run_diagnostics":RUN_DIAGNOSTICS, "run_final":RUN_FINAL_REMAINDER,
        "reserve_pope":RESERVE_POPE_IMAGES, "split_seed":SPLIT_SEED,
        "coverage_tolerance":COVERAGE_TOLERANCE_PP / 100.0,
        "dataset":DATASET, "external_manifest":EXTERNAL_MANIFEST, "external_image_root":EXTERNAL_IMAGE_ROOT,
        "auto_continue_after_smoke":True, "source_revision":actual_commit,
        "generation":{"max_new_tokens":180,"temperature":1.0,"top_p":1.0,"top_k":None,
                      "precision":"fp16","do_sample":True,"conv_mode":"llava_v1","prompt":"Describe this image."}
    }
    config_path = pathlib.Path("/content/tas-v7-config.json")
    config_path.write_text(json.dumps(configuration, indent=2))
    def run_phase(phase):
        subprocess.run([str(python), "-m", "experiments.tas_v7.cli", "--config",str(config_path),"--phase",phase], cwd=checkout,env=env,check=True)
    if DATASET == "amber":
        subprocess.run([str(python), "-m", "tas_runtime.bootstrap", "--root", "/content/tas-v7-envs", "--nlp-root", str(persistent / "nlp"), "--amber"], cwd=checkout, env=env, check=True)
    run_phase("plan")
except BaseException:
    import traceback
    traceback.print_exc()
    release_runtime()
    raise


## Prepare or run the study

First CPU session: approximately 17–18 GB of checkpoints plus annotations,
selected images and NLP resources; existing verified files are reused.
Only 30 train images are fetched, not all COCO train2017.

GPU sessions: lightweight preflight → local asset restoration → one model
load → baseline/view audit → correctness/throughput pilot → configured
stages. Per-image RNG makes retries/resume independent of job order. A
rejected optimization is recorded; baseline failures block larger runs.
Results use immutable chunks and perception caches use shards on Drive.


In [ ]:
try:
    phase_started = time.perf_counter()
    if effective_phase == "prepare":
        # Small NLP assets enable the offline scientific fixtures first.
        subprocess.run([str(python), "-m", "tas_runtime.bootstrap", "--root", "/content/tas-v7-envs", "--nlp-root", str(persistent / "nlp")], cwd=checkout, env=env, check=True)
        run_phase("check")
        run_phase("prepare")
        run_phase("score")
        display(Markdown("**CPU preparation complete.** Files are persistent. Switch to the A100 runtime and Run all again."))
    elif effective_phase == "run":
        run_phase("check")
        run_phase("run")
    elif effective_phase == "score":
        run_phase("score")
    elif effective_phase == "check":
        run_phase("check")
    else:
        raise ValueError("Unknown phase")
    print("Session phase seconds:", round(time.perf_counter() - phase_started, 1))
except BaseException:
    import traceback
    traceback.print_exc()
    release_runtime()
    raise


## Inspect saved reports

LEGACY and HARMONIZED are separate protocols; do not infer a matched
algorithm gain across them. CHAIR and annotation category recall do not
establish caption quality or preserved relations. Inspect ambiguous images,
captions, failures, coverage, length, timings and uncertainty together.
Reports include descriptive comparisons for paired captions with exactly
equal word counts. Batch latency and amortized seconds/image are separate.
CPU scoring/report regeneration is available with PHASE="score".


In [ ]:
try:
    import csv, html
    from IPython.display import Image as DisplayImage
    runs_root = persistent / "runs"
    for run_dir in sorted(runs_root.glob("*"))[-3:]:
        display(Markdown("### Run `" + run_dir.name + "`"))
        session = run_dir / "session.json"
        if session.exists():
            display(Markdown("```json\n" + session.read_text() + "\n```"))
        gallery = run_dir / "views.png"
        if gallery.exists():
            display(DisplayImage(filename=str(gallery)))
        for summary in sorted(run_dir.glob("*/summary.csv")):
            rows = list(csv.reader(summary.open()))
            table = "<table>" + "".join("<tr>" + "".join("<td>" + html.escape(value) + "</td>" for value in row) + "</tr>" for row in rows) + "</table>"
            display(Markdown("**" + summary.parent.name + "**"))
            display(HTML(table))
    print("Persistent artifacts:", persistent)
    print("Reuse this folder for later sessions/new idea notebooks. Do not delete frozen manifests or rewrite completed runs.")
except BaseException:
    import traceback
    traceback.print_exc()
    release_runtime()
    raise


## Release the session

With `AUTO_SHUTDOWN=True`, the final cell flushes Drive writes and releases
the runtime after preparation, evaluation or a planned budget stop. Errors
also invoke this cleanup before propagating. Reports remain on Drive.
Set it to False only when keeping the runtime for inspection.


In [ ]:
release_runtime()
